# Script_Diario V12 blindado (base V11 30-09-2026)
Cambios vs V10: pace alineado por días restantes; re-entreno mensual y refit con todo el historial; guard del día 1 y cuadre MAU App = subcats (assert); peso de pace propio para GPV; prorrateo de productos solo en días futuros y solo columnas GPV; factores manuales centralizados en CONFIG V11 (neutros en GPV); OB TC calibrado contra real reciente (sin x1.4/x1.7), con factor persistido para inicio de mes y sin negativos; Cyber oct-26 (Jira AD-532) con uplift residual sobre el modelo y descontado del pace; RGU sin día parcial y LE con el modelo; metas hardcodeadas con aviso; override revenue financiero atado a su mes; columnas rev_ci/co coherentes; auditoría a `kpitos.pipeline_audit` y reporte LE en Drive; se eliminan celdas duplicadas (reactive y limpieza MAU).


---
## V12 (blindaje) — cambios vs V11
Ver `forecast_hardening/REPORTE_REVISION.md`. Resumen: reloj único de Chile (`hoy_chile()`), compuerta de
publicación por grupos con chequeos entre tablas (`GATE`, modo `PIPELINE_MODE`=stage|live|dry_run),
corrección de features de lag del modelo V5 cuando el mes aún no tiene datos (día 1), escritura atómica de
`.pkl`, secreto de la CMF fuera del notebook, TMC con año actual, guardas de re-ejecución.
Primera celda de código = helper V12; última = publicación.

In [ ]:
# ==============================================================================
# V12 — BLINDAJE (ejecutar PRIMERO). Reloj único de Chile, compuerta de publicación y helpers.
# ------------------------------------------------------------------------------
# * hoy_chile()/ahora_chile(): UNA sola fuente de "hoy" (America/Santiago). Colab corre en UTC y
#   date.today()/Timestamp.now() sin zona saltaban de mes/día 3-4 h antes que Chile (entre las
#   20-21h y medianoche): tablas con meses mezclados. PIPELINE_NOW="YYYY-MM-DD HH:MM" fuerza la
#   fecha (reprocesos/pruebas).
# * GATE: intercepta pandas_gbq.to_gbq / bigquery load / DELETE del monitoreo. Modo (env
#   PIPELINE_MODE): stage (default: publica todo junto al final, por grupos, si valida) |
#   live (como V11, con validación por tabla) | dry_run (valida, no escribe).
# * V12_FIXES: cada corrección de lógica se puede apagar con PIPELINE_V12_OFF=nombre1,nombre2.
# ==============================================================================
import os, pickle, tempfile
import numpy as np
import pandas as pd

_OFF = {x.strip() for x in os.environ.get("PIPELINE_V12_OFF", "").split(",") if x.strip()}
V12_FIXES = {k: (k not in _OFF) for k in ["lag_mes_sin_datos", "tmc_anios", "pkl_atomico", "rgu_pkl_respaldo",
                                          "reloj_unico", "guardas_idempotencia"]}

def ahora_chile():
    """Timestamp tz-aware America/Santiago (o el forzado por PIPELINE_NOW)."""
    ov = os.environ.get("PIPELINE_NOW", "").strip()
    if ov:
        return pd.Timestamp(ov, tz="America/Santiago")
    return pd.Timestamp.now(tz="America/Santiago")

def hoy_chile():
    """Medianoche naive de la fecha de hoy en Chile."""
    return ahora_chile().tz_localize(None).normalize()

def ahora_utc_naive():
    """Instante actual en UTC naive (solo para marcas de auditoría; igual que Timestamp.now() en Colab)."""
    return ahora_chile().tz_convert("UTC").tz_localize(None)

def _dump_pickle_atomic(obj, path):
    """Escribe el .pkl a un temporal y lo reemplaza de una vez: una corrida cortada ya no deja un
    archivo vacío/corrupto (EOFError al siguiente load)."""
    if not V12_FIXES.get("pkl_atomico", True):
        pickle.dump(obj, open(path, "wb")); return
    d = os.path.dirname(path) or "."
    fd, tmp = tempfile.mkstemp(dir=d, suffix=".tmp")
    try:
        with os.fdopen(fd, "wb") as f:
            pickle.dump(obj, f)
        os.replace(tmp, path)
    except BaseException:
        if os.path.exists(tmp):
            os.remove(tmp)
        raise

def _columnas_del_modelo(model):
    """Nombres de features con los que se ENTRENÓ el modelo (None si no se pueden leer)."""
    est = getattr(model, "best_estimator_", model)
    for getter in (lambda e: e.feature_names_in_, lambda e: e.regressor_.feature_names_in_,
                   lambda e: e.get_booster().feature_names, lambda e: e.regressor_.get_booster().feature_names):
        try:
            v = getter(est)
            if v is not None and len(v):
                return [str(x) for x in v]
        except Exception:
            pass
    return None

# ---- compuerta de publicación (código de forecast_hardening/publish_gate.py, inline) ----

import json
import os
import re
import time
from collections import OrderedDict

import numpy as np
import pandas as pd

PROJECT_DEFAULT = "tenpo-bi-prod"

# grupo -> tablas que DEBEN salir juntas (todo o nada)
GROUPS: "OrderedDict[str, list[str]]" = OrderedDict([
    ("captacion", ["forecasting_le_captacion", "captaciones_monitoreo_le", "forecasting_le_captacion_90d"]),
    ("principales", ["forecasting_le_clientes_principales"]),
    ("tesoreria", ["flujos_tesoreria", "flujos_tesoreria_ajustados"]),
    ("seguros", ["forecasting_le_seguros", "forecasting_le_seguros_stock", "forecast_le_insurance"]),
    ("tarjetas", ["entrega_tarjetas"]),
    ("kpi_core", ["forecasting_rgu", "forecasting_mau_productos", "forecasting_mau_reactive_desglosado",
                  "forecasting_categorias", "forecasting_gpv_productos", "forecasting_gpv_app",
                  "forecasting_gpv_cca_ci", "forecasting_gpv_cca_co", "porc_insurance",
                  "forecasting_ob_tc", "ob_tenpo"]),
    ("revenue_fin", ["forecasting_revenue_financiero"]),
    ("referencias", ["tmc", "one_page_cuenta_remunerada"]),
    ("colocaciones", ["colocacion_90", "le_colocacion"]),
    ("auditoria", ["pipeline_audit"]),
])
_TABLE2GROUP = {t: g for g, ts in GROUPS.items() for t in ts}

# columnas que, junto a `fecha`, identifican una fila (para detectar duplicados)
_KEY_HELPERS = ["linea", "producto", "identificador", "category_type", "payment_method_recod", "segmento",
                "categoria", "mes_anterior", "fecha_ejecucion", "status", "tipo"]

_DML = re.compile(r"^\s*(DELETE|INSERT|UPDATE|MERGE|CREATE|DROP|TRUNCATE|ALTER)\b", re.I)


class PublishError(RuntimeError):
    pass


def short_name(table: str) -> str:
    return str(table).split(".")[-1]


def fq_name(table: str, project: str = PROJECT_DEFAULT) -> str:
    parts = str(table).replace("`", "").split(".")
    return ".".join(parts) if len(parts) == 3 else f"{project}.{'.'.join(parts)}"


def _type_class_bq(t: str) -> str:
    t = str(t).upper()
    if t in ("INTEGER", "INT64"):
        return "INT"
    if t in ("FLOAT", "FLOAT64", "NUMERIC", "BIGNUMERIC"):
        return "FLOAT"
    if t in ("BOOLEAN", "BOOL"):
        return "BOOL"
    if t in ("DATE", "DATETIME", "TIMESTAMP", "TIME"):
        return "TEMPORAL"
    return "STRING"


def _type_class_pd(s: pd.Series) -> str:
    if pd.api.types.is_bool_dtype(s):
        return "BOOL"
    if pd.api.types.is_integer_dtype(s):
        return "INT"
    if pd.api.types.is_float_dtype(s):
        return "FLOAT"
    if pd.api.types.is_datetime64_any_dtype(s):
        return "TEMPORAL"
    return "STRING"


class _DummyJob:
    def result(self, *a, **k):
        return None


class PublishGate:
    def __init__(self, mode: str = "stage", today=None, report_dir: str | None = None,
                 allow_schema_drift: bool = False, credentials_getter=None, log=print):
        assert mode in ("stage", "live", "dry_run"), mode
        self.mode = mode
        self.today = today
        self.report_dir = report_dir
        self.allow_schema_drift = allow_schema_drift
        self.creds = credentials_getter or (lambda: None)
        self.log = log
        self.ops: list[dict] = []          # todas las operaciones, en orden
        self.results: list[dict] = []      # resultado por grupo (se llena en commit)
        self._orig = {}
        self._baseline_cache: dict = {}
        self.staged: dict = {}             # tabla corta -> (nombre completo, DataFrame) de lo último comprometido
        self.issues_live: list[dict] = []

    # ------------------------------------------------------------------ enganche
    def install(self):
        import pandas_gbq
        prev = getattr(pandas_gbq, "_v12_gate", None)
        if prev is not None and prev is not self:
            # re-ejecutar la celda helper NO debe anidar compuertas: se desinstala la anterior
            if prev.ops:
                self.log(f"[GATE] !! la compuerta anterior tenía {len(prev.ops)} operaciones en cola sin publicar (se descartan)")
            prev.uninstall()
        pandas_gbq._v12_gate = self
        self._orig["to_gbq"] = pandas_gbq.to_gbq
        pandas_gbq.to_gbq = self._to_gbq
        try:
            from google.cloud import bigquery
            self._orig["load"] = bigquery.Client.load_table_from_dataframe
            self._orig["query"] = bigquery.Client.query
            gate = self

            def _load(client, dataframe, destination, *a, **k):
                return gate._capture_load(client, dataframe, destination, a, k)

            def _query(client, sql, *a, **k):
                if isinstance(sql, str) and _DML.match(sql):
                    return gate._capture_dml(client, sql, a, k)
                return gate._orig["query"](client, sql, *a, **k)

            bigquery.Client.load_table_from_dataframe = _load
            bigquery.Client.query = _query
        except Exception as e:  # pragma: no cover
            self.log(f"[GATE] !! no se pudo enganchar bigquery.Client: {e}")
        self.log(f"[GATE] instalada | modo={self.mode} | grupos={list(GROUPS)}")
        return self

    def uninstall(self):
        import pandas_gbq
        if getattr(pandas_gbq, "_v12_gate", None) is self:
            pandas_gbq._v12_gate = None
        if "to_gbq" in self._orig:
            pandas_gbq.to_gbq = self._orig["to_gbq"]
            self._orig.pop("to_gbq")
        if "load" in self._orig:
            from google.cloud import bigquery
            bigquery.Client.load_table_from_dataframe = self._orig.pop("load")
            bigquery.Client.query = self._orig.pop("query")

    # ------------------------------------------------------------------ captura
    def _to_gbq(self, dataframe, destination_table=None, project_id=None, if_exists="fail", *a, **k):
        table = fq_name(destination_table, project_id or PROJECT_DEFAULT)
        op = dict(kind="to_gbq", table=table, dest=destination_table, df=dataframe, mode=if_exists, args=a,
                  kwargs=dict(k, project_id=project_id, if_exists=if_exists), t=time.time())
        return self._register(op)

    def _capture_load(self, client, dataframe, destination, a, k):
        table = fq_name(str(destination))
        jc = k.get("job_config")
        disp = str(getattr(jc, "write_disposition", "WRITE_EMPTY"))
        op = dict(kind="load", table=table, df=dataframe, mode="append" if "APPEND" in disp else "replace",
                  client=client, args=a, kwargs=k, t=time.time())
        self._register(op)
        return _DummyJob()

    def _capture_dml(self, client, sql, a, k):
        m = re.search(r"`([\w.-]+)`", sql)
        table = m.group(1) if m else "<dml>"
        op = dict(kind="dml", table=fq_name(table) if m else table, df=None, mode="dml", client=client,
                  sql=sql, args=a, kwargs=k, t=time.time())
        self._register(op)
        return _DummyJob()

    def _register(self, op):
        op["short"] = short_name(op["table"])
        op["group"] = _TABLE2GROUP.get(op["short"], "otros:" + op["short"])
        if self.mode == "live":
            return self._live(op)
        self.ops.append(op)
        n = "" if op["df"] is None else f" {op['df'].shape[0]}x{op['df'].shape[1]}"
        self.log(f"[GATE:{self.mode}] en cola {op['kind']} -> {op['short']}{n} (grupo {op['group']})")
        return None

    def _live(self, op):
        if op["kind"] != "dml":
            issues = self.check_table(op)
            blocks = [i for i in issues if i["level"] == "BLOCK"]
            self.issues_live += [dict(i, table=op["short"]) for i in issues]
            for i in issues:
                self.log(f"[GATE:live] {i['level']} {op['short']}: {i['msg']}")
            if blocks:
                self.log(f"[GATE:live] !! {op['short']} NO se publica (se conserva la versión anterior)")
                return None
            self._align(op)
        return self._execute(op)

    # ------------------------------------------------------------------ línea base BQ
    def baseline(self, table):
        if table in self._baseline_cache:
            return self._baseline_cache[table]
        res = None
        try:
            from google.cloud import bigquery
            client = bigquery.Client(project=table.split(".")[0], credentials=self.creds())
            t = client.get_table(table)
            res = dict(schema={f.name: _type_class_bq(f.field_type) for f in t.schema}, rows=int(t.num_rows or 0))
        except Exception as e:
            res = dict(error=f"{type(e).__name__}: {str(e)[:80]}")
        self._baseline_cache[table] = res
        return res

    # ------------------------------------------------------------------ validación por tabla
    def check_table(self, op) -> list[dict]:
        df, out = op["df"], []

        def add(level, msg):
            out.append(dict(level=level, msg=msg))

        if df is None or len(df.columns) == 0:
            return [dict(level="BLOCK", msg="sin columnas")]
        if len(df) == 0:
            add("BLOCK", "DataFrame vacío")
            return out
        num = df.select_dtypes(include=[np.number])
        if len(num.columns) and np.isinf(num.to_numpy(dtype=float, na_value=np.nan)).any():
            add("BLOCK", "hay valores infinitos")
        if df.isna().all(axis=None):
            add("BLOCK", "todo NaN")
        if "fecha" in df.columns:
            key = ["fecha"] + [c for c in _KEY_HELPERS if c in df.columns]
            d = int(df.duplicated(subset=key).sum())
            if d:
                add("BLOCK", f"{d} filas duplicadas por {key}")
        neg_cols = [c for c in num.columns
                    if re.search(r"(pred|real|saldo|gpv|mau|rgu|ob_|trx)", c) and "meta" not in c and "delta" not in c
                    and (num[c] < 0).any()]
        if neg_cols:
            add("WARN", f"valores negativos en {neg_cols[:5]}")
        # línea base
        if op["kind"] != "dml":
            b = self.baseline(op["table"])
            if "error" in b:
                add("INFO", f"sin línea base en BQ ({b['error']})")
            else:
                ex = b["schema"]
                missing = [c for c in ex if c not in df.columns]
                extra = [c for c in df.columns if c not in ex]
                if missing and not self.allow_schema_drift:
                    add("BLOCK", f"faltan columnas existentes en BQ: {missing[:6]}")
                elif missing:
                    add("WARN", f"faltan columnas existentes en BQ: {missing[:6]} (drift permitido)")
                if extra:
                    add("WARN", f"columnas nuevas vs BQ: {extra[:6]}")
                for c in df.columns:
                    if c in ex:
                        a, bcls = _type_class_pd(df[c]), ex[c]
                        if a != bcls and {a, bcls} != {"INT", "FLOAT"} and not (a == "STRING" and bcls == "TEMPORAL") \
                                and not (a == "TEMPORAL" and bcls == "STRING"):
                            lvl = "WARN" if self.allow_schema_drift else "BLOCK"
                            add(lvl, f"tipo de '{c}': {a} vs BQ {bcls}")
                if b["rows"] and op["mode"] == "replace" and len(df) < 0.5 * b["rows"]:
                    add("BLOCK", f"filas {len(df)} < 50% de las vigentes ({b['rows']})")
        return out

    def _align(self, op):
        """Alinea tipos INT/FLOAT con la tabla vigente (sin tocar valores)."""
        df = op["df"]
        if df is None:
            return
        b = self.baseline(op["table"])
        if "error" in b:
            return
        df = df.copy()
        for c in df.columns:
            cls = b["schema"].get(c)
            if cls is None:
                continue
            cur = _type_class_pd(df[c])
            if cls == "INT" and cur == "FLOAT":
                v = df[c]
                if ((v.dropna() % 1) == 0).all():
                    df[c] = v.round().astype("Int64")
            elif cls == "FLOAT" and cur == "INT":
                df[c] = df[c].astype(float)
        op["df"] = df

    # ------------------------------------------------------------------ chequeos entre tablas
    def cross_checks(self, frames: dict) -> list[dict]:
        out = []

        def add(level, msg):
            out.append(dict(level=level, msg=msg))

        def F(name):
            return frames.get(name)

        core = {n: f for n, f in frames.items() if n in GROUPS["kpi_core"] and "fecha" in f.columns}
        # C1: todas las tablas del grupo cubren el mismo mes
        months = {}
        for n, f in core.items():
            fd = pd.to_datetime(f["fecha"], errors="coerce", dayfirst=False)
            months[n] = sorted({p for p in fd.dt.to_period("M").dropna().astype(str).unique()})
        ref = months.get("forecasting_gpv_app") or next(iter(months.values()), [])
        for n, m in months.items():
            if m != ref:
                add("BLOCK", f"{n} cubre {m[:3]} pero forecasting_gpv_app cubre {ref[:3]}")
        # C2: corte real/forecast parecido entre tablas
        def last_real(f, col):
            if col not in f.columns:
                return None
            fd = pd.to_datetime(f["fecha"], errors="coerce")
            s = pd.to_numeric(f[col], errors="coerce")
            ok = fd[s.notna()]
            return ok.max() if len(ok) else pd.NaT
        cut = {"forecasting_gpv_app": "gpv", "forecasting_categorias": "mau_app"}
        cuts = {n: last_real(F(n), c) for n, c in cut.items() if F(n) is not None}
        cuts = {n: v for n, v in cuts.items() if v is not None and not pd.isna(v)}
        if len(cuts) >= 2:
            vals = list(cuts.values())
            gap = (max(vals) - min(vals)).days
            if gap > 1:
                add("WARN", f"último día real difiere {gap}d entre tablas: { {k: str(v.date()) for k, v in cuts.items()} }")
        # C3: suma de productos == GPV App en días futuros (productos en MM, app en CLP)
        gp, ga = F("forecasting_gpv_productos"), F("forecasting_gpv_app")
        if gp is not None and ga is not None and "gpv_real_pred" in ga.columns:
            prod = [c for c in gp.columns if c.startswith("gpv_real_pred_") and "cumsum" not in c]
            a = ga.assign(fecha=pd.to_datetime(ga["fecha"])).set_index("fecha")
            p = gp.assign(fecha=pd.to_datetime(gp["fecha"])).set_index("fecha")[prod].apply(pd.to_numeric, errors="coerce")
            fut = a.index[pd.to_numeric(a["gpv"], errors="coerce").isna()] if "gpv" in a.columns else a.index[:0]
            fut = fut.intersection(p.index)
            if len(fut):
                lhs = p.loc[fut].sum(axis=1)
                rhs = pd.to_numeric(a.loc[fut, "gpv_real_pred"], errors="coerce") / 1e6
                rel = ((lhs - rhs).abs() / rhs.abs().replace(0, np.nan)).max()
                if rel > 0.05:
                    add("BLOCK", f"suma de productos vs GPV App en días futuros difiere {rel:.1%}")
                elif rel > 0.005:
                    add("WARN", f"suma de productos vs GPV App en días futuros difiere {rel:.2%}")
        # C4: MAU App vs subcategorías (acumulado del mes)
        c = F("forecasting_categorias")
        if c is not None and "mau_real_pred_app" in c.columns:
            parts = [f"mau_real_pred_{k}" for k in ("concurrent", "m0", "resurrected", "late_active")]
            if all(p in c.columns for p in parts):
                tot = pd.to_numeric(c["mau_real_pred_app"], errors="coerce").sum()
                sub = sum(pd.to_numeric(c[p], errors="coerce").sum() for p in parts)
                if tot > 0:
                    rel = abs(sub - tot) / tot
                    if rel > 0.02:
                        add("BLOCK", f"MAU App {tot:,.0f} vs subcategorías {sub:,.0f} ({rel:.2%})")
                    elif rel > 0.005:
                        add("WARN", f"MAU App {tot:,.0f} vs subcategorías {sub:,.0f} ({rel:.2%})")
        # C5: desglose reactive suma al concurrent por día
        r = F("forecasting_mau_reactive_desglosado")
        if r is not None and c is not None and "mau_real_pred_concurrent" in c.columns:
            cols = [x for x in r.columns if x not in ("fecha", "mau_prediccion_concurrent")]
            rr = r.assign(fecha=pd.to_datetime(r["fecha"])).set_index("fecha")[cols].apply(pd.to_numeric, errors="coerce").sum(axis=1)
            cc = c.assign(fecha=pd.to_datetime(c["fecha"])).set_index("fecha")["mau_real_pred_concurrent"].astype(float)
            j = pd.concat([rr, cc], axis=1, keys=["r", "c"]).dropna()
            if len(j) and (j["r"] - j["c"]).abs().max() > 1:
                add("WARN", f"desglose reactive no cuadra con concurrent (máx {(j['r'] - j['c']).abs().max():,.0f})")
        return out

    # ------------------------------------------------------------------ ejecución
    def _execute(self, op):
        if op["kind"] == "to_gbq":
            return self._orig["to_gbq"](op["df"], op["dest"], *op["args"], **op["kwargs"])
        if op["kind"] == "load":
            return self._orig["load"](op["client"], op["df"], op["table"], *op["args"], **op["kwargs"]).result()
        if op["kind"] == "dml":
            return self._orig["query"](op["client"], op["sql"], *op["args"], **op["kwargs"]).result()

    def commit_all(self, raise_on_hold: bool = True):
        t0 = time.time()
        order = list(GROUPS) + sorted({o["group"] for o in self.ops if o["group"] not in GROUPS})
        held = []
        for g in order:
            ops = [o for o in self.ops if o["group"] == g]
            if not ops:
                continue
            res = dict(group=g, tables=sorted({o["short"] for o in ops}), issues=[], status="OK", published=[])
            # duplicados 'replace' -> gana el último
            last = {}
            for i, o in enumerate(ops):
                if o["mode"] == "replace" and o["kind"] != "dml":
                    if o["short"] in last:
                        res["issues"].append(dict(level="INFO", msg=f"{o['short']} escrita más de una vez; gana la última"))
                    last[o["short"]] = i
            ops_eff = [o for i, o in enumerate(ops) if not (o["mode"] == "replace" and o["kind"] != "dml" and last[o["short"]] != i)]
            frames = {}
            for o in ops_eff:
                if o["kind"] != "dml" and o["mode"] == "replace":
                    self.staged[o["short"]] = (o["table"], o["df"])
            for o in ops_eff:
                if o["kind"] != "dml":
                    for it in self.check_table(o):
                        res["issues"].append(dict(it, table=o["short"]))
                    if o["mode"] == "replace":
                        frames[o["short"]] = o["df"]
            req = [t for t in GROUPS.get(g, []) if t not in {o["short"] for o in ops_eff}]
            if g in GROUPS and req:
                res["issues"].append(dict(level="BLOCK", msg=f"faltan tablas del grupo (no se generaron): {req}"))
            if g == "kpi_core":
                res["issues"] += self.cross_checks(frames)
            blocks = [i for i in res["issues"] if i["level"] == "BLOCK"]
            if blocks:
                res["status"] = "HOLD"
                held.append(g)
            elif self.mode == "dry_run":
                res["status"] = "OK(dry_run)"
            else:
                try:
                    for o in ops_eff:
                        if o["kind"] != "dml":
                            self._align(o)
                        self._execute(o)
                        res["published"].append(o["short"])
                except Exception as e:  # una falla de escritura a mitad de grupo debe ser visible
                    res["status"] = "ERROR"
                    res["issues"].append(dict(level="BLOCK", msg=f"falló la escritura: {type(e).__name__}: {str(e)[:200]}"))
                    held.append(g)
            self.results.append(res)
        self.report(t0)
        self.ops = []
        if held and raise_on_hold:
            raise PublishError(f"Grupos retenidos (se conserva la versión anterior): {held}. Ver reporte arriba.")
        return self.results

    def compare_with_bq(self, tol: float = 1e-6, skip=("captaciones_monitoreo_le", "pipeline_audit")):
        """Compara lo que ESTA corrida publicaría contra lo que hoy hay en BigQuery (p. ej. lo publicado por la V11
        el mismo día). Es la prueba de 'sin afectar la salida' con datos reales. Solo lectura."""
        import pandas_gbq
        rows = []
        for short, (fq, new) in sorted(self.staged.items()):
            if short in skip:
                continue
            try:
                old = pandas_gbq.read_gbq(f"SELECT * FROM `{fq}`", project_id=fq.split(".")[0], credentials=self.creds())
            except Exception as e:
                rows.append(dict(tabla=short, estado="SIN_TABLA_VIGENTE", detalle=f"{type(e).__name__}"))
                continue
            rows.append(self._cmp_frames(short, old, new, tol))
        res = pd.DataFrame(rows)
        self.log("\n" + "=" * 100 + "\n COMPARACIÓN vs BIGQUERY VIGENTE (¿la salida cambia?)\n" + "=" * 100)
        for r in rows:
            self.log(f"[{r['estado']:<12}] {r['tabla']:<40} {r.get('detalle', '')}")
        self.compare_result = res
        return res

    @staticmethod
    def _cmp_frames(short, old, new, tol):
        def norm(d):
            d = d.copy()
            for c in d.columns:
                if pd.api.types.is_datetime64_any_dtype(d[c]):
                    d[c] = d[c].dt.strftime("%Y-%m-%d")
            key = [c for c in ["fecha"] + _KEY_HELPERS if c in d.columns]
            if "fecha" in key:
                f = pd.to_datetime(d["fecha"], errors="coerce", dayfirst=True)
                d["_k"] = f.dt.strftime("%Y-%m-%d").fillna(d["fecha"].astype(str))
                key = ["_k"] + [c for c in key if c != "fecha"]
                d = d.sort_values(key, kind="stable")
            return d.reset_index(drop=True)
        try:
            a, b = norm(old), norm(new)
            det = []
            ca, cb = set(a.columns) - {"_k"}, set(b.columns) - {"_k"}
            if ca - cb:
                det.append(f"solo en BQ: {sorted(ca - cb)[:4]}")
            if cb - ca:
                det.append(f"columnas nuevas: {sorted(cb - ca)[:4]}")
            if len(a) != len(b):
                return dict(tabla=short, estado="DIFIERE", detalle=f"filas BQ={len(a)} nueva={len(b)}; " + "; ".join(det))
            bad = []
            for c in sorted(ca & cb):
                x, y = a[c], b[c]
                try:
                    xn, yn = pd.to_numeric(x, errors="raise").astype(float), pd.to_numeric(y, errors="raise").astype(float)
                    m = ~((xn.isna() & yn.isna()) | np.isclose(xn, yn, rtol=tol, atol=tol, equal_nan=True))
                except Exception:
                    m = x.astype(str).values != y.astype(str).values
                if m.any():
                    bad.append(f"{c}({int(m.sum())})")
            if bad:
                det.append(f"{len(bad)} columnas con valores distintos: {bad[:6]}")
            return dict(tabla=short, estado="DIFIERE" if bad or (ca - cb) else "IGUAL", detalle="; ".join(det))
        except Exception as e:
            return dict(tabla=short, estado="ERROR", detalle=f"{type(e).__name__}: {str(e)[:80]}")

    def report(self, t0=None):
        self.log("\n" + "=" * 100 + f"\n REPORTE DE PUBLICACIÓN | modo={self.mode} | hoy={self.today}\n" + "=" * 100)
        for r in self.results:
            self.log(f"[{r['status']:<11}] {r['group']:<14} tablas={r['tables']}")
            for i in r["issues"]:
                if i["level"] != "INFO":
                    self.log(f"      {i['level']:<5} {i.get('table', ''):<38} {i['msg']}")
        if self.report_dir:
            try:
                os.makedirs(self.report_dir, exist_ok=True)
                fn = os.path.join(self.report_dir, f"reporte_publicacion_{self.today or 'na'}.json")
                json.dump(self.results, open(fn, "w"), indent=1, default=str)
                self.log(f"[GATE] reporte guardado en {fn}")
            except Exception as e:
                self.log(f"[GATE] no se pudo guardar el reporte: {e}")


def get_pipeline_mode() -> str:
    m = os.environ.get("PIPELINE_MODE", "stage").strip().lower()
    return m if m in ("stage", "live", "dry_run") else "stage"


GATE = PublishGate(mode=get_pipeline_mode(), today=hoy_chile().date(),
                   allow_schema_drift=os.environ.get("PIPELINE_ALLOW_SCHEMA_DRIFT", "") == "1",
                   credentials_getter=lambda: globals().get("credentials")).install()
print(f"[V12] hoy_chile={hoy_chile().date()} | modo={GATE.mode} | fixes_off={sorted(_OFF) or '-'}")


In [ ]:
# ================== CAPTACIÓN v2 — MES EN CURSO (AGOSTO 2026) + MONITOREO ==================
# >>> BLOQUE COMPARTIDO: Desde aquí hasta "--- FIN BLOQUE COMPARTIDO ---" es IDÉNTICO al de 90d
# >>> La única diferencia es FORE_END (31d = fin de Agosto, 90d = CUTOFF_R2+1 + 89 días rolling)
import numpy as np
import pandas as pd
import pandas_gbq
import pydata_google_auth
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from xgboost import XGBRegressor
from google.cloud import bigquery
from pandas.tseries.offsets import MonthEnd

# ----------------------- Parámetros clave -----------------------
PROJECT_ID = "tenpo-bi-prod"
START_DATE = "2024-01-01"

# Tablas BigQuery
TABLE_MAIN    = "tenpo-bi-prod.kpitos.forecasting_le_captacion"
TABLE_MONITOR = "tenpo-bi-prod.kpitos.captaciones_monitoreo_le"

# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
# >>> DIFERENCIA 31d vs 90d: Solo cambia FORE_END
# >>> 31d: Agosto solamente (31 días)
# >>> 90d: Desde FORE_START hasta CUTOFF_R2+1+89 días (rolling)
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
# V11: mes objetivo AUTOMATICO (antes hardcodeado). Hora de Chile para no saltar de mes
# antes de tiempo en Colab (UTC).
_HOY_CL  = ahora_chile().tz_localize(None).normalize()
_MESES_ES = ["Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio", "Julio", "Agosto",
             "Septiembre", "Octubre", "Noviembre", "Diciembre"]
FORE_START = _HOY_CL.replace(day=1)
FORE_END   = FORE_START + MonthEnd(0)      # mes en curso completo
_DIAS_MES  = FORE_END.day
_M1, _M2, _M3 = [FORE_START - pd.DateOffset(months=k) for k in (1, 2, 3)]
_MES_TXT   = f"{_MESES_ES[FORE_START.month - 1]} {FORE_START.year}"
print(f"[V11 CAPTACION 31d] mes objetivo: {_MES_TXT} ({_DIAS_MES} dias) | ref: {_M3.date()}..{_M1.date()}")
FORE_IDX   = pd.date_range(FORE_START, FORE_END, freq="D")

# --- AJUSTES DE VOLATILIDAD Y EMPUJE ORGÁNICO (AFINADO) ---
MAX_DROP_PCT         = 0.0200
MAX_RISE_PCT_BASE    = 0.0070
MAX_RISE_PCT_FIN_MES = 0.0150

# Multiplicador orgánico: Le da más fuerza al modelo en los últimos días
FIN_MES_MULTIPLIER   = 1.15

TCA_MIN_STEP_MM    = 1.0
TCA_RISE_ABS_MAX   = 11.0
NOREM_FLOOR_MM     = 37_000.0
NOREM_SOFT_CEIL_MM = 47_000.0

# ----------------------- Autenticación -----------------------
scopes = [
    "https://www.googleapis.com/auth/bigquery",
    "https://www.googleapis.com/auth/cloud-platform",
]
credentials = pydata_google_auth.get_user_credentials(scopes)

# ----------------------- Utils -----------------------
def read_gbq_robusto(sql, project_id, credentials, fallback_location="southamerica-west1"):
    try:
        return pandas_gbq.read_gbq(sql, project_id=project_id, credentials=credentials)
    except Exception as e:
        if ("Not found: Dataset" in str(e)) and ("location" in str(e)):
            return pandas_gbq.read_gbq(
                sql, project_id=project_id, credentials=credentials,
                location=fallback_location
            )
        raise

def sanitize_float_columns(df):
    str_cols = {"fecha", "fecha_ejecucion", "estado_total", "estado_rem",
                "estado_norem", "estado_tca", "estado_mau_rem"}
    for col in df.columns:
        if col in str_cols:
            df[col] = df[col].astype(str)
            continue
        if df[col].dtype == object:
            converted = pd.to_numeric(df[col], errors="coerce")
            if converted.notna().sum() > len(converted) * 0.5:
                df[col] = converted.astype(float)
        elif pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].astype(float)
    return df

def write_bq_safe(df, table_id, project_id, credentials, write_disposition="WRITE_TRUNCATE"):
    client = bigquery.Client(project=project_id, credentials=credentials)
    df_clean = sanitize_float_columns(df.copy())

    job_config = bigquery.LoadJobConfig(
        write_disposition=getattr(bigquery.WriteDisposition, write_disposition),
        source_format=bigquery.SourceFormat.PARQUET,
        autodetect=True,
    )
    job = client.load_table_from_dataframe(df_clean, table_id, job_config=job_config)
    job.result()
    return len(df_clean)

# ------------------- DATA REAL (Queries) -------------------
# Lógica v2 alineada al reporte por segmento:
# - CPF/daily_balance se filtra por universo audiencia.
# - REM usa saldo de daily_balance, no interest_payment.
# - REM exige cuenta activa en última foto active=TRUE.
# - TCA usa usuarios XS y última Fecha_Proceso disponible dentro de 7 días.
# - Se usa SAFE_CAST + NULLIF para evitar error por Cupo_Total/saldo_dia vacío.
# - total_captaciones = saldo_total_mm + saldo_tca_mm.
sql_captacion_v2 = f"""
DECLARE start_date DATE DEFAULT DATE('{START_DATE}');
DECLARE end_date   DATE DEFAULT DATE('{(FORE_END + pd.Timedelta(days=1)):%Y-%m-%d}');

WITH
-- ============================================================
-- Universo audiencias: mismo criterio que reporte por segmento
-- ============================================================
base_customers AS (
  SELECT
    TRIM(CAST(user_id AS STRING)) AS user_id
  FROM `tenpo-bi-prod.users.users_audience_tags`
  WHERE fecha_ob_general IS NOT NULL
    AND (has_prepago_account IS TRUE OR has_tc_account IS TRUE)
    AND user_is_locked = FALSE
),

-- Universo REM: última foto con active = TRUE
activos_rem AS (
  SELECT
    TRIM(CAST(id AS STRING)) AS user_id,
    DATE(updated_at)         AS fecha_actividad
  FROM `business-data-raw.ingestor_paid_account_public.users`
  WHERE active = TRUE
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY id
    ORDER BY updated_at DESC, version DESC
  ) = 1
),

cal AS (
  SELECT day AS fecha
  FROM UNNEST(GENERATE_DATE_ARRAY(start_date, DATE_SUB(end_date, INTERVAL 1 DAY))) AS day
),

-- ============================================================
-- Saldo total CPF: filtrado por audiencia
-- ============================================================
base_cap AS (
  SELECT
    DATE_SUB(b.fecha, INTERVAL 1 DAY) AS fecha,
    TRIM(CAST(b.user AS STRING))      AS user_id,
    SAFE_DIVIDE(SAFE_CAST(NULLIF(TRIM(CAST(b.saldo_dia AS STRING)), '') AS NUMERIC), 1000000) AS val
  FROM `tenpo-bi-prod.balance.daily_balance` b
  INNER JOIN base_customers bc
    ON bc.user_id = TRIM(CAST(b.user AS STRING))
  WHERE DATE_SUB(b.fecha, INTERVAL 1 DAY) >= start_date
    AND DATE_SUB(b.fecha, INTERVAL 1 DAY) < end_date
    AND SAFE_CAST(NULLIF(TRIM(CAST(b.saldo_dia AS STRING)), '') AS NUMERIC) > 0
),
hist_cap AS (
  SELECT
    fecha,
    SUM(val) AS saldo_total_mm
  FROM base_cap
  GROUP BY 1
),

-- ============================================================
-- TCA: XS + última Fecha_Proceso disponible dentro de 7 días
-- ============================================================
users_tc_xs AS (
  SELECT
    CAST(individual_id AS STRING) AS id_persona,
    SAFE_CAST(user_id AS STRING)  AS user_id
  FROM `tenpo-bi-prod.kpitos.tipo_users_tc`
  WHERE name_segment = 'XS'
),
base_tca_daily AS (
  SELECT
    DATE(m.Fecha_Proceso) AS fecha_real,
    u.user_id,
    SAFE_DIVIDE(SAFE_CAST(NULLIF(TRIM(CAST(m.Cupo_Total AS STRING)), '') AS NUMERIC), 1000000) AS val
  FROM `operacionestc-operaciones-prod.MAE.Cuentas` m
  INNER JOIN users_tc_xs u USING(id_persona)
  WHERE DATE(m.Fecha_Proceso) >= DATE_SUB(start_date, INTERVAL 7 DAY)
    AND DATE(m.Fecha_Proceso) < end_date
    AND m.Status_id NOT IN ('200','6','14')
    AND SAFE_CAST(NULLIF(TRIM(CAST(m.Cupo_Total AS STRING)), '') AS NUMERIC) > 0
    AND m.id_cuenta NOT IN ('1','2','1733','1734','1735','1736','1737')
),
fechas_tca AS (
  SELECT
    c.fecha AS fecha_kpi,
    MAX(t.fecha_real) AS fecha_real
  FROM cal c
  LEFT JOIN (SELECT DISTINCT fecha_real FROM base_tca_daily) t
    ON t.fecha_real BETWEEN DATE_SUB(c.fecha, INTERVAL 7 DAY) AND c.fecha
  GROUP BY 1
),
hist_tca AS (
  SELECT
    f.fecha_kpi AS fecha,
    SUM(t.val) AS saldo_tca_mm
  FROM fechas_tca f
  LEFT JOIN base_tca_daily t
    ON t.fecha_real = f.fecha_real
  GROUP BY 1
),

-- ============================================================
-- Saldo REM: daily_balance filtrado por audiencia + REM activa
-- ============================================================
base_rem AS (
  SELECT
    DATE_SUB(b.fecha, INTERVAL 1 DAY) AS fecha,
    TRIM(CAST(b.user AS STRING))      AS user_id,
    SAFE_DIVIDE(SAFE_CAST(NULLIF(TRIM(CAST(b.saldo_dia AS STRING)), '') AS NUMERIC), 1000000) AS val
  FROM `tenpo-bi-prod.balance.daily_balance` b
  INNER JOIN base_customers bc
    ON bc.user_id = TRIM(CAST(b.user AS STRING))
  INNER JOIN activos_rem a
    ON a.user_id = TRIM(CAST(b.user AS STRING))
   AND a.fecha_actividad <= LAST_DAY(DATE_SUB(b.fecha, INTERVAL 1 DAY))
  WHERE DATE_SUB(b.fecha, INTERVAL 1 DAY) >= start_date
    AND DATE_SUB(b.fecha, INTERVAL 1 DAY) < end_date
    AND SAFE_CAST(NULLIF(TRIM(CAST(b.saldo_dia AS STRING)), '') AS NUMERIC) > 0
),
hist_rem AS (
  SELECT
    fecha,
    SUM(val) AS saldo_rem_mm
  FROM base_rem
  GROUP BY 1
),

-- ============================================================
-- MAU REM: misma fuente del reporte
-- ============================================================
hist_mau AS (
  SELECT
    fecha,
    COUNT(DISTINCT SAFE_CAST(user AS STRING)) AS mau_rem
  FROM `tenpo-bi-prod.kpitos.cohorts_cuenta_rem`
  WHERE fecha >= start_date
    AND fecha < end_date
  GROUP BY 1
)

SELECT
  c.fecha,
  COALESCE(cap.saldo_total_mm, 0) AS saldo_mm,
  COALESCE(rem.saldo_rem_mm, 0) AS saldo_rem_mm,
  GREATEST(COALESCE(cap.saldo_total_mm, 0) - COALESCE(rem.saldo_rem_mm, 0), 0) AS saldo_norem_mm,
  COALESCE(tca.saldo_tca_mm, 0) AS saldo_tca_mm,
  COALESCE(cap.saldo_total_mm, 0) + COALESCE(tca.saldo_tca_mm, 0) AS total_captaciones,
  COALESCE(mau.mau_rem, 0) AS mau_rem
FROM cal c
LEFT JOIN hist_cap cap USING(fecha)
LEFT JOIN hist_rem rem USING(fecha)
LEFT JOIN hist_tca tca USING(fecha)
LEFT JOIN hist_mau mau USING(fecha)
ORDER BY fecha
"""

df_hist_v2 = read_gbq_robusto(sql_captacion_v2, PROJECT_ID, credentials).drop_duplicates("fecha")
if df_hist_v2.empty:
    raise RuntimeError("La query integrada de captaciones v2 no devolvió filas.")

df_hist_v2["fecha"] = pd.to_datetime(df_hist_v2["fecha"]).dt.normalize()
for _c in ["saldo_mm", "saldo_rem_mm", "saldo_norem_mm", "saldo_tca_mm", "total_captaciones", "mau_rem"]:
    df_hist_v2[_c] = pd.to_numeric(df_hist_v2[_c], errors="coerce").astype(float)

# La query trae calendario completo hasta FORE_END; por eso el corte R/F debe basarse
# sólo en fechas con foto real positiva, no en el calendario rellenado con 0.
cap_real_dates_for_cutoff = pd.to_datetime(
    df_hist_v2.loc[df_hist_v2["saldo_mm"] > 0, "fecha"]
).dt.normalize()

# Mantener los mismos dataframes esperados por el bloque histórico/forecast.
df_cap = df_hist_v2[["fecha", "saldo_mm"]].copy()
df_rem = df_hist_v2[["fecha", "saldo_rem_mm"]].copy()
df_tca_raw = df_hist_v2[["fecha", "saldo_tca_mm"]].copy()
df_mau_rem_raw = df_hist_v2[["fecha", "mau_rem"]].copy()

# Evitar ceros estructurales por faltas de foto: se consideran nulos y se completan hacia adelante
# para variables de stock. La primera observación se completa hacia atrás si agosto parte antes de una foto.
for _df, _col in [(df_cap, "saldo_mm"), (df_rem, "saldo_rem_mm"), (df_tca_raw, "saldo_tca_mm")]:
    _df[_col] = _df[_col].replace(0, np.nan).ffill().bfill()

# ------------------- CORTES R/F -------------------
TODAY_CL = ahora_chile().normalize()
D2_NAT   = (TODAY_CL - pd.Timedelta(days=2)).tz_localize(None).normalize()

cap_dates = set(cap_real_dates_for_cutoff.tolist())
if len(cap_dates):
    real_dates = pd.to_datetime(sorted(list(cap_dates)))
    candidates = real_dates[(real_dates <= D2_NAT) & (real_dates <= FORE_END)]
    CUTOFF_R2 = candidates.max() if not candidates.empty else min(real_dates.max(), FORE_END)
else:
    CUTOFF_R2 = min(D2_NAT, FORE_END)

# Si el script se corre antes de que exista foto de agosto, fuerza todo agosto como forecast desde día 1.
if CUTOFF_R2 < FORE_START:
    CUTOFF_R2 = FORE_START - pd.Timedelta(days=1)

print(f"Cutoff R/F: Real hasta {CUTOFF_R2.strftime('%d-%m-%Y')}, Forecast desde {(CUTOFF_R2 + pd.Timedelta(days=1)).strftime('%d-%m-%Y')}")

# ------------------- TCA (SUAVIZADO) -------------------
tca_hist = df_tca_raw.drop_duplicates("fecha").set_index("fecha")["saldo_tca_mm"].sort_index()
tca_hist = tca_hist.replace(0, np.nan).ffill().bfill()
full_idx_tca = tca_hist.index.union(FORE_IDX).sort_values()
tca_combined = tca_hist.reindex(full_idx_tca).ffill().bfill()
tca_series = tca_combined.reindex(FORE_IDX).ffill().bfill().copy()

tca_deltas = tca_hist.diff().dropna()
pos_deltas = tca_deltas[tca_deltas > 0]
step_tca = float(np.median(pos_deltas.tail(7))) if not pos_deltas.empty else TCA_MIN_STEP_MM
step_tca = np.clip(step_tca * 0.90, TCA_MIN_STEP_MM, TCA_RISE_ABS_MAX)

last_val_tca = float(tca_combined.loc[:CUTOFF_R2].dropna().iloc[-1]) if not tca_combined.loc[:CUTOFF_R2].dropna().empty else float(tca_series.dropna().iloc[0] if not tca_series.dropna().empty else 9537.0)

for d in FORE_IDX:
    if d <= CUTOFF_R2:
        if pd.notna(tca_series.loc[d]):
            last_val_tca = float(tca_series.loc[d])
        else:
            tca_series.loc[d] = last_val_tca
    else:
        last_val_tca += step_tca
        tca_series.loc[d] = last_val_tca
tca_series = tca_series.astype(float)

# ------------------- TOTAL (FORECAST POR PERFIL DOM NORMALIZADO) -------------------
hist_series_raw = df_cap.set_index("fecha")["saldo_mm"].sort_index().replace(0, np.nan)
_full_hist_idx = pd.date_range(hist_series_raw.index.min(), max(FORE_END, CUTOFF_R2), freq="D")
hist_series = hist_series_raw.reindex(_full_hist_idx).ffill().bfill()
_start_candidates = hist_series.loc[:CUTOFF_R2].dropna() if CUTOFF_R2 >= hist_series.index.min() else pd.Series(dtype=float)
start_level = float(_start_candidates.iloc[-1]) if not _start_candidates.empty else float(hist_series.dropna().iloc[-1])

# ---- 1. Construir perfil DOM normalizado con meses de referencia ----
ref_months = [_M3, _M2, _M1]   # V11: 3 meses cerrados previos a FORE_START
dom_profiles = {}
for m_start in ref_months:
    m_end = m_start + pd.offsets.MonthEnd(1)
    idx_m = pd.date_range(m_start, m_end, freq="D")
    s_m   = hist_series.reindex(idx_m).ffill().bfill()
    if s_m.isna().all() or len(s_m) < 20:
        continue
    base_m = s_m.iloc[0]
    if base_m == 0:
        continue
    dom_profiles[m_start] = (s_m / base_m).values

if dom_profiles:
    keys = sorted(dom_profiles.keys())
    weights = np.arange(1, len(keys) + 1, dtype=float)
    max_len = max(len(v) for v in dom_profiles.values())
    profile_matrix = np.full((len(keys), max_len), np.nan)
    for i, k in enumerate(keys):
        v = dom_profiles[k]
        profile_matrix[i, :len(v)] = v
    dom_ratio_avg = np.average(
        np.where(np.isnan(profile_matrix),
                 np.nanmean(profile_matrix, axis=0),
                 profile_matrix),
        weights=weights, axis=0
    )
else:
    dom_ratio_avg = np.ones(_DIAS_MES)

# Suavizar el perfil DOM para eliminar escalones
dom_ratio_avg = pd.Series(dom_ratio_avg).rolling(3, center=True, min_periods=1).mean().values

# ---- 2. Referencias históricas de cierre ----
# V11: "jul/jun/may" = mes-1 / mes-2 / mes-3 respecto de FORE_START (antes fijos)
def _serie_mes(_m):
    _fin = _m + MonthEnd(0)
    return hist_series.loc[_m:_fin] if hist_series.index.min() <= _fin else pd.Series(dtype=float)
jul_series = _serie_mes(_M1)
jun_series = _serie_mes(_M2)
may_series = _serie_mes(_M3)

close_jul = float(jul_series.dropna().iloc[-1]) if not jul_series.dropna().empty else start_level
close_jun = float(jun_series.dropna().iloc[-1]) if not jun_series.dropna().empty else start_level
close_may = float(may_series.dropna().iloc[-1]) if not may_series.dropna().empty else close_jun

# Tendencia de cierre: promedio de los últimos 2 deltas (May→Jun y Jun→Jul)
delta_jun = close_jun - close_may
delta_jul = close_jul - close_jun
tendencia_cierre = (delta_jun + delta_jul) / 2.0

# Cierre Agosto: tendencia conservadora al 50%
close_agosto_target = close_jul + tendencia_cierre * 0.50

# ---- 3. Anclar implied_day1 ----
cutoff_day   = CUTOFF_R2.day
cutoff_level = start_level

ratio_at_cutoff = dom_ratio_avg[min(cutoff_day - 1, len(dom_ratio_avg) - 1)]
implied_day1_from_cutoff   = cutoff_level / ratio_at_cutoff if ratio_at_cutoff != 0 else cutoff_level
implied_day1_from_close_jul = close_jul

implied_day1_mid = max(implied_day1_from_cutoff, implied_day1_from_close_jul) * 0.980

profile_len = len(dom_ratio_avg)
close_ratio = dom_ratio_avg[min(_DIAS_MES - 1, profile_len - 1)]  # V11: ultimo dia del mes objetivo
implied_day1_close = close_agosto_target / close_ratio if close_ratio > 0 else close_agosto_target

TECHO_ABS = close_agosto_target * 1.003
PISO_ABS  = close_agosto_target * 0.900

# ---- 4. Construir serie forecast ----
total_base = pd.Series(index=FORE_IDX, dtype=float)

for d in FORE_IDX:
    if d <= CUTOFF_R2:
        if d in hist_series.index:
            total_base.loc[d] = float(hist_series.loc[d])
        else:
            total_base.loc[d] = cutoff_level
        continue

    day_idx = min(d.day - 1, profile_len - 1)
    ratio = dom_ratio_avg[day_idx]

    # Blend gradual entre mid y close: transición suave días 24→31
    if d.day <= 23:
        blend = 0.0  # 100% mid
    elif d.day >= _DIAS_MES:
        blend = 1.0  # 100% close
    else:
        blend = (d.day - 23) / float(_DIAS_MES - 23)  # V11: gradual 0→1 hasta el ultimo dia del mes

    implied_blended = implied_day1_mid * (1 - blend) + implied_day1_close * blend
    projected = implied_blended * ratio

    projected = min(projected, TECHO_ABS)
    projected = max(projected, PISO_ABS)
    total_base.loc[d] = projected

total_base = total_base.ffill().bfill()

# ------------------- NOREM / REM -------------------
hist_idx = pd.date_range(df_cap["fecha"].min(), max(CUTOFF_R2, df_cap["fecha"].min()), freq="D")
tot_h = df_cap.set_index("fecha")["saldo_mm"].replace(0, np.nan).reindex(hist_idx).ffill().bfill()
rem_h = df_rem.set_index("fecha")["saldo_rem_mm"].replace(0, np.nan).reindex(hist_idx).ffill().bfill()
norem_h = (tot_h - rem_h).clip(lower=0.0)

share_h = (norem_h / tot_h.replace(0, np.nan)).fillna(0.15)
recent_share = share_h.iloc[-180:].groupby(share_h.iloc[-180:].index.day).mean()

dom_profile = pd.Series(index=range(1, 32), dtype=float)
dom_profile.loc[recent_share.index] = recent_share.values
dom_profile = dom_profile.ffill().bfill().rolling(3, center=True, min_periods=1).mean()

norem_all = pd.Series(index=FORE_IDX, dtype=float)
rem_all   = pd.Series(index=FORE_IDX, dtype=float)

for d in FORE_IDX:
    if d <= CUTOFF_R2:
        continue
    sh  = float(dom_profile.loc[d.day])
    tot = float(total_base.loc[d])
    cand_norem = np.clip(tot * sh, NOREM_FLOOR_MM, NOREM_SOFT_CEIL_MM)
    norem_all.loc[d] = cand_norem
    rem_all.loc[d]   = max(0.0, tot - cand_norem)

# ------------------- MAU REM (XGBOOST) -------------------
# Viene desde la query integrada v2 para mantener el mismo criterio base.
df_mau_rem_raw = df_hist_v2[["fecha", "mau_rem"]].copy()

mau_rem_series = pd.Series(index=FORE_IDX, dtype=float)

start_mau         = 175000.0
model_xgb_mau     = None
pendiente_mau     = 1000.0
X_train_mau       = pd.DataFrame()
df_mau_clean      = pd.DataFrame()
hist_max_rise_mau = 3000.0
hist_max_drop_mau = -1000.0

if not df_mau_rem_raw.empty:
    df_mau_rem_raw["fecha"] = pd.to_datetime(df_mau_rem_raw["fecha"]).dt.normalize()
    df_mau_clean = df_mau_rem_raw.drop_duplicates("fecha").set_index("fecha").sort_index()
    df_mau_clean["mau_rem"] = pd.to_numeric(df_mau_clean["mau_rem"], errors="coerce").astype(float)
    df_mau_clean = df_mau_clean[df_mau_clean["mau_rem"] > 100000]

    if not df_mau_clean.empty:
        full_idx_mau = pd.date_range(df_mau_clean.index.min(), df_mau_clean.index.max(), freq="D")
        df_mau_clean = df_mau_clean.reindex(full_idx_mau).interpolate(method="linear")
        df_mau_clean.index.name = "fecha"

        fecha_max_mau = df_mau_clean.index.max()
        df_mau_train = df_mau_clean[
            df_mau_clean.index >= (fecha_max_mau - pd.Timedelta(days=180))
        ].copy()

        if not df_mau_train.empty:
            df_mau_train["delta_mau"]  = df_mau_train["mau_rem"].diff()
            df_mau_train["dia_mes"]    = df_mau_train.index.day
            df_mau_train["dia_semana"] = df_mau_train.index.dayofweek
            df_mau_train["es_finde"]   = df_mau_train["dia_semana"].isin([5, 6]).astype(int)
            df_mau_train = df_mau_train.dropna()

            X_train_mau = df_mau_train[["dia_mes", "dia_semana", "es_finde"]]
            y_train_mau = df_mau_train["delta_mau"]

            if not X_train_mau.empty:
                model_xgb_mau = XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=4, random_state=42)
                model_xgb_mau.fit(X_train_mau, y_train_mau)
                hist_max_rise_mau = float(y_train_mau.max() * 1.5)
                hist_max_drop_mau = float(y_train_mau.min() * 1.5)

            mau_smooth = df_mau_clean["mau_rem"].rolling(14, min_periods=1).mean()
            dias_t = min(180, len(mau_smooth) - 1)
            pendiente_mau = ((mau_smooth.iloc[-1] - mau_smooth.iloc[-dias_t]) / dias_t if dias_t > 0 else 1000.0)
            if pendiente_mau < 500.0:
                pendiente_mau = 1000.0

            start_mau = float(df_mau_clean["mau_rem"].iloc[-1])

curr_mau = start_mau
for d in FORE_IDX:
    if model_xgb_mau is not None and not X_train_mau.empty:
        X_pred_mau = pd.DataFrame({"dia_mes": [d.day], "dia_semana": [d.dayofweek], "es_finde": [1 if d.dayofweek in [5, 6] else 0]})
        pred_delta_mau = float(model_xgb_mau.predict(X_pred_mau)[0])
        pred_delta_mau = np.clip(pred_delta_mau, hist_max_drop_mau, hist_max_rise_mau)
    else:
        pred_delta_mau = pendiente_mau

    if d <= CUTOFF_R2 and not df_mau_clean.empty and d in df_mau_clean.index:
        curr_mau = float(df_mau_clean.loc[d, "mau_rem"])
    else:
        curr_mau += pred_delta_mau

    mau_rem_series.loc[d] = curr_mau

# ------------------- ARMADO DEL OUTPUT -------------------
out = pd.DataFrame(index=FORE_IDX)
out.index.name = "fecha"
mask_real = out.index <= CUTOFF_R2
mask_fore = ~mask_real

real_cap = df_cap.set_index("fecha")["saldo_mm"].replace(0, np.nan).reindex(FORE_IDX).ffill().bfill()
real_rem = df_rem.set_index("fecha")["saldo_rem_mm"].replace(0, np.nan).reindex(FORE_IDX).ffill().bfill()

out["saldo_total_mm"] = np.where(mask_real, real_cap, total_base.reindex(FORE_IDX).ffill().bfill())
out["saldo_rem_mm"]   = np.where(mask_real, real_rem, rem_all.reindex(FORE_IDX).ffill().bfill())
out["saldo_total_mm"] = pd.to_numeric(out["saldo_total_mm"], errors="coerce").replace(0, np.nan).bfill().ffill()
out["saldo_rem_mm"]   = pd.to_numeric(out["saldo_rem_mm"], errors="coerce").replace(0, np.nan).bfill().ffill()

out["saldo_norem_mm"] = (out["saldo_total_mm"] - out["saldo_rem_mm"]).clip(lower=0.0)
out["saldo_tca_mm"]   = tca_series
out["mau_rem"]         = mau_rem_series

out["saldo_total_real_mm"]         = np.where(mask_real, out["saldo_total_mm"], 0.0)
out["saldo_total_pronosticado_mm"] = np.where(mask_fore, out["saldo_total_mm"], 0.0)
out["saldo_rem_real_mm"]           = np.where(mask_real, out["saldo_rem_mm"], 0.0)
out["saldo_rem_pronosticado_mm"]   = np.where(mask_fore, out["saldo_rem_mm"], 0.0)
out["saldo_norem_real_mm"]         = np.where(mask_real, out["saldo_norem_mm"], 0.0)
out["saldo_norem_pronosticado_mm"] = np.where(mask_fore, out["saldo_norem_mm"], 0.0)
out["saldo_tca_real_mm"]           = np.where(mask_real, out["saldo_tca_mm"], 0.0)
out["saldo_tca_pronosticado_mm"]   = np.where(mask_fore, out["saldo_tca_mm"], 0.0)
out["mau_rem_real"]                = np.where(mask_real, out["mau_rem"], 0.0)
out["mau_rem_pronosticado"]        = np.where(mask_fore, out["mau_rem"], 0.0)

out["estado_total"]   = np.where(mask_real, "R", "F")
out["estado_rem"]     = np.where(mask_real, "R", "F")
out["estado_norem"]   = np.where(mask_real, "R", "F")
out["estado_tca"]     = np.where(mask_real, "R", "F")
out["estado_mau_rem"] = np.where(mask_real, "R", "F")

out["total_captaciones"] = out["saldo_total_mm"] + out["saldo_tca_mm"]

# Diagnóstico anti-cero: estas columnas son stocks; en forecast no deberían quedar en cero
# salvo que realmente no exista ninguna base histórica utilizable.
if mask_fore.any():
    _forecast_check = out.loc[mask_fore, ["saldo_total_mm", "saldo_rem_mm", "saldo_norem_mm", "saldo_tca_mm", "total_captaciones", "mau_rem"]]
    _zero_cols = [c for c in _forecast_check.columns if float(pd.to_numeric(_forecast_check[c], errors="coerce").fillna(0).abs().sum()) == 0.0]
    if _zero_cols:
        print(f"[WARN] Columnas forecast en cero: {_zero_cols}. Revisar fuente o filtros de query v2.")

# --- FIN BLOQUE COMPARTIDO ---

# ------------------- FORMATEO FINAL -------------------
dt_fechas = out.index.copy()
out = out.reset_index()
out["numfecha"] = out["fecha"].dt.day
out["fecha"]    = out["fecha"].dt.strftime("%d-%m-%Y")

cols_float = [
    c for c in out.columns
    if ("mm" in c or "mau" in c or "total" in c) and not c.startswith("estado")
]
for c in cols_float:
    out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0).astype(float)

# ------------------- PREVIEW CONSOLA -------------------
print("\n" + "=" * 85)
print(f" PREVIEW: PROYECCIÓN CAPTACIONES {_MES_TXT.upper()} ({_DIAS_MES} días) ".center(85, "="))
print("=" * 85)
preview_cols = [
    "fecha", "estado_total", "total_captaciones",
    "saldo_total_mm", "saldo_tca_mm", "saldo_rem_mm",
    "saldo_norem_mm", "mau_rem"
]
print(out[preview_cols].to_string(index=False, float_format=lambda x: f"{x:,.1f}"))
print("=" * 85 + "\n")

# ==============================================================================
# >>> GRÁFICO 1: COMPARATIVA INTRA-MES (día del mes superpuesto)
# ==============================================================================
print("Generando Gráfico Comparativo Intra-mes...")

hist_start = _M3
hist_end = FORE_START - pd.Timedelta(days=1)
idx_hist = pd.date_range(hist_start, hist_end, freq="D")

s_cap = df_cap.set_index("fecha")["saldo_mm"].reindex(idx_hist).ffill()
s_tca = df_tca_raw.set_index("fecha")["saldo_tca_mm"].reindex(idx_hist).ffill()
df_hist_plot = pd.DataFrame({
    "fecha": idx_hist,
    "total_captaciones": (s_cap + s_tca).values,
    "estado_total": "R"
})

df_fore_plot = pd.DataFrame({
    "fecha": dt_fechas,
    "total_captaciones": out["total_captaciones"].values,
    "estado_total": out["estado_total"].values
})

df_plot = pd.concat([df_hist_plot, df_fore_plot], ignore_index=True)
df_plot["dia_mes"] = df_plot["fecha"].dt.day
df_plot["mes_anio"] = df_plot["fecha"].dt.to_period("M")

colores = ['#ff7f0e', '#2ca02c', '#9467bd', '#2980B9']
plt.figure(figsize=(16, 8))
meses_unicos = sorted(df_plot["mes_anio"].unique())

for i, mes in enumerate(meses_unicos):
    df_mes = df_plot[df_plot["mes_anio"] == mes].sort_values("dia_mes")
    df_r = df_mes[df_mes["estado_total"] == "R"]
    df_f = df_mes[df_mes["estado_total"] == "F"]
    color_mes = colores[i % len(colores)]

    if not df_r.empty:
        grosor = 3.0 if str(mes) == FORE_START.strftime('%Y-%m') else 2.5
        plt.plot(df_r["dia_mes"], df_r["total_captaciones"],
                 color=color_mes, linewidth=grosor, linestyle='-', marker='o', markersize=4, alpha=0.9,
                 label=f"{mes} (Real)")
    if not df_f.empty:
        if not df_r.empty:
            df_conn = pd.concat([df_r.tail(1), df_f])
            plt.plot(df_conn["dia_mes"], df_conn["total_captaciones"],
                     color=color_mes, linewidth=4.0, linestyle='--', marker='o', markersize=4)
            plt.plot([], [], color=color_mes, linewidth=4.0, linestyle='--', label=f"{mes} (Forecast)")
        else:
            plt.plot(df_f["dia_mes"], df_f["total_captaciones"],
                     color=color_mes, linewidth=4.0, linestyle='--', marker='o', markersize=4,
                     label=f"{mes} (Forecast)")

plt.title(f"Comparativa Intra-mes: Evolución de Captaciones Totales (hasta {_MES_TXT})", fontsize=18, fontweight='bold', pad=20)
plt.xlabel(f"Día del Mes (1 al {_DIAS_MES})", fontsize=13)
plt.ylabel("Total Captaciones (MM CLP)", fontsize=13)
plt.xticks(range(1, 32))
plt.gca().yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: f'{int(x):,}'))
plt.grid(True, linestyle='--', alpha=0.5)
handles, labels = plt.gca().get_legend_handles_labels()
by_label = dict(zip(labels, handles))
plt.legend(by_label.values(), by_label.keys(), loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=12)
plt.tight_layout()
plt.savefig(f"meses_sobrepuestos_captaciones_{FORE_START:%Y_%m}.png", bbox_inches='tight')
print(f"--> Gráfico 1 guardado como 'meses_sobrepuestos_captaciones_{FORE_START:%Y_%m}.png'\n")

# ==============================================================================
# >>> GRÁFICO 2: DOW-ALIGNED — Curvas históricas alineadas por día de semana
# >>> Eje X = los 31 días de Agosto. Meses anteriores mapeados por DOW.
# ==============================================================================
print("Generando Gráfico DOW-aligned...")
import matplotlib.lines as mlines

DOW_NAMES_ES = ['Lun', 'Mar', 'Mié', 'Jue', 'Vie', 'Sáb', 'Dom']
HIST_COLORS  = ["#E67E22", "#27AE60", "#8E44AD"]

# --- Agosto: datos del forecast (Real + F) ---
ago_fechas = dt_fechas.copy()
ago_vals   = out["total_captaciones"].values.astype(float)
ago_estado = out["estado_total"].values
ago_days   = ago_fechas.day
ago_dow    = ago_fechas.weekday
ago_dow_occ = pd.Series(ago_dow).groupby(ago_dow).cumcount().values + 1  # 1er Lun, 2do Lun...

# --- Meses históricos alineados por DOW sobre el eje de Agosto ---
hist_months = [
    _M1, _M2, _M3,   # V11
]

hist_lines = []  # lista de (day_positions[], values[], label)
for m_start in hist_months:
    m_end = m_start + pd.offsets.MonthEnd(1)
    m_idx = pd.date_range(m_start, m_end, freq="D")
    s_cap_m = df_cap.set_index("fecha")["saldo_mm"].reindex(m_idx).ffill()
    s_tca_m = df_tca_raw.set_index("fecha")["saldo_tca_mm"].reindex(m_idx).ffill()
    total_m = (s_cap_m + s_tca_m).values

    if np.isnan(total_m).all():
        hist_lines.append(None)
        continue

    m_dow = m_idx.weekday
    m_dow_occ = pd.Series(m_dow).groupby(m_dow).cumcount().values + 1

    # Para cada día de Agosto, buscar en este mes el mismo DOW + ocurrencia
    aligned_days = []
    aligned_vals = []
    for i in range(len(ago_fechas)):
        mask = (m_dow == ago_dow[i]) & (m_dow_occ == ago_dow_occ[i])
        idxs = np.where(mask)[0]
        if len(idxs) > 0:
            aligned_days.append(ago_days[i])
            aligned_vals.append(total_m[idxs[0]])

    hist_lines.append((aligned_days, aligned_vals))

# --- Promedio histórico DOW-aligned ---
avg_vals = np.full(len(ago_fechas), np.nan)
for i in range(len(ago_fechas)):
    day_vals = []
    for hl in hist_lines:
        if hl is not None:
            for j, d in enumerate(hl[0]):
                if d == ago_days[i]:
                    day_vals.append(hl[1][j])
    if day_vals:
        avg_vals[i] = np.mean(day_vals)

# --- Plot ---
fig, ax = plt.subplots(figsize=(18, 8))

# Históricos (punteados)
for i, (hl, m_start) in enumerate(zip(hist_lines, hist_months)):
    if hl is not None:
        ax.plot(hl[0], hl[1], color=HIST_COLORS[i], linestyle='--', linewidth=1.5, alpha=0.50)

# Promedio (negro grueso)
ax.plot(ago_days, avg_vals, color='black', linewidth=3, alpha=0.85)

# Agosto Real
real_mask = np.array(ago_estado) == "R"
fc_mask   = np.array(ago_estado) == "F"

if real_mask.any():
    ax.plot(ago_days[real_mask], ago_vals[real_mask],
            color='#2980B9', linewidth=3, marker='o', markersize=5,
            markerfacecolor='#2980B9', markeredgecolor='white', markeredgewidth=0.8, zorder=5)

# Conector R→F
if real_mask.any() and fc_mask.any():
    last_r = np.where(real_mask)[0][-1]
    first_f = np.where(fc_mask)[0][0]
    ax.plot([ago_days[last_r], ago_days[first_f]], [ago_vals[last_r], ago_vals[first_f]],
            color='#2980B9', linewidth=2.5, linestyle='--', alpha=0.7, zorder=4)

# Agosto Forecast
if fc_mask.any():
    ax.plot(ago_days[fc_mask], ago_vals[fc_mask],
            color='#2980B9', linewidth=3, linestyle='--', marker='^', markersize=6,
            markerfacecolor='#E74C3C', markeredgecolor='white', markeredgewidth=0.8, zorder=5)

# Corte R/F
if real_mask.any() and fc_mask.any():
    corte_x = ago_days[np.where(real_mask)[0][-1]] + 0.5
    ax.axvline(x=corte_x, color='red', linestyle=':', linewidth=1.5, alpha=0.6)

# Sombreado fines de semana
for i in range(len(ago_fechas)):
    if ago_dow[i] >= 5:
        ax.axvspan(ago_days[i] - 0.4, ago_days[i] + 0.4, color='gray', alpha=0.08, zorder=0)

# Eje X: día + nombre DOW
x_labels = [f'{ago_days[i]}\n{DOW_NAMES_ES[ago_dow[i]]}' for i in range(len(ago_fechas))]
ax.set_xticks(ago_days)
ax.set_xticklabels(x_labels, fontsize=7)

# Leyenda
legend_elements = []
for i, m in enumerate(hist_months):
    legend_elements.append(mlines.Line2D([], [], color=HIST_COLORS[i], linestyle='--', linewidth=1.5, alpha=0.5, label=f'Real {m.strftime("%Y-%m")}'))
legend_elements.append(mlines.Line2D([], [], color='black', linewidth=3, alpha=0.85, label='Promedio 3 meses (DOW)'))
legend_elements.append(mlines.Line2D([], [], color='#2980B9', linewidth=3, marker='o', markersize=5, label='Ago 2026 — Real'))
legend_elements.append(mlines.Line2D([], [], color='#2980B9', linewidth=3, linestyle='--', marker='^', markersize=6, markerfacecolor='#E74C3C', markeredgecolor='white', label='Ago 2026 — Forecast'))
ax.legend(handles=legend_elements, loc='upper left', fontsize=9, framealpha=0.9)

ax.set_title(f'Captaciones {_MES_TXT} vs Histórico — Alineado por Día de Semana', fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel('Total Captaciones (MM CLP)', fontsize=12)
ax.set_xlabel(f'Día de {_MESES_ES[FORE_START.month - 1]}', fontsize=11)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: f'{int(x):,}'))
ax.grid(True, alpha=0.2, linestyle=':')
fig.tight_layout()
plt.savefig(f"captaciones_dow_aligned_{FORE_START:%Y_%m}.png", bbox_inches='tight', dpi=150)
print(f"--> Gráfico 2 guardado como 'captaciones_dow_aligned_{FORE_START:%Y_%m}.png'\n")

# ==============================================================================
# >>> SECCIÓN EXCLUSIVA 31d: ESCRITURA BQ (tabla principal + monitoreo)
# ==============================================================================
client = bigquery.Client(project=PROJECT_ID, credentials=credentials)

# --- 1. TABLA PRINCIPAL (TRUNCATE) ---
print(f"Escribiendo tabla principal: {TABLE_MAIN} ...")
n = write_bq_safe(out, TABLE_MAIN, PROJECT_ID, credentials, "WRITE_TRUNCATE")
print(f"OK. {TABLE_MAIN} actualizada ({n} filas). Cutoff: {CUTOFF_R2.strftime('%d-%m-%Y')}")

# --- 2. MONITOREO (FOTO DIARIA — última ejecución del día gana) ---
print("Procesando Snapshot de Monitoreo...")
fecha_hoy = ahora_chile().normalize().tz_localize(None)
fecha_hoy_iso = fecha_hoy.strftime("%Y-%m-%d")

df_monitor = out.copy()
df_monitor.insert(0, "fecha_ejecucion", fecha_hoy_iso)

write_mode = "WRITE_APPEND"

try:
    sql_check = f"""
        SELECT EXTRACT(MONTH FROM DATE(MAX(fecha_ejecucion))) AS mes
        FROM `{TABLE_MONITOR}`
    """
    last_month = pandas_gbq.read_gbq(
        sql_check, project_id=PROJECT_ID, credentials=credentials
    )["mes"].iloc[0]

    if last_month is not None and int(last_month) != fecha_hoy.month:
        print("Cambio de mes detectado. Reseteando tabla de monitoreo...")
        write_mode = "WRITE_TRUNCATE"
    else:
        query_delete = f"""
            DELETE FROM `{TABLE_MONITOR}`
            WHERE fecha_ejecucion = '{fecha_hoy_iso}'
        """
        client.query(query_delete).result()
except Exception as e:
    write_mode = "WRITE_TRUNCATE"

n_mon = write_bq_safe(df_monitor, TABLE_MONITOR, PROJECT_ID, credentials, write_mode)
print(f"Monitoreo guardado ({n_mon} filas, ejecución {fecha_hoy_iso})")
print(f"¡Actualización Exitosa de {_MES_TXT} ({_DIAS_MES} días) y Monitoreo!")

In [ ]:
# ================== CAPTACIÓN v2 — 90 DÍAS ROLLING (AGOSTO 2026) ==================
# >>> BLOQUE COMPARTIDO: Mantener idéntico al 31d en la lógica del primer mes
# >>> Diferencia 90d: FORE_END se calcula dinámicamente tras CUTOFF_R2
import numpy as np
import pandas as pd
import pandas_gbq
import pydata_google_auth
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from xgboost import XGBRegressor
from google.cloud import bigquery
from pandas.tseries.offsets import MonthEnd

# ----------------------- Parámetros clave -----------------------
PROJECT_ID = "tenpo-bi-prod"
START_DATE = "2024-01-01"

# Tablas BigQuery
TABLE_MAIN_90D = "tenpo-bi-prod.kpitos.forecasting_le_captacion_90d"

# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
# >>> 90d rolling para Agosto 2026
# >>> FORE_START queda fijo en el día 1 del mes.
# >>> FORE_END y FORE_IDX se calculan dinámicamente después de CUTOFF_R2.
# >>> Los días de Agosto deben coincidir con el script 31d.
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
# V11: mes objetivo AUTOMATICO (antes hardcodeado). Hora de Chile para no saltar de mes
# antes de tiempo en Colab (UTC).
_HOY_CL  = ahora_chile().tz_localize(None).normalize()
_MESES_ES = ["Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio", "Julio", "Agosto",
             "Septiembre", "Octubre", "Noviembre", "Diciembre"]
FORE_START = _HOY_CL.replace(day=1)
FIRST_MONTH_END = FORE_START + MonthEnd(0)
_DIAS_MES  = FIRST_MONTH_END.day
_M1, _M2, _M3 = [FORE_START - pd.DateOffset(months=k) for k in (1, 2, 3)]
_MES_TXT   = f"{_MESES_ES[FORE_START.month - 1]} {FORE_START.year}"
print(f"[V11 CAPTACION 90d] mes inicial: {_MES_TXT} | ref: {_M3.date()}..{_M1.date()}")

# La query real solo necesita datos hasta el cierre del mes / D-2.
# No se consultan datos futuros del horizonte 90d; esos se proyectan en Python.
TODAY_CL_PRE = ahora_chile().normalize()
D2_NAT_PRE   = (TODAY_CL_PRE - pd.Timedelta(days=2)).tz_localize(None).normalize()
SQL_END_DATE = max(FIRST_MONTH_END, D2_NAT_PRE) + pd.Timedelta(days=1)

# --- AJUSTES DE VOLATILIDAD Y EMPUJE ORGÁNICO (AFINADO) ---
MAX_DROP_PCT         = 0.0200
MAX_RISE_PCT_BASE    = 0.0070
MAX_RISE_PCT_FIN_MES = 0.0150

# Multiplicador orgánico: Le da más fuerza al modelo en los últimos días
FIN_MES_MULTIPLIER   = 1.15

TCA_MIN_STEP_MM    = 1.0
TCA_RISE_ABS_MAX   = 11.0
NOREM_FLOOR_MM     = 37_000.0
NOREM_SOFT_CEIL_MM = 47_000.0

# ----------------------- Autenticación -----------------------
scopes = [
    "https://www.googleapis.com/auth/bigquery",
    "https://www.googleapis.com/auth/cloud-platform",
]
credentials = pydata_google_auth.get_user_credentials(scopes)

# ----------------------- Utils -----------------------
def read_gbq_robusto(sql, project_id, credentials, fallback_location="southamerica-west1"):
    try:
        return pandas_gbq.read_gbq(sql, project_id=project_id, credentials=credentials)
    except Exception as e:
        if ("Not found: Dataset" in str(e)) and ("location" in str(e)):
            return pandas_gbq.read_gbq(
                sql, project_id=project_id, credentials=credentials,
                location=fallback_location
            )
        raise

def sanitize_float_columns(df):
    str_cols = {"fecha", "fecha_ejecucion", "estado_total", "estado_rem",
                "estado_norem", "estado_tca", "estado_mau_rem"}
    for col in df.columns:
        if col in str_cols:
            df[col] = df[col].astype(str)
            continue
        if df[col].dtype == object:
            converted = pd.to_numeric(df[col], errors="coerce")
            if converted.notna().sum() > len(converted) * 0.5:
                df[col] = converted.astype(float)
        elif pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].astype(float)
    return df

def write_bq_safe(df, table_id, project_id, credentials, write_disposition="WRITE_TRUNCATE"):
    client = bigquery.Client(project=project_id, credentials=credentials)
    df_clean = sanitize_float_columns(df.copy())

    job_config = bigquery.LoadJobConfig(
        write_disposition=getattr(bigquery.WriteDisposition, write_disposition),
        source_format=bigquery.SourceFormat.PARQUET,
        autodetect=True,
    )
    job = client.load_table_from_dataframe(df_clean, table_id, job_config=job_config)
    job.result()
    return len(df_clean)

# ------------------- DATA REAL (Queries) -------------------
# Lógica v2 alineada al reporte por segmento:
# - CPF/daily_balance se filtra por universo audiencia.
# - REM usa saldo de daily_balance, no interest_payment.
# - REM exige cuenta activa en última foto active=TRUE.
# - TCA usa usuarios XS y última Fecha_Proceso disponible dentro de 7 días.
# - Se usa SAFE_CAST + NULLIF para evitar error por Cupo_Total/saldo_dia vacío.
# - total_captaciones = saldo_total_mm + saldo_tca_mm.
sql_captacion_v2 = f"""
DECLARE start_date DATE DEFAULT DATE('{START_DATE}');
DECLARE end_date   DATE DEFAULT DATE('{SQL_END_DATE:%Y-%m-%d}');

WITH
-- ============================================================
-- Universo audiencias: mismo criterio que reporte por segmento
-- ============================================================
base_customers AS (
  SELECT
    TRIM(CAST(user_id AS STRING)) AS user_id
  FROM `tenpo-bi-prod.users.users_audience_tags`
  WHERE fecha_ob_general IS NOT NULL
    AND (has_prepago_account IS TRUE OR has_tc_account IS TRUE)
    AND user_is_locked = FALSE
),

-- Universo REM: última foto con active = TRUE
activos_rem AS (
  SELECT
    TRIM(CAST(id AS STRING)) AS user_id,
    DATE(updated_at)         AS fecha_actividad
  FROM `business-data-raw.ingestor_paid_account_public.users`
  WHERE active = TRUE
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY id
    ORDER BY updated_at DESC, version DESC
  ) = 1
),

cal AS (
  SELECT day AS fecha
  FROM UNNEST(GENERATE_DATE_ARRAY(start_date, DATE_SUB(end_date, INTERVAL 1 DAY))) AS day
),

-- ============================================================
-- Saldo total CPF: filtrado por audiencia
-- ============================================================
base_cap AS (
  SELECT
    DATE_SUB(b.fecha, INTERVAL 1 DAY) AS fecha,
    TRIM(CAST(b.user AS STRING))      AS user_id,
    SAFE_DIVIDE(SAFE_CAST(NULLIF(TRIM(CAST(b.saldo_dia AS STRING)), '') AS NUMERIC), 1000000) AS val
  FROM `tenpo-bi-prod.balance.daily_balance` b
  INNER JOIN base_customers bc
    ON bc.user_id = TRIM(CAST(b.user AS STRING))
  WHERE DATE_SUB(b.fecha, INTERVAL 1 DAY) >= start_date
    AND DATE_SUB(b.fecha, INTERVAL 1 DAY) < end_date
    AND SAFE_CAST(NULLIF(TRIM(CAST(b.saldo_dia AS STRING)), '') AS NUMERIC) > 0
),
hist_cap AS (
  SELECT
    fecha,
    SUM(val) AS saldo_total_mm
  FROM base_cap
  GROUP BY 1
),

-- ============================================================
-- TCA: XS + última Fecha_Proceso disponible dentro de 7 días
-- ============================================================
users_tc_xs AS (
  SELECT
    CAST(individual_id AS STRING) AS id_persona,
    SAFE_CAST(user_id AS STRING)  AS user_id
  FROM `tenpo-bi-prod.kpitos.tipo_users_tc`
  WHERE name_segment = 'XS'
),
base_tca_daily AS (
  SELECT
    DATE(m.Fecha_Proceso) AS fecha_real,
    u.user_id,
    SAFE_DIVIDE(SAFE_CAST(NULLIF(TRIM(CAST(m.Cupo_Total AS STRING)), '') AS NUMERIC), 1000000) AS val
  FROM `operacionestc-operaciones-prod.MAE.Cuentas` m
  INNER JOIN users_tc_xs u USING(id_persona)
  WHERE DATE(m.Fecha_Proceso) >= DATE_SUB(start_date, INTERVAL 7 DAY)
    AND DATE(m.Fecha_Proceso) < end_date
    AND m.Status_id NOT IN ('200','6','14')
    AND SAFE_CAST(NULLIF(TRIM(CAST(m.Cupo_Total AS STRING)), '') AS NUMERIC) > 0
    AND m.id_cuenta NOT IN ('1','2','1733','1734','1735','1736','1737')
),
fechas_tca AS (
  SELECT
    c.fecha AS fecha_kpi,
    MAX(t.fecha_real) AS fecha_real
  FROM cal c
  LEFT JOIN (SELECT DISTINCT fecha_real FROM base_tca_daily) t
    ON t.fecha_real BETWEEN DATE_SUB(c.fecha, INTERVAL 7 DAY) AND c.fecha
  GROUP BY 1
),
hist_tca AS (
  SELECT
    f.fecha_kpi AS fecha,
    SUM(t.val) AS saldo_tca_mm
  FROM fechas_tca f
  LEFT JOIN base_tca_daily t
    ON t.fecha_real = f.fecha_real
  GROUP BY 1
),

-- ============================================================
-- Saldo REM: daily_balance filtrado por audiencia + REM activa
-- ============================================================
base_rem AS (
  SELECT
    DATE_SUB(b.fecha, INTERVAL 1 DAY) AS fecha,
    TRIM(CAST(b.user AS STRING))      AS user_id,
    SAFE_DIVIDE(SAFE_CAST(NULLIF(TRIM(CAST(b.saldo_dia AS STRING)), '') AS NUMERIC), 1000000) AS val
  FROM `tenpo-bi-prod.balance.daily_balance` b
  INNER JOIN base_customers bc
    ON bc.user_id = TRIM(CAST(b.user AS STRING))
  INNER JOIN activos_rem a
    ON a.user_id = TRIM(CAST(b.user AS STRING))
   AND a.fecha_actividad <= LAST_DAY(DATE_SUB(b.fecha, INTERVAL 1 DAY))
  WHERE DATE_SUB(b.fecha, INTERVAL 1 DAY) >= start_date
    AND DATE_SUB(b.fecha, INTERVAL 1 DAY) < end_date
    AND SAFE_CAST(NULLIF(TRIM(CAST(b.saldo_dia AS STRING)), '') AS NUMERIC) > 0
),
hist_rem AS (
  SELECT
    fecha,
    SUM(val) AS saldo_rem_mm
  FROM base_rem
  GROUP BY 1
),

-- ============================================================
-- MAU REM: misma fuente del reporte
-- ============================================================
hist_mau AS (
  SELECT
    fecha,
    COUNT(DISTINCT SAFE_CAST(user AS STRING)) AS mau_rem
  FROM `tenpo-bi-prod.kpitos.cohorts_cuenta_rem`
  WHERE fecha >= start_date
    AND fecha < end_date
  GROUP BY 1
)

SELECT
  c.fecha,
  COALESCE(cap.saldo_total_mm, 0) AS saldo_mm,
  COALESCE(rem.saldo_rem_mm, 0) AS saldo_rem_mm,
  GREATEST(COALESCE(cap.saldo_total_mm, 0) - COALESCE(rem.saldo_rem_mm, 0), 0) AS saldo_norem_mm,
  COALESCE(tca.saldo_tca_mm, 0) AS saldo_tca_mm,
  COALESCE(cap.saldo_total_mm, 0) + COALESCE(tca.saldo_tca_mm, 0) AS total_captaciones,
  COALESCE(mau.mau_rem, 0) AS mau_rem
FROM cal c
LEFT JOIN hist_cap cap USING(fecha)
LEFT JOIN hist_rem rem USING(fecha)
LEFT JOIN hist_tca tca USING(fecha)
LEFT JOIN hist_mau mau USING(fecha)
ORDER BY fecha
"""

df_hist_v2 = read_gbq_robusto(sql_captacion_v2, PROJECT_ID, credentials).drop_duplicates("fecha")
if df_hist_v2.empty:
    raise RuntimeError("La query integrada de captaciones v2 no devolvió filas.")

df_hist_v2["fecha"] = pd.to_datetime(df_hist_v2["fecha"]).dt.normalize()
for _c in ["saldo_mm", "saldo_rem_mm", "saldo_norem_mm", "saldo_tca_mm", "total_captaciones", "mau_rem"]:
    df_hist_v2[_c] = pd.to_numeric(df_hist_v2[_c], errors="coerce").astype(float)

# La query trae calendario completo hasta FORE_END; por eso el corte R/F debe basarse
# sólo en fechas con foto real positiva, no en el calendario rellenado con 0.
cap_real_dates_for_cutoff = pd.to_datetime(
    df_hist_v2.loc[df_hist_v2["saldo_mm"] > 0, "fecha"]
).dt.normalize()

# Mantener los mismos dataframes esperados por el bloque histórico/forecast.
df_cap = df_hist_v2[["fecha", "saldo_mm"]].copy()
df_rem = df_hist_v2[["fecha", "saldo_rem_mm"]].copy()
df_tca_raw = df_hist_v2[["fecha", "saldo_tca_mm"]].copy()
df_mau_rem_raw = df_hist_v2[["fecha", "mau_rem"]].copy()

# Evitar ceros estructurales por faltas de foto: se consideran nulos y se completan hacia adelante
# para variables de stock. La primera observación se completa hacia atrás si agosto parte antes de una foto.
for _df, _col in [(df_cap, "saldo_mm"), (df_rem, "saldo_rem_mm"), (df_tca_raw, "saldo_tca_mm")]:
    _df[_col] = _df[_col].replace(0, np.nan).ffill().bfill()

# ------------------- CORTES R/F -------------------
TODAY_CL = ahora_chile().normalize()
D2_NAT   = (TODAY_CL - pd.Timedelta(days=2)).tz_localize(None).normalize()

cap_dates = set(cap_real_dates_for_cutoff.tolist())
if len(cap_dates):
    real_dates = pd.to_datetime(sorted(list(cap_dates)))
    # Para que el 90d sea comparable con el 31d, el corte del primer mes se limita a Agosto.
    candidates = real_dates[(real_dates <= D2_NAT) & (real_dates <= FIRST_MONTH_END)]
    CUTOFF_R2 = candidates.max() if not candidates.empty else min(real_dates.max(), FIRST_MONTH_END)
else:
    CUTOFF_R2 = min(D2_NAT, FIRST_MONTH_END)

# Si el script se corre antes de que exista foto de agosto, fuerza todo agosto como forecast desde día 1.
if CUTOFF_R2 < FORE_START:
    CUTOFF_R2 = FORE_START - pd.Timedelta(days=1)

# 90d: primer día forecast + 89 días = 90 días de forecast.
FORE_END = CUTOFF_R2 + pd.Timedelta(days=1) + pd.Timedelta(days=89)
FORE_IDX = pd.date_range(FORE_START, FORE_END, freq="D")

print(f"Cutoff R/F: Real hasta {CUTOFF_R2.strftime('%d-%m-%Y')}, Forecast desde {(CUTOFF_R2 + pd.Timedelta(days=1)).strftime('%d-%m-%Y')}")
print(f"Horizonte 90d: hasta {FORE_END.strftime('%d-%m-%Y')} ({len(FORE_IDX)} días en el índice; {(FORE_IDX > CUTOFF_R2).sum()} días forecast)")

# ------------------- TCA (SUAVIZADO) -------------------
tca_hist = df_tca_raw.drop_duplicates("fecha").set_index("fecha")["saldo_tca_mm"].sort_index()
tca_hist = tca_hist.replace(0, np.nan).ffill().bfill()
full_idx_tca = tca_hist.index.union(FORE_IDX).sort_values()
tca_combined = tca_hist.reindex(full_idx_tca).ffill().bfill()
tca_series = tca_combined.reindex(FORE_IDX).ffill().bfill().copy()

tca_deltas = tca_hist.diff().dropna()
pos_deltas = tca_deltas[tca_deltas > 0]
step_tca = float(np.median(pos_deltas.tail(7))) if not pos_deltas.empty else TCA_MIN_STEP_MM
step_tca = np.clip(step_tca * 0.90, TCA_MIN_STEP_MM, TCA_RISE_ABS_MAX)

last_val_tca = float(tca_combined.loc[:CUTOFF_R2].dropna().iloc[-1]) if not tca_combined.loc[:CUTOFF_R2].dropna().empty else float(tca_series.dropna().iloc[0] if not tca_series.dropna().empty else 9537.0)

for d in FORE_IDX:
    if d <= CUTOFF_R2:
        if pd.notna(tca_series.loc[d]):
            last_val_tca = float(tca_series.loc[d])
        else:
            tca_series.loc[d] = last_val_tca
    else:
        last_val_tca += step_tca
        tca_series.loc[d] = last_val_tca
tca_series = tca_series.astype(float)

# ------------------- TOTAL (FORECAST POR PERFIL DOM NORMALIZADO) -------------------
hist_series_raw = df_cap.set_index("fecha")["saldo_mm"].sort_index().replace(0, np.nan)
_full_hist_idx = pd.date_range(hist_series_raw.index.min(), max(FORE_END, CUTOFF_R2), freq="D")
hist_series = hist_series_raw.reindex(_full_hist_idx).ffill().bfill()
_start_candidates = hist_series.loc[:CUTOFF_R2].dropna() if CUTOFF_R2 >= hist_series.index.min() else pd.Series(dtype=float)
start_level = float(_start_candidates.iloc[-1]) if not _start_candidates.empty else float(hist_series.dropna().iloc[-1])

# ---- 1. Construir perfil DOM normalizado con meses de referencia ----
ref_months = [_M3, _M2, _M1]   # V11: 3 meses cerrados previos a FORE_START
dom_profiles = {}
for m_start in ref_months:
    m_end = m_start + pd.offsets.MonthEnd(1)
    idx_m = pd.date_range(m_start, m_end, freq="D")
    s_m   = hist_series.reindex(idx_m).ffill().bfill()
    if s_m.isna().all() or len(s_m) < 20:
        continue
    base_m = s_m.iloc[0]
    if base_m == 0:
        continue
    dom_profiles[m_start] = (s_m / base_m).values

if dom_profiles:
    keys = sorted(dom_profiles.keys())
    weights = np.arange(1, len(keys) + 1, dtype=float)
    max_len = max(len(v) for v in dom_profiles.values())
    profile_matrix = np.full((len(keys), max_len), np.nan)
    for i, k in enumerate(keys):
        v = dom_profiles[k]
        profile_matrix[i, :len(v)] = v
    dom_ratio_avg = np.average(
        np.where(np.isnan(profile_matrix),
                 np.nanmean(profile_matrix, axis=0),
                 profile_matrix),
        weights=weights, axis=0
    )
else:
    dom_ratio_avg = np.ones(_DIAS_MES)

# Suavizar el perfil DOM para eliminar escalones
dom_ratio_avg = pd.Series(dom_ratio_avg).rolling(3, center=True, min_periods=1).mean().values

# ---- 2. Referencias históricas de cierre ----
# V11: "jul/jun/may" = mes-1 / mes-2 / mes-3 respecto de FORE_START (antes fijos)
def _serie_mes(_m):
    _fin = _m + MonthEnd(0)
    return hist_series.loc[_m:_fin] if hist_series.index.min() <= _fin else pd.Series(dtype=float)
jul_series = _serie_mes(_M1)
jun_series = _serie_mes(_M2)
may_series = _serie_mes(_M3)

close_jul = float(jul_series.dropna().iloc[-1]) if not jul_series.dropna().empty else start_level
close_jun = float(jun_series.dropna().iloc[-1]) if not jun_series.dropna().empty else start_level
close_may = float(may_series.dropna().iloc[-1]) if not may_series.dropna().empty else close_jun

# Tendencia de cierre: promedio de los últimos 2 deltas (May→Jun y Jun→Jul)
delta_jun = close_jun - close_may
delta_jul = close_jul - close_jun
tendencia_cierre = (delta_jun + delta_jul) / 2.0

# Cierre Agosto: tendencia conservadora al 50%
close_agosto_target = close_jul + tendencia_cierre * 0.50

# ---- 3. Anclar implied_day1 ----
cutoff_day   = CUTOFF_R2.day
cutoff_level = start_level

ratio_at_cutoff = dom_ratio_avg[min(cutoff_day - 1, len(dom_ratio_avg) - 1)]
implied_day1_from_cutoff   = cutoff_level / ratio_at_cutoff if ratio_at_cutoff != 0 else cutoff_level
implied_day1_from_close_jul = close_jul

implied_day1_mid = max(implied_day1_from_cutoff, implied_day1_from_close_jul) * 0.980

profile_len = len(dom_ratio_avg)
close_ratio = dom_ratio_avg[min(_DIAS_MES - 1, profile_len - 1)]  # V11: ultimo dia del mes objetivo
implied_day1_close = close_agosto_target / close_ratio if close_ratio > 0 else close_agosto_target

TECHO_ABS = close_agosto_target * 1.003
PISO_ABS  = close_agosto_target * 0.900

# ---- 4. Construir serie forecast ----
# Agosto se proyecta con la misma fórmula del script 31d.
# Los meses siguientes reciclan la curva completa de Agosto + shift mensual conservador.
total_base = pd.Series(index=FORE_IDX, dtype=float)

# 4.1 Primer mes: Agosto 2026. Este tramo debe calzar con el script 31d.
for d in FORE_IDX:
    if d > FIRST_MONTH_END:
        continue

    if d <= CUTOFF_R2:
        if d in hist_series.index:
            total_base.loc[d] = float(hist_series.loc[d])
        else:
            total_base.loc[d] = cutoff_level
        continue

    day_idx = min(d.day - 1, profile_len - 1)
    ratio = dom_ratio_avg[day_idx]

    # Blend gradual entre mid y close: transición suave días 24→31
    if d.day <= 23:
        blend = 0.0  # 100% mid
    elif d.day >= _DIAS_MES:
        blend = 1.0  # 100% close
    else:
        blend = (d.day - 23) / float(_DIAS_MES - 23)  # V11: gradual 0→1 hasta el ultimo dia del mes

    implied_blended = implied_day1_mid * (1 - blend) + implied_day1_close * blend
    projected = implied_blended * ratio

    projected = min(projected, TECHO_ABS)
    projected = max(projected, PISO_ABS)
    total_base.loc[d] = projected

# Curva completa de Agosto para reutilizarla en meses posteriores.
ago_full = total_base.loc[(total_base.index >= FORE_START) & (total_base.index <= FIRST_MONTH_END)].copy()
ago_full = ago_full.ffill().bfill()

# 4.2 Meses siguientes del horizonte 90d.
# Usar la misma forma diaria de Agosto para que no cambie el primer mes,
# agregando un shift mensual conservador en base a la tendencia histórica.
monthly_shift = tendencia_cierre * 0.10
base_month_ord = FORE_START.to_period("M").ordinal

for d in FORE_IDX:
    if d <= FIRST_MONTH_END:
        continue

    month_offset = d.to_period("M").ordinal - base_month_ord
    equiv_day = min(d.day, FIRST_MONTH_END.day)
    ago_equiv_date = FORE_START + pd.Timedelta(days=equiv_day - 1)

    base_val = float(ago_full.loc[ago_equiv_date]) if ago_equiv_date in ago_full.index else float(ago_full.iloc[-1])
    total_base.loc[d] = base_val + (monthly_shift * month_offset)

total_base = total_base.ffill().bfill()

# ------------------- NOREM / REM -------------------
hist_idx = pd.date_range(df_cap["fecha"].min(), max(CUTOFF_R2, df_cap["fecha"].min()), freq="D")
tot_h = df_cap.set_index("fecha")["saldo_mm"].replace(0, np.nan).reindex(hist_idx).ffill().bfill()
rem_h = df_rem.set_index("fecha")["saldo_rem_mm"].replace(0, np.nan).reindex(hist_idx).ffill().bfill()
norem_h = (tot_h - rem_h).clip(lower=0.0)

share_h = (norem_h / tot_h.replace(0, np.nan)).fillna(0.15)
recent_share = share_h.iloc[-180:].groupby(share_h.iloc[-180:].index.day).mean()

dom_profile = pd.Series(index=range(1, 32), dtype=float)
dom_profile.loc[recent_share.index] = recent_share.values
dom_profile = dom_profile.ffill().bfill().rolling(3, center=True, min_periods=1).mean()

norem_all = pd.Series(index=FORE_IDX, dtype=float)
rem_all   = pd.Series(index=FORE_IDX, dtype=float)

for d in FORE_IDX:
    if d <= CUTOFF_R2:
        continue
    sh  = float(dom_profile.loc[d.day])
    tot = float(total_base.loc[d])
    cand_norem = np.clip(tot * sh, NOREM_FLOOR_MM, NOREM_SOFT_CEIL_MM)
    norem_all.loc[d] = cand_norem
    rem_all.loc[d]   = max(0.0, tot - cand_norem)

# ------------------- MAU REM (XGBOOST) -------------------
# Viene desde la query integrada v2 para mantener el mismo criterio base.
df_mau_rem_raw = df_hist_v2[["fecha", "mau_rem"]].copy()

mau_rem_series = pd.Series(index=FORE_IDX, dtype=float)

start_mau         = 175000.0
model_xgb_mau     = None
pendiente_mau     = 1000.0
X_train_mau       = pd.DataFrame()
df_mau_clean      = pd.DataFrame()
hist_max_rise_mau = 3000.0
hist_max_drop_mau = -1000.0

if not df_mau_rem_raw.empty:
    df_mau_rem_raw["fecha"] = pd.to_datetime(df_mau_rem_raw["fecha"]).dt.normalize()
    df_mau_clean = df_mau_rem_raw.drop_duplicates("fecha").set_index("fecha").sort_index()
    df_mau_clean["mau_rem"] = pd.to_numeric(df_mau_clean["mau_rem"], errors="coerce").astype(float)
    df_mau_clean = df_mau_clean[df_mau_clean["mau_rem"] > 100000]

    if not df_mau_clean.empty:
        full_idx_mau = pd.date_range(df_mau_clean.index.min(), df_mau_clean.index.max(), freq="D")
        df_mau_clean = df_mau_clean.reindex(full_idx_mau).interpolate(method="linear")
        df_mau_clean.index.name = "fecha"

        fecha_max_mau = df_mau_clean.index.max()
        df_mau_train = df_mau_clean[
            df_mau_clean.index >= (fecha_max_mau - pd.Timedelta(days=180))
        ].copy()

        if not df_mau_train.empty:
            df_mau_train["delta_mau"]  = df_mau_train["mau_rem"].diff()
            df_mau_train["dia_mes"]    = df_mau_train.index.day
            df_mau_train["dia_semana"] = df_mau_train.index.dayofweek
            df_mau_train["es_finde"]   = df_mau_train["dia_semana"].isin([5, 6]).astype(int)
            df_mau_train = df_mau_train.dropna()

            X_train_mau = df_mau_train[["dia_mes", "dia_semana", "es_finde"]]
            y_train_mau = df_mau_train["delta_mau"]

            if not X_train_mau.empty:
                model_xgb_mau = XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=4, random_state=42)
                model_xgb_mau.fit(X_train_mau, y_train_mau)
                hist_max_rise_mau = float(y_train_mau.max() * 1.5)
                hist_max_drop_mau = float(y_train_mau.min() * 1.5)

            mau_smooth = df_mau_clean["mau_rem"].rolling(14, min_periods=1).mean()
            dias_t = min(180, len(mau_smooth) - 1)
            pendiente_mau = ((mau_smooth.iloc[-1] - mau_smooth.iloc[-dias_t]) / dias_t if dias_t > 0 else 1000.0)
            if pendiente_mau < 500.0:
                pendiente_mau = 1000.0

            start_mau = float(df_mau_clean["mau_rem"].iloc[-1])

curr_mau = start_mau
for d in FORE_IDX:
    if model_xgb_mau is not None and not X_train_mau.empty:
        X_pred_mau = pd.DataFrame({"dia_mes": [d.day], "dia_semana": [d.dayofweek], "es_finde": [1 if d.dayofweek in [5, 6] else 0]})
        pred_delta_mau = float(model_xgb_mau.predict(X_pred_mau)[0])
        pred_delta_mau = np.clip(pred_delta_mau, hist_max_drop_mau, hist_max_rise_mau)
    else:
        pred_delta_mau = pendiente_mau

    if d <= CUTOFF_R2 and not df_mau_clean.empty and d in df_mau_clean.index:
        curr_mau = float(df_mau_clean.loc[d, "mau_rem"])
    else:
        curr_mau += pred_delta_mau

    mau_rem_series.loc[d] = curr_mau

# ------------------- ARMADO DEL OUTPUT -------------------
out = pd.DataFrame(index=FORE_IDX)
out.index.name = "fecha"
mask_real = out.index <= CUTOFF_R2
mask_fore = ~mask_real

real_cap = df_cap.set_index("fecha")["saldo_mm"].replace(0, np.nan).reindex(FORE_IDX).ffill().bfill()
real_rem = df_rem.set_index("fecha")["saldo_rem_mm"].replace(0, np.nan).reindex(FORE_IDX).ffill().bfill()

out["saldo_total_mm"] = np.where(mask_real, real_cap, total_base.reindex(FORE_IDX).ffill().bfill())
out["saldo_rem_mm"]   = np.where(mask_real, real_rem, rem_all.reindex(FORE_IDX).ffill().bfill())
out["saldo_total_mm"] = pd.to_numeric(out["saldo_total_mm"], errors="coerce").replace(0, np.nan).bfill().ffill()
out["saldo_rem_mm"]   = pd.to_numeric(out["saldo_rem_mm"], errors="coerce").replace(0, np.nan).bfill().ffill()

out["saldo_norem_mm"] = (out["saldo_total_mm"] - out["saldo_rem_mm"]).clip(lower=0.0)
out["saldo_tca_mm"]   = tca_series
out["mau_rem"]         = mau_rem_series

out["saldo_total_real_mm"]         = np.where(mask_real, out["saldo_total_mm"], 0.0)
out["saldo_total_pronosticado_mm"] = np.where(mask_fore, out["saldo_total_mm"], 0.0)
out["saldo_rem_real_mm"]           = np.where(mask_real, out["saldo_rem_mm"], 0.0)
out["saldo_rem_pronosticado_mm"]   = np.where(mask_fore, out["saldo_rem_mm"], 0.0)
out["saldo_norem_real_mm"]         = np.where(mask_real, out["saldo_norem_mm"], 0.0)
out["saldo_norem_pronosticado_mm"] = np.where(mask_fore, out["saldo_norem_mm"], 0.0)
out["saldo_tca_real_mm"]           = np.where(mask_real, out["saldo_tca_mm"], 0.0)
out["saldo_tca_pronosticado_mm"]   = np.where(mask_fore, out["saldo_tca_mm"], 0.0)
out["mau_rem_real"]                = np.where(mask_real, out["mau_rem"], 0.0)
out["mau_rem_pronosticado"]        = np.where(mask_fore, out["mau_rem"], 0.0)

out["estado_total"]   = np.where(mask_real, "R", "F")
out["estado_rem"]     = np.where(mask_real, "R", "F")
out["estado_norem"]   = np.where(mask_real, "R", "F")
out["estado_tca"]     = np.where(mask_real, "R", "F")
out["estado_mau_rem"] = np.where(mask_real, "R", "F")

out["total_captaciones"] = out["saldo_total_mm"] + out["saldo_tca_mm"]

# Diagnóstico anti-cero: estas columnas son stocks; en forecast no deberían quedar en cero
# salvo que realmente no exista ninguna base histórica utilizable.
if mask_fore.any():
    _forecast_check = out.loc[mask_fore, ["saldo_total_mm", "saldo_rem_mm", "saldo_norem_mm", "saldo_tca_mm", "total_captaciones", "mau_rem"]]
    _zero_cols = [c for c in _forecast_check.columns if float(pd.to_numeric(_forecast_check[c], errors="coerce").fillna(0).abs().sum()) == 0.0]
    if _zero_cols:
        print(f"[WARN] Columnas forecast en cero: {_zero_cols}. Revisar fuente o filtros de query v2.")

# --- FIN BLOQUE COMPARTIDO ---


# ------------------- FORMATEO FINAL -------------------
dt_fechas = out.index.copy()
out = out.reset_index()
out["numfecha"] = out["fecha"].dt.day
out["fecha"]    = out["fecha"].dt.strftime("%d-%m-%Y")

cols_float = [
    c for c in out.columns
    if ("mm" in c or "mau" in c or "total" in c) and not c.startswith("estado")
]
for c in cols_float:
    out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0).astype(float)

# ------------------- PREVIEW CONSOLA -------------------
print("\n" + "=" * 95)
print(f" PREVIEW: PROYECCIÓN CAPTACIONES 90d DESDE {_MES_TXT.upper()} ".center(95, "="))
print(f" Horizonte: {FORE_START.strftime('%d-%m-%Y')} → {FORE_END.strftime('%d-%m-%Y')} ".center(95, "="))
print("=" * 95)
preview_cols = [
    "fecha", "estado_total", "total_captaciones",
    "saldo_total_mm", "saldo_tca_mm", "saldo_rem_mm",
    "saldo_norem_mm", "mau_rem"
]
print(out[preview_cols].to_string(index=False, float_format=lambda x: f"{x:,.1f}"))
print("=" * 95 + "\n")

# Chequeo operativo: los días de Agosto son el tramo compartido con el script 31d.
first_month_rows = out[pd.to_datetime(out["fecha"], format="%d-%m-%Y").dt.to_period("M") == FORE_START.to_period("M")]
print(f"[CHECK] Filas del primer mes compartido con 31d: {len(first_month_rows)} días ({FORE_START.strftime('%Y-%m')}).")

# ==============================================================================
# >>> GRÁFICO 90d: Histórico reciente + Forecast rolling
# ==============================================================================
print("Generando Gráfico Rolling 90d...")

plt.figure(figsize=(18, 8))

# --- 1. Construir serie histórica desde Mayo hasta CUTOFF_R2 ---
hist_chart_start = _M3
hist_chart_idx   = pd.date_range(hist_chart_start, CUTOFF_R2, freq="D")

s_cap_hist = df_cap.set_index("fecha")["saldo_mm"].replace(0, np.nan).reindex(hist_chart_idx).ffill().bfill()
s_tca_hist = df_tca_raw.set_index("fecha")["saldo_tca_mm"].replace(0, np.nan).reindex(hist_chart_idx).ffill().bfill()
total_hist = (s_cap_hist + s_tca_hist).values

# Plot histórico real
plt.plot(hist_chart_idx, total_hist,
         color="#1f77b4", linewidth=2.5, linestyle="-", marker="o", markersize=2,
         label="Real (R)")

# --- 2. Forecast desde el output (solo días F) ---
mask_f = out["estado_total"] == "F"
dias_f = pd.to_datetime(out.loc[mask_f, "fecha"], format="%d-%m-%Y")

# Conector Real → Forecast
if mask_f.any():
    last_real_date = hist_chart_idx[-1]
    last_real_val  = total_hist[-1]
    first_f_date   = dias_f.iloc[0]
    first_f_val    = out.loc[mask_f, "total_captaciones"].iloc[0]

    plt.plot([last_real_date, first_f_date], [last_real_val, first_f_val],
             color="#ff7f0e", linewidth=3.0, linestyle="--", alpha=0.7)

    plt.plot(dias_f, out.loc[mask_f, "total_captaciones"],
             color="#ff7f0e", linewidth=3.0, linestyle="--", marker="o", markersize=2,
             label="Forecast (F)")

# --- 3. Línea de corte R/F ---
plt.axvline(x=CUTOFF_R2, color="red", linestyle=":", linewidth=1.5,
            label=f"Corte Real ({CUTOFF_R2.strftime('%d-%m-%Y')})")

# --- 4. Sombreado alterno de meses ---
ax = plt.gca()
all_dates_chart = pd.date_range(hist_chart_start, FORE_END, freq="D")
months_chart = all_dates_chart.to_period("M").unique()
for i, m in enumerate(months_chart):
    if i % 2 == 0:
        m_s = m.to_timestamp()
        m_e = (m + 1).to_timestamp()
        ax.axvspan(m_s, m_e, alpha=0.06, color="gray")

plt.title(f"Captaciones Totales — Rolling 90d desde {_MES_TXT}\n(Real desde {_M3:%b %Y} hasta {CUTOFF_R2.strftime('%d-%m-%Y')} | Forecast hasta {FORE_END.strftime('%d-%m-%Y')})",
          fontsize=16, fontweight="bold", pad=15)
plt.xlabel("Fecha", fontsize=12)
plt.ylabel("Total Captaciones (MM CLP)", fontsize=12)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: f"{int(x):,}"))
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend(loc="upper left", fontsize=11)
plt.tight_layout()
plt.savefig(f"captaciones_90d_{FORE_START:%Y_%m}_v2.png", bbox_inches="tight", dpi=150)
print(f"--> Gráfico guardado como 'captaciones_90d_{FORE_START:%Y_%m}_v2.png'\n")

# ==============================================================================
# >>> SECCIÓN EXCLUSIVA 90d: ESCRITURA BQ (tabla 90d — WRITE_TRUNCATE)
# ==============================================================================
print(f"Escribiendo tabla 90d: {TABLE_MAIN_90D} ...")
n = write_bq_safe(out, TABLE_MAIN_90D, PROJECT_ID, credentials, "WRITE_TRUNCATE")
print(f"OK. {TABLE_MAIN_90D} actualizada ({n} filas).")
print(f"Cutoff: {CUTOFF_R2.strftime('%d-%m-%Y')} | Horizonte hasta: {FORE_END.strftime('%d-%m-%Y')}")
print(f"¡Actualización Exitosa 90d {_MES_TXT} con lógica v2 de captaciones!")

In [ ]:
# ===================== LE CLIENTES PRINCIPALES — (R/F + LONG + modelo) — AGOSTO 2026 =====================
# - Target: Agosto 2026 (31 días)
# - Base: 12m train, recency weights
# - Salida en UNA MISMA TABLA con productos y métricas
# ====================================================================================================
#
# Versión v7 adaptada a AGOSTO 2026
# -----------------------------------------------
# Ajustes para agosto:
#   - TARGET_MONTH: 2026-08-01 (31 días)
#   - Calendario: Día 1=Sáb, Día 31=Lun (último día hábil = spike máximo)
#   - EOM: 26=Mié, 27=Jue, 28=Vie, 29=Sáb, 30=Dom, 31=Lun
#     Spike de cierre cae en el último día hábil (lunes 31), que además es el último día del mes = ideal
#     Sábado 29 y Domingo 30 quedan como valle de fin de semana antes del remate del lunes
#   - Gráfico mejorado: curva real+forecast continua, promedio 3m, y 3 meses individuales
# ====================================================================================================

import pandas as pd
import numpy as np
import pandas_gbq
import pydata_google_auth
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from pandas.tseries.offsets import MonthEnd
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from google.cloud import bigquery
from google.cloud.exceptions import NotFound

# ----------------------- Configuración -----------------------
PROJECT_ID  = "tenpo-bi-prod"
BQ_DEFAULT_LOCATION = "southamerica-west1"

START_DATE_SOW = pd.Timestamp("2025-05-01")
# V11: mes objetivo AUTOMATICO (antes hardcodeado). Hora de Chile para no saltar de mes
# antes de tiempo en Colab (UTC).
_HOY_CL  = ahora_chile().tz_localize(None).normalize()
_MESES_ES = ["Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio", "Julio", "Agosto",
             "Septiembre", "Octubre", "Noviembre", "Diciembre"]
TARGET_MONTH   = _HOY_CL.replace(day=1)   # V11: mes en curso automatico
_MES_TXT       = f"{_MESES_ES[TARGET_MONTH.month - 1]} {TARGET_MONTH.year}"
print(f"[V11 PRINCIPALES] mes objetivo: {_MES_TXT}")
END_DATE_SOW   = (TARGET_MONTH + MonthEnd(0) + pd.Timedelta(days=1))

DEST_TABLE      = "kpitos.forecasting_le_clientes_principales"

TRAIN_MONTHS_BACK   = 12
PROFILE_MONTHS_BACK = 12

MIN_TRAIN_ROWS = 8
EVAL_LAST_K    = 6
SEED           = 42

RECENCY_HALFLIFE_M          = 4
MOMENTUM_LOOKBACK_DAYS      = 14
MOMENTUM_PRIOR_ALPHA        = 0.35
MOMENTUM_PRIOR_CAP_UP       = 0.35   # antes 0.50 — muy asimétrico frente al -0.15 a la baja
MOMENTUM_PRIOR_CAP_DOWN     = 0.15
MOMENTUM_WEEKDAYS_ONLY      = True

# --------- PARÁMETROS CLAVE DEL NIVEL DIARIO (FORMA) ---------
BASE_RANGE_MIN       = 3000.0
BASE_RANGE_MAX       = 3500.0

CLIP_MIN             = 2500.0
CLIP_MAX             = 5500.0

DOW_SHAPE_AMPL             = 0.08   # bajado de 0.12 — la versión anterior generaba saltos como el del día 21
DOW_FLOOR_FACTOR           = 0.93   # bajado de 0.90 — un poco menos de caída permitida

INTRA_MONTH_TREND_STRENGTH = 0.01
JITTER_STRENGTH            = 0.02   # bajado de 0.04 — menos ruido sintético, que se apoye más en el patrón real

ENABLE_EOM_SPIKE    = True
EOM_SPIKE_START_DAY = 26

# V11: multiplicadores de cierre derivados del CALENDARIO del mes (antes: tabla fija de
# agosto). Regla = la de agosto: ultimo dia habil 1.35; los 3 habiles previos 1.18/1.10/1.05
# (hacia atras); sabado 0.90, domingo 0.90/0.95. Reproduce exactamente la tabla de agosto.
def _eom_multiplicadores(mes, desde=26):
    fin = (mes + MonthEnd(0)).day
    dias = list(range(desde, fin + 1))
    wd = {d: mes.replace(day=d).weekday() for d in dias}
    habiles = [d for d in dias if wd[d] < 5]
    ultimo = max(habiles) if habiles else fin
    out = {}
    for j, d in enumerate(sorted([h for h in habiles if h < ultimo], reverse=True)):
        out[d] = [1.18, 1.10, 1.05][j] if j < 3 else 1.05
    for d in dias:
        if d == ultimo:
            out[d] = 1.35
        elif wd[d] == 5:
            out[d] = 0.90
        elif wd[d] == 6:
            out[d] = 0.95
    return out, ultimo

EOM_SPIKE_MULTIPLIERS, _ULTIMO_HABIL = _eom_multiplicadores(TARGET_MONTH, 26)
print(f"[V11 PRINCIPALES] EOM multiplicadores: {EOM_SPIKE_MULTIPLIERS} | peak dia {_ULTIMO_HABIL}")

# Techo de seguridad para el ratio histórico EOM/mid-month (ver ENABLE_HIST_EOM_CALIBRATION).
# Evita que un mes atípico en la ventana de calibración (ej. un solo día con pico fuerte)
# infle el multiplicador de cierre más allá de lo razonable.
HIST_EOM_RATIO_CAP = 1.40

# Peso de la calibración histórica al combinarse con el multiplicador manual de la tabla.
# 0 = usar solo la tabla manual; 1 = usar solo el ratio histórico calibrado.
# Antes se usaba max(base_mult, calibrated_mult), lo que garantizaba que el número más
# grande de los dos siempre ganaba y podía superar el 1.60 manual sin techo.
HIST_EOM_BLEND_WEIGHT = 0.35

ENFORCE_EOM_MONOTONIC = False
DISABLE_JITTER_EOM = True

ENABLE_WARM_START          = True
WARM_START_DAYS            = 3
WARM_START_ALPHA           = 0.80
WARM_START_LOOKBACK_MONTHS = 4

ENABLE_HIST_EOM_CALIBRATION    = True
HIST_EOM_CALIBRATION_MONTHS    = 4
HIST_EOM_MID_DAYS              = (10, 20)
HIST_EOM_SPIKE_DAYS            = (26, (TARGET_MONTH + MonthEnd(0)).day)   # V11

ENABLE_FIRST_WEEK_ANCHOR       = True
FIRST_WEEK_ANCHOR_DAYS         = 8
FIRST_WEEK_ANCHOR_ALPHA        = 0.55
FIRST_WEEK_ANCHOR_LOOKBACK_M   = 4

DOW_FLOOR_LOOKBACK_WEEKS = 8
DOW_FLOOR_MIN_SAMPLES    = 3

FALLBACK_RECENT_MEAN_DAYS      = 14
FALLBACK_USE_RECENT_DAILY_MEAN = True
FALLBACK_MIN_DAILY             = 1.0
FALLBACK_RECENT_WINDOW_DAYS    = 7

ALPHA_META_MONTH       = 0.00
META_DAILY_FLOOR_ALPHA = 0.00

BOUNDARY_SMOOTH_DAYS     = 7
BOUNDARY_REF_RECENT_DAYS = 7
BOUNDARY_CAP_UP          = 1.38
BOUNDARY_CAP_DOWN        = 0.70

ENABLE_BLACK_FRIDAY    = False

ENABLE_SOFT_TOTAL_ANCHOR = True
SOFT_TOTAL_ALPHA         = 0.45   # antes 0.65 — pull más suave hacia el total del modelo mensual
SOFT_TOTAL_CAP_UP        = 1.20   # antes 1.40 — permitía inflar TODA la curva pendiente hasta un 40% de un solo golpe
SOFT_TOTAL_CAP_DOWN      = 0.75

RAMP_DAYS_FROM_LAST_REAL = 2

STANDARD_COLS = [
    "fecha", "dia", "producto", "valor_real_mm",
    "valor_pronosticado_mm", "valor_mm", "estado", "modelo",
]

# ----------------------- Auth & helpers -----------------------
scopes = [
    "https://www.googleapis.com/auth/bigquery",
    "https://www.googleapis.com/auth/cloud-platform",
]
credentials = pydata_google_auth.get_user_credentials(scopes)

def read_gbq_robusto(sql, project_id, credentials, fallback_location=BQ_DEFAULT_LOCATION):
    return pandas_gbq.read_gbq(sql, project_id=project_id, credentials=credentials)

def ensure_dataset_location(project_id, fq_table, credentials, default_location=BQ_DEFAULT_LOCATION):
    client = bigquery.Client(project=project_id, credentials=credentials)
    if "." not in fq_table: raise RuntimeError("DEST_TABLE debe ser 'dataset.tabla'.")
    dataset_id = fq_table.split(".")[0]
    ref = f"{project_id}.{dataset_id}"
    try:
        ds = client.get_dataset(ref)
        return ds.location or default_location
    except NotFound:
        ds = bigquery.Dataset(ref)
        ds.location = default_location
        client.create_dataset(ds, exists_ok=True)
        return default_location

def mape(a, f):
    a = np.array(a, dtype=float); f = np.array(f, dtype=float)
    m = (a != 0) & np.isfinite(a) & np.isfinite(f)
    return (np.abs(a[m]-f[m]) / np.abs(a[m])).mean()*100 if m.sum()>0 else np.inf

def recency_weights(n, half_life):
    if n <= 0: return np.array([])
    idx = np.arange(n, dtype=float); age = (n - 1) - idx
    w = 0.5 ** (age / max(half_life, 1e-9))
    return w / (w.sum() + 1e-12)

# ----------------------- Query SoW → Principalidad diaria alineada a query MTD -----------------------
# Concepto alineado a la query de referencia:
#   1) Principal = usuario con GPV App MTD / renta_estimada >= 0.5.
#   2) Se exige que el usuario exista en habitualidad_tenpo moda del mes, igual que la query final.
#   3) El denominador de principalidad es MAU App MTD con las mismas líneas y filtros base.
#   4) La serie diaria conserva el enfoque de forecast: cuenta el primer día en que cada usuario cruza SOW 50%.
sql_sow = f"""
DECLARE start_date DATE DEFAULT DATE('{START_DATE_SOW:%Y-%m-%d}');
DECLARE end_date   DATE DEFAULT DATE('{END_DATE_SOW:%Y-%m-%d}');

WITH base_economics AS (
  SELECT
    DATE(fecha) AS d,
    DATE_TRUNC(DATE(fecha), MONTH) AS mes,
    SAFE_CAST(user AS STRING) AS user_id,
    trx_id,
    monto,
    linea
  FROM `tenpo-bi-prod.economics.economics`
  WHERE DATE(fecha) >= start_date AND DATE(fecha) < end_date
    AND LOWER(IFNULL(nombre,"null")) NOT LIKE "%devol%"
    AND LOWER(IFNULL(actividad_nac_cd,"null")) NOT LIKE "%refund%"
),

gpv_diario AS (
  SELECT d, mes, user_id, SUM(monto) AS gpv_app_d
  FROM base_economics
  WHERE linea IN (SELECT linea FROM `tenpo-bi-prod.kpitos.lineas_gpv_app`)
  GROUP BY 1,2,3
),

renta_uniq AS (
  SELECT SAFE_CAST(user AS STRING) AS user_id, MAX(renta_estimada) AS renta_estimada
  FROM `tenpo-bi-prod.analytics_audience.user_renta_estimada`
  WHERE renta_estimada > 0
  GROUP BY 1
),

habitualidad_moda_mes AS (
  SELECT
    DATE_TRUNC(DATE(fecha_ejecucion), MONTH) AS mes,
    SAFE_CAST(user AS STRING) AS user_id,
    habitualidad,
    COUNT(DISTINCT DATE(fecha_ejecucion)) AS frequency
  FROM `tenpo-bi-prod.analytics_audience.habitualidad_tenpo`
  WHERE DATE(fecha_ejecucion) >= start_date
    AND DATE(fecha_ejecucion) < end_date
  GROUP BY 1,2,3
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY user_id, mes
    ORDER BY
      frequency DESC,
      CASE habitualidad
        WHEN 'engage'   THEN 1
        WHEN 'habit'    THEN 2
        WHEN 'aha'      THEN 3
        WHEN 'distract' THEN 4
        WHEN 'setup'    THEN 5
        ELSE 99
      END ASC
  ) = 1
),

cal AS (
  SELECT day AS d FROM UNNEST(GENERATE_DATE_ARRAY(start_date, DATE_SUB(end_date, INTERVAL 1 DAY))) AS day
),

usuarios_gpv AS (SELECT DISTINCT user_id FROM gpv_diario),

grid AS (
  SELECT u.user_id, c.d, DATE_TRUNC(c.d, MONTH) AS mes
  FROM usuarios_gpv u CROSS JOIN cal c
),

gpv_fill AS (
  SELECT g.user_id, g.d, g.mes, IFNULL(d.gpv_app_d, 0) AS gpv_app_d
  FROM grid g
  LEFT JOIN gpv_diario d
    ON d.user_id = g.user_id AND d.d = g.d
),

gpv_mtd AS (
  SELECT
    user_id,
    d AS fecha,
    mes,
    SUM(gpv_app_d) OVER (
      PARTITION BY user_id, mes
      ORDER BY d
      ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS gpv_app_mtd
  FROM gpv_fill
),

sow_user_d AS (
  SELECT
    m.user_id,
    m.fecha,
    m.mes,
    r.renta_estimada,
    SAFE_DIVIDE(m.gpv_app_mtd, r.renta_estimada) AS sow_mtd
  FROM gpv_mtd m
  LEFT JOIN renta_uniq r USING (user_id)
),

first_cross_month AS (
  SELECT
    s.user_id,
    s.mes,
    MIN(s.fecha) AS activation_date_month
  FROM sow_user_d s
  INNER JOIN habitualidad_moda_mes h
    ON h.user_id = s.user_id
   AND h.mes     = s.mes
  WHERE s.sow_mtd >= 0.5
  GROUP BY 1,2
),

first_cross_month_cnt AS (
  SELECT
    activation_date_month AS fecha,
    mes,
    COUNT(DISTINCT user_id) AS principals_activated_today_month
  FROM first_cross_month
  GROUP BY 1,2
),

principal_stock_mtd_daily AS (
  SELECT
    c.d AS fecha,
    DATE_TRUNC(c.d, MONTH) AS mes,
    COUNT(DISTINCT f.user_id) AS principal_stock_mtd_query
  FROM cal c
  LEFT JOIN first_cross_month f
    ON f.mes = DATE_TRUNC(c.d, MONTH)
   AND f.activation_date_month <= c.d
  GROUP BY 1,2
),

mau_users_daily AS (
  SELECT DISTINCT d, mes, user_id
  FROM base_economics
  WHERE linea IN (SELECT * FROM `tenpo-bi-prod.kpitos.lineas_mau_app`)
),

first_mau_month AS (
  SELECT user_id, mes, MIN(d) AS first_mau_date
  FROM mau_users_daily
  GROUP BY 1,2
),

mau_app_mtd_daily AS (
  SELECT
    c.d AS fecha,
    DATE_TRUNC(c.d, MONTH) AS mes,
    COUNT(DISTINCT f.user_id) AS mau_app_mtd_query,
    COUNT(DISTINCT IF(r.user_id IS NOT NULL, f.user_id, NULL)) AS mau_con_renta_mtd
  FROM cal c
  LEFT JOIN first_mau_month f
    ON f.mes = DATE_TRUNC(c.d, MONTH)
   AND f.first_mau_date <= c.d
  LEFT JOIN renta_uniq r
    ON r.user_id = f.user_id
  GROUP BY 1,2
)

SELECT
  c.d AS fecha,
  DATE_TRUNC(c.d, MONTH) AS mes,
  IFNULL(fcm.principals_activated_today_month, 0) AS principals_activated_today_month,
  IFNULL(ps.principal_stock_mtd_query, 0) AS principal_stock_mtd_query,
  IFNULL(mau.mau_app_mtd_query, 0) AS mau_app_mtd_query,
  IFNULL(mau.mau_con_renta_mtd, 0) AS mau_con_renta
FROM cal c
LEFT JOIN first_cross_month_cnt fcm
  ON fcm.fecha = c.d
 AND fcm.mes   = DATE_TRUNC(c.d, MONTH)
LEFT JOIN principal_stock_mtd_daily ps
  ON ps.fecha = c.d
 AND ps.mes   = DATE_TRUNC(c.d, MONTH)
LEFT JOIN mau_app_mtd_daily mau
  ON mau.fecha = c.d
 AND mau.mes   = DATE_TRUNC(c.d, MONTH)
ORDER BY fecha
"""

df = read_gbq_robusto(sql_sow, PROJECT_ID, credentials)
if df.empty: raise RuntimeError("La query de SoW/Principalidad no devolvió filas.")

df["fecha"] = pd.to_datetime(df["fecha"]).dt.normalize()
df["mes"]   = pd.to_datetime(df["mes"]).dt.to_period("M").dt.to_timestamp()
for col in ["principals_activated_today_month", "principal_stock_mtd_query", "mau_app_mtd_query", "mau_con_renta"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)

# ----------------------- MAU App MTD alineado a query de referencia -----------------------
df_target = df[df["mes"] == TARGET_MONTH].copy()

max_app_mtd = df_target["mau_app_mtd_query"].max() if not df_target.empty else pd.NA
mau_app_mes = float(max_app_mtd) if pd.notna(max_app_mtd) else np.nan

max_renta = df_target["mau_con_renta"].max() if not df_target.empty else pd.NA
mau_con_renta_mes = float(max_renta) if pd.notna(max_renta) else np.nan

# ----------------------- Modelado mensual -----------------------
monthly = df.groupby("mes", as_index=False)["principals_activated_today_month"].sum().rename(columns={"principals_activated_today_month": "valor_mensual"}).sort_values("mes").reset_index(drop=True)
m_train_full = monthly[monthly["mes"] < TARGET_MONTH].copy()
train_start  = (TARGET_MONTH + MonthEnd(-TRAIN_MONTHS_BACK)).to_period("M").to_timestamp()
m_train      = m_train_full[m_train_full["mes"] >= train_start].copy()

def month_features(dfm, value_col="valor_mensual"):
    f = dfm.copy().sort_values("mes").reset_index(drop=True)
    f[value_col] = pd.to_numeric(f[value_col], errors="coerce").astype(float)
    f["t"] = np.arange(len(f), dtype=float)
    mo = f["mes"].dt.month.astype(float)
    f["mo_sin"] = np.sin(2*np.pi * mo / 12.0)
    f["mo_cos"] = np.cos(2*np.pi * mo / 12.0)
    for lag in [1, 2, 3, 6, 12]: f[f"{value_col}_lag{lag}"] = f[value_col].shift(lag)
    for w in [3, 6, 12]: f[f"{value_col}_rm{w}"] = f[value_col].rolling(w).mean().shift(1)
    return f

f_train   = month_features(m_train, "valor_mensual")
feat_cols = [c for c in f_train.columns if c not in ["mes", "valor_mensual"]]
f_train   = f_train.dropna(subset=feat_cols, how="any")
sample_weight = recency_weights(len(f_train), RECENCY_HALFLIFE_M) if len(f_train) > 0 else None

winner = "NaiveLast"
if f_train.empty or len(f_train) < MIN_TRAIN_ROWS:
    forecast_month_total_model = float(m_train["valor_mensual"].iloc[-1]) if not m_train.empty else 0.0
else:
    X, y = f_train[feat_cols].values, f_train["valor_mensual"].values
    lin = LinearRegression().fit(X, y, sample_weight=sample_weight)
    xgb = XGBRegressor(n_estimators=350, learning_rate=0.05, max_depth=4, subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0, random_state=SEED).fit(X, y, sample_weight=sample_weight)
    k = min(EVAL_LAST_K, len(f_train))
    e_lin, e_xgb = mape(y[-k:], lin.predict(X[-k:])), mape(y[-k:], xgb.predict(X[-k:]))
    winner = "LinReg" if e_lin <= e_xgb else "XGB"
    model  = lin if winner == "LinReg" else xgb
    m_plus = pd.concat([m_train, pd.DataFrame({"mes": [TARGET_MONTH], "valor_mensual": [np.nan]})], ignore_index=True).sort_values("mes").reset_index(drop=True)
    f_plus = month_features(m_plus, "valor_mensual")
    rowX   = f_plus[f_plus["mes"] == TARGET_MONTH][feat_cols].fillna(0.0).values
    forecast_month_total_model = float(model.predict(rowX)[0])

cal_all = pd.date_range(TARGET_MONTH, TARGET_MONTH + MonthEnd(0), freq="D")
DAYS_IN_MONTH = len(cal_all)
real_mtd = df_target.set_index("fecha")["principals_activated_today_month"].sort_index()

def momentum_prior_v6(series, lookback_days=14, weekdays_only=True):
    if series.empty: return np.nan
    s = series[series > 0].copy()
    if len(s) < 5: return np.nan
    if weekdays_only:
        s = s[s.index.weekday < 5]
        if len(s) < 4: return np.nan
    last_end = s.index.max()
    last_start = last_end - pd.Timedelta(days=lookback_days - 1)
    prev_start = last_start - pd.Timedelta(days=lookback_days)
    prev_end   = last_start - pd.Timedelta(days=1)
    s_last = s[(s.index >= last_start) & (s.index <= last_end)].sum()
    s_prev = s[(s.index >= prev_start) & (s.index <= prev_end)].sum()
    return np.nan if s_prev <= 0 else s_last / s_prev

rr = momentum_prior_v6(real_mtd, MOMENTUM_LOOKBACK_DAYS, MOMENTUM_WEEKDAYS_ONLY)
forecast_momentum_prior = (forecast_month_total_model * (1.0 + np.clip(rr - 1.0, -MOMENTUM_PRIOR_CAP_DOWN, MOMENTUM_PRIOR_CAP_UP))) if np.isfinite(rr) else np.nan
forecast_month_final = ((1.0 - MOMENTUM_PRIOR_ALPHA) * forecast_month_total_model + MOMENTUM_PRIOR_ALPHA * forecast_momentum_prior) if np.isfinite(forecast_momentum_prior) else forecast_month_total_model

# ========================= METAS AGOSTO/2026 (placeholder — actualizar con meta comercial real cuando esté disponible) =========================
# V11: metas diarias ACUMULADAS de principales (mantenidos + subidos engage + otros subidos)
# por mes. Fuente oct-26: "Octubre-26 Calculadora Tenpo Performance (metas oct final v10)",
# hoja "Metas Diarias (Octubre)-26", columna AD (= acumulado W+X+Y; total mes 96.052). Agregar cada mes
# al armar las metas; si falta el mes, no hay meta (ALPHA_META_MONTH=0: no afecta el LE).
METAS_PRINCIPALES_ACUM = {
    "2026-10": [1778, 4629, 7664, 10826, 14515, 18389, 22137, 25235, 28537, 31799, 34854, 37702, 40949, 44427, 47445, 50262, 53157, 55946, 58710, 61728, 65097, 68137, 70742, 73744, 77064, 80219, 83023, 86429, 89659, 92223, 96052],
}
_acum = METAS_PRINCIPALES_ACUM.get(TARGET_MONTH.strftime("%Y-%m"), [])
if not _acum:
    print(f"[V11 PRINCIPALES] !! sin meta diaria cargada para {TARGET_MONTH:%Y-%m}")
meta_df = pd.DataFrame({
    "fecha": pd.date_range(TARGET_MONTH, periods=len(_acum), freq="D"),
    "acum": pd.Series(_acum, dtype=float),
}).set_index("fecha").sort_index()
meta_eom = float(meta_df["acum"].iloc[-1]) if not meta_df.empty else np.nan
if np.isfinite(meta_eom) and ALPHA_META_MONTH > 0:
    forecast_month_final = (1.0 - ALPHA_META_MONTH) * forecast_month_final + ALPHA_META_MONTH * meta_eom

# ----------------------- Reales hasta D-1 -----------------------
hoy_cl_aware = ahora_chile().normalize()
hoy_cl = hoy_cl_aware.tz_convert(None)
ayer_cl = hoy_cl - pd.Timedelta(days=1)

real_dates  = set([d for d in cal_all if d <= ayer_cl])
pending_all = [d for d in cal_all if d not in real_dates]
real_mtd_full_for_sum = real_mtd.reindex(cal_all, fill_value=0.0).astype(float)
sum_real = float(real_mtd_full_for_sum.loc[list(real_dates)].sum()) if real_dates else 0.0

# =====================================================================
# Promedios históricos por día del mes
# =====================================================================
hist_day_of_month_avg = {}
meses_anchor = [TARGET_MONTH - pd.DateOffset(months=i) for i in range(1, max(WARM_START_LOOKBACK_MONTHS, FIRST_WEEK_ANCHOR_LOOKBACK_M) + 1)]
for day_num in range(1, DAYS_IN_MONTH + 1):
    vals = []
    for m in meses_anchor:
        d_check = m + pd.Timedelta(days=day_num - 1)
        if d_check.month == m.month and d_check in df["fecha"].values:
            v = float(df.loc[df["fecha"] == d_check, "principals_activated_today_month"].sum())
            if v > 0: vals.append(v)
    if vals: hist_day_of_month_avg[day_num] = float(np.mean(vals))

# Ratio histórico EOM
hist_eom_ratio = np.nan
if ENABLE_HIST_EOM_CALIBRATION:
    meses_eom_cal = [TARGET_MONTH - pd.DateOffset(months=i) for i in range(1, HIST_EOM_CALIBRATION_MONTHS + 1)]
    ratios = []
    for m in meses_eom_cal:
        df_m = df[df["mes"] == m].copy()
        if df_m.empty: continue
        df_m["dia"] = df_m["fecha"].dt.day
        mid_vals = df_m[(df_m["dia"] >= HIST_EOM_MID_DAYS[0]) & (df_m["dia"] <= HIST_EOM_MID_DAYS[1])]["principals_activated_today_month"]
        eom_vals = df_m[(df_m["dia"] >= HIST_EOM_SPIKE_DAYS[0])]["principals_activated_today_month"]
        mid_avg = float(mid_vals.mean()) if len(mid_vals) > 0 else np.nan
        eom_avg = float(eom_vals.mean()) if len(eom_vals) > 0 else np.nan
        if np.isfinite(mid_avg) and mid_avg > 0 and np.isfinite(eom_avg): ratios.append(eom_avg / mid_avg)
    if ratios:
        hist_eom_ratio_raw = float(np.mean(ratios))
        hist_eom_ratio = min(hist_eom_ratio_raw, HIST_EOM_RATIO_CAP)
        print(f"[INFO] Ratio histórico EOM/mid-month (últimos {HIST_EOM_CALIBRATION_MONTHS}m): {hist_eom_ratio_raw:.3f} (aplicado con cap {HIST_EOM_RATIO_CAP}: {hist_eom_ratio:.3f})")

# ======================= FORECAST DIARIO =======================
model_daily_series = pd.Series(0.0, index=cal_all)
if real_dates:
    model_daily_series.loc[list(real_dates)] = real_mtd_full_for_sum.loc[list(real_dates)].astype(float)

df_hist = df[df["fecha"] <= ayer_cl].copy()
df_hist["dow"] = df_hist["fecha"].dt.weekday
recent_start = ayer_cl - pd.Timedelta(weeks=DOW_FLOOR_LOOKBACK_WEEKS)

dow_recent_means = {}
for dow in range(7):
    s = df_hist[(df_hist["dow"] == dow) & (df_hist["fecha"] >= recent_start)]["principals_activated_today_month"]
    s = s[s > 0]
    dow_recent_means[dow] = float(s.mean()) if len(s) >= DOW_FLOOR_MIN_SAMPLES else np.nan

recent_14 = real_mtd_full_for_sum[real_mtd_full_for_sum.index <= ayer_cl].tail(FALLBACK_RECENT_MEAN_DAYS)
recent_14 = recent_14[recent_14 > 0]
fallback_recent_mean = float(recent_14.mean()) if len(recent_14) > 0 else np.nan

valid_means = [v for v in dow_recent_means.values() if np.isfinite(v) and v > 0]
if not valid_means and np.isfinite(fallback_recent_mean) and fallback_recent_mean > 0: valid_means = [fallback_recent_mean]

dow_level = {dow: 0.5 for dow in range(7)}
if valid_means:
    mn, mx = min(valid_means), max(valid_means)
    if mx - mn >= 1e-6:
        for dow in range(7):
            base = dow_recent_means.get(dow, np.nan)
            if not np.isfinite(base) or base <= 0: base = fallback_recent_mean if np.isfinite(fallback_recent_mean) and fallback_recent_mean > 0 else (mn+mx)/2
            dow_level[dow] = float(np.clip((base - mn) / (mx - mn), 0.0, 1.0))

center    = 0.5 * (BASE_RANGE_MIN + BASE_RANGE_MAX)
first_day = cal_all[0]
last_day  = cal_all[-1]
span_days = max((last_day - first_day).days, 1)

# Día peak del spike para agosto: día 31 (lunes, último día hábil Y último día)
EOM_PEAK_DAY = _ULTIMO_HABIL   # V11: ultimo dia habil del mes objetivo

for d in pending_all:
    dow, dia = d.weekday(), d.day
    if dia >= EOM_SPIKE_START_DAY:
        # En la zona de cierre, el patrón día-de-semana (miércoles a lunes, incl. el valle de
        # fin de semana) ya está codificado a mano en EOM_SPIKE_MULTIPLIERS. Aplicar además el
        # dow_factor genérico duplicaba ese efecto: el viernes se inflaba dos veces y el
        # sábado/domingo se desinflaba dos veces, generando la caída exagerada del día 29.
        dow_factor = 1.0
    else:
        lvl = dow_level.get(dow, 0.5)
        dow_factor = max(1.0 + DOW_SHAPE_AMPL * (2 * lvl - 1.0), DOW_FLOOR_FACTOR)
    pos = (d - first_day).days / span_days
    trend_factor = 1.0 + INTRA_MONTH_TREND_STRENGTH * (2 * pos - 1.0)
    jitter_factor = 1.0
    if not (DISABLE_JITTER_EOM and dia >= EOM_SPIKE_START_DAY):
        rng = np.random.RandomState(SEED + dia)
        jitter_factor = 1.0 + JITTER_STRENGTH * rng.uniform(-1.0, 1.0)

    v = center * dow_factor * trend_factor * jitter_factor

    if ENABLE_EOM_SPIKE and dia >= EOM_SPIKE_START_DAY:
        base_mult = EOM_SPIKE_MULTIPLIERS.get(dia, 1.0)
        if ENABLE_HIST_EOM_CALIBRATION and np.isfinite(hist_eom_ratio) and hist_eom_ratio > 1.0:
            if dia <= EOM_PEAK_DAY:
                days_in = max(dia - EOM_SPIKE_START_DAY + 1, 1)
                total_to_peak = EOM_PEAK_DAY - EOM_SPIKE_START_DAY + 1
                progress = days_in / total_to_peak
                # Para días de fin de semana (sáb/dom), no aplicar calibración ascendente
                if dow >= 5:  # sábado o domingo
                    progress = 0.0
            else:
                progress = 0.0
            if progress > 0:
                calibrated_mult = 1.0 + progress * (hist_eom_ratio - 1.0)
                # Blend en vez de max(): la calibración histórica modula el multiplicador
                # manual, pero ya no puede superarlo sin límite.
                final_mult = (1.0 - HIST_EOM_BLEND_WEIGHT) * base_mult + HIST_EOM_BLEND_WEIGHT * calibrated_mult
            else:
                final_mult = base_mult
        else:
            final_mult = base_mult
        v *= final_mult

    if ENABLE_WARM_START and dia <= WARM_START_DAYS and dia in hist_day_of_month_avg:
        hist_avg = hist_day_of_month_avg[dia]
        if hist_avg > 0: v = WARM_START_ALPHA * hist_avg + (1.0 - WARM_START_ALPHA) * v
    elif ENABLE_FIRST_WEEK_ANCHOR and dia > WARM_START_DAYS and dia <= FIRST_WEEK_ANCHOR_DAYS and dia in hist_day_of_month_avg:
        hist_avg = hist_day_of_month_avg[dia]
        if hist_avg > 0: v = FIRST_WEEK_ANCHOR_ALPHA * hist_avg + (1.0 - FIRST_WEEK_ANCHOR_ALPHA) * v

    model_daily_series.loc[d] = float(np.clip(v, CLIP_MIN, CLIP_MAX))

# Suavizado rampa corta
if real_dates:
    last_real_date = max(real_dates)
    last_real_val = float(model_daily_series.loc[last_real_date])
    if last_real_val > 0:
        pending_sorted = sorted(pending_all)
        for i, d in enumerate(pending_sorted):
            if i < RAMP_DAYS_FROM_LAST_REAL:
                w = (i + 1) / (RAMP_DAYS_FROM_LAST_REAL + 1)
                model_daily_series.loc[d] = float((1.0 - w) * last_real_val + w * float(model_daily_series.loc[d]))

pending = [d for d in cal_all if d > ayer_cl]
sum_future_raw = float(model_daily_series.loc[pending].sum()) if len(pending) > 0 else 0.0
proj_total_raw = float(sum_real + sum_future_raw)

if ENABLE_SOFT_TOTAL_ANCHOR and len(pending) > 0 and np.isfinite(forecast_month_final) and forecast_month_final > 0 and proj_total_raw > 0:
    ratio_target = float(forecast_month_final / proj_total_raw)
    soft_ratio_raw = 1.0 + SOFT_TOTAL_ALPHA * (ratio_target - 1.0)
    soft_ratio = float(np.clip(soft_ratio_raw, SOFT_TOTAL_CAP_DOWN, SOFT_TOTAL_CAP_UP))
    print(f"[DIAG SOFT-ANCHOR] proj_total_raw (forma diaria, antes de anclar): {proj_total_raw:,.0f} | "
          f"FC mensual objetivo: {forecast_month_final:,.0f} | ratio_target: {ratio_target:.3f} | "
          f"soft_ratio sin cap: {soft_ratio_raw:.3f} | soft_ratio aplicado: {soft_ratio:.3f} "
          f"(cap=[{SOFT_TOTAL_CAP_DOWN},{SOFT_TOTAL_CAP_UP}])")
    model_daily_series.loc[pending] = (model_daily_series.loc[pending].astype(float) * soft_ratio).clip(CLIP_MIN, CLIP_MAX)

sum_future = float(model_daily_series.loc[pending].sum()) if len(pending) > 0 else 0.0
proj_total_final = float(sum_real + sum_future)

# ----------------------- Salidas LONG -----------------------
rows = []
modelo = f"{winner}_v7_range{int(BASE_RANGE_MIN)}-{int(BASE_RANGE_MAX)}_eom{EOM_SPIKE_START_DAY}_momWD{int(MOMENTUM_PRIOR_ALPHA*100)}_soft{int(SOFT_TOTAL_ALPHA*100)}_warm{int(WARM_START_ALPHA*100)}"

for d in cal_all:
    is_real = d in real_dates
    val_real = float(real_mtd_full_for_sum.get(d, 0.0)) if is_real else 0.0
    val_fc   = 0.0 if is_real else float(model_daily_series.get(d, 0.0))
    val_all  = val_real if is_real else val_fc
    estado   = "R" if is_real else ("F" if val_fc > 0 else "")
    rows.append([d, int(d.day), "clientes_principales", val_real, val_fc, val_all, estado, modelo])

out_total = pd.DataFrame(rows, columns=["fecha_dt","dia","producto","valor_real_mm","valor_pronosticado_mm","valor_mm","estado","modelo"])
out_total["fecha"] = out_total["fecha_dt"].dt.strftime("%d-%m-%Y")
out_total = out_total.drop(columns=["fecha_dt"]).sort_values(["producto", "dia"]).reset_index(drop=True)

# ----------------------- Desglose mantenidos/engage/otros -----------------------
sql_seg_src = f"""
SELECT DATE(fecha) AS fecha, DATE_TRUNC(DATE(mes), MONTH) AS mes,
  CAST(habitualidad_anterior AS STRING) AS habitualidad_anterior,
  CAST(principals_activated_today_month AS FLOAT64) AS principals_activated_today_month
FROM `tenpo-bi-prod.kpitos.principalidad_diaria`
WHERE DATE(fecha) >= DATE('{(TARGET_MONTH + MonthEnd(-PROFILE_MONTHS_BACK)).date()}')
  AND DATE(fecha) < DATE('{(TARGET_MONTH + MonthEnd(0) + pd.Timedelta(days=1)).date()}')
"""
seg_src = read_gbq_robusto(sql_seg_src, PROJECT_ID, credentials)
if seg_src.empty: raise RuntimeError("principalidad_diaria no devolvió filas.")
seg_src["fecha"] = pd.to_datetime(seg_src["fecha"]).dt.normalize()
seg_src["mes"] = pd.to_datetime(seg_src["mes"]).dt.to_period("M").dt.to_timestamp()

def _is_engage(s): return s.fillna("").str.lower().str.contains("engage")
def _is_principal_prev(s): return s.fillna("").str.lower().str.contains("principal")

seg_target = seg_src[seg_src["mes"] == TARGET_MONTH].copy()
seg_target["is_engage"] = _is_engage(seg_target["habitualidad_anterior"])
seg_target["is_principal"] = _is_principal_prev(seg_target["habitualidad_anterior"])

share_start = ayer_cl - pd.Timedelta(weeks=8)
seg_recent = seg_src[(seg_src["fecha"] >= share_start) & (seg_src["fecha"] <= ayer_cl)].copy()
if seg_recent.empty: seg_recent = seg_src[seg_src["fecha"] < TARGET_MONTH].copy()
seg_recent["is_engage"] = _is_engage(seg_recent["habitualidad_anterior"])
seg_recent["is_principal"] = _is_principal_prev(seg_recent["habitualidad_anterior"])
seg_recent["is_otros"] = ~(seg_recent["is_engage"] | seg_recent["is_principal"])
seg_recent["dow"] = seg_recent["fecha"].dt.weekday

def _agg_share(col):
    b = seg_recent.groupby(["dow", col])["principals_activated_today_month"].sum().reset_index().rename(columns={"principals_activated_today_month": "n"})
    t = seg_recent.groupby("dow")["principals_activated_today_month"].sum().reset_index().rename(columns={"principals_activated_today_month": "tot_dow"})
    m = b.merge(t, on="dow", how="left"); m["share"] = np.where(m["tot_dow"]>0, m["n"]/m["tot_dow"], np.nan)
    return m

shares_eng, shares_prin, shares_otros = _agg_share("is_engage"), _agg_share("is_principal"), _agg_share("is_otros")

def _gs(col):
    g = seg_recent.groupby(col)["principals_activated_today_month"].sum().reset_index().rename(columns={"principals_activated_today_month":"n"})
    t = float(g["n"].sum()); return float(g.loc[g[col]==True,"n"].sum()/t) if t>0 else np.nan

sg_p, sg_e, sg_o = _gs("is_principal"), _gs("is_engage"), _gs("is_otros")

def get_shares(dow):
    sp = float(shares_prin.loc[(shares_prin["dow"]==dow)&(shares_prin["is_principal"]==True),"share"].sum()) if not shares_prin.empty else np.nan
    se = float(shares_eng.loc[(shares_eng["dow"]==dow)&(shares_eng["is_engage"]==True),"share"].sum()) if not shares_eng.empty else np.nan
    so = float(shares_otros.loc[(shares_otros["dow"]==dow)&(shares_otros["is_otros"]==True),"share"].sum()) if not shares_otros.empty else np.nan
    if not np.isfinite(sp) or sp<=0.001: sp = sg_p if np.isfinite(sg_p) else 0.60
    if not np.isfinite(se) or se<=0.001: se = sg_e if np.isfinite(sg_e) else 0.30
    if not np.isfinite(so): so = sg_o if np.isfinite(sg_o) else 0.10
    s = sp+se+so
    return (sp/s, se/s, so/s) if s>0 else (0.60, 0.30, 0.10)

rows_two = []
for _, r in out_total[["fecha","dia","estado","modelo"]].iterrows():
    d_str, dia, estado = r["fecha"], int(r["dia"]), r["estado"]
    d = pd.to_datetime(d_str, format="%d-%m-%Y")
    total_dia = float(out_total.loc[(out_total["fecha"]==d_str)&(out_total["producto"]=="clientes_principales"),"valor_mm"].sum())
    if estado == "R":
        ds = seg_target[seg_target["fecha"]==d].copy(); ds["is_otros"]=~(ds["is_engage"]|ds["is_principal"])
        pr = float(ds.loc[ds["is_principal"]==True,"principals_activated_today_month"].sum())
        er = float(ds.loc[ds["is_engage"]==True,"principals_activated_today_month"].sum())
        ot = float(ds.loc[ds["is_otros"]==True,"principals_activated_today_month"].sum())
        if pr==0 and er==0 and total_dia>0: sp,se,so=get_shares(d.weekday()); pr,er,ot=total_dia*sp,total_dia*se,total_dia*so
        else: ot += total_dia-(pr+er+ot)
        for prod,val in [("__desde_principal",pr),("__desde_engage",er),("__desde_otros",ot)]:
            rows_two.append([d_str,dia,f"clientes_principales{prod}",val,0.0,val,"R",r["modelo"]])
    else:
        sp,se,so=get_shares(d.weekday()); pf,ef,of_=total_dia*sp,total_dia*se,total_dia*so; of_+=total_dia-(pf+ef+of_)
        for prod,val in [("__desde_principal",pf),("__desde_engage",ef),("__desde_otros",of_)]:
            rows_two.append([d_str,dia,f"clientes_principales{prod}",0.0,val,val,"F",r["modelo"]])

out_two = pd.DataFrame(rows_two, columns=STANDARD_COLS)

# ----------------------- Stock, Principalidad, Cobertura -----------------------
# Denominador oficial de principalidad: MAU App MTD alineado a la query de referencia.
mau_daily_series = df_target.set_index("fecha")["mau_app_mtd_query"].sort_index().astype(float)
mau_mes = float(mau_daily_series.max()) if not mau_daily_series.empty else mau_app_mes

# Se conserva MAU con renta como métrica de cobertura, pero ya no es el denominador principal.
mau_renta_daily_series = df_target.set_index("fecha")["mau_con_renta"].sort_index().astype(float) if "mau_con_renta" in df_target.columns else pd.Series(dtype=float)
real_mtd_full = real_mtd_full_for_sum.reindex(cal_all, fill_value=0.0).astype(float)
cumsum_real = real_mtd_full.cumsum()
model_full = model_daily_series.reindex(cal_all, fill_value=0.0).astype(float)
cumsum_all = model_full.cumsum()

rows_pr, rows_stock, rows_pr_app = [], [], []
for d in cal_all:
    ds, dia = d.strftime("%d-%m-%Y"), int(d.day)
    dn = mau_mes if np.isfinite(mau_mes) and mau_mes>0 else np.nan
    da = mau_app_mes if np.isfinite(mau_app_mes) and mau_app_mes>0 else np.nan
    nrc, nac = float(cumsum_real.get(d,0.0)), float(cumsum_all.get(d,0.0))
    if d <= ayer_cl:
        rows_pr.append([ds,dia,"principalidad_real",nrc/dn if np.isfinite(dn) else 0,0,nrc/dn if np.isfinite(dn) else 0,"R",modelo])
        rows_stock.append([ds,dia,"clientes_principales_stock",nrc,0,nrc,"R",modelo])
        rows_pr_app.append([ds,dia,"principalidad_sobre_app",nrc/da if np.isfinite(da) else 0,0,nrc/da if np.isfinite(da) else 0,"R",modelo])
    else:
        rows_pr.append([ds,dia,"principalidad_real",0,nac/dn if np.isfinite(dn) else 0,nac/dn if np.isfinite(dn) else 0,"F",modelo])
        rows_stock.append([ds,dia,"clientes_principales_stock",0,nac,nac,"F" if nac>0 else "",modelo])
        rows_pr_app.append([ds,dia,"principalidad_sobre_app",0,nac/da if np.isfinite(da) else 0,nac/da if np.isfinite(da) else 0,"F",modelo])

out_pr = pd.DataFrame(rows_pr, columns=STANDARD_COLS)
out_stock = pd.DataFrame(rows_stock, columns=STANDARD_COLS)
out_pr_app = pd.DataFrame(rows_pr_app, columns=STANDARD_COLS)

rows_act = [[r["fecha"],r["dia"],"clientes_principales_activados",r["valor_real_mm"],r["valor_pronosticado_mm"],r["valor_mm"],r["estado"],r["modelo"]] for _,r in out_total.iterrows()]
out_act = pd.DataFrame(rows_act, columns=STANDARD_COLS)

rows_mau, rows_mau_renta, rows_mau_app, rows_cov = [], [], [], []
cov_val = (mau_con_renta_mes/mau_app_mes) if (np.isfinite(mau_con_renta_mes) and mau_con_renta_mes>0 and np.isfinite(mau_app_mes) and mau_app_mes>0) else 0
for d in cal_all:
    ds, dia = d.strftime("%d-%m-%Y"), int(d.day)
    md = float(mau_daily_series.get(d, np.nan))
    if not np.isfinite(md) or md<=0: md = mau_mes if np.isfinite(mau_mes) and mau_mes>0 else 0

    mr = float(mau_renta_daily_series.get(d, np.nan)) if not mau_renta_daily_series.empty else np.nan
    if not np.isfinite(mr) or mr<=0: mr = mau_con_renta_mes if np.isfinite(mau_con_renta_mes) and mau_con_renta_mes>0 else 0

    ma = float(mau_app_mes) if np.isfinite(mau_app_mes) and mau_app_mes>0 else 0
    rows_mau.append([ds,dia,"mau_app_mtd_query",0,0,md,"R" if md>0 else "",modelo])
    rows_mau_renta.append([ds,dia,"mau_con_renta",0,0,mr,"R" if mr>0 else "",modelo])
    rows_mau_app.append([ds,dia,"mau_total_app",0,0,ma,"R" if ma>0 else "",modelo])
    rows_cov.append([ds,dia,"cobertura_renta",cov_val,0,cov_val,"R" if cov_val>0 else "",modelo])

out_mau = pd.DataFrame(rows_mau, columns=STANDARD_COLS)
out_mau_renta = pd.DataFrame(rows_mau_renta, columns=STANDARD_COLS)
out_mau_app = pd.DataFrame(rows_mau_app, columns=STANDARD_COLS)
out_cov = pd.DataFrame(rows_cov, columns=STANDARD_COLS)

# ----------------------- Unir -----------------------
out_all = pd.concat([out_total, out_two, out_pr, out_stock, out_act, out_mau, out_mau_renta, out_mau_app, out_pr_app, out_cov], ignore_index=True).sort_values(["producto","dia"]).reset_index(drop=True)
display_map = {"clientes_principales":"Principales Diario","clientes_principales__desde_principal":"Principales Mantenidos","clientes_principales__desde_engage":"Desde Engage → Principal","clientes_principales__desde_otros":"Desde Otros → Principal","principalidad_real":"% Principalidad (MAU App MTD query)","principalidad_sobre_app":"% Principalidad (MAU App MTD query)","clientes_principales_activados":"Principales Activados (diario)","clientes_principales_stock":"Stock Principales (acumulado)","mau_con_renta":"MAU con renta", "mau_app_mtd_query":"MAU App MTD (query)","mau_total_app":"MAU total app","cobertura_renta":"Cobertura de renta"}
out_all["producto_display"] = out_all["producto"].map(display_map).fillna(out_all["producto"])

# ----------------------- BQ Write -----------------------
def ensure_numeric_float64(df_in, cols): return df_in.astype({c:"float64" for c in cols})
BQ_LOCATION = ensure_dataset_location(PROJECT_ID, DEST_TABLE, credentials)
schema_main = [{"name":"fecha","type":"STRING"},{"name":"dia","type":"INTEGER"},{"name":"producto","type":"STRING"},{"name":"producto_display","type":"STRING"},{"name":"valor_real_mm","type":"FLOAT"},{"name":"valor_pronosticado_mm","type":"FLOAT"},{"name":"valor_mm","type":"FLOAT"},{"name":"estado","type":"STRING"},{"name":"modelo","type":"STRING"}]
out_all = ensure_numeric_float64(out_all, ["valor_real_mm","valor_pronosticado_mm","valor_mm"])
pandas_gbq.to_gbq(out_all.astype({"fecha":"string","dia":"int64","producto":"string","producto_display":"string","valor_real_mm":"float64","valor_pronosticado_mm":"float64","valor_mm":"float64","estado":"string","modelo":"string"}), DEST_TABLE, project_id=PROJECT_ID, if_exists="replace", location=BQ_LOCATION, table_schema=schema_main)

# ----------------------- Tabla Resumen -----------------------
print("\n"+"="*200)
print(f" RESUMEN DIARIO - CLIENTES PRINCIPALES Y MÉTRICAS ({_MES_TXT.upper()}) ".center(200))
print("="*200)
pivot_df = out_all.pivot_table(index="fecha",columns="producto",values="valor_mm",aggfunc="sum").reset_index()
estado_df = out_all[out_all["producto"]=="clientes_principales"][["fecha","estado"]].rename(columns={"estado":"estado_total"})
st = pivot_df.merge(estado_df, on="fecha", how="left")
st["fecha_dt"] = pd.to_datetime(st["fecha"], format="%d-%m-%Y")
st = st.sort_values("fecha_dt").drop(columns=["fecha_dt"])
for ac, sc in [("clientes_principales_acum","clientes_principales"),("desde_principal_acum","clientes_principales__desde_principal"),("desde_engage_acum","clientes_principales__desde_engage"),("desde_otros_acum","clientes_principales__desde_otros")]:
    if sc in st.columns: st[ac] = st[sc].cumsum()
co = ["fecha","estado_total","clientes_principales","clientes_principales_acum","clientes_principales_activados","clientes_principales__desde_principal","desde_principal_acum","clientes_principales__desde_engage","desde_engage_acum","clientes_principales__desde_otros","desde_otros_acum","clientes_principales_stock","principalidad_real","principalidad_sobre_app","cobertura_renta","mau_app_mtd_query","mau_con_renta","mau_total_app"]
co = [c for c in co if c in st.columns]; st = st[co]
for c in co:
    if c not in ["fecha","estado_total"]:
        if c in ["principalidad_real","principalidad_sobre_app","cobertura_renta"]: st[c] = st[c].apply(lambda x: f"{x:,.4f}" if pd.notnull(x) else "")
        else: st[c] = st[c].apply(lambda x: f"{x:,.1f}" if pd.notnull(x) else "")
pd.set_option("display.max_columns", None); pd.set_option("display.width", 300)
print(st.to_string(index=False))
print("="*200+"\n")
print(f"[RESUMEN v7-AGO] Total mes proyectado: {proj_total_final:,.0f}")
print(f"[RESUMEN v7-AGO] Sum real: {sum_real:,.0f} | Sum forecast: {sum_future:,.0f}")
print(f"[RESUMEN v7-AGO] Modelo: {winner} | FC mensual (pre-mom): {forecast_month_total_model:,.0f} | FC final: {forecast_month_final:,.0f}")
print(f"[RESUMEN v7-AGO] Momentum ratio (weekdays): {rr:.4f}" if np.isfinite(rr) else "[RESUMEN v7-AGO] Momentum: N/A")


# ----------------------- Comparación contra query oficial MTD -----------------------
sql_principalidad_ref = f"""
DECLARE mes_actual            DATE DEFAULT DATE('{TARGET_MONTH:%Y-%m-%d}');
DECLARE mes_anterior          DATE DEFAULT DATE_TRUNC(DATE_SUB(mes_actual, INTERVAL 1 MONTH), MONTH);
DECLARE fin_mes_anterior      DATE DEFAULT DATE_SUB(mes_actual, INTERVAL 1 DAY);
DECLARE dia_ayer              DATE DEFAULT LEAST(DATE_SUB(CURRENT_DATE("America/Santiago"), INTERVAL 1 DAY), LAST_DAY(mes_actual));
DECLARE dia_ayer_mes_anterior DATE DEFAULT DATE_SUB(dia_ayer, INTERVAL 1 MONTH);

WITH
base_economics AS (
  SELECT
    DATE(fecha)                AS fecha,
    DATE_TRUNC(DATE(fecha), MONTH) AS mes,
    SAFE_CAST(user AS STRING)  AS user_id,
    trx_id,
    monto,
    linea
  FROM `tenpo-bi-prod.economics.economics`
  WHERE LOWER(IFNULL(nombre, "null"))           NOT LIKE "%devol%"
    AND LOWER(IFNULL(actividad_nac_cd, "null")) NOT LIKE "%refund%"
    AND DATE(fecha) BETWEEN mes_anterior AND dia_ayer
),
base_marcada AS (
  SELECT b.*, 'MTD' AS tipo_periodo
  FROM base_economics b
  WHERE b.fecha BETWEEN mes_actual   AND dia_ayer
     OR b.fecha BETWEEN mes_anterior AND dia_ayer_mes_anterior
  UNION ALL
  SELECT b.*, 'Cerrado' AS tipo_periodo
  FROM base_economics b
  WHERE b.fecha BETWEEN mes_anterior AND fin_mes_anterior
),
base_gpv_sow AS (
  SELECT tipo_periodo, mes, user_id, SUM(monto) AS gpv
  FROM base_marcada
  WHERE linea IN (SELECT * FROM `tenpo-bi-prod.kpitos.lineas_gpv_app`)
  GROUP BY 1,2,3
),
base_renta_estimada AS (
  SELECT SAFE_CAST(user AS STRING) AS user_id, MAX(renta_estimada) AS renta_estimada
  FROM `tenpo-bi-prod.analytics_audience.user_renta_estimada`
  WHERE renta_estimada > 0
  GROUP BY 1
),
analisis_sow_user AS (
  SELECT g.tipo_periodo, g.mes, g.user_id, g.gpv, r.renta_estimada,
         SAFE_DIVIDE(g.gpv, r.renta_estimada) AS sow
  FROM base_gpv_sow g
  LEFT JOIN base_renta_estimada r USING (user_id)
),
principales AS (
  SELECT tipo_periodo, mes, user_id, 'principal' AS habitualidad
  FROM analisis_sow_user
  WHERE sow >= 0.5
),
habitualidad_moda_mes AS (
  SELECT
    DATE_TRUNC(DATE(fecha_ejecucion), MONTH) AS mes,
    SAFE_CAST(user AS STRING) AS user_id,
    habitualidad,
    COUNT(DISTINCT DATE(fecha_ejecucion)) AS frequency
  FROM `tenpo-bi-prod.analytics_audience.habitualidad_tenpo`
  WHERE DATE(fecha_ejecucion) BETWEEN mes_anterior AND CURRENT_DATE("America/Santiago")
  GROUP BY 1,2,3
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY user_id, mes
    ORDER BY
      frequency DESC,
      CASE habitualidad
        WHEN 'engage'   THEN 1
        WHEN 'habit'    THEN 2
        WHEN 'aha'      THEN 3
        WHEN 'distract' THEN 4
        WHEN 'setup'    THEN 5
        ELSE 99
      END ASC
  ) = 1
),
habitualidad_final AS (
  SELECT
    tp AS tipo_periodo,
    h.mes,
    h.user_id,
    CASE WHEN p.habitualidad = 'principal' THEN 'principal' ELSE h.habitualidad END AS habitualidad
  FROM habitualidad_moda_mes h
  CROSS JOIN UNNEST(['MTD', 'Cerrado']) AS tp
  LEFT JOIN principales p
    ON h.user_id = p.user_id
   AND h.mes     = p.mes
   AND tp        = p.tipo_periodo
  WHERE h.habitualidad IS NOT NULL
),
base_con_habitualidad AS (
  SELECT
    b.tipo_periodo,
    b.mes,
    b.user_id,
    b.trx_id,
    b.monto,
    b.linea,
    CASE
      WHEN h.habitualidad = 'principal' THEN '1- Principal'
      WHEN h.habitualidad = 'engage'    THEN '2- Engage'
      ELSE                                   '3- Otros'
    END AS habitualidad
  FROM base_marcada b
  LEFT JOIN habitualidad_final h
    ON b.user_id      = h.user_id
   AND b.mes          = h.mes
   AND b.tipo_periodo = h.tipo_periodo
),
kpi_mau AS (
  SELECT tipo_periodo, mes AS fecha, habitualidad, 'App' AS tipo_reporte,
         COUNT(DISTINCT user_id) AS mau
  FROM base_con_habitualidad
  WHERE linea IN (SELECT * FROM `tenpo-bi-prod.kpitos.lineas_mau_app`)
  GROUP BY 1,2,3,4
)
SELECT tipo_periodo, fecha, habitualidad, tipo_reporte, 'MAU' AS kpi,
       CAST(mau AS FLOAT64) AS valor
FROM kpi_mau
ORDER BY tipo_reporte, fecha DESC, tipo_periodo, habitualidad, kpi
"""

try:
    ref_pr = read_gbq_robusto(sql_principalidad_ref, PROJECT_ID, credentials)
    if not ref_pr.empty:
        ref_pr["fecha"] = pd.to_datetime(ref_pr["fecha"]).dt.to_period("M").dt.to_timestamp()
        m_ref = (
            (ref_pr["tipo_periodo"] == "MTD") &
            (ref_pr["fecha"] == TARGET_MONTH) &
            (ref_pr["tipo_reporte"] == "App") &
            (ref_pr["kpi"] == "MAU")
        )
        ref_mtd = ref_pr[m_ref].copy()
        ref_principal_mau = float(ref_mtd.loc[ref_mtd["habitualidad"] == "1- Principal", "valor"].sum())
        ref_app_mau = float(ref_mtd["valor"].sum())
        ref_principalidad = ref_principal_mau / ref_app_mau if ref_app_mau > 0 else np.nan

        fecha_comp = min(ayer_cl, cal_all[-1])
        script_stock = float(cumsum_real.reindex(cal_all, fill_value=0.0).loc[:fecha_comp].iloc[-1]) if fecha_comp in cal_all else np.nan
        script_mau = float(mau_daily_series.reindex(cal_all).ffill().loc[fecha_comp]) if fecha_comp in cal_all else np.nan
        script_principalidad = script_stock / script_mau if script_mau > 0 else np.nan

        print("\n" + "="*120)
        print(" COMPARACIÓN PRINCIPALIDAD VS QUERY OFICIAL MTD ".center(120))
        print("="*120)
        print(f"Fecha comparación: {fecha_comp.strftime('%d-%m-%Y')}")
        print(f"Query oficial MTD  | Principal MAU App: {ref_principal_mau:,.0f} | MAU App: {ref_app_mau:,.0f} | Principalidad: {ref_principalidad:,.4f}")
        print(f"Script forecast    | Principal stock:   {script_stock:,.0f} | MAU App: {script_mau:,.0f} | Principalidad: {script_principalidad:,.4f}")
        print(f"Delta principalidad: {(script_principalidad - ref_principalidad):,.6f}" if np.isfinite(script_principalidad) and np.isfinite(ref_principalidad) else "Delta principalidad: N/A")
        print("="*120 + "\n")
except Exception as e:
    print(f"[WARN] No se pudo ejecutar comparación contra query oficial MTD: {e}")

# =================================================================================================
# === GRÁFICO MEJORADO: Real+Forecast continuo, Promedio 3m, 3 meses individuales ================
# =================================================================================================
print("\nGenerando visualización mejorada...")

meses_hist = [TARGET_MONTH - pd.DateOffset(months=i) for i in range(1, 4)]  # Últimos 3 meses
HIST_COLORS = ["#E67E22", "#27AE60", "#8E44AD"]  # Naranja, Verde, Púrpura — fáciles de distinguir
HIST_LABELS = []

fig, ax = plt.subplots(figsize=(16, 8))
hist_curves = []

# --- 1. Curvas individuales de los últimos 3 meses (punteadas, colores distinguibles) ---
for i, m in enumerate(meses_hist):
    df_m = df[df["mes"] == m].copy()
    if not df_m.empty:
        df_m["dia_del_mes"] = df_m["fecha"].dt.day
        df_m = df_m.sort_values("dia_del_mes")
        label = f'Real {m.strftime("%Y-%m")}'
        ax.plot(df_m["dia_del_mes"], df_m["principals_activated_today_month"],
                label=label, color=HIST_COLORS[i], alpha=0.50, linestyle='--', linewidth=1.5)
        hist_curves.append(df_m.set_index("dia_del_mes")["principals_activated_today_month"])

# --- 2. Promedio de los últimos 3 meses (línea negra gruesa) ---
if hist_curves:
    df_hist_all = pd.concat(hist_curves, axis=1)
    mean_curve = df_hist_all.mean(axis=1).dropna()
    ax.plot(mean_curve.index, mean_curve.values,
            label='Promedio últimos 3 meses', color='black', linewidth=3, alpha=0.85)

# --- 3. Curva continua: Real (sólida azul) + Forecast (sólida azul punteada) ---
out_act_sorted = out_act.sort_values("dia")
dias_all   = out_act_sorted["dia"].values
vals_all   = out_act_sorted["valor_mm"].values
estados    = out_act_sorted["estado"].values

# Separar real y forecast
dias_real = [d for d, e in zip(dias_all, estados) if e == "R"]
vals_real = [v for v, e in zip(vals_all, estados) if e == "R"]
dias_fc   = [d for d, e in zip(dias_all, estados) if e == "F"]
vals_fc   = [v for v, e in zip(vals_all, estados) if e == "F"]

# Tramo real (línea sólida azul con marcadores)
if dias_real:
    ax.plot(dias_real, vals_real, color='#2980B9', linewidth=3, marker='o',
            markersize=5, markerfacecolor='#2980B9', markeredgecolor='white',
            markeredgewidth=0.8, zorder=5)

# Conectar último real con primer forecast (línea de transición)
if dias_real and dias_fc:
    ax.plot([dias_real[-1], dias_fc[0]], [vals_real[-1], vals_fc[0]],
            color='#2980B9', linewidth=2.5, linestyle='--', alpha=0.7, zorder=4)

# Tramo forecast (línea punteada azul con triángulos)
if dias_fc:
    ax.plot(dias_fc, vals_fc, color='#2980B9', linewidth=3, linestyle='--',
            marker='^', markersize=6, markerfacecolor='#E74C3C', markeredgecolor='white',
            markeredgewidth=0.8, zorder=5)

# --- 4. Línea vertical de corte Real/Forecast ---
if dias_real and dias_fc:
    corte_x = dias_real[-1] + 0.5
    ax.axvline(x=corte_x, color='red', linestyle=':', linewidth=1.5, alpha=0.7)
    ax.text(corte_x + 0.3, ax.get_ylim()[1] * 0.95, 'Corte\nReal/FC',
            fontsize=8, color='red', alpha=0.8, ha='left', va='top',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='red', alpha=0.7))

# --- Leyenda personalizada ---
legend_elements = []
for i, m in enumerate(meses_hist):
    legend_elements.append(mlines.Line2D([], [], color=HIST_COLORS[i], linestyle='--', linewidth=1.5, alpha=0.5, label=f'Real {m.strftime("%Y-%m")}'))
legend_elements.append(mlines.Line2D([], [], color='black', linewidth=3, alpha=0.85, label='Promedio últimos 3 meses'))
legend_elements.append(mlines.Line2D([], [], color='#2980B9', linewidth=3, marker='o', markersize=5, label=f'{_MES_TXT} — Real (R)'))
legend_elements.append(mlines.Line2D([], [], color='#2980B9', linewidth=3, linestyle='--', marker='^', markersize=6, markerfacecolor='#E74C3C', label=f'{_MES_TXT} — Forecast (F)'))

ax.legend(handles=legend_elements, loc='upper left', fontsize=9, framealpha=0.9)

ax.set_title(f'Activación Diaria Principales: {_MES_TXT} (Real + Forecast) vs Histórico 3 Meses', fontsize=14, pad=15, fontweight='bold')
ax.set_xlabel('Día del Mes', fontsize=12)
ax.set_ylabel('Clientes Principales Activados (Diario)', fontsize=12)
ax.set_xticks(range(1, DAYS_IN_MONTH + 1))
ax.grid(True, alpha=0.25, linestyle=':')
fig.tight_layout()

plt.savefig(f'forecast_principales_{TARGET_MONTH:%Y_%m}_vs_historico.png', dpi=300)
plt.show()
print(f"Proceso finalizado. Gráfico 'forecast_principales_{TARGET_MONTH:%Y_%m}_vs_historico.png' guardado con éxito.")

In [ ]:

# =====================================================================================
# GRÁFICO DOW-ALIGNED: Comparación día-de-semana vs día-de-semana
# =====================================================================================
# En vez de comparar día 1 vs día 1 (que pueden ser Lun vs Jue), este gráfico
# compara el 1er Lunes vs 1er Lunes, 1er Martes vs 1er Martes, etc.
#
# Usa los mismos datos ya cargados del script principal (df con la data histórica
# y out_act con el forecast de junio). Ejecutar DESPUÉS del script principal.
# =====================================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.ticker as mticker

# ─── Configuración (ajustar si se cambia de mes) ───
TARGET_MONTH_CHART = TARGET_MONTH   # V11: mismo mes que la celda de principales (antes junio fijo)
N_HIST_MONTHS      = 3  # Últimos 3 meses

DOW_NAMES_ES = ['Lun', 'Mar', 'Mié', 'Jue', 'Vie', 'Sáb', 'Dom']
HIST_COLORS  = ["#E67E22", "#27AE60", "#8E44AD"]  # Naranja, Verde, Púrpura

# ─── Función: asignar posición semana-dow a cada día de un mes ───
def assign_week_dow_position(dates_series, values_series):
    """
    Para cada día del mes, asigna:
      - week_of_month: 1-5 (semana dentro del mes)
      - dow: 0-6 (Lun-Dom)
      - dow_occurrence: 1er Lun, 2do Lun, etc.
      - sequential_pos: posición secuencial 1..N para el eje X
    """
    result = pd.DataFrame({
        "fecha": dates_series,
        "valor": values_series,
    }).copy()
    result["dia"] = result["fecha"].dt.day
    result["dow"] = result["fecha"].dt.weekday  # 0=Lun
    result["dow_name"] = result["dow"].map(lambda x: DOW_NAMES_ES[x])

    # Calcular la N-ésima ocurrencia de cada DOW en el mes
    result["dow_occurrence"] = result.groupby("dow").cumcount() + 1  # 1er Lun, 2do Lun...

    # Posición secuencial: semana * 7 + dow (para ordenar cronológicamente)
    # Pero mejor usar simplemente el orden del día en el mes
    result["seq_pos"] = range(1, len(result) + 1)

    # Label para el eje X: "S1-Lun", "S1-Mar", etc.
    result["week_label"] = "S" + result["dow_occurrence"].astype(str)
    result["x_label"] = result["week_label"] + "-" + result["dow_name"]

    return result

# ─── Función: alinear un mes histórico al target por DOW ───
def align_month_to_target_by_dow(df_target_month, df_hist_month):
    """
    Para cada día del mes target, busca el día equivalente en el mes histórico
    (mismo DOW, misma ocurrencia: 1er Lun↔1er Lun, 2do Mar↔2do Mar, etc.)
    """
    aligned = []
    for _, row_t in df_target_month.iterrows():
        dow = row_t["dow"]
        occ = row_t["dow_occurrence"]
        seq = row_t["seq_pos"]

        match = df_hist_month[
            (df_hist_month["dow"] == dow) &
            (df_hist_month["dow_occurrence"] == occ)
        ]
        if not match.empty:
            aligned.append({
                "seq_pos": seq,
                "x_label": row_t["x_label"],
                "valor": float(match.iloc[0]["valor"]),
                "dow": dow,
            })

    return pd.DataFrame(aligned)

# ─── Preparar datos del mes target (Junio 2026) ───
# out_act viene del script principal — tiene Real + Forecast
out_act_sorted = out_act.sort_values("dia").copy()
out_act_sorted["fecha"] = pd.to_datetime(out_act_sorted["fecha"], format="%d-%m-%Y")

target_data = assign_week_dow_position(
    out_act_sorted["fecha"],
    out_act_sorted["valor_mm"]
)
target_data["estado"] = out_act_sorted["estado"].values

# ─── Preparar datos históricos (alineados por DOW) ───
meses_hist = [TARGET_MONTH_CHART - pd.DateOffset(months=i) for i in range(1, N_HIST_MONTHS + 1)]
hist_aligned_list = []
hist_raw_list = []

for m in meses_hist:
    df_m = df[df["mes"] == m].copy()
    if df_m.empty:
        hist_aligned_list.append(None)
        hist_raw_list.append(None)
        continue

    df_m = df_m.sort_values("fecha")
    raw = assign_week_dow_position(df_m["fecha"], df_m["principals_activated_today_month"])
    hist_raw_list.append(raw)

    aligned = align_month_to_target_by_dow(target_data, raw)
    hist_aligned_list.append(aligned)

# ─── Calcular promedio de los 3 meses (DOW-aligned) ───
avg_aligned = target_data[["seq_pos", "x_label", "dow"]].copy()
valor_cols = []
for i, aligned in enumerate(hist_aligned_list):
    if aligned is not None and not aligned.empty:
        col_name = f"val_{i}"
        merged = avg_aligned.merge(aligned[["seq_pos", "valor"]].rename(columns={"valor": col_name}), on="seq_pos", how="left")
        avg_aligned = merged
        valor_cols.append(col_name)

if valor_cols:
    avg_aligned["promedio"] = avg_aligned[valor_cols].mean(axis=1)
else:
    avg_aligned["promedio"] = np.nan

# ═══════════════════════════════════════════════════════════════════
# GRÁFICO 1: Comparación DOW-aligned — todos los meses superpuestos
# ═══════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(18, 8))

# --- Meses históricos (punteados) ---
for i, (aligned, m) in enumerate(zip(hist_aligned_list, meses_hist)):
    if aligned is not None and not aligned.empty:
        ax.plot(aligned["seq_pos"], aligned["valor"],
                color=HIST_COLORS[i], linestyle='--', linewidth=1.5, alpha=0.50,
                label=f'Real {m.strftime("%Y-%m")}')

# --- Promedio 3 meses (negro grueso) ---
if not avg_aligned["promedio"].isna().all():
    ax.plot(avg_aligned["seq_pos"], avg_aligned["promedio"],
            color='black', linewidth=3, alpha=0.85, label='Promedio 3 meses (DOW-aligned)')

# --- Mes target: Real + Forecast ---
real_mask = target_data["estado"] == "R"
fc_mask   = target_data["estado"] == "F"

if real_mask.any():
    ax.plot(target_data.loc[real_mask, "seq_pos"], target_data.loc[real_mask, "valor"],
            color='#2980B9', linewidth=3, marker='o', markersize=5,
            markerfacecolor='#2980B9', markeredgecolor='white', markeredgewidth=0.8, zorder=5)

if real_mask.any() and fc_mask.any():
    last_real_idx = target_data.loc[real_mask].index[-1]
    first_fc_idx  = target_data.loc[fc_mask].index[0]
    ax.plot([target_data.loc[last_real_idx, "seq_pos"], target_data.loc[first_fc_idx, "seq_pos"]],
            [target_data.loc[last_real_idx, "valor"], target_data.loc[first_fc_idx, "valor"]],
            color='#2980B9', linewidth=2.5, linestyle='--', alpha=0.7, zorder=4)

if fc_mask.any():
    ax.plot(target_data.loc[fc_mask, "seq_pos"], target_data.loc[fc_mask, "valor"],
            color='#2980B9', linewidth=3, linestyle='--', marker='^', markersize=6,
            markerfacecolor='#E74C3C', markeredgecolor='white', markeredgewidth=0.8, zorder=5)

# --- Línea de corte ---
if real_mask.any() and fc_mask.any():
    corte_x = float(target_data.loc[real_mask, "seq_pos"].max()) + 0.5
    ax.axvline(x=corte_x, color='red', linestyle=':', linewidth=1.5, alpha=0.6)

# --- Sombreado de fines de semana ---
for _, row in target_data.iterrows():
    if row["dow"] >= 5:  # Sáb o Dom
        ax.axvspan(row["seq_pos"] - 0.4, row["seq_pos"] + 0.4,
                   color='gray', alpha=0.08, zorder=0)

# --- Eje X: etiquetas DOW ---
ax.set_xticks(target_data["seq_pos"].values)
# Etiquetas compactas: solo DOW name, con separadores de semana
x_labels = []
prev_occ = 0
for _, row in target_data.iterrows():
    occ = row["dow_occurrence"]
    if occ != prev_occ:
        x_labels.append(f'─S{int(occ)}─\n{row["dow_name"]}')
        prev_occ = occ
    else:
        x_labels.append(row["dow_name"])

ax.set_xticklabels(x_labels, fontsize=7, rotation=0)

# --- Leyenda ---
legend_elements = []
for i, m in enumerate(meses_hist):
    legend_elements.append(mlines.Line2D([], [], color=HIST_COLORS[i], linestyle='--', linewidth=1.5, alpha=0.5, label=f'Real {m.strftime("%Y-%m")}'))
legend_elements.append(mlines.Line2D([], [], color='black', linewidth=3, alpha=0.85, label='Promedio 3 meses'))
legend_elements.append(mlines.Line2D([], [], color='#2980B9', linewidth=3, marker='o', markersize=5, label=f'{TARGET_MONTH_CHART.strftime("%b %Y")} — Real'))
legend_elements.append(mlines.Line2D([], [], color='#2980B9', linewidth=3, linestyle='--', marker='^', markersize=6, markerfacecolor='#E74C3C', markeredgecolor='white', label=f'{TARGET_MONTH_CHART.strftime("%b %Y")} — Forecast'))

ax.legend(handles=legend_elements, loc='upper left', fontsize=9, framealpha=0.9)
ax.set_title(f'Activación Diaria: {TARGET_MONTH_CHART.strftime("%B %Y")} vs Histórico — Alineado por Día de Semana', fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel('Clientes Principales Activados (Diario)', fontsize=12)
ax.set_xlabel('Semana del Mes — Día de Semana', fontsize=11)
ax.grid(True, alpha=0.2, linestyle=':')
fig.tight_layout()
plt.savefig(f'forecast_principales_{TARGET_MONTH_CHART:%Y_%m}_dow_aligned.png', dpi=300)
plt.show()
print(f"Gráfico DOW-aligned guardado: 'forecast_principales_{TARGET_MONTH_CHART:%Y_%m}_dow_aligned.png'")

# --- Preparar datos para la tabla resumen ---
dow_hist_vals = [[] for _ in range(7)]
dow_target_vals = [[] for _ in range(7)]

for aligned_df in hist_aligned_list:
    if aligned_df is not None and not aligned_df.empty:
        for _, row in aligned_df.iterrows():
            dow_hist_vals[row['dow']].append(row['valor'])

for _, row in target_data.iterrows():
    dow_target_vals[row['dow']].append(row['valor'])

# ═══════════════════════════════════════════════════════════════════
# TABLA: Resumen comparativo por DOW
# ═══════════════════════════════════════════════════════════════════
print("\n" + "="*90)
print(f" COMPARACIÓN POR DÍA DE SEMANA: {TARGET_MONTH_CHART:%m-%Y} vs Promedio 3 Meses ".center(90))
print("="*90)
print(f"{'DOW':<6} {'Prom Hist':>10} {'Prom Jun':>10} {'Δ':>8} {'Δ%':>7}  {'N hist':>6} {'N jun':>6}")
print("-"*65)

for dow in range(7):
    h_vals = dow_hist_vals[dow]
    t_vals = dow_target_vals[dow]
    h_avg = np.mean(h_vals) if h_vals else np.nan
    t_avg = np.mean(t_vals) if t_vals else np.nan
    delta = t_avg - h_avg if np.isfinite(h_avg) and np.isfinite(t_avg) else np.nan
    pct = (delta / h_avg * 100) if np.isfinite(h_avg) and h_avg > 0 and np.isfinite(delta) else np.nan

    print(f"{DOW_NAMES_ES[dow]:<6} {h_avg:>10,.0f} {t_avg:>10,.0f} {delta:>+8,.0f} {pct:>+6.1f}%  {len(h_vals):>6} {len(t_vals):>6}"
          if np.isfinite(h_avg) and np.isfinite(t_avg) else
          f"{DOW_NAMES_ES[dow]:<6} {'N/A':>10} {'N/A':>10} {'':>8} {'':>7}  {len(h_vals):>6} {len(t_vals):>6}")

print("="*90)


In [ ]:
import pydata_google_auth
import pandas as pd
import numpy as np
import requests
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV
import xgboost as xgb
import matplotlib.pyplot as plt
import matplotlib.ticker as tkr
import seaborn as sns
import warnings
import pandas_gbq
warnings.filterwarnings('ignore')
import pickle
from google.colab import drive
drive.mount('/content/drive')

from google.colab import auth
auth.authenticate_user()
print("Autenticado exitosamente en Google Cloud")

SCOPES = [
            'https://www.googleapis.com/auth/cloud-platform',
            'https://www.googleapis.com/auth/drive',
        ]
credentials = pydata_google_auth.get_user_credentials(
            SCOPES,
            auth_local_webserver=True,
        )

In [ ]:
mes = "Consolidado"
archivo_metas = pd.read_excel(f"/content/drive/.shortcut-targets-by-id/10Ha9KHrUn7dPQ7dGoiEovdVoRhmr5MSZ/Planificación Financiera/Cierres/Metas/Metas KPI ({mes}).xlsx", sheet_name="Consolidado")
primera_columna = archivo_metas.columns[0]
archivo_metas.columns = [primera_columna] + pd.to_datetime(archivo_metas.columns[1:]).strftime('%Y-%m-%d').tolist()

In [ ]:
# =================================================================================================
# CONFIGURACIÓN DE FECHAS Y METAS
# =================================================================================================
from datetime import date, timedelta
import pandas as pd

directorio_salida = "/shortcut-targets-by-id/10Ha9KHrUn7dPQ7dGoiEovdVoRhmr5MSZ/Planificación Financiera/KPItos Data/modelos"

hoy = hoy_chile().date()
ayer = hoy - timedelta(days=1)
inicio_mes = hoy.replace(day=1)
inicio_anio = date(hoy.year, 1, 1)

# --- 1. FECHAS DE ENTRENAMIENTO (Ahora son móviles hasta "ayer") ---
fechas_entrenamiento = ("2023-01-01", ayer.strftime("%Y-%m-%d"))
fechas_entrenamiento_tc = ("2024-07-01", ayer.strftime("%Y-%m-%d"))
fechas_entrenamiento_tc_xs = ("2024-10-01", ayer.strftime("%Y-%m-%d"))

# (Mantenemos tu proyección estática original por si otras celdas la usan)
fechas_proyeccion = ("2023-10-01", "2026-04-01", "2026-04-30")

# --- 2. CONFIGURACIÓN DE METAS Y ALGORITMOS ---
columna_fecha = archivo_metas.filter(like=fechas_proyeccion[1]).columns[0]
meta_gpv_tc_vir = archivo_metas.loc[archivo_metas['KPI'] == "gpv_tc_vir", columna_fecha].values[0]
meta_gpv_tc_fis = archivo_metas.loc[archivo_metas['KPI'] == "gpv_tc_fis", columna_fecha].values[0]
meta_gpv_rec = 1

algoritmo_gpv_tc_fis = "gpv_tc_fis"
algoritmo_gpv_tc_vir = "gpv_tc_vir"
algoritmo_gpv_rec = "gpv_rec"

# --- 3. HORIZONTES DE PROYECCIÓN MÓVILES ---
# A. (8 Semanas)
fin_8_semanas = hoy + timedelta(weeks=8) - timedelta(days=1)
fechas_proyeccion_8 = (
    "2023-10-01",
    inicio_mes.strftime("%Y-%m-%d"),
    fin_8_semanas.strftime("%Y-%m-%d"),
)

# 2. Horizonte 90 Días Móviles
fin_90_dias = hoy + timedelta(days=90)
fechas_proyeccion_90 = (
    "2023-10-01",
    inicio_mes.strftime("%Y-%m-%d"),
    fin_90_dias.strftime("%Y-%m-%d"),
)

print(f"Entrenamiento termina el: {fechas_entrenamiento[1]}")
print(f"Horizonte 8 semanas termina el: {fechas_proyeccion_8[2]}")
print(f"Horizonte 90 días termina el:   {fechas_proyeccion_90[2]}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
QUERY = """SELECT * FROM `tenpo-bi-prod.kpitos.modelos_forecasting_gpv_productos`
UNION ALL SELECT * FROM `tenpo-bi-prod.kpitos.modelos_forecasting_gpv_recaudacion`"""
gpvs_productos = pd.read_gbq(QUERY,use_bqstorage_api=True, project_id='tenpo-bi-prod', credentials=credentials, dialect='standard')
gpvs_productos['fecha'] = pd.to_datetime(gpvs_productos['fecha'], format="%Y-%m-%d")

gpv_tc_fis = gpvs_productos[gpvs_productos['linea'] == "credit_card_physical"].drop(columns={'linea'})
gpv_tc_vir = gpvs_productos[gpvs_productos['linea'] == "credit_card_virtual"].drop(columns={'linea'})
gpv_rec = gpvs_productos[gpvs_productos['linea'] == "recaudacion"].drop(columns={'linea'})


In [ ]:
class ForecastProducto:

    def __init__(self, nombre_algoritmo, meta, fechas_entrenamiento, fechas_proyeccion, directorio):
        """
        Inicializa la clase ForecastProducto.

        Args:
            nombre_algoritmo (str): Nombre del algoritmo a usar
            meta (float): Meta objetivo
            fechas_entrenamiento (tuple): (fecha_inicio, fecha_fin) para entrenamiento
            fechas_proyeccion (tuple): (fecha_inicio, inicio_mes, fin_mes) para proyección
            directorio (str): Directorio donde guardar/cargar modelos
        """
        self.nombre_algoritmo = nombre_algoritmo
        self.meta = meta
        self.fecha_train_inicio, self.fecha_train_final = fechas_entrenamiento
        self.fecha_proyeccion_inicio, self.inicio_mes, self.fin_mes = fechas_proyeccion
        self.directorio = directorio

        # Atributos que se inicializan durante el procesamiento
        self.model = None
        self.kpi_merge = None
        self.kpi_dummies = None
        self.model_columns = None

    def _cargar_feriados(self):
        """
        Carga datos de feriados desde archivo Excel.

        Returns:
            pd.DataFrame: DataFrame con columnas fecha, feriado, irrenunciable
        """
        ruta_feriados = "/content/drive/.shortcut-targets-by-id/10Ha9KHrUn7dPQ7dGoiEovdVoRhmr5MSZ/Planificación Financiera/Cierres/Proyecciones/feriados.xlsx"
        feriados = pd.read_excel(ruta_feriados)
        return feriados[['fecha', 'feriado', 'irrenunciable']]

    def _agregar_variables_temporales(self, df):
        """
        Agrega variables temporales al DataFrame.

        Args:
            df (pd.DataFrame): DataFrame con columna fecha

        Returns:
            pd.DataFrame: DataFrame con variables temporales agregadas
        """
        # Crear variables temporales básicas
        df['dia_semana'] = df['fecha'].dt.dayofweek
        df['dia'] = df['fecha'].dt.day
        df['mes'] = df['fecha'].dt.month
        df['year'] = df['fecha'].dt.year

        # Mapear días de la semana a nombres
        dias_mapping = {
            0: 'Mon', 1: 'Tues', 2: 'Weds', 3: 'Thurs',
            4: 'Fri', 5: 'Sat', 6: 'Sun'
        }
        df['dia_semana'] = df['dia_semana'].map(dias_mapping)

        # Convertir feriados a booleanos
        df['feriado'] = df['feriado'].astype(bool)
        df['irrenunciable'] = df['irrenunciable'].astype(bool)

        return df

    def _marcar_activacion_tc(self, grupo):
        """
        Marca días de activación de tarjeta de crédito basado en reglas de negocio.

        Args:
            grupo (pd.DataFrame): Grupo de datos por año/mes

        Returns:
            pd.DataFrame: Grupo con columna activacion_tc marcada
        """
        grupo = grupo.sort_values('dia')
        grupo['activacion_tc'] = False

        # Buscar primer día hábil después del día 5 y 20
        for dia_limite in [5, 20]:
            grupo_dia = grupo[grupo['dia'] >= dia_limite]

            for idx, row in grupo_dia.iterrows():
                # Verificar si es día hábil (no fin de semana ni feriado)
                if row['dia_semana'] not in ['Sun', 'Sat'] and not row['feriado']:
                    if not grupo.loc[idx, 'activacion_tc']:
                        grupo.loc[idx, 'activacion_tc'] = True
                    break

        return grupo

    def _preparar_datos(self, kpi):
        """
        Prepara los datos para el entrenamiento del modelo.

        Args:
            kpi (pd.DataFrame): DataFrame con datos del KPI
        """
        # Crear rango completo de fechas
        fechas = pd.date_range('2023-01-01', '2026-12-31')
        kpi = kpi.merge(pd.DataFrame({'fecha': fechas}), on='fecha', how='outer')

        # Agregar feriados
        feriados = self._cargar_feriados()
        kpi = kpi.merge(feriados, on='fecha', how='outer').fillna(0)

        # Agregar variables temporales
        kpi = self._agregar_variables_temporales(kpi)

        # Identificar columna objetivo
        columnas_conocidas = {
            'fecha', 'dia_semana', 'dia', 'mes', 'year',
            'real', 'feriado', 'irrenunciable', 'activacion_tc'
        }
        target_col_name = list(set(kpi.columns) - columnas_conocidas)[0]
        kpi = kpi.rename(columns={target_col_name: 'target_kpi'})

        # Marcar activaciones de TC por grupo año/mes
        kpi = kpi.groupby(['year', 'mes']).apply(self._marcar_activacion_tc).reset_index(drop=True)

        # Guardar datos procesados
        self.kpi_merge = kpi
        self.kpi_modelo = kpi[
            (kpi['fecha'] >= self.fecha_train_inicio) &
            (kpi['fecha'] <= self.fecha_train_final)
        ]
        self.kpi_dummies = pd.get_dummies(self.kpi_modelo.drop(columns='real'))

    def _entrenar_modelo(self):
        """
        Entrena el modelo XGBoost con búsqueda de hiperparámetros.
        """
        # Preparar datos para entrenamiento
        X = self.kpi_dummies.drop(['target_kpi', 'fecha'], axis=1)
        y = self.kpi_dummies['target_kpi']
        self.model_columns = X.columns.tolist()

        # División train/test
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=123
        )

        # Configurar modelo base
        modelo = xgb.XGBRegressor(random_state=123)

        # Definir grilla de hiperparámetros
        param_grid = {
            'nthread': [4],
            'objective': ['reg:squarederror'],
            'learning_rate': [0.001, 0.01, 0.03, 0.05, 0.07, 0.1],
            'max_depth': [1, 2, 3, 4, 5, 6, 7],
            'min_child_weight': [4],
            'subsample': [0.7],
            'colsample_bytree': [0.7],
            'n_estimators': [1000, 1500]
        }

        # Búsqueda de hiperparámetros
        grid = GridSearchCV(
            modelo, param_grid, cv=2, n_jobs=5, verbose=True
        )
        grid.fit(X_train, y_train)

        # Mostrar resultados del entrenamiento
        print(f"Best score: {grid.best_score_}")
        print(f"Best params: {grid.best_params_}")
        print(f"Train accuracy: {grid.score(X_train, y_train)}")
        print(f"Test accuracy: {grid.score(X_test, y_test)}")

        # Guardar modelo
        self.model = grid
        # Ensure the directory exists before saving
        import os
        os.makedirs(self.directorio, exist_ok=True)
        filename = f"{self.directorio}/xgb_{self.nombre_algoritmo}.pkl"
        _dump_pickle_atomic(grid, filename)

    def _aplicar_reglas_negocio(self, kpi_test):
        """
        Aplica reglas de negocio específicas según el algoritmo.

        Args:
            kpi_test (pd.DataFrame): DataFrame con predicciones

        Returns:
            pd.DataFrame: DataFrame con reglas de negocio aplicadas
        """
        # Regla específica para insurance
        if "insurance" in self.nombre_algoritmo:
            mask_insurance = (
                (kpi_test['fecha'].dt.day > 24) &
                (kpi_test['target_kpi_prediccion'] > 1)
            )
            kpi_test.loc[mask_insurance, 'target_kpi_prediccion'] = 0

        return kpi_test

    def _calcular_metricas_acumuladas(self, kpi_test):
        """
        Calcula métricas acumuladas y aplica la meta.

        Args:
            kpi_test (pd.DataFrame): DataFrame con predicciones

        Returns:
            pd.DataFrame: DataFrame con métricas calculadas
        """
        # Combinar predicción con datos reales
        kpi_test['target_kpi_real_pred'] = kpi_test['target_kpi_prediccion']
        mask_real = kpi_test['real'] == 1
        kpi_test.loc[mask_real, 'target_kpi_real_pred'] = kpi_test.loc[mask_real, 'target_kpi']

        # Aplicar límite mínimo
        kpi_test["target_kpi_prediccion"] = kpi_test["target_kpi_prediccion"].clip(lower=1)
        kpi_test["target_kpi_real_pred"] = kpi_test["target_kpi_real_pred"].clip(lower=1)

        # Calcular acumulados mensuales
        kpi_test['periodo'] = kpi_test['fecha'].dt.to_period("M")
        kpi_test['target_kpi_prediccion_cumsum'] = kpi_test.groupby('periodo')['target_kpi_prediccion'].cumsum()
        kpi_test['target_kpi_real_pred_cumsum'] = kpi_test.groupby('periodo')['target_kpi_real_pred'].cumsum()

        # Crear columnas auxiliares para manejo de datos reales
        kpi_test['target_kpi_real_pred_cumsum2'] = kpi_test['target_kpi_real_pred_cumsum']
        kpi_test.loc[mask_real, 'target_kpi_real_pred_cumsum2'] = 0
        kpi_test.loc[~mask_real, 'target_kpi_real_pred_cumsum'] = 0

        # Reemplazar ceros con NaN y filtrar por fechas
        kpi_test = kpi_test.replace(0, np.nan)
        kpi_test = kpi_test[
            (kpi_test['fecha'] >= self.inicio_mes) &
            (kpi_test['fecha'] <= self.fin_mes)
        ]

        # Aplicar meta
        kpi_test['meta'] = self.meta
        total_prediccion = kpi_test['target_kpi_prediccion'].sum()
        kpi_test['target_kpi_prediccion_meta'] = (
            kpi_test['target_kpi_prediccion'] / total_prediccion
        ) * self.meta
        kpi_test['target_kpi_prediccion_meta_cumsum'] = kpi_test['target_kpi_prediccion_meta'].cumsum()

        return kpi_test

    def _limpiar_columnas(self, kpi_test):
        """
        Elimina columnas auxiliares que no son necesarias en el output final.

        Args:
            kpi_test (pd.DataFrame): DataFrame con todas las columnas

        Returns:
            pd.DataFrame: DataFrame con columnas limpias
        """
        columnas_eliminar = [
            'real', 'dia_semana', 'dia', 'mes', 'year',
            'feriado', 'irrenunciable', 'activacion_tc', 'periodo'
        ]
        kpi_test.drop(columns=columnas_eliminar, inplace=True, errors='ignore')
        return kpi_test

    def _renombrar_columnas(self, kpi_test):
        """
        Renombra las columnas según las convenciones del algoritmo.

        Args:
            kpi_test (pd.DataFrame): DataFrame con nombres originales

        Returns:
            pd.DataFrame: DataFrame con columnas renombradas
        """
        columnas = kpi_test.columns.tolist()

        # Determinar sufijo según tipo de algoritmo
        if self.nombre_algoritmo.startswith(("ob_tc", "ob_tenpo")):
            sufijo = self.nombre_algoritmo[self.nombre_algoritmo.rfind('_'):]
        else:
            sufijo = f"_{self.nombre_algoritmo[4:]}"

        # Aplicar sufijo a todas las columnas excepto fecha
        nuevas_columnas = [columnas[0]] + [f"{col}{sufijo}" for col in columnas[1:]]
        kpi_test.columns = nuevas_columnas

        # Reemplazar 'target_kpi' con prefijo del algoritmo
        if self.nombre_algoritmo.startswith(("ob_tc", "ob_tenpo")):
            reemplazo = self.nombre_algoritmo[:2]
        else:
            reemplazo = self.nombre_algoritmo[:3]

        kpi_test.columns = kpi_test.columns.str.replace('target_kpi', reemplazo)

        # Ajustes específicos para ob_tc
        if self.nombre_algoritmo.startswith("ob_tc"):
            nuevas_columnas = []
            for col in kpi_test.columns:
                if col.startswith("meta") and not col.startswith("meta_ob_tc"):
                    nueva_col = col.replace("meta", "meta_ob_tc", 1)
                else:
                    nueva_col = col
                nuevas_columnas.append(nueva_col)
            kpi_test.columns = nuevas_columnas

        # Ajustes específicos para ob_tenpo
        if self.nombre_algoritmo == "ob_tenpo":
            nuevas_columnas = []
            for col in kpi_test.columns:
                if (col.startswith("ob_") and
                    not col.startswith("ob_tenpo") and
                    col.endswith("_tenpo")):
                    base = col[len("ob_"):-len("_tenpo")]
                    nueva_col = f"ob_tenpo_{base}"
                    nuevas_columnas.append(nueva_col)
                else:
                    nuevas_columnas.append(col)
            kpi_test.columns = nuevas_columnas

        # Ajuste específico para ob_tenp
        if self.nombre_algoritmo == "ob_tenp":
            kpi_test = kpi_test.rename(columns={"meta_tenpo": "meta"})

        return kpi_test

    def proyectar(self, kpi):
        """
        Realiza la proyección usando el modelo entrenado.

        Args:
            kpi (pd.DataFrame): DataFrame con datos del KPI

        Returns:
            pd.DataFrame: DataFrame con proyecciones y métricas calculadas
        """
        # Preparar datos si no se han preparado
        if self.kpi_merge is None or self.kpi_dummies is None:
            self._preparar_datos(kpi)

        # Cargar o entrenar modelo
        if self.model is None:
            try:
                filename = f"{self.directorio}/xgb_{self.nombre_algoritmo}.pkl"
                self.model = pickle.load(open(filename, 'rb'))

                # Obtener columnas del modelo desde los datos preparados
                temp_X = self.kpi_dummies.drop(['target_kpi', 'fecha'], axis=1)
                self.model_columns = temp_X.columns.tolist()
                # V12: si el pkl trae sus propias features, usarlas (un pkl de otra version/feature-set
                # ya no rompe el predict; las faltantes se rellenan con 0 en reindex).
                _mc = _columnas_del_modelo(self.model)
                if _mc:
                    self.model_columns = _mc
            except Exception as _e:
                print(f"[V12] pkl ausente/ilegible ({type(_e).__name__}): re-entrenando {self.nombre_algoritmo}")
                self._entrenar_modelo()

        # Preparar datos de prueba
        kpi_test = self.kpi_merge[self.kpi_merge['fecha'] >= self.fecha_proyeccion_inicio]
        test_dummies = pd.get_dummies(kpi_test).drop(
            columns=['fecha', 'target_kpi', 'real'], errors='ignore'
        )

        # Asegurar que las columnas coincidan con el modelo
        test_dummies = test_dummies.reindex(columns=self.model_columns, fill_value=0)

        # Realizar predicción
        pred = self.model.predict(test_dummies)
        pred_df = pd.DataFrame(pred.astype(np.int64), columns=['target_kpi_prediccion'])
        pred_df['fecha'] = kpi_test['fecha'].values
        kpi_test = kpi_test.merge(pred_df, on='fecha')

        # Aplicar reglas de negocio
        kpi_test = self._aplicar_reglas_negocio(kpi_test)

        # Calcular métricas acumuladas
        kpi_test = self._calcular_metricas_acumuladas(kpi_test)

        # Limpiar columnas auxiliares
        kpi_test = self._limpiar_columnas(kpi_test)

        # Renombrar columnas según convenciones
        kpi_test = self._renombrar_columnas(kpi_test)

        return kpi_test

In [ ]:
#-------------------------------------

# --- forecast_gpv_tc_fis_8 ---
forecast_gpv_tc_fis_90 = ForecastProducto(nombre_algoritmo=algoritmo_gpv_tc_fis, meta=meta_gpv_tc_fis, fechas_entrenamiento=fechas_entrenamiento_tc, fechas_proyeccion=fechas_proyeccion_90, directorio=directorio_salida)
forecast_gpv_tc_fis_90 = forecast_gpv_tc_fis_90.proyectar(gpv_tc_fis)
#--- Ajuste y cálculo de acumulados ---
mask = forecast_gpv_tc_fis_90['gpv_tc_fis'].isna()
forecast_gpv_tc_fis_90.loc[mask, 'gpv_prediccion_tc_fis'] *= 1.2
forecast_gpv_tc_fis_90['gpv_prediccion_tc_fis'] = forecast_gpv_tc_fis_90['gpv_prediccion_tc_fis'].round(0)
forecast_gpv_tc_fis_90.loc[mask, 'gpv_real_pred_tc_fis'] = forecast_gpv_tc_fis_90.loc[mask, 'gpv_prediccion_tc_fis']
forecast_gpv_tc_fis_90['gpv_prediccion_cumsum_tc_fis'] = forecast_gpv_tc_fis_90['gpv_prediccion_tc_fis'].cumsum()
cumsum_real = forecast_gpv_tc_fis_90['gpv_real_pred_tc_fis'].cumsum()
forecast_gpv_tc_fis_90.loc[mask, 'gpv_real_pred_cumsum2_tc_fis'] = cumsum_real[mask]
#-------------------------------------

# --- forecast_gpv_tc_vir_8 ---
forecast_gpv_tc_vir_90 = ForecastProducto(nombre_algoritmo=algoritmo_gpv_tc_vir, meta=meta_gpv_tc_vir, fechas_entrenamiento=fechas_entrenamiento_tc, fechas_proyeccion=fechas_proyeccion_90, directorio=directorio_salida)
forecast_gpv_tc_vir_90 = forecast_gpv_tc_vir_90.proyectar(gpv_tc_vir)
#--- Ajuste y cálculo de acumulados ---
mask = forecast_gpv_tc_vir_90['gpv_tc_vir'].isna()
forecast_gpv_tc_vir_90.loc[mask, 'gpv_prediccion_tc_vir'] *= 1.2
forecast_gpv_tc_vir_90['gpv_prediccion_tc_vir'] = forecast_gpv_tc_vir_90['gpv_prediccion_tc_vir'].round(0)
forecast_gpv_tc_vir_90.loc[mask, 'gpv_real_pred_tc_vir'] = forecast_gpv_tc_vir_90.loc[mask, 'gpv_prediccion_tc_vir']
forecast_gpv_tc_vir_90['gpv_prediccion_cumsum_tc_vir'] = forecast_gpv_tc_vir_90['gpv_prediccion_tc_vir'].cumsum()
cumsum_real = forecast_gpv_tc_vir_90['gpv_real_pred_tc_vir'].cumsum()
forecast_gpv_tc_vir_90.loc[mask, 'gpv_real_pred_cumsum2_tc_vir'] = cumsum_real[mask]


# --- forecast_gpv_tc_vir_8 ---
forecast_gpv_rec_90 = ForecastProducto(nombre_algoritmo=algoritmo_gpv_rec, meta=meta_gpv_rec, fechas_entrenamiento=fechas_entrenamiento_tc, fechas_proyeccion=fechas_proyeccion_90, directorio=directorio_salida)
forecast_gpv_rec_90 = forecast_gpv_rec_90.proyectar(gpv_rec)
#--- Ajuste y cálculo de acumulados ---
mask = forecast_gpv_rec_90['gpv_rec'].isna()
forecast_gpv_rec_90.loc[mask, 'gpv_prediccion_rec'] *= 1.2
forecast_gpv_rec_90['gpv_prediccion_rec'] = forecast_gpv_rec_90['gpv_prediccion_rec'].round(0)
forecast_gpv_rec_90.loc[mask, 'gpv_real_pred_rec'] = forecast_gpv_rec_90.loc[mask, 'gpv_prediccion_rec']
forecast_gpv_rec_90['gpv_prediccion_cumsum_rec'] = forecast_gpv_rec_90['gpv_prediccion_rec'].cumsum()
cumsum_real = forecast_gpv_rec_90['gpv_real_pred_rec'].cumsum()
forecast_gpv_rec_90.loc[mask, 'gpv_real_pred_cumsum2_rec'] = cumsum_real[mask]
#-------------------------------------

In [ ]:
gpv_tc_proy = (
    forecast_gpv_tc_vir_90.set_index("fecha")["gpv_real_pred_tc_vir"]
    .add(
        forecast_gpv_tc_fis_90.set_index("fecha")["gpv_real_pred_tc_fis"],
        fill_value=0
    )
    .to_frame(name="gpv_tc_real_proy")
    .reset_index()
)
flujos_tesoreria = pd.merge(gpv_tc_proy,forecast_gpv_rec_90[['fecha',"gpv_real_pred_rec"]], on='fecha')

cutoff_real = hoy_chile().date() - timedelta(days=1)
flujos_tesoreria["status"] = np.where(
    flujos_tesoreria["fecha"].dt.date <= cutoff_real,
    "R",  # Real
    "F"   # Forecast
)

project = 'tenpo-bi-prod'
pandas_gbq.to_gbq(
    flujos_tesoreria, 'kpitos.flujos_tesoreria', project_id=project, if_exists='replace',
)

In [ ]:
# =================================================================================================
# TESORERÍA
# =================================================================================================

import pandas as pd
import numpy as np
import pandas_gbq
from datetime import date

print("--- 1. Preparando entorno y feriados ---")

ruta_base = "/content/drive/.shortcut-targets-by-id/10Ha9KHrUn7dPQ7dGoiEovdVoRhmr5MSZ/Planificación Financiera/Cierres/Proyecciones/"

try:
    feriados_cl = pd.read_excel(f"{ruta_base}feriados.xlsx")
    feriados_us = pd.read_excel(f"{ruta_base}feriados_gringos.xlsx")
    print("Archivos cargados exitosamente.")
except FileNotFoundError as e:
    print(f"Error: No se encontró el archivo:\n{e}")

feriados_cl = feriados_cl[['fecha']].assign(es_feriado_cl=True)
feriados_us = feriados_us[['fecha']].assign(es_feriado_us=True)
feriados_cl['fecha'] = pd.to_datetime(feriados_cl['fecha'])
feriados_us['fecha'] = pd.to_datetime(feriados_us['fecha'])

def ajustar_flujo_tesoreria(df_input, col_valor):
    df = df_input.copy()
    if not np.issubdtype(df['fecha'].dtype, np.datetime64):
        df['fecha'] = pd.to_datetime(df['fecha'])

    df = df.merge(feriados_cl, on='fecha', how='left').merge(feriados_us, on='fecha', how='left')
    df[['es_feriado_cl', 'es_feriado_us']] = df[['es_feriado_cl', 'es_feriado_us']].fillna(False)

    df['dia_semana'] = df['fecha'].dt.dayofweek
    df['es_inhabil'] = (df['dia_semana'] >= 5) | (df['es_feriado_cl']) | (df['es_feriado_us'])

    valores_ajustados = []
    acumulado = 0

    for i, row in df.iterrows():
        valor = row[col_valor] if pd.notnull(row[col_valor]) else 0
        if row['es_inhabil']:
            acumulado += valor
            valores_ajustados.append(0)
        else:
            nuevo_valor = valor + acumulado
            valores_ajustados.append(nuevo_valor)
            acumulado = 0

    df[f"{col_valor}_tesoreria"] = valores_ajustados
    return df[['fecha', f"{col_valor}_tesoreria"]]

# =================================================================================================
# 2. PROCESAMIENTO TC Y RECAUDACIÓN
# =================================================================================================
print("--- 2. Procesando TC y Recaudación ---")

df_tc_base = pd.merge(
    forecast_gpv_tc_fis_90[['fecha', 'gpv_real_pred_tc_fis']],
    forecast_gpv_tc_vir_90[['fecha', 'gpv_real_pred_tc_vir']],
    on='fecha', how='outer'
).fillna(0)
df_tc_base['gpv_tc_real_proy'] = df_tc_base['gpv_real_pred_tc_fis'] + df_tc_base['gpv_real_pred_tc_vir']

df_rec_base = forecast_gpv_rec_90[['fecha', 'gpv_real_pred_rec']].copy()

df_tes_tc = ajustar_flujo_tesoreria(df_tc_base, 'gpv_tc_real_proy')
df_tes_rec = ajustar_flujo_tesoreria(df_rec_base, 'gpv_real_pred_rec')

# =================================================================================================
# 3. PROCESAMIENTO PREPAGO
# =================================================================================================
print("--- 3. Generando Proyección de Prepago ---")

gpv_prp_fis = gpvs_productos[gpvs_productos['linea'] == "mastercard_physical"].drop(columns={'linea'})
gpv_prp_vir = gpvs_productos[gpvs_productos['linea'] == "mastercard"].drop(columns={'linea'})

f_prp_fis = ForecastProducto("gpv_prp_fis", 1, fechas_entrenamiento, fechas_proyeccion_90, directorio_salida).proyectar(gpv_prp_fis)
f_prp_vir = ForecastProducto("gpv_prp_vir", 1, fechas_entrenamiento, fechas_proyeccion_90, directorio_salida).proyectar(gpv_prp_vir)

df_prp_total = pd.merge(
    f_prp_fis[['fecha', 'gpv_real_pred_prp_fis']],
    f_prp_vir[['fecha', 'gpv_real_pred_prp_vir']],
    on='fecha', how='outer'
).fillna(0)
df_prp_total['gpv_prp_total'] = df_prp_total['gpv_real_pred_prp_fis'] + df_prp_total['gpv_real_pred_prp_vir']

df_tes_prp = ajustar_flujo_tesoreria(df_prp_total, 'gpv_prp_total')

# =================================================================================================
# 4. CÁLCULO DE RATIOS CROSSBORDER
# =================================================================================================
print("--- 4. Calculando Ratios Crossborder ---")

fecha_max_real = gpvs_productos['fecha'].max()
fecha_inicio_calc = fecha_max_real - pd.DateOffset(months=3)
df_hist_recent = gpvs_productos[gpvs_productos['fecha'] >= fecha_inicio_calc].copy()

df_pivot = df_hist_recent.pivot_table(index='fecha', columns='linea', values='gpv', aggfunc='sum').fillna(0)
for col in ['mastercard', 'mastercard_physical', 'crossborder', 'credit_card_virtual', 'credit_card_physical', 'credit_card_crossborder']:
    if col not in df_pivot.columns: df_pivot[col] = 0.0

vol_xb_prp = df_pivot['crossborder'].sum()
vol_tot_prp = (df_pivot['mastercard'] + df_pivot['mastercard_physical']).sum()
ratio_prp = vol_xb_prp / vol_tot_prp if vol_tot_prp > 0 else 0

vol_xb_tc = df_pivot['credit_card_crossborder'].sum()
vol_tot_tc = (df_pivot['credit_card_virtual'] + df_pivot['credit_card_physical']).sum()
ratio_tc = vol_xb_tc / vol_tot_tc if vol_tot_tc > 0 else 0

if ratio_tc == 0:
    val_xb_ref = 167447698.0
    val_tc_ref = 1923443015.0
    ratio_tc = val_xb_ref / val_tc_ref

df_tes_prp['gpv_xb_tesoreria'] = df_tes_prp['gpv_prp_total_tesoreria'] * ratio_prp
df_tes_tc['gpv_tc_xb_tesoreria'] = df_tes_tc['gpv_tc_real_proy_tesoreria'] * ratio_tc

# =================================================================================================
# 5. CONSOLIDACIÓN FINAL
# =================================================================================================
print("--- 5. Generando Tabla Final ---")

df_final = df_tes_tc.merge(df_tes_rec, on='fecha', how='outer')
df_final = df_final.merge(df_tes_prp, on='fecha', how='outer')

# ---> FILTRO MÓVIL: Desde el 1 de Enero del año actual en adelante
inicio_anio_pd = pd.to_datetime(date(hoy_chile().date().year, 1, 1))
df_final = df_final[df_final['fecha'] >= inicio_anio_pd].sort_values('fecha').reset_index(drop=True)

df_final.rename(columns={
    'gpv_tc_real_proy_tesoreria': 'proyeccion_gpv_tc',
    'gpv_tc_xb_tesoreria': 'proyeccion_gpv_tc_xb',
    'gpv_real_pred_rec_tesoreria': 'proyeccion_recaudacion_tc',
    'gpv_prp_total_tesoreria': 'proyeccion_gpv_prepago',
    'gpv_xb_tesoreria': 'proyeccion_gpv_prepago_xb'
}, inplace=True)

df_final['status'] = 'R'

cols_finales = [
    'fecha', 'status',
    'proyeccion_gpv_tc',
    'proyeccion_gpv_tc_xb',
    'proyeccion_recaudacion_tc',
    'proyeccion_gpv_prepago',
    'proyeccion_gpv_prepago_xb'
]

for col in cols_finales:
    if col not in df_final.columns: df_final[col] = 0
df_final = df_final[cols_finales].fillna(0)

print(f"\nVista previa:")
display(df_final.head(5))

nombre_tabla = 'kpitos.flujos_tesoreria_ajustados'
print(f"\nGuardando en BigQuery: {nombre_tabla} ...")
pandas_gbq.to_gbq(df_final, nombre_tabla, project_id='tenpo-bi-prod', if_exists='replace')
print("¡Tabla actualizada correctamente en BigQuery!")

In [ ]:
# ==========================================
# LE SEGUROS — Forecast (XGB) + Reglas + Escritura BQ — JULIO 2026
# ==========================================
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import pandas_gbq
import pydata_google_auth
from pandas.tseries.offsets import MonthEnd, MonthBegin
from google.cloud import bigquery
from google.cloud.exceptions import NotFound
from xgboost import XGBRegressor

# ---------- Parámetros generales ----------
PROJECT_ID   = "tenpo-bi-prod"
DATASET_ID   = "kpitos"
LOCATION_FALLBACK = "southamerica-west1"

START_DATE   = pd.Timestamp("2023-01-01").normalize()

# --- AGOSTO 2026 (31 días) ---
# Calendario: día 1=Mié, día 5=Dom (billing), día 20=Lun (billing), día 31=Vie.
FORE_START   = ahora_chile().tz_localize(None).normalize().replace(day=1)   # V11: mes en curso
FORE_END     = (FORE_START + MonthEnd(0)).normalize()  # 2026-08-31

D_MINUS_1 = (ahora_chile().normalize().tz_localize(None) - pd.Timedelta(days=1))

TRAIN_WINDOW_DAYS = 180
CAP_UP_ECO   = 0.10
CAP_DOWN_ECO = 0.10
CAP_UP_STK   = 0.06
CAP_DOWN_STK = 0.06

ALPHA_ANCHOR_VOL = 0.60
ALPHA_ANCHOR_STD = 0.50
HARD_UP_MULT   = 1.50
HARD_DOWN_MULT = 0.25
VERY_LOW_LEVEL = 5.0

# --- PARAMETROS ESTRICTOS (NO TOCAR) ---
BILLING_DAYS = [5, 20]

# ===========================
# OBJETIVO JULIO 2026
#
# - El script queda en julio completo (31 días).
# - Los targets definitivos se recalculan dinámicamente desde BQ usando los
#   últimos 2 meses cerrados previos a FORE_START (para julio: mayo + junio).
# - Este bloque hardcodeado queda solo como fallback si la lectura dinámica
#   falla o viene fuera de rango.
#
# Fallback histórico disponible originalmente (base histórica, no target de julio):
#   revenue_TC    : 95.6045 MM
#   revenue_NO_TC :  3.2622 MM
#   revenue_total : 98.8667 MM
#   gpv_TC        : 251.8754 MM
#   gpv_NO_TC     :  15.8061 MM
#   gpv_total     : 267.6815 MM
# ===========================

# Fallback base — se mantiene para contingencia, no como fuente principal de julio
_BASE_REV_TC    = 95.6045
_BASE_REV_NOTC  =  3.2622
_BASE_REV_TOTAL = 98.8667
_BASE_GPV_TC    = 251.8754
_BASE_GPV_NOTC  =  15.8061
_BASE_GPV_TOTAL = 267.6815

_BASE_REV_TC_PCT   = _BASE_REV_TC   / _BASE_REV_TOTAL
_BASE_REV_NOTC_PCT = _BASE_REV_NOTC / _BASE_REV_TOTAL
_BASE_GPV_TC_PCT   = _BASE_GPV_TC   / _BASE_GPV_TOTAL
_BASE_GPV_NOTC_PCT = _BASE_GPV_NOTC / _BASE_GPV_TOTAL

# Fallback julio: crecimiento conservador si no se logra recalcular con mayo+junio
_JUL_GROWTH     = 1.012
_JUL_REV_TOTAL  = _BASE_REV_TOTAL * _JUL_GROWTH
_JUL_GPV_TOTAL  = _BASE_GPV_TOTAL * _JUL_GROWTH

ADJUST_TARGETS = {
    "gpv":           _JUL_GPV_TOTAL,
    "gpv_TC":        _JUL_GPV_TOTAL * _BASE_GPV_TC_PCT,
    "gpv_NO_TC":     _JUL_GPV_TOTAL * _BASE_GPV_NOTC_PCT,
    "mau_NO_TC":     None,   # dinámico desde BQ
    "mau_TC":        None,   # dinámico desde BQ
    "revenue_NO_TC": _JUL_REV_TOTAL * _BASE_REV_NOTC_PCT,
    "revenue_TC":    _JUL_REV_TOTAL * _BASE_REV_TC_PCT,
    "revenue_total": _JUL_REV_TOTAL,
    "trx":           None,
}

ADJUST_ELASTICITY = {
    "mau_NO_TC": 0.06,
    "mau_TC":    0.65,
    "revenue_NO_TC": 0.80,
    "revenue_TC":    0.95
}

SOFT_TOLERANCE_PCT = {
    "mau_NO_TC": 0.00,
    "mau_TC": 0.02,
    "revenue_NO_TC": 0.00,
    "revenue_TC": 0.08,
    "revenue_total": 0.00
}
OVERSHOOT_MIN = {"revenue_total": 0.000}
OVERSHOOT_MAX = {"revenue_total": 0.000}
RELAX_TC_BILLING_IF_NO_F = True
MAX_TREND_BIAS_PCT = {
    "mau_NO_TC": 0.02,
    "mau_TC":    0.08,
    "revenue_NO_TC": 0.06,
    "revenue_TC":    0.05
}

WEIGHT_RECENCY_STRENGTH = 0.45
F_FLOORS = {"gpv": 0.48, "trx": 220.50, "mau_NO_TC": 0.0}
REV_NOTC_FLOOR_PER_F = 0.02

NOTC_SHAPE_ACTIVE = False
NOTC_SHAPE_ELAST  = 1.00
NOTC_SHAPE_TOL    = 0.05
NOTC_USE_DOW      = True

FORCE_TARGETS = True

# Split histórico billing days — se mantiene por patrón de facturación 5/20
# Día 5 : ~12.6% del revenue_TC | ~11.6% del mau_TC
# Día 20: ~87.4% del revenue_TC | ~88.4% del mau_TC
# Necesario porque el XGB penaliza sábados/domingos aunque sean días de facturación
_TC_REV_DAY5_PCT  = 0.126
_TC_REV_DAY20_PCT = 0.874
_TC_MAU_DAY5_PCT  = 0.116
_TC_MAU_DAY20_PCT = 0.884

# ---------- Auth ----------
scopes = [
    "https://www.googleapis.com/auth/bigquery",
    "https://www.googleapis.com/auth/cloud-platform"
]
credentials = pydata_google_auth.get_user_credentials(scopes)

# ---------- IO helpers ----------
def ensure_dataset(project_id, dataset_id, credentials):
    client = bigquery.Client(project=project_id, credentials=credentials)
    try:
        ds = client.get_dataset(f"{project_id}.{dataset_id}")
    except NotFound as e:
        raise RuntimeError(f"No existe dataset {project_id}.{dataset_id}.") from e
    return client, ds.location

def get_dataset_location_safe(project_id, dataset_id, credentials):
    try:
        client = bigquery.Client(project=project_id, credentials=credentials)
        ds = client.get_dataset(f"{project_id}.{dataset_id}")
        return ds.location
    except Exception:
        return None

def read_gbq_robusto(sql, project_id, credentials,
                     primary_location: str | None = None,
                     fallback_location: str | None = None):
    if primary_location:
        try:
            return pandas_gbq.read_gbq(
                sql, project_id=project_id,
                credentials=credentials,
                location=primary_location
            )
        except Exception:
            pass
    try:
        return pandas_gbq.read_gbq(
            sql, project_id=project_id,
            credentials=credentials
        )
    except Exception:
        pass
    if fallback_location:
        return pandas_gbq.read_gbq(
            sql, project_id=project_id,
            credentials=credentials,
            location=fallback_location
        )
    return pandas_gbq.read_gbq(sql, project_id=project_id, credentials=credentials)

def to_bq(df: pd.DataFrame, table_fqtn: str, schema: list):
    if df.empty:
        print(f"[WARN] vacío → no se escribe {table_fqtn}.")
        return
    client, location = ensure_dataset(PROJECT_ID, DATASET_ID, credentials)
    if "fecha" in df.columns:
        df = df.copy()
        df["fecha"] = pd.to_datetime(df["fecha"]).dt.date
    dataset_table = table_fqtn.split(f"{PROJECT_ID}.")[-1]
    pandas_gbq.to_gbq(
        df,
        destination_table=dataset_table,
        project_id=PROJECT_ID,
        if_exists="replace",
        location=location,
        table_schema=schema
    )
    print(f"[OK] escrito {table_fqtn}: {len(df):,} filas (location={location})")

def _coerce_numeric(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    obj_cols = out.select_dtypes(include=["object", "string"]).columns
    for col in obj_cols:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    return out

# ---------- Features & model ----------
def _make_calendar_features(idx, is_tc):
    idx = pd.DatetimeIndex(pd.to_datetime(idx).normalize())
    df = pd.DataFrame(index=idx)
    df["dow"] = idx.dayofweek
    df["dom"] = idx.day
    df["month"] = idx.month
    df["wom"] = ((idx.day - 1)//7)+1
    df["is_bom"] = (idx.day == 1).astype(int)
    df["is_eom"] = (idx.is_month_end).astype(int)
    df["is_bill_5_20"] = df["dom"].isin(BILLING_DAYS).astype(int) if is_tc else 0
    df = pd.concat([df, pd.get_dummies(df["dow"], prefix="dow", drop_first=False)], axis=1)
    return df

def _add_lag_roll(y):
    df = pd.DataFrame({"y": y})
    for L in [1, 7, 14, 28]:
        df[f"lag{L}"] = y.shift(L)
    for W in [7, 14, 28]:
        df[f"rm{W}"] = y.rolling(W, min_periods=max(1, W//2)).mean()
    df["d_lag7"]  = df["y"] - df["lag7"]
    df["d_lag14"] = df["y"] - df["lag14"]
    return df

def _winsorize(s, p_high=0.99):
    if s.dropna().empty:
        return s
    return s.clip(upper=float(s.quantile(p_high)))

def _clamp(prev, nxt, up_cap, down_cap):
    if np.isnan(nxt):
        nxt = prev
    delta = nxt - prev
    if delta > 0:
        delta = min(delta, prev * up_cap)
    else:
        delta = max(delta, -prev * down_cap)
    return max(0.0, prev + delta)

def _bounds(x, med14):
    if np.isnan(med14) or med14 <= 0:
        return max(0.0, x)
    return float(np.clip(x, med14 * HARD_DOWN_MULT, med14 * HARD_UP_MULT))

def _decay_weights(index, cutoff, hl=45):
    age = (cutoff - pd.DatetimeIndex(index)).days.astype(float)
    age = np.maximum(age, 0.0)
    return np.power(0.5, age / hl)

def _is_tc(name: str) -> bool:
    n = name.lower()
    return ("_tc" in n) or ("revenue_tc" in n) or ("gpv_tc" in n)

def xgb_forecast_series(s_hist, fore_idx, up_cap, down_cap, ident_name, xgb_params=None):
    s_hist = s_hist.copy()
    s_hist.index = pd.DatetimeIndex(s_hist.index).normalize()
    s_hist = s_hist.astype(float)
    fore_idx = pd.DatetimeIndex(pd.to_datetime(fore_idx)).normalize()
    if fore_idx.size == 0 or s_hist.dropna().empty:
        return pd.Series(index=fore_idx, dtype=float)

    train_start = max(s_hist.index.min(), D_MINUS_1 - pd.Timedelta(days=TRAIN_WINDOW_DAYS))
    s_train = s_hist.loc[(s_hist.index >= train_start) & (s_hist.index <= D_MINUS_1)].dropna()
    if s_train.size < 12:
        last_val = s_hist.loc[s_hist.index <= D_MINUS_1].dropna().iloc[-1] if s_hist.dropna().size else 0.0
        out = pd.Series(index=fore_idx, dtype=float)
        prev = float(last_val)
        med14 = s_hist.loc[s_hist.index <= D_MINUS_1].tail(14).median()
        for d in fore_idx:
            prev = _bounds(_clamp(prev, prev, up_cap, down_cap), med14)
            out.loc[d] = prev
        return out

    full_idx = pd.date_range(min(train_start, fore_idx.min()), fore_idx.max(), freq="D").normalize()
    is_tc = _is_tc(ident_name)
    cal = _make_calendar_features(full_idx, is_tc)
    tmp = pd.DataFrame(index=full_idx)
    tmp["y"] = s_hist.reindex(full_idx)
    feats = _add_lag_roll(tmp["y"])
    X = pd.concat([cal, feats.drop(columns=["y"])], axis=1)

    y_train = tmp.loc[(tmp.index >= train_start) & (tmp.index <= D_MINUS_1), "y"].astype(float).clip(lower=0.0)
    q99 = y_train.quantile(0.99)
    if pd.notna(q99):
        y_train = y_train.clip(upper=float(q99))
    mask = y_train.notna() & np.isfinite(y_train.values)
    X_train = X.loc[y_train.index[mask]]
    y_train = y_train.loc[mask]
    w = _decay_weights(y_train.index, cutoff=D_MINUS_1, hl=45)

    if xgb_params is None:
        xgb_params = dict(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=5,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_alpha=0.0,
            reg_lambda=1.2,
            random_state=2025,
            objective="reg:squarederror"
        )
    model = XGBRegressor(**xgb_params)
    model.fit(X_train.fillna(0.0), np.log1p(y_train.values), sample_weight=w)

    out = pd.Series(index=fore_idx, dtype=float)
    last_real = float(s_hist.loc[s_hist.index <= D_MINUS_1].dropna().iloc[-1])
    prev = last_real
    rm7 = s_hist.loc[s_hist.index <= D_MINUS_1].tail(7).mean()
    med14 = s_hist.loc[s_hist.index <= D_MINUS_1].tail(14).median()
    alpha = ALPHA_ANCHOR_VOL if is_tc else ALPHA_ANCHOR_STD

    y_sim = tmp["y"].copy()
    for d in fore_idx:
        d0 = pd.Timestamp(d).normalize()
        feats_d = _add_lag_roll(y_sim.reindex(pd.DatetimeIndex(X.index))).loc[d0]
        x_raw = pd.concat([cal.loc[d0], feats_d.drop(labels=["y"])], axis=0).to_frame().T
        x_d = _coerce_numeric(x_raw).fillna(0.0)

        raw = prev if ((med14 if not np.isnan(med14) else 0.0) < VERY_LOW_LEVEL) else float(np.expm1(model.predict(x_d)[0]))
        base = rm7 if not np.isnan(rm7) else prev
        blended = alpha * base + (1 - alpha) * raw
        val = _bounds(_clamp(prev, blended, up_cap, down_cap), med14)
        out.loc[d0] = val
        prev = val
        y_sim.loc[d0] = val
        last6 = s_hist.loc[s_hist.index <= D_MINUS_1].tail(6).values
        rm7 = pd.Series(list(last6) + [val]).mean()
    return out

# ---------- SQL MAU/Revenue/GPV ----------
MAU_SQL = f"""
WITH raw AS (
  SELECT
    DATE(fecha) AS fecha,
    LOWER(IFNULL(nombre, "nulo")) AS nombre,
    CAST(user AS STRING) AS user_id,
    SUM(monto)/1e6 AS gpv,
    COUNT(trx_id)     AS trx,
    SUM(revenue)/1e6  AS revenue
  FROM `{PROJECT_ID}.economics.economics`
  WHERE linea = "insurance"
    AND fecha >= DATE("{START_DATE.date()}")
    AND fecha <= DATE("{FORE_END.date()}")
  GROUP BY 1,2,3
),
map AS (
  SELECT
    fecha,
    CASE
      WHEN nombre LIKE "%asistencia familiar%"               THEN "insurance_multiasistencia"
      WHEN nombre LIKE "%auto contenido%"                    THEN "insurance_autocontenido"
      WHEN nombre LIKE "%fallecimiento y telemedicina%"      THEN "insurance_vida"
      WHEN nombre LIKE "%soap%"                              THEN "insurance_soap"
      WHEN nombre LIKE "%saldo deudor%"                      THEN "insurance_desgravamen"
      WHEN nombre LIKE "%cesantia%"                          THEN "insurance_cesantia"
      WHEN nombre LIKE "%incapacidad temporal%"              THEN "insurance_cesantia"
      WHEN nombre LIKE "%telemedicina%"                      THEN "insurance_telemedicina"
      ELSE "insurance_otro"
    END AS sublinea,
    user_id,
    gpv,
    trx,
    revenue
  FROM raw
),
mau_no_tc AS (
  SELECT fecha, COUNT(DISTINCT user_id) AS mau_no_tc
  FROM map
  WHERE sublinea LIKE "insurance_%"
    AND sublinea NOT IN ("insurance_desgravamen","insurance_cesantia")
  GROUP BY 1
),
mau_tc AS (
  SELECT fecha, COUNT(DISTINCT user_id) AS mau_tc
  FROM map
  WHERE sublinea IN ("insurance_desgravamen","insurance_cesantia")
  GROUP BY 1
),
rev_no_tc AS (
  SELECT fecha, SUM(revenue) AS revenue_no_tc
  FROM map
  WHERE sublinea LIKE "insurance_%"
    AND sublinea NOT IN ("insurance_desgravamen","insurance_cesantia")
  GROUP BY 1
),
rev_tc AS (
  SELECT fecha, SUM(revenue) AS revenue_tc
  FROM map
  WHERE sublinea IN ("insurance_desgravamen","insurance_cesantia")
  GROUP BY 1
),
gpv_no_tc AS (
  SELECT fecha, SUM(gpv) AS gpv_no_tc
  FROM map
  WHERE sublinea LIKE "insurance_%"
    AND sublinea NOT IN ("insurance_desgravamen","insurance_cesantia")
  GROUP BY 1
),
gpv_tc AS (
  SELECT fecha, SUM(gpv) AS gpv_tc
  FROM map
  WHERE sublinea IN ("insurance_desgravamen","insurance_cesantia")
  GROUP BY 1
),
tot_trx AS (
  SELECT fecha, SUM(trx) AS trx
  FROM raw
  GROUP BY 1
),
mau_rows AS (
  SELECT mnt.fecha, "mau_NO_TC" AS identificador, CAST(mnt.mau_no_tc AS FLOAT64) AS valor FROM mau_no_tc AS mnt
  UNION ALL
  SELECT mt.fecha, "mau_TC" AS identificador, CAST(mt.mau_tc AS FLOAT64) AS valor FROM mau_tc AS mt
),
rev_rows AS (
  SELECT rnt.fecha, "revenue_NO_TC" AS identificador, CAST(rnt.revenue_no_tc AS FLOAT64) AS valor FROM rev_no_tc AS rnt
  UNION ALL
  SELECT rtc.fecha, "revenue_TC"    AS identificador, CAST(rtc.revenue_tc    AS FLOAT64) AS valor FROM rev_tc AS rtc
),
gpv_rows AS (
  SELECT gnt.fecha, "gpv_NO_TC" AS identificador, CAST(gnt.gpv_no_tc AS FLOAT64) AS valor FROM gpv_no_tc AS gnt
  UNION ALL
  SELECT gtc.fecha, "gpv_TC"    AS identificador, CAST(gtc.gpv_tc    AS FLOAT64) AS valor FROM gpv_tc  AS gtc
),
trx_rows AS (
  SELECT t.fecha, "trx" AS identificador, CAST(t.trx AS FLOAT64) AS valor FROM tot_trx AS t
)
SELECT * FROM mau_rows
UNION ALL SELECT * FROM rev_rows
UNION ALL SELECT * FROM gpv_rows
UNION ALL SELECT * FROM trx_rows
ORDER BY 1,2
"""

# ---------- SQL VALIDACIÓN MAU_TC ----------
VALIDATE_MAU_TC_SQL = f"""
WITH raw AS (
  SELECT DATE(fecha) AS fecha, LOWER(IFNULL(nombre,"nulo")) AS nombre, CAST(user AS STRING) AS user_id
  FROM `{PROJECT_ID}.economics.economics`
  WHERE linea = "insurance"
    AND fecha >= DATE("{FORE_START.date()}")
    AND fecha <= DATE("{D_MINUS_1.date()}")
),
map AS (
  SELECT fecha,
         CASE
           WHEN nombre LIKE "%saldo deudor%"         THEN "insurance_desgravamen"
           WHEN nombre LIKE "%cesantia%"             THEN "insurance_cesantia"
           WHEN nombre LIKE "%incapacidad temporal%" THEN "insurance_cesantia"
           ELSE "other"
         END AS sublinea,
         user_id
  FROM raw
)
SELECT fecha, CAST(COUNT(DISTINCT user_id) AS FLOAT64) AS mau_tc_real
FROM map
WHERE sublinea IN ("insurance_desgravamen","insurance_cesantia")
GROUP BY 1
ORDER BY 1
"""

# ---------- STOCK ----------
STOCK_SQL = f"""
WITH base AS (
  SELECT DATE(fecha) AS fecha,
         LOWER(CAST(category_type AS STRING)) AS cat,
         CAST(payment_method_recod AS STRING) AS payment_method_recod,
         SUM(stockTotal) AS stock_total
  FROM `{PROJECT_ID}.insurance.stock`
  WHERE fecha >= DATE("{START_DATE.date()}") AND fecha <= DATE("{FORE_END.date()}")
  GROUP BY 1,2,3
),
tag AS (
  SELECT fecha,
         CASE WHEN cat LIKE "%cesantia%" OR cat LIKE "%desgravamen%" OR cat LIKE "%saldo deudor%" THEN "TC" ELSE "NO_TC" END AS category_type,
         payment_method_recod,
         SUM(stock_total) AS stock_total
  FROM base
  GROUP BY 1,2,3
)
SELECT fecha, category_type, payment_method_recod, stock_total FROM tag ORDER BY 1,2,3
"""

# ---------- Targets dinámicos últimos 2 meses ----------
# Para julio toma mayo + junio como referencia de momentum y proyecta julio.
def compute_dynamic_targets_last2m():
    global ADJUST_TARGETS

    ref_end   = FORE_START - pd.Timedelta(days=1)       # para julio: 2026-06-30
    ref_start = (FORE_START - MonthBegin(2)).normalize() # para julio: 2026-05-01

    ECON_LOC = get_dataset_location_safe(PROJECT_ID, "economics", credentials) or "US"

    dyn_sql = f"""
    WITH raw AS (
      SELECT
        DATE(fecha) AS fecha,
        LOWER(IFNULL(nombre, "nulo")) AS nombre,
        CAST(user AS STRING) AS user_id,
        SUM(monto)/1e6 AS gpv,
        COUNT(trx_id)     AS trx,
        SUM(revenue)/1e6  AS revenue
      FROM `{PROJECT_ID}.economics.economics`
      WHERE linea = "insurance"
        AND fecha >= DATE("{ref_start.date()}")
        AND fecha <= DATE("{ref_end.date()}")
      GROUP BY 1,2,3
    ),
    map AS (
      SELECT
        fecha,
        CASE
          WHEN nombre LIKE "%asistencia familiar%"               THEN "insurance_multiasistencia"
          WHEN nombre LIKE "%auto contenido%"                    THEN "insurance_autocontenido"
          WHEN nombre LIKE "%fallecimiento y telemedicina%"      THEN "insurance_vida"
          WHEN nombre LIKE "%soap%"                              THEN "insurance_soap"
          WHEN nombre LIKE "%saldo deudor%"                      THEN "insurance_desgravamen"
          WHEN nombre LIKE "%cesantia%"                          THEN "insurance_cesantia"
          WHEN nombre LIKE "%incapacidad temporal%"              THEN "insurance_cesantia"
          WHEN nombre LIKE "%telemedicina%"                      THEN "insurance_telemedicina"
          ELSE "insurance_otro"
        END AS sublinea,
        user_id,
        gpv,
        trx,
        revenue
      FROM raw
    ),
    mau_no_tc AS (
      SELECT fecha, COUNT(DISTINCT user_id) AS mau_no_tc
      FROM map
      WHERE sublinea LIKE "insurance_%"
        AND sublinea NOT IN ("insurance_desgravamen","insurance_cesantia")
      GROUP BY 1
    ),
    mau_tc AS (
      SELECT fecha, COUNT(DISTINCT user_id) AS mau_tc
      FROM map
      WHERE sublinea IN ("insurance_desgravamen","insurance_cesantia")
      GROUP BY 1
    ),
    rev_no_tc AS (
      SELECT fecha, SUM(revenue) AS revenue_no_tc
      FROM map
      WHERE sublinea LIKE "insurance_%"
        AND sublinea NOT IN ("insurance_desgravamen","insurance_cesantia")
      GROUP BY 1
    ),
    rev_tc AS (
      SELECT fecha, SUM(revenue) AS revenue_tc
      FROM map
      WHERE sublinea IN ("insurance_desgravamen","insurance_cesantia")
      GROUP BY 1
    ),
    gpv_no_tc AS (
      SELECT fecha, SUM(gpv) AS gpv_no_tc
      FROM map
      WHERE sublinea LIKE "insurance_%"
        AND sublinea NOT IN ("insurance_desgravamen","insurance_cesantia")
      GROUP BY 1
    ),
    gpv_tc AS (
      SELECT fecha, SUM(gpv) AS gpv_tc
      FROM map
      WHERE sublinea IN ("insurance_desgravamen","insurance_cesantia")
      GROUP BY 1
    ),
    mau_rows AS (
      SELECT mnt.fecha, "mau_NO_TC" AS identificador, CAST(mnt.mau_no_tc AS FLOAT64) AS valor
      FROM mau_no_tc AS mnt
      UNION ALL
      SELECT mt.fecha, "mau_TC" AS identificador, CAST(mt.mau_tc AS FLOAT64) AS valor
      FROM mau_tc AS mt
    ),
    rev_rows AS (
      SELECT rnt.fecha, "revenue_NO_TC" AS identificador, CAST(rnt.revenue_no_tc AS FLOAT64) AS valor
      FROM rev_no_tc AS rnt
      UNION ALL
      SELECT rtc.fecha, "revenue_TC" AS identificador, CAST(rtc.revenue_tc AS FLOAT64) AS valor
      FROM rev_tc AS rtc
    ),
    gpv_rows AS (
      SELECT gnt.fecha, "gpv_NO_TC" AS identificador, CAST(gnt.gpv_no_tc AS FLOAT64) AS valor
      FROM gpv_no_tc AS gnt
      UNION ALL
      SELECT gtc.fecha, "gpv_TC" AS identificador, CAST(gtc.gpv_tc AS FLOAT64) AS valor
      FROM gpv_tc AS gtc
    )
    SELECT * FROM mau_rows
    UNION ALL SELECT * FROM rev_rows
    UNION ALL SELECT * FROM gpv_rows
    ORDER BY 1,2
    """

    df_ref = read_gbq_robusto(
        dyn_sql,
        PROJECT_ID,
        credentials,
        primary_location=ECON_LOC,
        fallback_location=None
    )

    if df_ref.empty:
        print(f"[WARN] Targets dinámicos: sin datos entre {ref_start.date()} y {ref_end.date()}. Se usa fallback.")
        return

    df_ref["fecha"] = pd.to_datetime(df_ref["fecha"]).dt.normalize()
    df_ref["mes"]   = df_ref["fecha"].dt.to_period("M").dt.to_timestamp()

    monthly = (df_ref
               .groupby(["mes", "identificador"], as_index=False)["valor"]
               .sum())

    # Boost de crecimiento natural mensual ~2-3%
    TREND_BOOST = 0.85

    def _monthly_series(ident):
        return (monthly[monthly["identificador"] == ident]
                .set_index("mes")["valor"]
                .sort_index())

    def _trend_target_last2(ident):
        s = _monthly_series(ident)
        if s.empty:
            return None
        last = float(s.iloc[-1])
        if len(s) == 1:
            return last * 1.02   # +2% si solo hay un mes
        prev = float(s.iloc[-2])
        delta = last - prev
        # Aplicar momentum positivo; si es negativo se amortigua
        if delta >= 0:
            projected = last + delta * TREND_BOOST
        else:
            projected = last + delta * 0.30   # amortigua caídas al 30%
        return float(max(projected, last * 0.97))   # floor: no bajar más de 3%

    def _last_closed_value(ident):
        s = _monthly_series(ident)
        return float(s.iloc[-1]) if not s.empty else None

    mau_no_tc = _trend_target_last2("mau_NO_TC")
    mau_tc    = _trend_target_last2("mau_TC")
    rev_no_tc = _trend_target_last2("revenue_NO_TC")
    rev_tc    = _trend_target_last2("revenue_TC")
    gpv_notc  = _trend_target_last2("gpv_NO_TC")
    gpv_tc    = _trend_target_last2("gpv_TC")

    # Referencias reales del último mes cerrado para validación y proporciones.
    last_rev_tc   = _last_closed_value("revenue_TC")
    last_rev_ntc  = _last_closed_value("revenue_NO_TC")
    last_gpv_tc   = _last_closed_value("gpv_TC")
    last_gpv_ntc  = _last_closed_value("gpv_NO_TC")

    last_rev_total = ((last_rev_tc or 0.0) + (last_rev_ntc or 0.0)) if (last_rev_tc is not None or last_rev_ntc is not None) else _BASE_REV_TOTAL
    last_gpv_total = ((last_gpv_tc or 0.0) + (last_gpv_ntc or 0.0)) if (last_gpv_tc is not None or last_gpv_ntc is not None) else _BASE_GPV_TOTAL

    # REVENUE: usar targets dinámicos de julio si están cerca del último mes cerrado.
    if rev_tc is not None and rev_no_tc is not None:
        rev_total_dyn = rev_tc + rev_no_tc
        ref_total = last_rev_total if last_rev_total > 0 else _BASE_REV_TOTAL
        if abs(rev_total_dyn - ref_total) / ref_total < 0.25:
            ADJUST_TARGETS["revenue_TC"]    = rev_tc
            ADJUST_TARGETS["revenue_NO_TC"] = rev_no_tc
            ADJUST_TARGETS["revenue_total"] = rev_total_dyn
        else:
            print(f"[WARN] Momentum revenue BQ={rev_total_dyn:.2f} vs último mes cerrado={ref_total:.2f} "
                  f"— fuera de ±25%. Se mantiene fallback.")

    # MAU: validar que los valores son estructuralmente plausibles.
    # TC se concentra en días 5/20; NO_TC debe tener base diaria mínima.
    if mau_tc is not None and mau_tc > 80000:
        ADJUST_TARGETS["mau_TC"] = mau_tc
    else:
        print(f"[WARN] mau_TC BQ={mau_tc} inusualmente bajo — XGB decide libremente.")
    if mau_no_tc is not None and mau_no_tc > 1000:
        ADJUST_TARGETS["mau_NO_TC"] = mau_no_tc

    # GPV: usar targets dinámicos directos por TC/NO_TC para julio.
    if gpv_tc is not None and gpv_notc is not None:
        gpv_total_dyn = gpv_tc + gpv_notc
        ref_total = last_gpv_total if last_gpv_total > 0 else _BASE_GPV_TOTAL
        if abs(gpv_total_dyn - ref_total) / ref_total < 0.30:
            ADJUST_TARGETS["gpv"]       = gpv_total_dyn
            ADJUST_TARGETS["gpv_TC"]    = gpv_tc
            ADJUST_TARGETS["gpv_NO_TC"] = gpv_notc
        else:
            print(f"[WARN] GPV BQ={gpv_total_dyn:.1f} vs último mes cerrado={ref_total:.1f} — mantiene fallback.")

    print(f"\n[INFO] Targets {FORE_START:%Y-%m} definitivos (ref BQ últimos 2 meses: {ref_start.date()} → {ref_end.date()}):")
    for k in ["mau_NO_TC","mau_TC","revenue_NO_TC","revenue_TC","revenue_total","gpv","gpv_TC","gpv_NO_TC"]:
        v = ADJUST_TARGETS.get(k)
        if v is not None:
            print(f"  {k:14s}: {v:,.3f}")
    rev_tc_v  = ADJUST_TARGETS.get("revenue_TC")  or 0.0
    rev_tot_v = ADJUST_TARGETS.get("revenue_total") or 1.0
    print(f"\n  [CHECK] revenue TC  / total : {rev_tc_v/rev_tot_v*100:.1f}%")
    print(f"  [CHECK] revenue NTC / total : {(rev_tot_v-rev_tc_v)/rev_tot_v*100:.1f}%")


# ---------- Build R/F ----------
def run_mau_xgb():
    ECONOMICS_LOC = get_dataset_location_safe(PROJECT_ID, "economics", credentials) or "US"
    df = read_gbq_robusto(MAU_SQL, PROJECT_ID, credentials, primary_location=ECONOMICS_LOC, fallback_location=None)
    if df.empty:
        cols = ["fecha","dia","identificador","valor_real","valor_pronosticado","estado","valor_total","valor_acum_mtd"]
        return pd.DataFrame(columns=cols)
    df["fecha"] = pd.to_datetime(df["fecha"]).dt.normalize()
    df["valor"] = pd.to_numeric(df["valor"], errors="coerce").astype(float)
    df_mes = df[(df["fecha"] >= FORE_START) & (df["fecha"] <= FORE_END)].copy()
    if df_mes.empty:
        cols = ["fecha","dia","identificador","valor_real","valor_pronosticado","estado","valor_total","valor_acum_mtd"]
        return pd.DataFrame(columns=cols)

    df_r = df_mes[df_mes["fecha"] <= D_MINUS_1.normalize()].copy()
    fore_start = max(FORE_START, D_MINUS_1.normalize() + pd.Timedelta(days=1))
    fore_idx = pd.date_range(fore_start.normalize(), FORE_END.normalize(), freq="D").normalize()

    preds = []
    for uid, g_all in df.groupby("identificador", sort=True):
        g_all = g_all.copy()
        g_all["fecha"] = pd.to_datetime(g_all["fecha"]).dt.normalize()
        s_all = g_all.drop_duplicates("fecha").set_index("fecha").sort_index()["valor"]
        s_hist = s_all.loc[s_all.index <= D_MINUS_1.normalize()]
        if fore_idx.size > 0:
            yhat = xgb_forecast_series(s_hist, fore_idx, CAP_UP_ECO, CAP_DOWN_ECO, uid)
        else:
            yhat = pd.Series(index=fore_idx, dtype=float)
        if not yhat.empty:
            preds.append(pd.DataFrame({
                "fecha": yhat.index,
                "identificador": uid,
                "valor_pronosticado": yhat.values
            }))
    df_f = pd.concat(preds, ignore_index=True) if preds else pd.DataFrame(columns=["fecha","identificador","valor_pronosticado"])

    out = (df_r.rename(columns={"valor": "valor_real"})
             .merge(df_f, on=["fecha","identificador"], how="outer"))
    out["estado"] = np.where(out["fecha"] <= D_MINUS_1.normalize(), "R", "F")
    out["valor_total"] = np.where(out["estado"].eq("R"),
                                  out["valor_real"].astype(float),
                                  out["valor_pronosticado"].astype(float))
    out = out.sort_values(["identificador","fecha"]).reset_index(drop=True)
    out["valor_acum_mtd"] = out.groupby("identificador")["valor_total"].cumsum()
    out["dia"] = pd.to_datetime(out["fecha"]).dt.day.astype(int)
    return out[["fecha","dia","identificador","valor_real","valor_pronosticado","estado","valor_total","valor_acum_mtd"]]

def run_stock_xgb():
    INSURANCE_LOC = get_dataset_location_safe(PROJECT_ID, "insurance", credentials) or LOCATION_FALLBACK
    df = read_gbq_robusto(STOCK_SQL, PROJECT_ID, credentials, primary_location=INSURANCE_LOC, fallback_location=None)
    if df.empty:
        cols = ["fecha","dia","category_type","payment_method_recod","stock_real","stock_pronosticado","estado","stock_total","stock_acum_mtd"]
        return pd.DataFrame(columns=cols)
    df["fecha"] = pd.to_datetime(df["fecha"]).dt.normalize()
    df["stock_total"] = pd.to_numeric(df["stock_total"], errors="coerce").astype(float)
    df_mes = df[(df["fecha"] >= FORE_START) & (df["fecha"] <= FORE_END)].copy()
    if df_mes.empty:
        cols = ["fecha","dia","category_type","payment_method_recod","stock_real","stock_pronosticado","estado","stock_total","stock_acum_mtd"]
        return pd.DataFrame(columns=cols)
    df_r = df_mes[df_mes["fecha"] <= D_MINUS_1.normalize()].copy()
    fore_start = max(FORE_START, D_MINUS_1.normalize() + pd.Timedelta(days=1))
    fore_idx = pd.date_range(fore_start.normalize(), FORE_END.normalize(), freq="D").normalize()

    preds = []
    keys = df[["category_type","payment_method_recod"]].drop_duplicates().itertuples(index=False, name=None)
    for cat, pm in sorted(keys):
        part = df[(df["category_type"] == cat) & (df["payment_method_recod"] == pm)][["fecha","stock_total"]].copy()
        s_all = part.drop_duplicates("fecha").set_index("fecha").sort_index()["stock_total"]
        s_hist = s_all.loc[s_all.index <= D_MINUS_1.normalize()]
        if fore_idx.size > 0:
            yhat = xgb_forecast_series(s_hist, fore_idx, CAP_UP_STK, CAP_DOWN_STK, f"{cat}|{pm}")
        else:
            yhat = pd.Series(index=fore_idx, dtype=float)
        if not yhat.empty:
            preds.append(pd.DataFrame({
                "fecha": yhat.index,
                "category_type": cat,
                "payment_method_recod": pm,
                "stock_pronosticado": yhat.values
            }))
    df_f = pd.concat(preds, ignore_index=True) if preds else pd.DataFrame(columns=["fecha","category_type","payment_method_recod","stock_pronosticado"])

    out = (df_r.rename(columns={"stock_total": "stock_real"})
             .merge(df_f, on=["fecha","category_type","payment_method_recod"], how="outer"))
    out["estado"] = np.where(out["fecha"] <= D_MINUS_1.normalize(), "R", "F")
    out["stock_total"] = np.where(out["estado"].eq("R"),
                                  out["stock_real"].astype(float),
                                  out["stock_pronosticado"].astype(float))
    out = out.sort_values(["category_type","payment_method_recod","fecha"]).reset_index(drop=True)
    out["stock_acum_mtd"] = out.groupby(["category_type","payment_method_recod"])["stock_total"].cumsum()
    out["dia"] = pd.to_datetime(out["fecha"]).dt.day.astype(int)
    return out[["fecha","dia","category_type","payment_method_recod","stock_real","stock_pronosticado","estado","stock_total","stock_acum_mtd"]]

# ---------- Reglas base ----------
def enforce_tc_billing(df):
    df = df.copy()
    df["fecha"] = pd.to_datetime(df["fecha"]).dt.normalize()
    # V11: el dia 1 (sin reales del mes) estas columnas llegan como object -> cumsum fallaba
    for _c in ["valor_real", "valor_pronosticado", "valor_total"]:
        if _c in df.columns:
            df[_c] = pd.to_numeric(df[_c], errors="coerce")
    dom = df["fecha"].dt.day
    is_tc = df["identificador"].isin(["mau_TC","revenue_TC"])
    is_F  = df["estado"].eq("F")
    not_bill = ~dom.isin(BILLING_DAYS)
    idx0 = df.index[is_tc & is_F & not_bill]
    df.loc[idx0, ["valor_pronosticado","valor_total"]] = 0.0
    df = df.sort_values(["identificador","fecha"])
    df["valor_acum_mtd"] = df.groupby("identificador")["valor_total"].cumsum()
    return df

def set_f_floors(df):
    df = df.copy()
    df["fecha"] = pd.to_datetime(df["fecha"]).dt.normalize()
    is_F = df["estado"].eq("F")
    for ident, lvl in F_FLOORS.items():
        idx = df.index[(df["identificador"] == ident) & is_F]
        df.loc[idx, ["valor_pronosticado","valor_total"]] = np.maximum(
            df.loc[idx, ["valor_pronosticado","valor_total"]].fillna(0.0),
            float(lvl)
        )
    df = df.sort_values(["identificador","fecha"])
    df["valor_acum_mtd"] = df.groupby("identificador")["valor_total"].cumsum()
    return df

# ---------- Helpers soft ----------
def _recent_trend_bias(series: pd.Series, max_bias_pct: float) -> float:
    s = series.dropna()
    if s.size < 5:
        return 0.0
    x = np.arange(len(s))
    x = (x - x.mean()) / (x.std() + 1e-9)
    y = s.values
    beta = np.dot(x, y - y.mean()) / (np.dot(x, x) + 1e-9)
    bias = max(0.0, min(max_bias_pct, beta / (abs(y.mean()) + 1e-9)))
    return float(bias)

def _weights_for_F(df_rows: pd.DataFrame) -> np.ndarray:
    if df_rows.empty:
        return np.array([])
    vals = df_rows["valor_total"].fillna(0.0).values
    vals = np.maximum(vals, 1e-9)
    dom = pd.to_datetime(df_rows["fecha"]).dt.day.values.astype(float)
    dim = pd.to_datetime(df_rows["fecha"]).dt.days_in_month.values.astype(float)
    recency = 1.0 + WEIGHT_RECENCY_STRENGTH * (dom / dim)
    w = vals * recency
    w = w / w.sum() if w.sum() > 0 else np.full_like(w, 1.0 / len(w))
    return w

# ---------- Ajustes suaves ----------
def adjust_revenue_soft(df):
    df = df.copy()
    df["fecha"] = pd.to_datetime(df["fecha"]).dt.normalize()
    dom = df["fecha"].dt.day
    is_F = df["estado"].eq("F")

    tgt_total = ADJUST_TARGETS.get("revenue_total")
    tgt_tc    = ADJUST_TARGETS.get("revenue_TC")
    tgt_ntc   = ADJUST_TARGETS.get("revenue_NO_TC")
    if tgt_total is None:
        return df

    if tgt_tc is None and tgt_ntc is None:
        tgt_ntc = tgt_total
        tgt_tc = 0.0
    elif tgt_tc is None:
        tgt_tc = max(0.0, tgt_total - tgt_ntc)
    elif tgt_ntc is None:
        tgt_ntc = max(0.0, tgt_total - tgt_tc)

    idx_tc_F_strict = df.index[(df["identificador"]=="revenue_TC") & is_F & dom.isin(BILLING_DAYS)]
    tc_has_F_slots = (len(idx_tc_F_strict) > 0)
    idx_tc_F = df.index[(df["identificador"]=="revenue_TC") & is_F] if ((not tc_has_F_slots) and RELAX_TC_BILLING_IF_NO_F) else idx_tc_F_strict
    idx_ntc_F = df.index[(df["identificador"]=="revenue_NO_TC") & is_F]

    s_r_tc  = df[(df["identificador"]=="revenue_TC") & df["estado"].eq("R")].set_index("fecha")["valor_total"].sort_index()
    s_r_ntc = df[(df["identificador"]=="revenue_NO_TC") & df["estado"].eq("R")].set_index("fecha")["valor_total"].sort_index()
    bias_tc   = _recent_trend_bias(s_r_tc.tail(14),  MAX_TREND_BIAS_PCT.get("revenue_TC", 0.0))
    bias_ntc  = _recent_trend_bias(s_r_ntc.tail(14), MAX_TREND_BIAS_PCT.get("revenue_NO_TC", 0.0))

    cur_tc  = float(df[df["identificador"]=="revenue_TC"]["valor_total"].sum())
    cur_ntc = float(df[df["identificador"]=="revenue_NO_TC"]["valor_total"].sum())

    elast_tc  = ADJUST_ELASTICITY.get("revenue_TC", 0.95)
    elast_ntc = ADJUST_ELASTICITY.get("revenue_NO_TC", 0.80)

    step_tc  = cur_tc  + (tgt_tc  - cur_tc)  * elast_tc  * (1.0 + bias_tc)
    step_ntc = cur_ntc + (tgt_ntc - cur_ntc) * elast_ntc * (1.0 + bias_ntc)

    if len(idx_tc_F):
        need_tc = max(0.0, step_tc - cur_tc)
        if need_tc > 0:
            rows_tc_F = df.loc[idx_tc_F, ["fecha","valor_total"]].copy()
            w_tc = _weights_for_F(rows_tc_F)
            inc_tc = need_tc * w_tc
            df.loc[idx_tc_F, "valor_pronosticado"] = df.loc[idx_tc_F, "valor_pronosticado"].fillna(0.0) + inc_tc
            df.loc[idx_tc_F, "valor_total"]        = df.loc[idx_tc_F, "valor_total"].fillna(0.0) + inc_tc

    if len(idx_ntc_F):
        df.loc[idx_ntc_F, ["valor_pronosticado","valor_total"]] = np.maximum(
            df.loc[idx_ntc_F, ["valor_pronosticado","valor_total"]].fillna(0.0),
            REV_NOTC_FLOOR_PER_F
        )
        cur_ntc = float(df[df["identificador"]=="revenue_NO_TC"]["valor_total"].sum())
        need_ntc = step_ntc - cur_ntc
        rows_ntc_F = df.loc[idx_ntc_F, ["fecha","valor_total"]].copy()
        w_ntc = _weights_for_F(rows_ntc_F)
        if need_ntc < 0:
            dec = (-need_ntc) * w_ntc
            new_p = np.minimum(
                df.loc[idx_ntc_F, "valor_pronosticado"].values,
                np.maximum(df.loc[idx_ntc_F, "valor_pronosticado"].values - dec, REV_NOTC_FLOOR_PER_F)
            )
            new_t = np.minimum(
                df.loc[idx_ntc_F, "valor_total"].values,
                np.maximum(df.loc[idx_ntc_F, "valor_total"].values - dec, REV_NOTC_FLOOR_PER_F)
            )
            df.loc[idx_ntc_F, "valor_pronosticado"] = new_p
            df.loc[idx_ntc_F, "valor_total"]        = new_t
        elif need_ntc > 0:
            inc = need_ntc * w_ntc
            df.loc[idx_ntc_F, "valor_pronosticado"] += inc
            df.loc[idx_ntc_F, "valor_total"]        += inc

    df = df.sort_values(["identificador","fecha"]).reset_index(drop=True)
    df["valor_acum_mtd"] = df.groupby("identificador")["valor_total"].cumsum()
    return df

def adjust_mau_soft(df):
    df = df.copy()
    df["fecha"] = pd.to_datetime(df["fecha"]).dt.normalize()
    is_F = df["estado"].eq("F")
    for ident in ["mau_NO_TC","mau_TC"]:
        tgt = ADJUST_TARGETS.get(ident, None)
        if tgt is None:
            continue
        cur = float(df[df["identificador"]==ident]["valor_total"].sum())
        r   = float(df[(df["identificador"]==ident) & df["estado"].eq("R")]["valor_total"].sum())
        if tgt <= r + 1e-9:
            continue
        elast = ADJUST_ELASTICITY.get(ident, 0.5)
        s_r = df[(df["identificador"]==ident) & df["estado"].eq("R")].set_index("fecha")["valor_total"].sort_index()
        bias = _recent_trend_bias(s_r.tail(14), MAX_TREND_BIAS_PCT.get(ident, 0.0))
        step = cur + (tgt - cur) * elast * (1.0 + bias)

        if ident == "mau_TC":
            mask_F = (df["identificador"]==ident) & is_F & (
                df["fecha"].dt.day.isin(BILLING_DAYS) | (RELAX_TC_BILLING_IF_NO_F & (~df["fecha"].dt.day.isin(BILLING_DAYS)))
            )
        else:
            mask_F = (df["identificador"]==ident) & is_F

        idx_F = df.index[mask_F]
        if not len(idx_F):
            continue

        rows = df.loc[idx_F, ["fecha","valor_total"]].copy()
        w = _weights_for_F(rows)
        delta = step - cur
        if delta > 0:
            inc = delta * w
            df.loc[idx_F, "valor_pronosticado"] = df.loc[idx_F, "valor_pronosticado"].fillna(0.0) + inc
            df.loc[idx_F, "valor_total"]        = df.loc[idx_F, "valor_total"].fillna(0.0) + inc

        if ident=="mau_NO_TC":
            df.loc[idx_F, "valor_pronosticado"] = np.maximum(df.loc[idx_F, "valor_pronosticado"], F_FLOORS.get("mau_NO_TC", 0.0))
            df.loc[idx_F, "valor_total"]        = np.maximum(df.loc[idx_F, "valor_total"],        F_FLOORS.get("mau_NO_TC", 0.0))

    df = df.sort_values(["identificador","fecha"])
    df["valor_acum_mtd"] = df.groupby("identificador")["valor_total"].cumsum()
    return df

def shape_revenue_notc_after20(df):
    return df

# ---------- HARD CAPS ----------
def _cap_F_to_budget(df, ident, budget, per_f_floor=0.0, only_tc_520=False):
    d = df.copy()
    is_F = d["estado"].eq("F") & (d["identificador"]==ident)
    if only_tc_520:
        dmask = d["fecha"].dt.day.isin(BILLING_DAYS)
        is_F = is_F & dmask
    idx = d.index[is_F]
    if len(idx) == 0:
        return d
    f_vals = d.loc[idx, "valor_total"].fillna(0.0).values.astype(float)
    floors = np.full_like(f_vals, per_f_floor, dtype=float)
    f_vals = np.maximum(f_vals, floors)
    cur_F_sum = float(f_vals.sum())
    budget = float(max(0.0, budget))
    if cur_F_sum <= budget + 1e-12:
        d.loc[idx, "valor_pronosticado"] = f_vals
        d.loc[idx, "valor_total"]        = f_vals
        return d
    weights = f_vals / (cur_F_sum if cur_F_sum > 0 else 1.0)
    new_f = floors + max(0.0, budget - floors.sum()) * np.maximum(weights, 0.0)
    if budget < floors.sum() - 1e-12:
        new_f = floors
    d.loc[idx, "valor_pronosticado"] = new_f
    d.loc[idx, "valor_total"]        = new_f
    return d

def reconcile_hard_caps(df):
    d = df.copy()
    d["fecha"] = pd.to_datetime(d["fecha"]).dt.normalize()
    d = d.sort_values(["identificador","fecha"]).reset_index(drop=True)
    d["valor_acum_mtd"] = d.groupby("identificador")["valor_total"].cumsum()
    return d

# ---------- FORZADO ----------
def _distribute_budget(values: np.ndarray, budget: float) -> np.ndarray:
    n = len(values)
    if n == 0:
        return values
    base = np.asarray(values, dtype=float)
    base = np.nan_to_num(base, nan=0.0, posinf=0.0, neginf=0.0)
    total = base.sum()
    if total <= 1e-12:
        return np.full(n, budget / n if n > 0 else 0.0, dtype=float)
    w = base / total
    return budget * w

def force_projected_to_month_targets(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    d = df.copy()
    d["fecha"] = pd.to_datetime(d["fecha"]).dt.normalize()

    for ident in ["mau_NO_TC","mau_TC","revenue_NO_TC","revenue_TC","gpv_TC","gpv_NO_TC"]:
        tgt = ADJUST_TARGETS.get(ident, None)
        if tgt is None:
            continue

        mask_ident = d["identificador"].eq(ident)
        cur_R  = float(d[mask_ident & d["estado"].eq("R")]["valor_total"].fillna(0.0).sum())
        idx_F  = d.index[mask_ident & d["estado"].eq("F")]

        if len(idx_F) == 0:
            continue

        budget_F = max(0.0, float(tgt) - cur_R)

        # GPV: distribuir proporcionalmente entre todos los días F
        if ident.startswith("gpv"):
            idx_F    = d.index[mask_ident & d["estado"].eq("F")]
            base_vals = d.loc[idx_F, "valor_total"].fillna(0.0).values.astype(float)
            new_vals  = _distribute_budget(base_vals, budget_F)
            d.loc[idx_F, "valor_pronosticado"] = new_vals
            d.loc[idx_F, "valor_total"]        = new_vals
            continue

        # revenue_TC y mau_TC: split fijo día5/día20 basado en histórico
        # Evita que el XGB distorsione la asignación cuando 5 o 20 cae en fin de semana
        if ident in ("revenue_TC", "mau_TC"):
            pct_d5  = _TC_REV_DAY5_PCT  if ident == "revenue_TC" else _TC_MAU_DAY5_PCT
            pct_d20 = _TC_REV_DAY20_PCT if ident == "revenue_TC" else _TC_MAU_DAY20_PCT

            idx_d5  = d.index[mask_ident & d["estado"].eq("F") & d["fecha"].dt.day.eq(5)]
            idx_d20 = d.index[mask_ident & d["estado"].eq("F") & d["fecha"].dt.day.eq(20)]

            if len(idx_d5) > 0 and len(idx_d20) > 0:
                # Ambos días pendientes: aplicar split histórico
                v_d5  = budget_F * pct_d5
                v_d20 = budget_F * pct_d20
                d.loc[idx_d5,  ["valor_pronosticado","valor_total"]] = v_d5
                d.loc[idx_d20, ["valor_pronosticado","valor_total"]] = v_d20
                print(f"  [SPLIT] {ident}: día5={v_d5:.2f}  día20={v_d20:.2f}  "
                      f"({pct_d5*100:.1f}%/{pct_d20*100:.1f}%)")
                continue
            elif len(idx_d20) > 0:
                # Solo queda día 20
                d.loc[idx_d20, ["valor_pronosticado","valor_total"]] = budget_F
                print(f"  [SPLIT] {ident}: solo día20={budget_F:.2f}")
                continue
            elif len(idx_d5) > 0:
                # Solo queda día 5
                d.loc[idx_d5, ["valor_pronosticado","valor_total"]] = budget_F
                print(f"  [SPLIT] {ident}: solo día5={budget_F:.2f}")
                continue

        # mau_NO_TC y revenue_NO_TC: distribución proporcional entre todos los días F
        idx_F_520 = d.index[mask_ident & d["estado"].eq("F")]
        if len(idx_F_520) == 0:
            continue
        base_vals = d.loc[idx_F_520, "valor_total"].fillna(0.0).values.astype(float)
        new_vals  = _distribute_budget(base_vals, budget_F)
        d.loc[idx_F_520, "valor_pronosticado"] = new_vals
        d.loc[idx_F_520, "valor_total"]        = new_vals

    d = d.sort_values(["identificador","fecha"]).reset_index(drop=True)
    d["valor_acum_mtd"] = d.groupby("identificador")["valor_total"].cumsum()
    return d

# ---------- Esquemas / Tablas ----------
SCHEMA_MAU = [
    {"name":"fecha","type":"DATE"},
    {"name":"dia","type":"INTEGER"},
    {"name":"identificador","type":"STRING"},
    {"name":"valor_real","type":"FLOAT"},
    {"name":"valor_pronosticado","type":"FLOAT"},
    {"name":"estado","type":"STRING"},
    {"name":"valor_total","type":"FLOAT"},
    {"name":"valor_acum_mtd","type":"FLOAT"},
]
SCHEMA_STOCK = [
    {"name":"fecha","type":"DATE"},
    {"name":"dia","type":"INTEGER"},
    {"name":"category_type","type":"STRING"},
    {"name":"payment_method_recod","type":"STRING"},
    {"name":"stock_real","type":"FLOAT"},
    {"name":"stock_pronosticado","type":"FLOAT"},
    {"name":"estado","type":"STRING"},
    {"name":"stock_total","type":"FLOAT"},
    {"name":"stock_acum_mtd","type":"FLOAT"},
]
TABLE_MAU_OUT   = f"{PROJECT_ID}.{DATASET_ID}.forecasting_le_seguros"
TABLE_STOCK_OUT = f"{PROJECT_ID}.{DATASET_ID}.forecasting_le_seguros_stock"

def print_le_report(df):
    le = df.groupby("identificador")["valor_total"].sum().to_dict()
    rv_total = le.get("revenue_NO_TC", 0.0) + le.get("revenue_TC", 0.0)
    print("\n=== Totales (R+F) ===")
    for k in ["mau_NO_TC","mau_TC","revenue_NO_TC","revenue_TC"]:
        if k in le:
            print(f"{k:14s}: {le[k]:,.2f}")
    print(f"{'revenue_total':14s}: {rv_total:,.2f}")

def validate_mau_tc(out_mau_df: pd.DataFrame):
    ECONOMICS_LOC = get_dataset_location_safe(PROJECT_ID, "economics", credentials) or "US"
    df_true = read_gbq_robusto(
        VALIDATE_MAU_TC_SQL,
        PROJECT_ID,
        credentials,
        primary_location=ECONOMICS_LOC,
        fallback_location=None
    )
    if out_mau_df.empty or df_true.empty:
        print("\n[VALIDACIÓN MAU_TC] No hay datos para validar.")
        return

    out_mau = out_mau_df.copy()
    out_mau["fecha"] = pd.to_datetime(out_mau["fecha"]).dt.normalize()
    df_true = df_true.copy()
    df_true["fecha"] = pd.to_datetime(df_true["fecha"]).dt.normalize()

    start_month = ahora_chile().normalize().tz_localize(None).replace(day=1)
    mask = (
        (out_mau["identificador"]=="mau_TC") &
        (out_mau["estado"]=="R") &
        (out_mau["fecha"]>=start_month)
    )
    pipe = (out_mau.loc[mask, ["fecha","valor_total"]]
            .groupby("fecha", as_index=False).sum()
            .rename(columns={"valor_total":"mau_tc_pipe"}))

    pipe["mau_tc_pipe"] = pd.to_numeric(pipe["mau_tc_pipe"], errors="coerce")
    df_true["mau_tc_real"]  = pd.to_numeric(df_true["mau_tc_real"],  errors="coerce")

    chk = pipe.merge(df_true, on="fecha", how="outer").sort_values("fecha")
    chk["delta"] = chk["mau_tc_pipe"].fillna(0.0) - chk["mau_tc_real"].fillna(0.0)

    print("\n=== VALIDACIÓN MAU_TC (R del mes corriente) ===")
    print(chk.fillna(0).to_string(index=False))

# ============================================================
# Escritura de kpitos.forecast_le_insurance (R hasta D-1)
# ============================================================
INS_TABLE_OUT = f"{PROJECT_ID}.{DATASET_ID}.forecast_le_insurance"

SCHEMA_INSURANCE_FLAT = [
    {"name":"fecha","type":"DATE"},
    {"name":"linea","type":"STRING"},
    {"name":"dia_semana","type":"STRING"},
    {"name":"dia","type":"INTEGER"},
    {"name":"mes","type":"INTEGER"},
    {"name":"year","type":"INTEGER"},
    {"name":"gpv","type":"FLOAT"},
    {"name":"real","type":"INTEGER"},
]

_DOW_MAP_INS = {0:"Mon", 1:"Tue", 2:"Wed", 3:"Thu", 4:"Fri", 5:"Sat", 6:"Sun"}
def _weekday_abbr_single_ins(ts) -> str:
    return _DOW_MAP_INS[pd.Timestamp(ts).dayofweek]

ECON_LOC_SAFE = get_dataset_location_safe(PROJECT_ID, "economics", credentials) or "US"

HIST_SQL_INS = f"""
WITH raw AS (
  SELECT
    DATE(fecha) AS fecha,
    LOWER(IFNULL(nombre, "nulo")) AS nombre,
    SUM(monto) AS gpv_clp
  FROM `{PROJECT_ID}.economics.economics`
  WHERE linea = "insurance"
    AND fecha >= DATE("{START_DATE.date()}")
    AND fecha <= DATE("{D_MINUS_1.date()}")
  GROUP BY 1,2
),
tag AS (
  SELECT
    fecha,
    CASE
      WHEN nombre LIKE "%saldo deudor%" OR nombre LIKE "%desgravamen%" OR nombre LIKE "%cesantia%"
           OR nombre LIKE "%incapacidad temporal%" THEN "TC"
      ELSE "NO_TC"
    END AS tipo,
    gpv_clp
  FROM raw
),
agg AS (
  SELECT fecha,
         SUM(IF(tipo="TC", gpv_clp, 0))     AS gpv_tc_clp,
         SUM(IF(tipo="NO_TC", gpv_clp, 0))  AS gpv_notc_clp
  FROM tag
  GROUP BY 1
)
SELECT fecha, gpv_tc_clp, gpv_notc_clp
FROM agg
ORDER BY fecha
"""

def build_and_write_insurance_flat():
    print(f"=== Insurance FLAT (R hasta D-1) → {INS_TABLE_OUT} ===")
    df = read_gbq_robusto(
        HIST_SQL_INS,
        PROJECT_ID,
        credentials,
        primary_location=ECON_LOC_SAFE,
        fallback_location=None
    )
    if df.empty:
        print("[WARN] No hay histórico en economics para insurance. Se omite escritura.")
        return

    df["fecha"] = pd.to_datetime(df["fecha"]).dt.normalize()
    rows = []
    for r in df.itertuples(index=False):
        fecha = pd.Timestamp(r.fecha)
        dow = _weekday_abbr_single_ins(fecha)
        rows.append(dict(
            fecha=fecha, linea="insurance_credito",
            dia_semana=dow, dia=int(fecha.day), mes=int(fecha.month), year=int(fecha.year),
            gpv=float(getattr(r, "gpv_tc_clp") or 0.0), real=1
        ))
        rows.append(dict(
            fecha=fecha, linea="insurance_no_credito",
            dia_semana=dow, dia=int(fecha.day), mes=int(fecha.month), year=int(fecha.year),
            gpv=float(getattr(r, "gpv_notc_clp") or 0.0), real=1
        ))

    out = (pd.DataFrame(rows)
             .sort_values(["fecha","linea"])
             .reset_index(drop=True)
             [["fecha","linea","dia_semana","dia","mes","year","gpv","real"]])

    to_bq(out, INS_TABLE_OUT, SCHEMA_INSURANCE_FLAT)
    print(f"[OK] Insurance FLAT escrito: {len(out):,} filas (hasta {D_MINUS_1.date()}).")

# ---------- MAIN ----------
def main():
    print(f"=== LE SEGUROS — Mes={FORE_START.strftime('%Y-%m')} | D-1={D_MINUS_1.date()} ===")
    print(f"    Periodo forecast: {FORE_START.date()} → {FORE_END.date()} ({(FORE_END - FORE_START).days + 1} días)")

    # 1) Targets dinámicos: momentum de mayo + junio → proyecta julio
    compute_dynamic_targets_last2m()

    # 2) Distribuir GPV con proporciones fallback si no vienen de BQ
    gpv_total = ADJUST_TARGETS.get("gpv")
    if gpv_total and ADJUST_TARGETS.get("gpv_TC") is None:
        ADJUST_TARGETS["gpv_TC"]    = gpv_total * _BASE_GPV_TC_PCT    # 94.1% TC (fallback histórico)
        ADJUST_TARGETS["gpv_NO_TC"] = gpv_total * _BASE_GPV_NOTC_PCT  # 5.9% NO_TC (fallback histórico)

    print(f"\n[INFO] === Targets definitivos {FORE_START:%Y-%m} ===")
    print(f"  Base fallback histórica: revenue total {_BASE_REV_TOTAL:.2f}M | GPV {_BASE_GPV_TOTAL:.1f}M")
    print(f"  Crecimiento fallback aplicado: {(_JUL_GROWTH-1)*100:.1f}%")
    for k in ["mau_NO_TC","mau_TC","revenue_NO_TC","revenue_TC","revenue_total","gpv","gpv_TC","gpv_NO_TC"]:
        v = ADJUST_TARGETS.get(k)
        if v is not None:
            print(f"  {k:14s}: {v:,.3f}")
    rev_tc_v  = ADJUST_TARGETS.get("revenue_TC")  or 0.0
    rev_tot_v = ADJUST_TARGETS.get("revenue_total") or 1.0
    print(f"  Proporción TC/total : {rev_tc_v/rev_tot_v*100:.1f}%")
    print(f"  Proporción NTC/total: {(rev_tot_v-rev_tc_v)/rev_tot_v*100:.1f}%")

    # 3) R/F MAU + revenue + gpv
    out_mau = run_mau_xgb()
    out_mau = enforce_tc_billing(out_mau)
    out_mau = set_f_floors(out_mau)
    out_mau = adjust_revenue_soft(out_mau)
    out_mau = adjust_mau_soft(out_mau)
    out_mau = shape_revenue_notc_after20(out_mau)
    out_mau = reconcile_hard_caps(out_mau)

    if FORCE_TARGETS:
        out_mau = force_projected_to_month_targets(out_mau)

    to_bq(out_mau, TABLE_MAU_OUT, SCHEMA_MAU)

    # 4) STOCK
    out_stock = run_stock_xgb()
    to_bq(out_stock, TABLE_STOCK_OUT, SCHEMA_STOCK)

    print_le_report(out_mau)
    validate_mau_tc(out_mau)
    print(f"[OK] Proceso completo {FORE_START:%Y-%m}.")

# ============================================================
# Runner
# ============================================================
if __name__ == "__main__":
    try:
        main()
    finally:
        build_and_write_insurance_flat()

In [ ]:
"""
LE Seguros — Gráfico comparativo por día del mes
Día 1 vs día 1, día 2 vs día 2, etc.
Últimos 3 meses reales (economics.economics) + forecast junio (forecasting_le_seguros)
"""
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas_gbq
import pydata_google_auth

# ----------------------------------------------------------
# 0. AUTH
# ----------------------------------------------------------
scopes = [
    "https://www.googleapis.com/auth/bigquery",
    "https://www.googleapis.com/auth/cloud-platform",
]
credentials = pydata_google_auth.get_user_credentials(scopes)
PROJECT_ID = "tenpo-bi-prod"

# ----------------------------------------------------------
# 1. HISTÓRICO — economics.economics (últimos 3 meses)
# ----------------------------------------------------------
SQL_HIST = """
SELECT
  DATE(fecha)                            AS fecha,
  FORMAT_DATE('%Y-%m', fecha)            AS mes_anio,
  EXTRACT(DAY FROM fecha)               AS dia_mes,
  SUM(revenue) / 1e6                    AS revenue
FROM `tenpo-bi-prod.economics.economics`
WHERE linea = 'insurance'
  AND fecha >= DATE_TRUNC(
        DATE_SUB(CURRENT_DATE('America/Santiago'), INTERVAL 3 MONTH), MONTH)
  AND fecha <  DATE_TRUNC(CURRENT_DATE('America/Santiago'), MONTH)
GROUP BY 1, 2, 3
ORDER BY 1
"""

# ----------------------------------------------------------
# 2. FORECAST JUNIO — forecasting_le_seguros
# ----------------------------------------------------------
SQL_FORE = """
SELECT
  DATE(fecha)                                       AS fecha,
  FORMAT_DATE('%Y-%m', fecha)                       AS mes_anio,
  EXTRACT(DAY FROM fecha)                           AS dia_mes,
  CASE
    WHEN LOWER(TRIM(CAST(estado AS STRING))) LIKE 'f%' THEN 'Forecast'
    ELSE 'Real'
  END                                               AS tipo,
  SUM(valor_total)                                  AS revenue
FROM `tenpo-bi-prod.kpitos.forecasting_le_seguros`
WHERE FORMAT_DATE('%Y-%m', fecha) = FORMAT_DATE('%Y-%m', CURRENT_DATE('America/Santiago'))
  AND identificador IN ('revenue_TC', 'revenue_NO_TC')
GROUP BY 1, 2, 3, 4
ORDER BY 1
"""

print("Descargando histórico (economics.economics)...")
df_hist = pandas_gbq.read_gbq(SQL_HIST, project_id=PROJECT_ID, credentials=credentials)
df_hist["fecha"]  = pd.to_datetime(df_hist["fecha"])
df_hist["tipo"]   = "Real"
print(f"  {len(df_hist)} filas | meses: {sorted(df_hist['mes_anio'].unique())}")

print("Descargando forecast del mes (forecasting_le_seguros)...")
df_fore = pandas_gbq.read_gbq(SQL_FORE, project_id=PROJECT_ID, credentials=credentials)
df_fore["fecha"] = pd.to_datetime(df_fore["fecha"])
print(f"  {len(df_fore)} filas")

# ----------------------------------------------------------
# 3. UNIR
# ----------------------------------------------------------
df_all = pd.concat([df_hist, df_fore], ignore_index=True)

meses_hist = sorted(df_hist["mes_anio"].unique())[-3:]
mes_actual  = df_fore["mes_anio"].iloc[0] if not df_fore.empty else ""

# ----------------------------------------------------------
# 4. COLORES Y ESTILOS
# ----------------------------------------------------------
ESTILOS = {
    meses_hist[0]: dict(color="#27AE60", lw=1.6, dashes=(5, 3),     alpha=0.85),
    meses_hist[1]: dict(color="#8E44AD", lw=1.6, dashes=(2, 2),     alpha=0.85),
    meses_hist[2]: dict(color="#1A6EA8", lw=2.0, dashes=(6, 2, 1, 2), alpha=0.90),
    mes_actual:    dict(color="#E05A2B", lw=2.8, dashes=None,        alpha=1.00),
}

# ----------------------------------------------------------
# 5. GRÁFICO
# ----------------------------------------------------------
fig, ax = plt.subplots(figsize=(16, 7))
fig.patch.set_facecolor("#FAFAFA")
ax.set_facecolor("#FAFAFA")

# --- Meses históricos ---
for mes in meses_hist:
    cfg = ESTILOS[mes]
    d   = df_all[(df_all["mes_anio"] == mes) & (df_all["tipo"] == "Real")].sort_values("dia_mes")
    kw  = dict(color=cfg["color"], linewidth=cfg["lw"],
               alpha=cfg["alpha"], label=mes, zorder=3)
    if cfg["dashes"]:
        kw["dashes"] = cfg["dashes"]
    ax.plot(d["dia_mes"].tolist(), d["revenue"].tolist(), **kw)

# --- Promedio histórico ---
df_h3 = df_all[(df_all["mes_anio"].isin(meses_hist)) & (df_all["tipo"] == "Real")]
prom  = (df_h3.groupby("dia_mes", as_index=False)["revenue"]
               .mean()
               .sort_values("dia_mes"))
ax.plot(prom["dia_mes"].tolist(), prom["revenue"].tolist(),
        color="#2C3E50", linewidth=2.2, linestyle=":",
        marker="o", markersize=3, markevery=5,
        label="Promedio 3 meses", zorder=5)

# --- Junio: Real (sólido) + Forecast (discontinuo) ---
if mes_actual:
    cfg    = ESTILOS[mes_actual]
    d_real = df_all[(df_all["mes_anio"] == mes_actual) & (df_all["tipo"] == "Real")].sort_values("dia_mes")
    d_fore = df_all[(df_all["mes_anio"] == mes_actual) & (df_all["tipo"] == "Forecast")].sort_values("dia_mes")

    if not d_real.empty:
        ax.plot(d_real["dia_mes"].tolist(), d_real["revenue"].tolist(),
                color=cfg["color"], linewidth=cfg["lw"], zorder=7,
                label=f"{mes_actual} (Real)")

    if not d_fore.empty:
        xf = d_fore["dia_mes"].tolist()
        yf = d_fore["revenue"].tolist()
        if not d_real.empty:
            xf = [d_real["dia_mes"].iloc[-1]] + xf
            yf = [d_real["revenue"].iloc[-1]]  + yf
        ax.plot(xf, yf,
                color=cfg["color"], linewidth=cfg["lw"],
                linestyle="--", zorder=7,
                label=f"{mes_actual} (Forecast)")

# ----------------------------------------------------------
# 6. LÍNEAS VERTICALES en días de facturación 5 y 20
# ----------------------------------------------------------
for bd in [5, 20]:
    ax.axvline(bd, color="#AAAAAA", linewidth=0.9, linestyle="--", zorder=1)
    ax.text(bd + 0.3, ax.get_ylim()[1] * 0.97,
            f"día {bd}", fontsize=8, color="#888", va="top")

# ----------------------------------------------------------
# 7. ESTÉTICA
# ----------------------------------------------------------
ax.set_xticks(range(1, 32))
ax.xaxis.set_minor_locator(mticker.NullLocator())
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.1f}M"))
ax.set_xlabel("Día del mes", fontsize=11, color="#444")
ax.set_ylabel("Revenue diario (MM CLP)", fontsize=11, color="#444")
ax.set_title(
    "LE Seguros — Revenue Diario por Número de Día\n"
    "Últimos 3 meses reales · Promedio · Forecast mes en curso",
    fontsize=13, fontweight="bold", color="#222", pad=14
)
ax.set_xlim(0.5, 31.5)
ax.grid(True, alpha=0.2, linestyle="--")
for spine in ax.spines.values():
    spine.set_visible(False)
ax.legend(loc="upper left", fontsize=9.5, framealpha=0.9,
          edgecolor="#CCC", ncol=2)

plt.tight_layout()
plt.savefig("le_seguros_dia_comparativo.png", dpi=150, bbox_inches="tight")
plt.show()
print("Gráfico guardado como 'le_seguros_dia_comparativo.png'")

In [ ]:
#######################################################
#        ENTREGA TARJETAS FÍSICAS (Forecast — JULIO 2026)
#######################################################

import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from datetime import datetime, timedelta
import pytz
import warnings
import pydata_google_auth

warnings.filterwarnings('ignore')

# ---------------------------------------------------------
# 0. AUTENTICACIÓN Y CONTROL DE FECHAS (D-1)
# ---------------------------------------------------------
print("0. Autenticando con Google Cloud y fijando fechas...")
scopes = [
    "https://www.googleapis.com/auth/bigquery",
    "https://www.googleapis.com/auth/cloud-platform",
]
credentials = pydata_google_auth.get_user_credentials(scopes)

tz_cl = pytz.timezone('America/Santiago')

# Mes objetivo fijo: AGOSTO 2026
# V11: mes de D-1 (el dia 1 proyecta/cierra el mes anterior en vez de lanzar error)
_D1_CL = ahora_chile().tz_localize(None).normalize() - pd.Timedelta(days=1)
_MESES_ES = ["Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio", "Julio", "Agosto",
             "Septiembre", "Octubre", "Noviembre", "Diciembre"]
FORE_START = _D1_CL.replace(day=1)
FORE_END   = (FORE_START + pd.offsets.MonthEnd(0)).normalize()
FORE_LABEL = f"{_MESES_ES[FORE_START.month - 1]} {FORE_START.year}"
FORE_MONTH_STR = FORE_START.strftime('%Y-%m')

# Corte R/F: real hasta D-1 Chile, acotado al mes objetivo
hoy_real = ahora_chile().to_pydatetime().date()
ayer_real = hoy_real - timedelta(days=1)
D_MINUS_1 = min(pd.Timestamp(ayer_real).normalize(), FORE_END)

# Mantener nombres usados aguas abajo
hoy  = hoy_real
ayer = D_MINUS_1.date()

if D_MINUS_1 < FORE_START:
    raise RuntimeError(f"Aún no hay días reales dentro de {FORE_LABEL}. D_MINUS_1={D_MINUS_1.date()}")

# ---------------------------------------------------------
# 1. EXTRACCIÓN (1 AÑO COMPLETO)
# ---------------------------------------------------------
PROJECT_ID_READ  = 'tenpo-operaciones'
PROJECT_ID_WRITE = 'tenpo-bi-prod'

query_ops = f"""
SELECT DATE(fecha_carga) AS fecha, COUNT(*) AS cantidad_tarjetas
FROM `tenpo-operaciones.OPD3.OP`
WHERE TIPOFAC = "3000"
  AND NOMCOMRED IN ('DELIVERYCARD-RETIRO', 'CASHOUTPHYSICALCARD')
  AND DATE(fecha_carga) BETWEEN DATE('{(D_MINUS_1 - pd.Timedelta(days=365)).date()}')
                            AND DATE('{D_MINUS_1.date()}')
GROUP BY 1
"""
query_feriados = """
SELECT DATE(fecha) AS fecha
FROM `operacionestc-operaciones-prod.Sandbox.Dias_Feriados_Chile`
"""

print("1. Descargando datos históricos (hasta D-1)...")
df_ops      = pd.read_gbq(query_ops,      project_id=PROJECT_ID_READ, credentials=credentials, location='US')
df_feriados = pd.read_gbq(query_feriados, project_id=PROJECT_ID_READ, credentials=credentials, location='US')

df_ops['fecha'] = pd.to_datetime(df_ops['fecha'])
PRECIO_TARJETA  = 7490

f_max           = pd.to_datetime(ayer)
f_min_1_year    = f_max - timedelta(days=365)
f_min_3_months  = f_max - timedelta(days=90)

df_ops = df_ops[df_ops['fecha'] >= f_min_1_year]
df = pd.DataFrame({'fecha': pd.date_range(f_min_1_year, f_max)})
df = df.merge(df_ops, on='fecha', how='left')
df['cantidad_tarjetas'] = df['cantidad_tarjetas'].fillna(0)

df['d_semana']       = df['fecha'].dt.dayofweek
df['d_mes']          = df['fecha'].dt.day
df['mes']            = df['fecha'].dt.month
df['fin_semana']     = df['d_semana'].isin([5, 6]).astype(int)
df['feriado']        = df['fecha'].isin(pd.to_datetime(df_feriados['fecha'])).astype(int)
df['pago']           = df['d_mes'].isin([30, 31, 1, 2, 15, 16]).astype(int)
df['revenue_diario'] = df['cantidad_tarjetas'] * PRECIO_TARJETA
df['mes_anio']       = df['fecha'].dt.strftime('%Y-%m')

# ---------------------------------------------------------
# FECHAS DE REFERENCIA — JULIO 2026 FIJO
# ---------------------------------------------------------
end_of_month    = FORE_END
mes_actual_str  = FORE_MONTH_STR

# Mes anterior: primer día del mes objetivo - 1 día → último día del mes anterior
primer_dia_actual  = FORE_START
mes_anterior_str   = (primer_dia_actual - pd.Timedelta(days=1)).strftime('%Y-%m')

# ---------------------------------------------------------
# ANCHOR: promedio de los últimos 4 meses completos
# Anchor sobre últimos 4 meses completos previos a julio
# → evita depender solo del cierre de junio y suaviza outliers
# ---------------------------------------------------------
meses_completos = sorted([m for m in df['mes_anio'].unique() if m < mes_actual_str])[-4:]

cierres = {}
for m in meses_completos:
    cierres[m] = int(df[df['mes_anio'] == m]['cantidad_tarjetas'].sum())

print(f"\n   Cierres históricos usados como anchor:")
for m, v in cierres.items():
    rev = v * PRECIO_TARJETA / 1e6
    print(f"     {m}: {v:,} tarjetas  (${rev:.1f}M)")

PROMEDIO_CIERRE   = int(np.mean(list(cierres.values())))
CRECIMIENTO_MES   = 1.01   # +1% sobre el promedio de 4 meses completos previos a julio
TARGET_VOL_JULIO  = int(PROMEDIO_CIERRE * CRECIMIENTO_MES)

print(f"\n   Promedio 4 meses : {PROMEDIO_CIERRE:,} tarjetas  (${PROMEDIO_CIERRE*PRECIO_TARJETA/1e6:.1f}M)")
print(f"   Target {FORE_LABEL} +{(CRECIMIENTO_MES-1)*100:.0f}%  : {TARGET_VOL_JULIO:,} tarjetas  (${TARGET_VOL_JULIO*PRECIO_TARJETA/1e6:.1f}M)")

# ---------------------------------------------------------
# 2. ENTRENAMIENTO XGB
# ---------------------------------------------------------
print("\n2. Entrenando modelo XGBoost...")
df_train = df[df['fecha'] >= f_min_3_months].copy()

features = ['d_semana', 'd_mes', 'mes', 'fin_semana', 'feriado', 'pago']
X = df_train[features]
y = df_train['cantidad_tarjetas']

model = xgb.XGBRegressor(
    n_estimators=120,
    learning_rate=0.04,
    max_depth=3,
    reg_lambda=1.5,
    gamma=0.5,
    subsample=0.85,
    colsample_bytree=0.85,
    random_state=42
)
model.fit(X, y)

# ---------------------------------------------------------
# 3. FORECAST HASTA FIN DE JULIO
# Forma del XGB escalada al target anclado sobre el promedio 4 meses
# ---------------------------------------------------------
print(f"3. Proyectando desde {f_max + timedelta(days=1)} hasta {end_of_month.date()}...")

fechas_futuras = pd.date_range(start=f_max + timedelta(days=1), end=end_of_month)
futuro = pd.DataFrame({'fecha': fechas_futuras})
futuro['d_semana']  = futuro['fecha'].dt.dayofweek
futuro['d_mes']     = futuro['fecha'].dt.day
futuro['mes']       = futuro['fecha'].dt.month
futuro['fin_semana']= futuro['d_semana'].isin([5, 6]).astype(int)
futuro['feriado']   = futuro['fecha'].isin(pd.to_datetime(df_feriados['fecha'])).astype(int)
futuro['pago']      = futuro['d_mes'].isin([30, 31, 1, 2, 15, 16]).astype(int)

if len(futuro) > 0:
    raw_pred = model.predict(futuro[features]).clip(min=0)
else:
    raw_pred = np.array([], dtype=float)

# Volumen real de julio ya disponible (días reales hasta D-1)
vol_real_julio = int(df[df['mes_anio'] == mes_actual_str]['cantidad_tarjetas'].sum())

# Escalar forma XGB para que total real + forecast = target
vol_pendiente = max(0.0, TARGET_VOL_JULIO - vol_real_julio)
raw_sum       = raw_pred.sum()
escala        = (vol_pendiente / raw_sum) if raw_sum > 0 else 1.0

if len(futuro) > 0:
    futuro['forecast_volumen'] = (raw_pred * escala).clip(min=0).round(0).astype(int)
    futuro['forecast_revenue'] = futuro['forecast_volumen'] * PRECIO_TARJETA
else:
    futuro['forecast_volumen'] = pd.Series(dtype='int64')
    futuro['forecast_revenue'] = pd.Series(dtype='float64')

print(f"   Vol real mes hasta D-1   : {vol_real_julio:,} tarjetas")
print(f"   Target total del mes     : {TARGET_VOL_JULIO:,} tarjetas")
print(f"   Vol pendiente forecast    : {int(vol_pendiente):,} tarjetas")
print(f"   Factor escala XGB         : {escala:.3f}x")

# ---------------------------------------------------------
# 4. CONSOLIDACIÓN
# ---------------------------------------------------------
print("4. Consolidando historia y forecast...")

df_hist = df[['fecha', 'cantidad_tarjetas', 'revenue_diario', 'mes_anio']].copy()
df_hist.rename(columns={'cantidad_tarjetas': 'volumen', 'revenue_diario': 'revenue'}, inplace=True)
df_hist['tipo'] = 'Real'

df_futuro = futuro[['fecha', 'forecast_volumen', 'forecast_revenue']].copy()
df_futuro.rename(columns={'forecast_volumen': 'volumen', 'forecast_revenue': 'revenue'}, inplace=True)
df_futuro['tipo']     = 'Forecast'
df_futuro['mes_anio'] = df_futuro['fecha'].dt.strftime('%Y-%m')

df_total = pd.concat([df_hist, df_futuro]).reset_index(drop=True)
df_total['dia_del_mes'] = df_total['fecha'].dt.day

df_total['acumulado_volumen_mes'] = df_total.groupby('mes_anio')['volumen'].cumsum()
df_total['acumulado_revenue_mes'] = df_total.groupby('mes_anio')['revenue'].cumsum()

# ---------------------------------------------------------
# 5. GRÁFICO: últimos 4 meses + promedio + forecast julio
# ---------------------------------------------------------
print("\n5. Generando gráfico...")

todos_meses = sorted(df_total['mes_anio'].unique())
meses_ant   = [m for m in todos_meses if m < mes_actual_str]
ultimos_4   = meses_ant[-4:] if len(meses_ant) >= 4 else meses_ant

COLORES_HIST = ['#27AE60', '#8E44AD', '#D4AC0D', '#1A6EA8']
ESTILOS_HIST = [
    (1.6, (4, 3)),
    (1.6, (2, 2)),
    (1.8, (6, 2, 1, 2)),
    (2.2, None),
]

fig, ax = plt.subplots(figsize=(15, 7))
fig.patch.set_facecolor('#FAFAFA')
ax.set_facecolor('#FAFAFA')

# Curvas históricas
for i, mes in enumerate(ultimos_4):
    datos_mes = df_total[df_total['mes_anio'] == mes].sort_values('dia_del_mes')
    lw, dashes = ESTILOS_HIST[i]
    kwargs = dict(color=COLORES_HIST[i], linewidth=lw, alpha=0.9, label=mes, zorder=3)
    if dashes:
        kwargs['dashes'] = dashes
    ax.plot(datos_mes['dia_del_mes'], datos_mes['acumulado_revenue_mes'] / 1e6, **kwargs)

# Promedio 4 meses
df_hist_4m   = df_total[df_total['mes_anio'].isin(ultimos_4) & (df_total['tipo'] == 'Real')]
promedio_dia = (df_hist_4m.groupby('dia_del_mes')['acumulado_revenue_mes']
                           .mean().reset_index().sort_values('dia_del_mes'))
ax.plot(promedio_dia['dia_del_mes'], promedio_dia['acumulado_revenue_mes'] / 1e6,
        color='#2C3E50', linewidth=2.5, linestyle=':', label='Promedio 4 meses',
        zorder=4, marker='o', markersize=3, markevery=5)

# Julio: Real + Forecast
datos_julio    = df_total[df_total['mes_anio'] == mes_actual_str].sort_values('dia_del_mes')
julio_real     = datos_julio[datos_julio['tipo'] == 'Real']
julio_forecast = datos_julio[datos_julio['tipo'] == 'Forecast']

if not julio_real.empty:
    ax.plot(julio_real['dia_del_mes'], julio_real['acumulado_revenue_mes'] / 1e6,
            color='#E05A2B', linewidth=3, label=f'{FORE_LABEL} (Real)', zorder=6)

if not julio_forecast.empty:
    if not julio_real.empty:
        conector = julio_real.iloc[[-1]].copy()
        conector['tipo'] = 'Forecast'
        julio_forecast = pd.concat([conector, julio_forecast])
    ax.plot(julio_forecast['dia_del_mes'], julio_forecast['acumulado_revenue_mes'] / 1e6,
            color='#E05A2B', linewidth=3, linestyle='--',
            label=f'{FORE_LABEL} (Forecast)', zorder=6)

# Anotar cierre proyectado
cierre_julio = datos_julio['acumulado_revenue_mes'].max()
ultimo_dia   = datos_julio['dia_del_mes'].max()
ax.annotate(f"  ${cierre_julio/1e6:.1f}M",
            xy=(ultimo_dia, cierre_julio / 1e6),
            fontsize=10, fontweight='bold', color='#E05A2B', va='center')

ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.1f}M'))
ax.set_xlabel('Día del mes', fontsize=11, color='#444')
ax.set_ylabel('Revenue acumulado (MM CLP)', fontsize=11, color='#444')
ax.set_title(f'Entrega de Tarjetas — Revenue Acumulado Mensual\nÚltimos 4 meses · Promedio · Forecast {FORE_LABEL}',
             fontsize=13, fontweight='bold', color='#222', pad=14)
ax.set_xticks(range(1, 32))
ax.tick_params(colors='#555')
ax.grid(True, alpha=0.25, linestyle='--')
for spine in ax.spines.values():
    spine.set_visible(False)
ax.legend(loc='upper left', fontsize=9.5, framealpha=0.85, edgecolor='#ccc')

plt.tight_layout()
plt.savefig(f'entrega_tarjetas_{FORE_START:%Y_%m}.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"   Gráfico guardado como 'entrega_tarjetas_{FORE_START:%Y_%m}.png'")

# ---------------------------------------------------------
# 6. RESUMEN EN CONSOLA
# ---------------------------------------------------------
cierre_proyectado = df_total[df_total['mes_anio'] == mes_actual_str]['acumulado_revenue_mes'].max()
cierre_vol        = df_total[df_total['mes_anio'] == mes_actual_str]['acumulado_volumen_mes'].max()
print("\n" + "="*56)
print(f"  CIERRE PROYECTADO {FORE_LABEL.upper()}")
print(f"  Anchor           : promedio 4 meses = {PROMEDIO_CIERRE:,} tarjetas")
print(f"  Crecimiento      : +{(CRECIMIENTO_MES-1)*100:.0f}%")
print(f"  Volumen proyectado: {int(cierre_vol):,} tarjetas")
print(f"  Revenue proyectado: ${cierre_proyectado:,.0f}  (${cierre_proyectado/1e6:.2f}M)")
print("="*56 + "\n")

# ---------------------------------------------------------
# 7. EXPORTACIÓN A BIGQUERY
# ---------------------------------------------------------
print("7. Exportando a BigQuery...")

out_bq = df_total[[
    'fecha', 'mes_anio', 'dia_del_mes', 'tipo',
    'volumen', 'acumulado_volumen_mes',
    'revenue', 'acumulado_revenue_mes'
]].copy()

out_bq['fecha'] = pd.to_datetime(out_bq['fecha'])
for c in ['volumen', 'acumulado_volumen_mes', 'revenue', 'acumulado_revenue_mes']:
    out_bq[c] = out_bq[c].fillna(0.0).astype(float)

try:
    out_bq.to_gbq(
        destination_table="kpitos.entrega_tarjetas",
        project_id=PROJECT_ID_WRITE,
        if_exists='replace',
        credentials=credentials
    )
    print(f"[OK] BigQuery reemplazado con éxito → {PROJECT_ID_WRITE}.kpitos.entrega_tarjetas")
except Exception as e:
    print(f"[ERROR] Al escribir en BQ: {e}")

In [ ]:
import pydata_google_auth
import pandas as pd
import numpy as np
import requests
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV
import xgboost as xgb
import matplotlib.pyplot as plt
import matplotlib.ticker as tkr
import seaborn as sns
import warnings
import pandas_gbq
warnings.filterwarnings('ignore')
import pickle
from google.colab import drive
drive.mount('/content/drive')

from google.colab import auth
auth.authenticate_user()
print("Autenticado exitosamente en Google Cloud")

SCOPES = [
            'https://www.googleapis.com/auth/cloud-platform',
            'https://www.googleapis.com/auth/drive',
        ]
credentials = pydata_google_auth.get_user_credentials(
            SCOPES,
            auth_local_webserver=True,
        )

In [ ]:
QUERY_UF = """
SELECT
    DATE(MAX(verified_date)) AS fecha,
    rate as valor_uf
FROM `business-data-raw.ingestor_exchange_rate_exchange_rate.unit_rate`
WHERE code = "UF"
GROUP BY EXTRACT(YEAR FROM verified_date), rate, EXTRACT(MONTH FROM verified_date)
ORDER BY fecha DESC
"""
df_uf = pd.read_gbq(QUERY_UF, use_bqstorage_api=True, project_id='tenpo-bi-prod', credentials=credentials, dialect='standard')
df_uf['fecha'] = pd.to_datetime(df_uf['fecha'], format="%Y-%m-%d")
ultimo_valor_uf = df_uf['valor_uf'].iloc[0]
print(f"UF cargada: {ultimo_valor_uf}")

In [ ]:
mes = "Consolidado"
archivo_metas = pd.read_excel(f"/content/drive/.shortcut-targets-by-id/10Ha9KHrUn7dPQ7dGoiEovdVoRhmr5MSZ/Planificación Financiera/Cierres/Metas/Metas KPI ({mes}).xlsx", sheet_name="Consolidado")
primera_columna = archivo_metas.columns[0]

# Create a new list for the updated column names
new_columns = [primera_columna]

# Iterate through the rest of the columns and attempt to convert them to datetime
for col in archivo_metas.columns[1:]:
    try:
        # Try converting to datetime and format it
        new_columns.append(pd.to_datetime(col).strftime('%Y-%m-%d'))
    except (ValueError, TypeError):
        # If conversion fails, keep the original column name
        new_columns.append(col)

# Assign the new list of column names back to the DataFrame
archivo_metas.columns = new_columns

In [ ]:
directorio_salida = "/content/drive/.shortcut-targets-by-id/10Ha9KHrUn7dPQ7dGoiEovdVoRhmr5MSZ/Planificación Financiera/KPItos Data/modelos"
# =============================================================================
# FECHAS DEL PIPELINE -- auto-calculadas desde hoy, con override opcional.
# =============================================================================
# Comportamiento:
#   - Por default (FECHAS_OVERRIDE["activo"]=False) las fechas se derivan de la
#     fecha de hoy: entrenamiento hasta el fin del mes anterior, proyeccion
#     sobre el mes en curso.
#   - Si necesitas forzar una ventana especifica (incidente de ingesta,
#     reprocesamiento historico), activa el override.
#   - Los inicios de ventana (2024-01-01 / 2024-07-01 / 2024-10-01) se dejan
#     fijos porque corresponden al arranque de cada linea (MAU, TC, TC xs).
#     Si necesitas cambiarlos, editalos en la seccion FECHAS_INICIO.
# =============================================================================
from datetime import date, timedelta
import calendar as _cal

FECHAS_INICIO = {
    "mau_gpv":  "2024-01-01",   # arranque general
    "tc":       "2024-07-01",   # arranque TC
    "tc_xs":    "2024-10-01",   # arranque TC xs
    "proyeccion": "2023-10-01", # lower bound de carga
}

FECHAS_OVERRIDE = {
    "activo": False,
    # Descomentar solo si activo=True:
    # "fechas_entrenamiento":        ("2024-01-01", "2026-07-31"),
    # "fechas_entrenamiento_tc":     ("2024-07-01", "2026-07-31"),
    # "fechas_entrenamiento_tc_xs":  ("2024-10-01", "2026-07-31"),
    # "fechas_proyeccion":           ("2023-10-01", "2026-08-01", "2026-08-31"),
    # "motivo": "reprocesamiento por incidente de ingesta",
}

if FECHAS_OVERRIDE.get("activo"):
    fechas_entrenamiento        = FECHAS_OVERRIDE["fechas_entrenamiento"]
    fechas_entrenamiento_tc     = FECHAS_OVERRIDE["fechas_entrenamiento_tc"]
    fechas_entrenamiento_tc_xs  = FECHAS_OVERRIDE["fechas_entrenamiento_tc_xs"]
    fechas_proyeccion           = FECHAS_OVERRIDE["fechas_proyeccion"]
    print(f"[FECHAS OVERRIDE ACTIVO] motivo: {FECHAS_OVERRIDE.get('motivo', '(sin motivo)')}")
else:
    _hoy = hoy_chile().date()
    _ini_mes_actual = _hoy.replace(day=1)
    _fin_mes_ant    = _ini_mes_actual - timedelta(days=1)
    _fin_mes_actual = _ini_mes_actual.replace(
        day=_cal.monthrange(_hoy.year, _hoy.month)[1])
    _fmt = "%Y-%m-%d"

    fechas_entrenamiento       = (FECHAS_INICIO["mau_gpv"], _fin_mes_ant.strftime(_fmt))
    fechas_entrenamiento_tc    = (FECHAS_INICIO["tc"],      _fin_mes_ant.strftime(_fmt))
    fechas_entrenamiento_tc_xs = (FECHAS_INICIO["tc_xs"],   _fin_mes_ant.strftime(_fmt))
    fechas_proyeccion          = (
        FECHAS_INICIO["proyeccion"],
        _ini_mes_actual.strftime(_fmt),
        _fin_mes_actual.strftime(_fmt),
    )

print(f"[FECHAS] Entrenamiento MAU/GPV: {fechas_entrenamiento}")
print(f"[FECHAS] Entrenamiento TC:      {fechas_entrenamiento_tc}")
print(f"[FECHAS] Entrenamiento TC xs:   {fechas_entrenamiento_tc_xs}")
print(f"[FECHAS] Proyeccion:            {fechas_proyeccion}")
# =============================================================================

meta_ob_tenpo = 60000

# --- Resolucion de la columna del mes en el excel de metas, con validacion ---
_cols_match = archivo_metas.filter(like=fechas_proyeccion[1]).columns.tolist()
if not _cols_match:
    raise RuntimeError(
        f"El excel de metas no tiene columna que contenga '{fechas_proyeccion[1]}'.\n"
        f"Cols disponibles: {list(archivo_metas.columns)[:20]}...\n"
        f"Accion: actualiza archivo_metas con la columna del mes en curso, "
        f"o activa FECHAS_OVERRIDE para forzar un mes anterior."
    )
columna_fecha = _cols_match[0]
print(f"[METAS] Columna del mes usada: {columna_fecha}")

meta_gpv_app = archivo_metas.loc[archivo_metas['KPI'] == "gpv_app", columna_fecha].values[0]

meta_mau_app = archivo_metas.loc[archivo_metas['KPI'] == "mau_app", columna_fecha].values[0]

##----------------------
# FIX: se comento este hardcode que pisaba la meta leida del Excel (era leftover ene-26)
# meta_mau_app = 1029816

# --- VARIABLES RGU ---
try:
    meta_rgu = archivo_metas.loc[archivo_metas['KPI'] == "rgu", columna_fecha].values[0]
except IndexError:
    meta_rgu = 1950000
    print("[V11 METAS] !! 'rgu' no esta en el Consolidado; meta_rgu = 1.950.000 (respaldo, NO es la meta real)")

meta_mau_concurrent = archivo_metas.loc[archivo_metas['KPI'] == "mau_concurrent", columna_fecha].values[0]
meta_mau_m0 = archivo_metas.loc[archivo_metas['KPI'] == "mau_m0", columna_fecha].values[0]
meta_mau_resurrected = archivo_metas.loc[archivo_metas['KPI'] == "mau_resurrected", columna_fecha].values[0]
meta_mau_late_active = archivo_metas.loc[archivo_metas['KPI'] == "mau_late_active", columna_fecha].values[0]

meta_mau_prp_vir = archivo_metas.loc[archivo_metas['KPI'] == "mau_prp_vir", columna_fecha].values[0]
meta_mau_prp_fis = archivo_metas.loc[archivo_metas['KPI'] == "mau_prp_fis", columna_fecha].values[0]
meta_mau_prp_uni = archivo_metas.loc[archivo_metas['KPI'] == "mau_prp", columna_fecha].values[0]
meta_gpv_prp_vir = archivo_metas.loc[archivo_metas['KPI'] == "gpv_prp_vir", columna_fecha].values[0]
meta_gpv_prp_fis = archivo_metas.loc[archivo_metas['KPI'] == "gpv_prp_fis", columna_fecha].values[0]
# meta_gpv_prp_uni = archivo_metas.loc[archivo_metas['KPI'] == "gpv_prp", columna_fecha].values[0]

meta_mau_tc_vir = archivo_metas.loc[archivo_metas['KPI'] == "mau_tc_vir", columna_fecha].values[0]
meta_mau_tc_fis = archivo_metas.loc[archivo_metas['KPI'] == "mau_tc_fis", columna_fecha].values[0]
meta_mau_tc_tot = archivo_metas.loc[archivo_metas['KPI'] == "mau_tc", columna_fecha].values[0]
meta_mau_tc_tar = archivo_metas.loc[archivo_metas['KPI'] == "mau_tc", columna_fecha].values[0]
meta_gpv_tc_vir = archivo_metas.loc[archivo_metas['KPI'] == "gpv_tc_vir", columna_fecha].values[0]
meta_gpv_tc_fis = archivo_metas.loc[archivo_metas['KPI'] == "gpv_tc_fis", columna_fecha].values[0]

meta_mau_insurance = archivo_metas.loc[archivo_metas['KPI'] == "mau_insurance", columna_fecha].values[0]
meta_gpv_insurance = archivo_metas.loc[archivo_metas['KPI'] == "gpv_insurance", columna_fecha].values[0]

meta_mau_pdc = archivo_metas.loc[archivo_metas['KPI'] == "mau_pdc", columna_fecha].values[0]
meta_gpv_pdc = archivo_metas.loc[archivo_metas['KPI'] == "gpv_pdc", columna_fecha].values[0]

meta_mau_crossborder = archivo_metas.loc[archivo_metas['KPI'] == "mau_crossborder", columna_fecha].values[0]
meta_gpv_crossborder = archivo_metas.loc[archivo_metas['KPI'] == "gpv_crossborder", columna_fecha].values[0]

meta_mau_paypal = archivo_metas.loc[archivo_metas['KPI'] == "mau_paypal", columna_fecha].values[0]
meta_gpv_paypal = archivo_metas.loc[archivo_metas['KPI'] == "gpv_paypal", columna_fecha].values[0]

meta_mau_paypal_abonos = archivo_metas.loc[archivo_metas['KPI'] == "mau_paypal_abonos", columna_fecha].values[0]
meta_gpv_paypal_abonos = archivo_metas.loc[archivo_metas['KPI'] == "gpv_paypal_abonos", columna_fecha].values[0]

meta_mau_top_ups = archivo_metas.loc[archivo_metas['KPI'] == "mau_top_ups", columna_fecha].values[0]
meta_gpv_top_ups = archivo_metas.loc[archivo_metas['KPI'] == "gpv_top_ups", columna_fecha].values[0]

meta_mau_p2p = archivo_metas.loc[archivo_metas['KPI'] == "mau_p2p", columna_fecha].values[0]
meta_gpv_p2p = archivo_metas.loc[archivo_metas['KPI'] == "gpv_p2p", columna_fecha].values[0]

meta_ob_tc_large = archivo_metas.loc[archivo_metas['KPI'] == "ob_tc_large", columna_fecha].values[0]
meta_ob_tc_medium = archivo_metas.loc[archivo_metas['KPI'] == "ob_tc_medium", columna_fecha].values[0]
meta_ob_tc_small = archivo_metas.loc[archivo_metas['KPI'] == "ob_tc_small", columna_fecha].values[0]
meta_ob_tc_xs = archivo_metas.loc[archivo_metas['KPI'] == "ob_tc_xs", columna_fecha].values[0]

# V11: metas que no estan en el Consolidado -> se buscan por nombre y si no estan
# se usa el valor de respaldo CON AVISO (antes quedaban hardcodeadas en silencio).
# Ojo: estas metas solo alimentan las columnas meta_*; no escalan la prediccion.
def _meta_o_respaldo(kpi, respaldo):
    fila = archivo_metas.loc[archivo_metas['KPI'] == kpi, columna_fecha]
    if len(fila) and pd.notnull(fila.values[0]):
        return fila.values[0]
    print(f"[V11 METAS] !! '{kpi}' no esta en el Consolidado ({columna_fecha}); uso respaldo {respaldo:,}")
    return respaldo

meta_trx_ci_bestado = _meta_o_respaldo("trx_ci_bestado", 3000000)
meta_trx_ci_obancos = _meta_o_respaldo("trx_ci_obancos", 2500000)
meta_gpv_ci_bestado = _meta_o_respaldo("gpv_ci_bestado", 70000000000)
meta_gpv_ci_obancos = _meta_o_respaldo("gpv_ci_obancos", 280000000000)
meta_trx_co_bestado = _meta_o_respaldo("trx_co_bestado", 3000000)
meta_trx_co_obancos = _meta_o_respaldo("trx_co_obancos", 2500000)
meta_gpv_co_bestado = _meta_o_respaldo("gpv_co_bestado", 70000000000)
meta_gpv_co_obancos = _meta_o_respaldo("gpv_co_obancos", 280000000000)
meta_gpv_insurance_credito    = _meta_o_respaldo("gpv_insurance_credito", 199000000)
meta_gpv_insurance_no_credito = _meta_o_respaldo("gpv_insurance_no_credito", 10880000)


algoritmo_gpv_app = "gpv_app"
algoritmo_gpv_paypal = "gpv_paypal"
algoritmo_gpv_paypal_abonos = "gpv_paypal_abonos"
algoritmo_gpv_prp_fis = "gpv_prp_fis"
algoritmo_gpv_prp_vir = "gpv_prp_vir"
algoritmo_gpv_pdc = "gpv_pdc"
algoritmo_gpv_insurance = "gpv_insurance"
algoritmo_gpv_insurance_credito = "gpv_insurance_credito"
algoritmo_gpv_insurance_no_credito = "gpv_insurance_no_credito"
algoritmo_gpv_crossborder = "gpv_crossborder"
algoritmo_gpv_gpv_p2p = "gpv_gpv_p2p"
algoritmo_gpv_top_ups = "gpv_top_ups"
algoritmo_gpv_cash_in_savings = "gpv_cash_in_savings"
algoritmo_gpv_investment_tyba = "gpv_investment_tyba"
algoritmo_gpv_tc_fis = "gpv_tc_fis"
algoritmo_gpv_tc_vir = "gpv_tc_vir"
algoritmo_gpv_p2p = "gpv_p2p"
algoritmo_trx_ci_bestado = "trx_ci_bestado"
algoritmo_trx_ci_obancos = "trx_ci_obancos"
algoritmo_gpv_ci_bestado = "gpv_ci_bestado"
algoritmo_gpv_ci_obancos = "gpv_ci_obancos"
algoritmo_trx_co_bestado = "trx_co_bestado"
algoritmo_trx_co_obancos = "trx_co_obancos"
algoritmo_gpv_co_bestado = "gpv_co_bestado"
algoritmo_gpv_co_obancos = "gpv_co_obancos"
algoritmo_mau_app = "mau_app"
algoritmo_rgu = "rgu"
algoritmo_mau_concurrent = "mau_concurrent"
algoritmo_mau_m0 = "mau_m0"
algoritmo_mau_resurrected = "mau_resurrected"
algoritmo_mau_late_active = "mau_late_active"

algoritmo_mau_tc_fis = "mau_tc_fis"
algoritmo_mau_tc_vir = "mau_tc_vir"
algoritmo_mau_tc_tar = "mau_tc_tar"
algoritmo_mau_prp_fis = "mau_prp_fis"
algoritmo_mau_prp_vir = "mau_prp_vir"
algoritmo_mau_prp_uni = "mau_prp_uni"
algoritmo_mau_p2p = "mau_p2p"
algoritmo_mau_insurance = "mau_insurance"
algoritmo_mau_crossborder = "mau_crossborder"
algoritmo_mau_paypal = "mau_paypal"
algoritmo_mau_paypal_abonos = "mau_paypal_abonos"
algoritmo_mau_top_ups = "mau_top_ups"
algoritmo_mau_pdc = "mau_pdc"

algoritmo_ob_tc_large = "ob_tc_large"
algoritmo_ob_tc_medium = "ob_tc_medium"
algoritmo_ob_tc_small = "ob_tc_small"
algoritmo_ob_tc_xs = "ob_tc_xs"
algoritmo_ob_tenpo = "ob_tenpo"

algoritmo_rev_atm_int = "rev_atm_int"
algoritmo_rev_atm_nac = "rev_atm_nac"

In [ ]:
# ==============================================================================
# V11: CONFIGURACION CENTRAL (todo lo que antes estaba hardcodeado celda por celda)
# ==============================================================================
from datetime import date as _date_v11

INICIO_MES_ACTUAL = hoy_chile().date().replace(day=1)

# 1) Re-entrenamiento: si el .pkl es anterior al dia 1 del mes en curso se re-entrena
#    (antes nunca se re-entrenaba mientras existiera el pkl). REENTRENAR_SIEMPRE fuerza.
REENTRENO_MENSUAL  = True
REENTRENAR_SIEMPRE = False
REENTRENO_CLASE_BASE = False   # True = tambien sub-segmentos reactive y ATM (lento: grilla 84x2)

# 2) Pace V11: promedia la curva MTD/cierre de los ultimos N meses cerrados.
#    Alineacion por KPI: hasta el dia UMBRAL se compara por dia calendario (pesa el
#    dia 1); despues, por DIAS RESTANTES (evita el sesgo 30 vs 31 dias).
#    Backtest jun-ago 26 (error abs. medio al cierre, dia >= 21): GPV 3.4% -> 0.7%,
#    MAU 0.70% -> 0.33%. GPV gana alineando por dias restantes en todo el mes.
PACE_MESES_REF = 3
PACE_UMBRAL_CALENDARIO = {"mau_nuevos": 10, "gpv_mm": 0}

# 3) Factores manuales GPV por producto. Como luego se prorratea al total GPV App,
#    solo mueven el MIX; en V11 quedan neutros (1.0). trx CI/CO no se prorratean.
FACTORES_GPV = {
    "tc_fis": 1.0, "tc_vir": 1.0, "prp_fis": 1.0, "prp_vir": 1.0, "pdc": 1.0,
    "crossborder": 1.0, "paypal": 1.0, "paypal_abonos": 1.0, "top_ups": 1.0,
    "p2p": 1.0, "cash_in_savings": 1.0, "investment_tyba": 1.0,
    "trx_ci_bestado": 1.03, "trx_ci_obancos": 1.03,
    "trx_co_bestado": 1.03, "trx_co_obancos": 1.03,
}

# 4) Eventos (Cyber). UPLIFT = volumen del dia / dia normal del mismo dia de semana.
#    Fuente acciones: Jira AD-532 (2026.10 Cyber): TP 40% cashback 04/10 22h-07/10
#    (AD-528), TC cuotas sin interes 3/6/10 04-12/10 (AD-544), OBTC fase 1 05-11/10 y
#    fase 2 12-25/10 (AD-535), pre-cyber aumento cupo/avances 1-5/10 (AD-540).
#    Uplifts medidos en el Cyber oct-25 (6-8 oct, lun-mie: mismo calendario que 2026)
#    contra la mediana del mismo dia de semana +-2..5 semanas.
#    El modelo YA trae parte del salto (aprende "dia 5-7 de octubre"): _aplicar_cyber
#    solo agrega el RESIDUO uplift / uplift_implicito_del_modelo (nunca < 1).
#    El pace descuenta el exceso Cyber del MTD (si no, sobreestima el cierre d6-d20).
import pandas as pd
CYBER_UPLIFT_GPV = {"2026-10-05": 1.65, "2026-10-06": 1.45, "2026-10-07": 1.45, "2026-10-08": 1.13}
CYBER_UPLIFT_PROD = {   # mix: TC sube mas que prepago en Cyber (oct-25)
    "tc":  {"2026-10-05": 1.94, "2026-10-06": 1.64, "2026-10-07": 1.84, "2026-10-08": 1.11},
    "prp": {"2026-10-05": 1.54, "2026-10-06": 1.34, "2026-10-07": 1.39, "2026-10-08": 1.11},
}
CYBER_UPLIFT_MAU = {}   # oct-25 no movio el MAU del mes (solo adelanta actividad) -> sin ajuste


def _aplicar_cyber(df, col_real, col_pred, uplifts, label):
    """Lleva la prediccion de cada dia Cyber futuro al uplift objetivo, descontando el
    uplift que el modelo ya trae (pred / mediana del mismo dia de semana en semanas
    vecinas no-Cyber). Modifica df in place."""
    if not uplifts:
        return
    f = pd.to_datetime(df['fecha'])
    cy = {pd.Timestamp(x) for x in uplifts}
    for d_str, u in uplifts.items():
        d = pd.Timestamp(d_str)
        i = f == d
        if not i.any() or df.loc[i, col_real].notna().any():
            continue   # fuera del mes o ya ocurrio
        vec = [d + pd.Timedelta(days=k) for k in (7, 14, 21, -7, -14)]
        vec = [x for x in vec if x not in cy and (f == x).any()]
        if not vec:
            continue
        base = df.loc[f.isin(vec), col_pred].median()
        pred = df.loc[i, col_pred].iloc[0]
        if base <= 0 or pred <= 0:
            continue
        impl = pred / base
        fac = max(1.0, u / impl)
        df.loc[i, col_pred] = round(float(pred) * fac)
        print(f"[V11 CYBER {label}] {d.date()} uplift obj={u:.2f} | modelo trae={impl:.2f} | factor={fac:.3f}")

# 5) OB TC: calibracion del modelo contra el real reciente (reemplaza x1.4 / x1.7)
OB_CALIB_DIAS  = 14
OB_CALIB_CLIP  = (0.3, 1.5)
OB_CALIB_MIN_DIAS = 5   # con menos dias reales en el mes se usa el ultimo factor guardado
# Semilla para la 1a corrida V11 (sin json en Drive): real/modelo ultimos 14 dias al 29-09-2026
OB_CALIB_DEFAULT = {"large": 0.553, "medium": 0.397, "small": 0.377, "xs": 0.766}

# 6) RGU: si el modelo no entrega prediccion, respaldo de altas diarias
RGU_RESPALDO_DIA = 2500

# 7) Revenue financiero TC: override manual SOLO vale para el mes indicado
REV_FIN_OVERRIDE = {"mes": "2026-09", "valor": 2552}

# 8) Tolerancia de cuadre MAU App vs suma de subcategorias
TOL_CUADRE_MAU = 0.005

print(f"[V11 CONFIG] mes={INICIO_MES_ACTUAL} | reentreno_mensual={REENTRENO_MENSUAL} "
      f"| pace_meses_ref={PACE_MESES_REF} | cyber_gpv={list(CYBER_UPLIFT_GPV)}")


In [ ]:
# ==============================================================================
# V11: PACE TARGETS alineados por DIAS RESTANTES
# ------------------------------------------------------------------------------
# V8-V10 comparaban el dia k del mes actual con el dia k de UN solo mes anterior. Si
# los meses tienen distinto largo (30 vs 31) el pace se sesga ~1 dia de volumen:
# sep-26 al d29 daba GPV 144.853 MM vs ~141k reales (+3%). Ahora:
#   ratio_m = MTD_m(dia equivalente) / cierre_m   (ver PACE_UMBRAL_CALENDARIO)
#   PACE    = MTD_actual / promedio(ratio_m) sobre los ultimos PACE_MESES_REF meses
# Sin dias reales en el mes (corrida del dia 1) -> PACE = None (se usa el modelo).
# ==============================================================================
import calendar as _cal_p
import pandas as pd
import pandas_gbq

_hoy_p  = hoy_chile()
_ayer_p = _hoy_p - pd.Timedelta(days=1)
_ini_p  = _hoy_p.replace(day=1)
_ini_ref = (_ini_p - pd.DateOffset(months=PACE_MESES_REF)).date()

QUERY_PACE = f'''
WITH b AS (
  SELECT DATE(fecha) AS f, SAFE_CAST(user AS STRING) AS u, monto, linea
  FROM `tenpo-bi-prod.economics.economics`
  WHERE LOWER(IFNULL(nombre, "null")) NOT LIKE "%devol%"
    AND LOWER(IFNULL(actividad_nac_cd, "null")) NOT LIKE "%refund%"
    AND DATE(fecha) BETWEEN '{_ini_ref}' AND '{_ayer_p.date()}'
),
primera AS (
  SELECT u, DATE_TRUNC(f, MONTH) AS m, MIN(f) AS f1
  FROM b WHERE linea IN (SELECT * FROM `tenpo-bi-prod.kpitos.lineas_mau_app`)
  GROUP BY 1, 2
),
mau AS (SELECT f1 AS f, COUNT(*) AS mau_nuevos FROM primera GROUP BY 1),
gpv AS (SELECT f, SUM(monto)/1e6 AS gpv_mm FROM b
        WHERE linea IN (SELECT * FROM `tenpo-bi-prod.kpitos.lineas_gpv_app`) GROUP BY 1)
SELECT f AS fecha, IFNULL(mau_nuevos, 0) AS mau_nuevos, IFNULL(gpv_mm, 0) AS gpv_mm
FROM mau FULL JOIN gpv USING (f)
ORDER BY f
'''

print('[V11 PACE] Consultando serie diaria para pace alineado...')
_df_pace = pandas_gbq.read_gbq(QUERY_PACE, project_id='tenpo-bi-prod', credentials=credentials, dialect='standard')
_df_pace['fecha'] = pd.to_datetime(_df_pace['fecha'])
_df_pace['mes'] = _df_pace['fecha'].dt.to_period('M')

_dias_mes_act = _cal_p.monthrange(_hoy_p.year, _hoy_p.month)[1]
_hay_real_mes = _ayer_p >= _ini_p
_dia_corte    = _ayer_p.day if _hay_real_mes else 0
DIAS_RESTANTES_MES = _dias_mes_act - _dia_corte      # global: la usan MAU/anclaje

def _pace_alineado(col):
    """Devuelve (pace, ratio_prom, detalle) para la columna col."""
    if not _hay_real_mes:
        return None, None, 'sin dias reales en el mes'
    act = _df_pace[_df_pace['mes'] == _ini_p.to_period('M')]
    mtd = act[act['fecha'] <= _ayer_p][col].sum()
    ratios = []
    for m in sorted(_df_pace['mes'].unique()):
        if m >= _ini_p.to_period('M'):
            continue
        x = _df_pace[_df_pace['mes'] == m].sort_values('fecha')
        n = m.days_in_month
        if len(x) < n:              # mes incompleto en la ventana
            continue
        if _dia_corte <= PACE_UMBRAL_CALENDARIO.get(col, 0):
            k = min(_dia_corte, n)       # mismo dia calendario
        else:
            k = n - DIAS_RESTANTES_MES   # mismos dias por delante
        if k < 1:
            continue
        ratios.append((str(m), x[col].iloc[:k].sum() / x[col].sum()))
    ratios = ratios[-PACE_MESES_REF:]
    if not ratios or mtd <= 0:
        return None, None, 'sin meses de referencia'
    r = sum(v for _, v in ratios) / len(ratios)
    # V11: exceso por eventos (Cyber). Los meses de referencia no lo tienen, asi que se
    # saca del MTD antes de escalar y se suma aparte (ocurrido + esperado).
    ups = {'gpv_mm': CYBER_UPLIFT_GPV, 'mau_nuevos': CYBER_UPLIFT_MAU}.get(col, {})
    exc_real = exc_fut = 0.0
    for d_str, u in ups.items():
        d = pd.Timestamp(d_str)
        if d.to_period('M') == _ini_p.to_period('M') and d <= _ayer_p:
            v = act.loc[act['fecha'] == d, col].sum()
            exc_real += v * (1 - 1 / u)
    base = (mtd - exc_real) / r
    for d_str, u in ups.items():
        d = pd.Timestamp(d_str)
        if d.to_period('M') == _ini_p.to_period('M') and d > _ayer_p:
            exc_fut += (u - 1) * base / _dias_mes_act
    if exc_real or exc_fut:
        print(f"[V11 PACE {col}] exceso evento: ocurrido={exc_real:,.0f} | esperado={exc_fut:,.0f}")
    return base + exc_real + exc_fut, r, ratios

_p_mau, _r_mau, _det_mau = _pace_alineado('mau_nuevos')
_p_gpv, _r_gpv, _det_gpv = _pace_alineado('gpv_mm')
PACE_MAU_APP = int(_p_mau) if _p_mau else None
PACE_GPV_APP = float(_p_gpv) if _p_gpv else None

print(f'[V11 PACE] dia_corte={_dia_corte} | dias_restantes={DIAS_RESTANTES_MES} | ref={_det_mau}')
print(f'[V11 PACE] PACE_MAU_APP = {PACE_MAU_APP}  (ratio prom {_r_mau})')
print(f'[V11 PACE] PACE_GPV_APP = {None if PACE_GPV_APP is None else round(PACE_GPV_APP):} MM  (ratio prom {_r_gpv})')
if PACE_MAU_APP is None:
    print('[V11 PACE] !! Sin pace: MAU y GPV usaran solo el modelo.')


In [ ]:
QUERY = """SELECT * FROM `tenpo-bi-prod.kpitos.modelos_forecasting_gpv_productos`
UNION ALL SELECT * FROM `tenpo-bi-prod.kpitos.modelos_forecasting_gpv_insurance`"""
gpvs_productos = pd.read_gbq(QUERY,use_bqstorage_api=True, project_id='tenpo-bi-prod', credentials=credentials, dialect='standard')
gpvs_productos['fecha'] = pd.to_datetime(gpvs_productos['fecha'], format="%Y-%m-%d")
gpv_app = gpvs_productos[~gpvs_productos['linea'].isin(['paypal','paypal_abonos'])]
gpv_app = gpv_app.groupby(['fecha','dia_semana','dia','mes','year','real'])['gpv'].sum().reset_index()

gpv_tc_fis = gpvs_productos[gpvs_productos['linea'] == "credit_card_physical"].drop(columns={'linea'})
gpv_tc_vir = gpvs_productos[gpvs_productos['linea'] == "credit_card_virtual"].drop(columns={'linea'})

gpv_prp_fis = gpvs_productos[gpvs_productos['linea'] == "mastercard_physical"].drop(columns={'linea'})
gpv_prp_vir = gpvs_productos[gpvs_productos['linea'] == "mastercard"].drop(columns={'linea'})

gpv_pdc = gpvs_productos[gpvs_productos['linea'] == "utility_payments"].drop(columns={'linea'})
gpv_paypal = gpvs_productos[gpvs_productos['linea'] == "paypal"].drop(columns={'linea'})
gpv_paypal_abonos = gpvs_productos[gpvs_productos['linea'] == "paypal_abonos"].drop(columns={'linea'})
gpv_crossborder = gpvs_productos[gpvs_productos['linea'] == "crossborder"].drop(columns={'linea'})
gpv_cash_in_savings = gpvs_productos[gpvs_productos['linea'] == "cash_in_savings"].drop(columns={'linea'})
gpv_investment_tyba = gpvs_productos[gpvs_productos['linea'] == "investment_tyba"].drop(columns={'linea'})
gpv_investment_tyba['gpv'] = gpv_investment_tyba['gpv'].replace({0:0.000000000000001})
gpv_p2p = gpvs_productos[gpvs_productos['linea'] == "p2p"].drop(columns={'linea'})
gpv_top_ups = gpvs_productos[gpvs_productos['linea'] == "top_ups"].drop(columns={'linea'})
gpv_p2p = gpvs_productos[gpvs_productos['linea'] == "p2p"].drop(columns={'linea'})

gpv_insurance = gpvs_productos[gpvs_productos['linea'] == "insurance"].drop(columns={'linea'})
gpv_insurance_credito = gpvs_productos[gpvs_productos['linea'] == "insurance_credito"].drop(columns={'linea'})
gpv_insurance_no_credito = gpvs_productos[gpvs_productos['linea'] == "insurance_no_credito"].drop(columns={'linea'})


In [ ]:
QUERY = """SELECT * FROM `tenpo-bi-prod.kpitos.modelos_forecating_trx_cca`"""
kpi_ci = pd.read_gbq(QUERY,use_bqstorage_api=True, project_id='tenpo-bi-prod', credentials=credentials, dialect='standard')
kpi_ci['fecha'] = pd.to_datetime(kpi_ci['fecha'], format="%Y-%m-%d")

trx_ci_bestado = kpi_ci[kpi_ci['cico_banco_tienda_origen'] == 'BANCO ESTADO'].drop(columns=['cico_banco_tienda_origen','gpv_ci'])
trx_ci_obancos = kpi_ci[kpi_ci['cico_banco_tienda_origen'] == 'OTROS BANCOS'].drop(columns=['cico_banco_tienda_origen','gpv_ci'])

gpv_ci_bestado = kpi_ci[kpi_ci['cico_banco_tienda_origen'] == 'BANCO ESTADO'].drop(columns=['cico_banco_tienda_origen','trx_ci'])
gpv_ci_obancos = kpi_ci[kpi_ci['cico_banco_tienda_origen'] == 'OTROS BANCOS'].drop(columns=['cico_banco_tienda_origen','trx_ci'])

In [ ]:
##########CASH OUT
QUERY = """SELECT * FROM `tenpo-bi-prod.kpitos.modelos_forecating_trx_cca_co`"""
kpi_co = pd.read_gbq(QUERY,use_bqstorage_api=True, project_id='tenpo-bi-prod', credentials=credentials, dialect='standard')
kpi_co['fecha'] = pd.to_datetime(kpi_co['fecha'], format="%Y-%m-%d")

trx_co_bestado = kpi_co[kpi_co['cico_banco_tienda_destino'] == 'BANCO ESTADO'].drop(columns=['cico_banco_tienda_destino','gpv_co'])
trx_co_obancos = kpi_co[kpi_co['cico_banco_tienda_destino'] == 'OTROS BANCOS'].drop(columns=['cico_banco_tienda_destino','gpv_co'])

gpv_co_bestado = kpi_co[kpi_co['cico_banco_tienda_destino'] == 'BANCO ESTADO'].drop(columns=['cico_banco_tienda_destino','trx_co'])
gpv_co_obancos = kpi_co[kpi_co['cico_banco_tienda_destino'] == 'OTROS BANCOS'].drop(columns=['cico_banco_tienda_destino','trx_co'])

In [ ]:
# =============================================================================
# EXTRACCIÓN DATA HISTÓRICA REVENUE ATM (NACIONAL E INTERNACIONAL)
# =============================================================================
QUERY_ATM = """
WITH base_retiros AS (
  SELECT
    DATE(fecha) AS fecha,
    DATE_TRUNC(fecha, MONTH) AS mes_calendario,
    user,
    nombre,
    trx_id,
    CASE
      WHEN nombre = 'Cashout ATM Redbank' THEN
        ROW_NUMBER() OVER(PARTITION BY user, DATE_TRUNC(fecha, MONTH), nombre ORDER BY fecha)
      ELSE 0
    END AS nro_retiro_nacional
  FROM `tenpo-bi-prod.economics.economics_performance`
  WHERE linea = 'cash_out'
    AND nombre IN ('Cashout ATM Internacional', 'Cashout ATM Redbank')
)
SELECT
  fecha,
  SUM(CASE WHEN nombre = 'Cashout ATM Internacional' THEN 3000 ELSE 0 END) AS rev_atm_int,
  SUM(CASE WHEN nombre = 'Cashout ATM Redbank' AND nro_retiro_nacional > 4 THEN ROUND(500 / 1.19, 2) ELSE 0 END) AS rev_atm_nac
FROM base_retiros
WHERE fecha <= DATE_SUB(CURRENT_DATE("America/Santiago"), INTERVAL 1 DAY)
GROUP BY 1
ORDER BY 1 DESC
"""
# 1. Leer data desde BQ
df_atm_crudo = pd.read_gbq(QUERY_ATM, use_bqstorage_api=True, project_id='tenpo-bi-prod', credentials=credentials, dialect='standard')
df_atm_crudo['fecha'] = pd.to_datetime(df_atm_crudo['fecha'], format="%Y-%m-%d")

# 2. Separar en dos dataframes para el ForecastProducto y duplicar el target como 'real'
df_rev_atm_int = df_atm_crudo[['fecha', 'rev_atm_int']].copy()
df_rev_atm_int['real'] = df_rev_atm_int['rev_atm_int']

df_rev_atm_nac = df_atm_crudo[['fecha', 'rev_atm_nac']].copy()
df_rev_atm_nac['real'] = df_rev_atm_nac['rev_atm_nac']

# 3. ---> PEGAMOS AQUÍ EL CÁLCULO DE METAS <---
dias_del_mes = df_rev_atm_int['fecha'].dt.daysinmonth.iloc[0]

meta_rev_atm_int = df_rev_atm_int['real'].mean() * dias_del_mes
meta_rev_atm_nac = df_rev_atm_nac['real'].mean() * dias_del_mes

In [ ]:
QUERY = """SELECT * FROM `tenpo-bi-prod.kpitos.modelos_forecasting_mau_app`"""
mau_inicial = pd.read_gbq(QUERY,use_bqstorage_api=True, project_id='tenpo-bi-prod', credentials=credentials, dialect='standard')
mau_inicial['fecha'] = pd.to_datetime(mau_inicial['fecha'], format="%Y-%m-%d")

mau = mau_inicial.copy()
mau_app = mau.groupby(['fecha', 'dia_semana', 'dia', 'mes', 'year','real']).agg({'mau':'sum'}).reset_index()
mau_concurrent = mau[mau['tipo_mau_mes_2'] == "concurrent_user"].drop(columns={'tipo_mau_mes_2'})
mau_resurrected = mau[mau['tipo_mau_mes_2'] == "resurrected_user"].drop(columns={'tipo_mau_mes_2'})
mau_late_active = mau[mau['tipo_mau_mes_2'] == "late_active_user"].drop(columns={'tipo_mau_mes_2'})
mau_m0 = mau[mau['tipo_mau_mes_2'] == "active_user_m0"].drop(columns={'tipo_mau_mes_2'})

In [ ]:
### Query productos sin TC
QUERY = """SELECT * FROM `tenpo-bi-prod.kpitos.modelos_forecasting_mau_productos_sin_tc`"""
mau_productos = pd.read_gbq(QUERY,use_bqstorage_api=True, project_id='tenpo-bi-prod', credentials=credentials, dialect='standard')
mau_productos['fecha'] = pd.to_datetime(mau_productos['fecha'], format="%Y-%m-%d")

### Query TC
QUERY = """SELECT * FROM `tenpo-bi-prod.kpitos.modelos_forecasting_mau_productos_tc`"""
mau_tc_productos = pd.read_gbq(QUERY,use_bqstorage_api=True, project_id='tenpo-bi-prod', credentials=credentials, dialect='standard')
mau_tc_productos['fecha'] = pd.to_datetime(mau_tc_productos['fecha'], format="%Y-%m-%d")

# --- INICIO CORRECCIÓN ---
# Convertir 'mau' a numérico y forzar agregación explícita
mau_tc_productos['mau'] = pd.to_numeric(mau_tc_productos['mau'], errors='coerce')
groupby_cols = ['fecha','dia_semana','dia','mes','year','real']

mau_tc_fis = mau_tc_productos[mau_tc_productos['linea'] == "credit_card_physical"].drop(columns='linea').groupby(groupby_cols).agg({'mau':'sum'}).reset_index()
mau_tc_vir = mau_tc_productos[mau_tc_productos['linea'] == "credit_card_virtual"].drop(columns='linea').groupby(groupby_cols).agg({'mau':'sum'}).reset_index()
mau_tc_tar = mau_tc_productos[mau_tc_productos['linea'] == "credit_card_tarjeta_unico"].drop(columns='linea').groupby(groupby_cols).agg({'mau':'sum'}).reset_index()
mau_tc_tot = mau_tc_productos[mau_tc_productos['linea'] == "credit_card_unico"].drop(columns='linea').groupby(groupby_cols).agg({'mau':'sum'}).reset_index()
# --- FIN CORRECCIÓN ---

mau_prp_fis = mau_productos[mau_productos['linea'] == "mastercard_physical"].drop(columns='linea')
mau_prp_vir = mau_productos[mau_productos['linea'] == "mastercard"].drop(columns='linea')
mau_prp_uni = mau_productos[mau_productos['linea'] == "mastercard_unico"].drop(columns='linea')
mau_p2p = mau_productos[mau_productos['linea'] == "p2p_unico"].drop(columns='linea')
mau_insurance = mau_productos[mau_productos['linea'] == "insurance"].drop(columns='linea')
mau_crossborder = mau_productos[mau_productos['linea'] == "crossborder"].drop(columns='linea')
mau_paypal = mau_productos[mau_productos['linea'] == "paypal"].drop(columns='linea')
mau_paypal_abonos = mau_productos[mau_productos['linea'] == "paypal_abonos"].drop(columns='linea')
mau_top_ups = mau_productos[mau_productos['linea'] == "top_ups"].drop(columns='linea')
mau_pdc = mau_productos[mau_productos['linea'] == "utility_payments"].drop(columns='linea')

In [ ]:
QUERY = """SELECT * FROM `tenpo-bi-prod.kpitos.modelos_forecasting_ob_tc`"""
ob_tc = pd.read_gbq(QUERY,use_bqstorage_api=True, project_id='tenpo-bi-prod', credentials=credentials, dialect='standard')
ob_tc['fecha'] = pd.to_datetime(ob_tc['fecha'], format="%Y-%m-%d")

ob_tc_total = ob_tc.groupby(['fecha','dia_semana','dia','mes','year','real'])['ob'].sum().reset_index()
ob_tc_large = ob_tc[ob_tc['segmento'] == 'Large'].drop(columns=['segmento'])
ob_tc_medium = ob_tc[ob_tc['segmento'] == 'Medium'].drop(columns=['segmento'])
ob_tc_small = ob_tc[ob_tc['segmento'] == 'Small'].drop(columns=['segmento'])
ob_tc_xs = ob_tc[(ob_tc['segmento'] == 'XS') & (ob_tc['fecha'] >= '2024-09-01')].drop(columns=['segmento'])

In [ ]:
QUERY = """SELECT
  DATE(fecha_ob_general) AS fecha,
  FORMAT_DATE('%a', fecha_ob_general) as dia_semana,
  EXTRACT(DAY FROM fecha_ob_general) as dia,
  EXTRACT(MONTH FROM fecha_ob_general) as mes,
  EXTRACT(YEAR FROM fecha_ob_general) as year,
  1 AS real,
  COUNT(DISTINCT user_id) AS mau
FROM `tenpo-bi-prod.users.users_audience_tags`
WHERE fecha_ob_general is not null
AND DATE(fecha_ob_general) <= DATE_SUB(CURRENT_DATE("America/Santiago"), INTERVAL 1 DAY)
GROUP BY 1,2,3,4,5,6
ORDER BY 1 DESC"""
ob_tenpo = pd.read_gbq(QUERY,use_bqstorage_api=True, project_id='tenpo-bi-prod', credentials=credentials, dialect='standard')
ob_tenpo['fecha'] = pd.to_datetime(ob_tenpo['fecha'], format="%Y-%m-%d")

In [ ]:
class ForecastProducto:


    def __init__(self, nombre_algoritmo, meta, fechas_entrenamiento, fechas_proyeccion, directorio):

        self.nombre_algoritmo = nombre_algoritmo
        self.meta = meta
        self.fecha_train_inicio, self.fecha_train_final = fechas_entrenamiento
        self.fecha_proyeccion_inicio, self.inicio_mes, self.fin_mes = fechas_proyeccion
        self.directorio = directorio

        # Atributos que se inicializan durante el procesamiento
        self.model = None
        self.kpi_merge = None
        self.kpi_dummies = None
        self.model_columns = None

    def _cargar_feriados(self):
        """
        Carga datos de feriados desde archivo Excel.

        Returns:
            pd.DataFrame: DataFrame con columnas fecha, feriado, irrenunciable
        """
        ruta_feriados = "/content/drive/.shortcut-targets-by-id/10Ha9KHrUn7dPQ7dGoiEovdVoRhmr5MSZ/Planificación Financiera/Cierres/Proyecciones/feriados.xlsx"
        feriados = pd.read_excel(ruta_feriados)
        return feriados[['fecha', 'feriado', 'irrenunciable']]

    def _agregar_variables_temporales(self, df):
        """
        Agrega variables temporales al DataFrame.

        Args:
            df (pd.DataFrame): DataFrame con columna fecha

        Returns:
            pd.DataFrame: DataFrame con variables temporales agregadas
        """
        # Crear variables temporales básicas
        df['dia_semana'] = df['fecha'].dt.dayofweek
        df['dia'] = df['fecha'].dt.day
        df['mes'] = df['fecha'].dt.month
        df['year'] = df['fecha'].dt.year

        # Mapear días de la semana a nombres
        dias_mapping = {
            0: 'Mon', 1: 'Tues', 2: 'Weds', 3: 'Thurs',
            4: 'Fri', 5: 'Sat', 6: 'Sun'
        }
        df['dia_semana'] = df['dia_semana'].map(dias_mapping)

        # Convertir feriados a booleanos
        df['feriado'] = df['feriado'].astype(bool)
        df['irrenunciable'] = df['irrenunciable'].astype(bool)

        return df

    def _marcar_activacion_tc(self, grupo):
        """
        Marca días de activación de tarjeta de crédito basado en reglas de negocio.

        Args:
            grupo (pd.DataFrame): Grupo de datos por año/mes

        Returns:
            pd.DataFrame: Grupo con columna activacion_tc marcada
        """
        grupo = grupo.sort_values('dia')
        grupo['activacion_tc'] = False

        # Buscar primer día hábil después del día 5 y 20
        for dia_limite in [5, 20]:
            grupo_dia = grupo[grupo['dia'] >= dia_limite]

            for idx, row in grupo_dia.iterrows():
                # Verificar si es día hábil (no fin de semana ni feriado)
                if row['dia_semana'] not in ['Sun', 'Sat'] and not row['feriado']:
                    if not grupo.loc[idx, 'activacion_tc']:
                        grupo.loc[idx, 'activacion_tc'] = True
                    break

        return grupo

    def _preparar_datos(self, kpi):
        """
        Prepara los datos para el entrenamiento del modelo.

        Args:
            kpi (pd.DataFrame): DataFrame con datos del KPI
        """
        # Crear rango completo de fechas
        fechas = pd.date_range('2023-01-01', '2026-12-31')
        kpi = kpi.merge(pd.DataFrame({'fecha': fechas}), on='fecha', how='outer')

        # Agregar feriados
        feriados = self._cargar_feriados()
        kpi = kpi.merge(feriados, on='fecha', how='outer').fillna(0)

        # Agregar variables temporales
        kpi = self._agregar_variables_temporales(kpi)

        # Identificar columna objetivo
        columnas_conocidas = {
            'fecha', 'dia_semana', 'dia', 'mes', 'year',
            'real', 'feriado', 'irrenunciable', 'activacion_tc'
        }
        target_col_name = list(set(kpi.columns) - columnas_conocidas)[0]
        kpi = kpi.rename(columns={target_col_name: 'target_kpi'})

        # Marcar activaciones de TC por grupo año/mes
        kpi = kpi.groupby(['year', 'mes']).apply(self._marcar_activacion_tc).reset_index(drop=True)

        # Guardar datos procesados
        self.kpi_merge = kpi
        self.kpi_modelo = kpi[
            (kpi['fecha'] >= self.fecha_train_inicio) &
            (kpi['fecha'] <= self.fecha_train_final)
        ]
        self.kpi_dummies = pd.get_dummies(self.kpi_modelo.drop(columns='real'))

    def _entrenar_modelo(self):
        """
        Entrena el modelo XGBoost con búsqueda de hiperparámetros.
        """
        # Preparar datos para entrenamiento
        X = self.kpi_dummies.drop(['target_kpi', 'fecha'], axis=1)
        y = self.kpi_dummies['target_kpi']
        self.model_columns = X.columns.tolist()

        # División train/test
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=123
        )

        # Configurar modelo base
        modelo = xgb.XGBRegressor(random_state=123)

        # Definir grilla de hiperparámetros
        param_grid = {
            'nthread': [4],
            'objective': ['reg:squarederror'],
            'learning_rate': [0.001, 0.01, 0.03, 0.05, 0.07, 0.1],
            'max_depth': [1, 2, 3, 4, 5, 6, 7],
            'min_child_weight': [4],
            'subsample': [0.7],
            'colsample_bytree': [0.7],
            'n_estimators': [1000, 1500]
        }

        # Búsqueda de hiperparámetros
        grid = GridSearchCV(
            modelo, param_grid, cv=2, n_jobs=5, verbose=True
        )
        grid.fit(X_train, y_train)

        # Mostrar resultados del entrenamiento
        print(f"Best score: {grid.best_score_}")
        print(f"Best params: {grid.best_params_}")
        print(f"Train accuracy: {grid.score(X_train, y_train)}")
        print(f"Test accuracy: {grid.score(X_test, y_test)}")

        # Guardar modelo
        self.model = grid
        filename = f"{self.directorio}/xgb_{self.nombre_algoritmo}.pkl"
        _dump_pickle_atomic(grid, filename)

    def _aplicar_reglas_negocio(self, kpi_test):
        """
        Aplica reglas de negocio específicas según el algoritmo.

        Args:
            kpi_test (pd.DataFrame): DataFrame con predicciones

        Returns:
            pd.DataFrame: DataFrame con reglas de negocio aplicadas
        """
        # Regla específica para insurance
        if "insurance" in self.nombre_algoritmo:
            mask_insurance = (
                (kpi_test['fecha'].dt.day > 24) &
                (kpi_test['target_kpi_prediccion'] > 1)
            )
            kpi_test.loc[mask_insurance, 'target_kpi_prediccion'] = 0

        return kpi_test

    def _calcular_metricas_acumuladas(self, kpi_test):
        """
        Calcula métricas acumuladas y aplica la meta.

        Args:
            kpi_test (pd.DataFrame): DataFrame con predicciones

        Returns:
            pd.DataFrame: DataFrame con métricas calculadas
        """
        # Combinar predicción con datos reales
        kpi_test['target_kpi_real_pred'] = kpi_test['target_kpi_prediccion']
        mask_real = kpi_test['real'] == 1
        kpi_test.loc[mask_real, 'target_kpi_real_pred'] = kpi_test.loc[mask_real, 'target_kpi']

        # Aplicar límite mínimo
        kpi_test["target_kpi_prediccion"] = kpi_test["target_kpi_prediccion"].clip(lower=1)
        kpi_test["target_kpi_real_pred"] = kpi_test["target_kpi_real_pred"].clip(lower=1)

        # Calcular acumulados mensuales
        kpi_test['periodo'] = kpi_test['fecha'].dt.to_period("M")
        kpi_test['target_kpi_prediccion_cumsum'] = kpi_test.groupby('periodo')['target_kpi_prediccion'].cumsum()
        kpi_test['target_kpi_real_pred_cumsum'] = kpi_test.groupby('periodo')['target_kpi_real_pred'].cumsum()

        # Crear columnas auxiliares para manejo de datos reales
        kpi_test['target_kpi_real_pred_cumsum2'] = kpi_test['target_kpi_real_pred_cumsum']
        kpi_test.loc[mask_real, 'target_kpi_real_pred_cumsum2'] = 0
        kpi_test.loc[~mask_real, 'target_kpi_real_pred_cumsum'] = 0

        # Reemplazar ceros con NaN y filtrar por fechas
        kpi_test = kpi_test.replace(0, np.nan)
        kpi_test = kpi_test[
            (kpi_test['fecha'] >= self.inicio_mes) &
            (kpi_test['fecha'] <= self.fin_mes)
        ]

        # Aplicar meta
        kpi_test['meta'] = self.meta
        total_prediccion = kpi_test['target_kpi_prediccion'].sum()
        kpi_test['target_kpi_prediccion_meta'] = (
            kpi_test['target_kpi_prediccion'] / total_prediccion
        ) * self.meta
        kpi_test['target_kpi_prediccion_meta_cumsum'] = kpi_test['target_kpi_prediccion_meta'].cumsum()

        return kpi_test

    def _limpiar_columnas(self, kpi_test):
        """
        Elimina columnas auxiliares que no son necesarias en el output final.

        Args:
            kpi_test (pd.DataFrame): DataFrame con todas las columnas

        Returns:
            pd.DataFrame: DataFrame con columnas limpias
        """
        columnas_eliminar = [
            'real', 'dia_semana', 'dia', 'mes', 'year',
            'feriado', 'irrenunciable', 'activacion_tc', 'periodo'
        ]
        kpi_test.drop(columns=columnas_eliminar, inplace=True, errors='ignore')
        return kpi_test

    def _renombrar_columnas(self, kpi_test):
        """
        Renombra las columnas según las convenciones del algoritmo.

        Args:
            kpi_test (pd.DataFrame): DataFrame con nombres originales

        Returns:
            pd.DataFrame: DataFrame con columnas renombradas
        """
        columnas = kpi_test.columns.tolist()

        # Determinar sufijo según tipo de algoritmo
        if self.nombre_algoritmo.startswith(("ob_tc", "ob_tenpo")):
            sufijo = self.nombre_algoritmo[self.nombre_algoritmo.rfind('_'):]
        else:
            sufijo = f"_{self.nombre_algoritmo[4:]}"

        # Aplicar sufijo a todas las columnas excepto fecha
        nuevas_columnas = [columnas[0]] + [f"{col}{sufijo}" for col in columnas[1:]]
        kpi_test.columns = nuevas_columnas

        # Reemplazar 'target_kpi' con prefijo del algoritmo
        if self.nombre_algoritmo.startswith(("ob_tc", "ob_tenpo")):
            reemplazo = self.nombre_algoritmo[:2]
        else:
            reemplazo = self.nombre_algoritmo[:3]

        kpi_test.columns = kpi_test.columns.str.replace('target_kpi', reemplazo)

        # Ajustes específicos para ob_tc
        if self.nombre_algoritmo.startswith("ob_tc"):
            nuevas_columnas = []
            for col in kpi_test.columns:
                if col.startswith("meta") and not col.startswith("meta_ob_tc"):
                    nueva_col = col.replace("meta", "meta_ob_tc", 1)
                else:
                    nueva_col = col
                nuevas_columnas.append(nueva_col)
            kpi_test.columns = nuevas_columnas

        # Ajustes específicos para ob_tenpo
        if self.nombre_algoritmo == "ob_tenpo":
            nuevas_columnas = []
            for col in kpi_test.columns:
                if (col.startswith("ob_") and
                    not col.startswith("ob_tenpo") and
                    col.endswith("_tenpo")):
                    base = col[len("ob_"):-len("_tenpo")]
                    nueva_col = f"ob_tenpo_{base}"
                    nuevas_columnas.append(nueva_col)
                else:
                    nuevas_columnas.append(col)
            kpi_test.columns = nuevas_columnas

        # Ajuste específico para ob_tenp
        if self.nombre_algoritmo == "ob_tenp":
            kpi_test = kpi_test.rename(columns={"meta_tenpo": "meta"})

        return kpi_test

    def proyectar(self, kpi):
        """
        Realiza la proyección usando el modelo entrenado.

        Args:
            kpi (pd.DataFrame): DataFrame con datos del KPI

        Returns:
            pd.DataFrame: DataFrame con proyecciones y métricas calculadas
        """
        # Preparar datos si no se han preparado
        if self.kpi_merge is None or self.kpi_dummies is None:
            self._preparar_datos(kpi)

        # V11: re-entreno mensual. Antes, si el .pkl existia NUNCA se re-entrenaba.
        if self.model is None:
            import os
            from datetime import date as _d
            _pkl = f"{self.directorio}/xgb_{self.nombre_algoritmo}.pkl"
            _viejo = (os.path.exists(_pkl) and
                      _d.fromtimestamp(os.path.getmtime(_pkl)) < hoy_chile().date().replace(day=1))
            _es_base = type(self).__name__ == "ForecastProducto"
            _aplica = (not _es_base) or globals().get("REENTRENO_CLASE_BASE", False)
            if _aplica and (globals().get("REENTRENAR_SIEMPRE", False) or (
                    globals().get("REENTRENO_MENSUAL", True) and _viejo)):
                print(f"[V11 REENTRENO] {self.nombre_algoritmo}: pkl anterior al mes en curso -> re-entrenando")
                self._entrenar_modelo()

        # Cargar o entrenar modelo
        if self.model is None:
            try:
                filename = f"{self.directorio}/xgb_{self.nombre_algoritmo}.pkl"
                self.model = pickle.load(open(filename, 'rb'))

                # Obtener columnas del modelo desde los datos preparados
                temp_X = self.kpi_dummies.drop(['target_kpi', 'fecha'], axis=1)
                self.model_columns = temp_X.columns.tolist()
                # V12: si el pkl trae sus propias features, usarlas (un pkl de otra version/feature-set
                # ya no rompe el predict; las faltantes se rellenan con 0 en reindex).
                _mc = _columnas_del_modelo(self.model)
                if _mc:
                    self.model_columns = _mc
            except Exception as _e:
                print(f"[V12] pkl ausente/ilegible ({type(_e).__name__}): re-entrenando {self.nombre_algoritmo}")
                self._entrenar_modelo()

        # Preparar datos de prueba
        kpi_test = self.kpi_merge[self.kpi_merge['fecha'] >= self.fecha_proyeccion_inicio]
        test_dummies = pd.get_dummies(kpi_test).drop(
            columns=['fecha', 'target_kpi', 'real'], errors='ignore'
        )

        # Asegurar que las columnas coincidan con el modelo
        test_dummies = test_dummies.reindex(columns=self.model_columns, fill_value=0)

        # Realizar predicción
        pred = self.model.predict(test_dummies)
        pred_df = pd.DataFrame(pred.astype(np.int64), columns=['target_kpi_prediccion'])
        pred_df['fecha'] = kpi_test['fecha'].values
        kpi_test = kpi_test.merge(pred_df, on='fecha')

        # Aplicar reglas de negocio
        kpi_test = self._aplicar_reglas_negocio(kpi_test)

        # Calcular métricas acumuladas
        kpi_test = self._calcular_metricas_acumuladas(kpi_test)

        # Limpiar columnas auxiliares
        kpi_test = self._limpiar_columnas(kpi_test)

        # Renombrar columnas según convenciones
        kpi_test = self._renombrar_columnas(kpi_test)

        return kpi_test

## ForecastProductoV2: entrenamiento honesto

Clase derivada que sólo sobrescribe `_entrenar_modelo`. Cambios respecto a la base:

1. **`TimeSeriesSplit(n_splits=5)`** en vez de `train_test_split` aleatorio → sin leakage temporal.
2. **Scoring por MAPE** (`neg_mean_absolute_percentage_error`) → alineado con la métrica de negocio.
3. **Holdout temporal** de los últimos `holdout_dias` días reservado fuera del grid → métrica final honesta.
4. **Grilla reducida** a las combinaciones que sí convergen (sin `lr=0.001` con `n_estimators=1000`).
5. Guarda el `.pkl` como `xgb_<algoritmo>.pkl` igual que la base; usar `nombre_algoritmo` con sufijo `_v2` para no pisarlo.

Además se redefine `_refit_y_predecir` para que respete la clase del template (`type(template)`),
así el backtest funciona tanto con V1 como con V2.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_percentage_error


class ForecastProductoV2(ForecastProducto):
    """Versión con entrenamiento metodológicamente honesto."""

    def __init__(self, *args, holdout_dias=30, **kwargs):
        super().__init__(*args, **kwargs)
        self.holdout_dias = holdout_dias
        self.holdout_metrics = None

    def _entrenar_modelo(self):
        # 1. Ordenar por fecha (crucial para TimeSeriesSplit)
        df = self.kpi_dummies.sort_values('fecha').reset_index(drop=True)
        X = df.drop(['target_kpi', 'fecha'], axis=1)
        y = df['target_kpi']
        fechas = df['fecha']
        self.model_columns = X.columns.tolist()

        # 2. Holdout temporal: últimos `holdout_dias` días
        n_h = self.holdout_dias
        X_tr, X_h = X.iloc[:-n_h], X.iloc[-n_h:]
        y_tr, y_h = y.iloc[:-n_h], y.iloc[-n_h:]
        f_h       = fechas.iloc[-n_h:]

        # 3. Grilla reducida (12 combos × 5 folds = 60 fits)
        param_grid = {
            'learning_rate':   [0.05, 0.1],
            'max_depth':       [3, 4, 5],
            'n_estimators':    [500, 1000],
            'min_child_weight':[4],
            'subsample':       [0.8],
            'colsample_bytree':[0.8],
            'objective':       ['reg:squarederror'],
        }

        tscv = TimeSeriesSplit(n_splits=5)
        modelo = xgb.XGBRegressor(random_state=123)
        grid = GridSearchCV(
            modelo, param_grid, cv=tscv,
            scoring='neg_mean_absolute_percentage_error',
            n_jobs=5, verbose=True,
        )
        grid.fit(X_tr, y_tr)

        # 4. Evaluar en holdout temporal (métrica honesta)
        pred_h = grid.predict(X_h)
        mask = y_h > 0
        mape  = mean_absolute_percentage_error(y_h[mask], pred_h[mask]) * 100
        sesgo = ((pred_h[mask] - y_h[mask]) / y_h[mask]).mean() * 100
        self.holdout_metrics = {
            'mape_pct': round(mape, 2),
            'sesgo_pct': round(sesgo, 2),
            'n_dias': int(mask.sum()),
            'desde': str(f_h.min().date()),
            'hasta': str(f_h.max().date()),
        }

        print(f"Best CV score (neg MAPE): {grid.best_score_:.4f}")
        print(f"Best params: {grid.best_params_}")
        print(f"Holdout temporal ({self.holdout_metrics['desde']} a {self.holdout_metrics['hasta']}):")
        print(f"  MAPE  : {mape:.2f}%")
        print(f"  Sesgo : {sesgo:+.2f}%")

        self.model = grid
        filename = f"{self.directorio}/xgb_{self.nombre_algoritmo}.pkl"
        _dump_pickle_atomic(grid, filename)


# --- redefinir _refit_y_predecir para que respete la clase del template ---
def _refit_y_predecir(template, kpi_df, fecha_train_inicio, fecha_corte,
                      horizonte_dias, usar_grid_existente=True):
    inicio_pred = pd.Timestamp(fecha_corte) + pd.Timedelta(days=1)
    fin_pred    = inicio_pred + pd.Timedelta(days=horizonte_dias - 1)

    fcst_class = type(template)            # usa V1 o V2 según el template
    init_kwargs = dict(
        nombre_algoritmo=template.nombre_algoritmo,
        meta=template.meta,
        fechas_entrenamiento=(str(fecha_train_inicio), str(fecha_corte)),
        fechas_proyeccion=(str(inicio_pred),
                           str(inicio_pred.replace(day=1)),
                           str(fin_pred)),
        directorio=template.directorio,
    )
    if isinstance(template, ForecastProductoV2):
        init_kwargs['holdout_dias'] = template.holdout_dias
    fcst = fcst_class(**init_kwargs)
    fcst._preparar_datos(kpi_df.copy())

    X = fcst.kpi_dummies.drop(['target_kpi', 'fecha'], axis=1)
    y = fcst.kpi_dummies['target_kpi']

    pkl_path = f"{template.directorio}/xgb_{template.nombre_algoritmo}.pkl"
    best_params = pickle.load(open(pkl_path, 'rb')).best_params_
    model = xgb.XGBRegressor(**best_params, random_state=123)
    model.fit(X, y)

    fcst.model = model
    fcst.model_columns = X.columns.tolist()

    test = fcst.kpi_merge[
        (fcst.kpi_merge['fecha'] >= inicio_pred) &
        (fcst.kpi_merge['fecha'] <= fin_pred)
    ]
    test_dummies = (
        pd.get_dummies(test)
          .drop(columns=['fecha', 'target_kpi', 'real'], errors='ignore')
          .reindex(columns=fcst.model_columns, fill_value=0)
    )
    pred = model.predict(test_dummies).astype(np.int64)
    return pd.DataFrame({
        'fecha': test['fecha'].values,
        'pred': pred,
        'fecha_corte': pd.Timestamp(fecha_corte),
    })


## ForecastProductoV3: V2 + log-transform del target

Ataca el síntoma de **`last_estimated` alto al inicio del mes que se desinfla** y el **sesgo positivo sistemático** del backtest.

**Diagnóstico**: el target tiene picos (día 1 del mes ≈ 10× el resto). Con `reg:squarederror`, esos picos arrastran la predicción de los días planos hacia arriba (RMSE pesa los errores al cuadrado).

**Cambio**: envuelve el `XGBRegressor` en `TransformedTargetRegressor(log1p / expm1)`. Entrena en log-space y predice en escala original automáticamente.

Hereda todo lo de V2 (`TimeSeriesSplit` + scoring por MAPE + holdout temporal).

También redefine `_refit_y_predecir` para usar `sklearn.base.clone(best_estimator)` en vez de reconstruir `XGBRegressor` desde `best_params`. Eso lo hace polimórfico: funciona con V1, V2 y V3 sin tocar nada.

In [ ]:
from sklearn.compose import TransformedTargetRegressor
from sklearn.base import clone


class ForecastProductoV3(ForecastProductoV2):
    """V2 + target en log-space para domar picos."""

    def _entrenar_modelo(self):
        df = self.kpi_dummies.sort_values('fecha').reset_index(drop=True)
        X = df.drop(['target_kpi', 'fecha'], axis=1)
        y = df['target_kpi'].clip(lower=0)  # evita negativos por si acaso
        fechas = df['fecha']
        self.model_columns = X.columns.tolist()

        n_h = self.holdout_dias
        X_tr, X_h = X.iloc[:-n_h], X.iloc[-n_h:]
        y_tr, y_h = y.iloc[:-n_h], y.iloc[-n_h:]
        f_h       = fechas.iloc[-n_h:]

        base = xgb.XGBRegressor(random_state=123)
        modelo = TransformedTargetRegressor(
            regressor=base, func=np.log1p, inverse_func=np.expm1,
        )

        # mismos rangos que V2 pero con prefijo regressor__
        param_grid = {
            'regressor__learning_rate':   [0.05, 0.1],
            'regressor__max_depth':       [3, 4, 5],
            'regressor__n_estimators':    [500, 1000],
            'regressor__min_child_weight':[4],
            'regressor__subsample':       [0.8],
            'regressor__colsample_bytree':[0.8],
            'regressor__objective':       ['reg:squarederror'],
        }

        tscv = TimeSeriesSplit(n_splits=5)
        grid = GridSearchCV(
            modelo, param_grid, cv=tscv,
            scoring='neg_mean_absolute_percentage_error',
            n_jobs=5, verbose=True,
        )
        grid.fit(X_tr, y_tr)

        # holdout temporal — predict ya devuelve escala original
        pred_h = grid.predict(X_h)
        mask = y_h > 0
        mape  = mean_absolute_percentage_error(y_h[mask], pred_h[mask]) * 100
        sesgo = ((pred_h[mask] - y_h[mask]) / y_h[mask]).mean() * 100
        self.holdout_metrics = {
            'mape_pct': round(mape, 2),
            'sesgo_pct': round(sesgo, 2),
            'n_dias': int(mask.sum()),
            'desde': str(f_h.min().date()),
            'hasta': str(f_h.max().date()),
        }
        print(f"Best CV score (neg MAPE log-space): {grid.best_score_:.4f}")
        print(f"Best params: {grid.best_params_}")
        print(f"Holdout temporal ({self.holdout_metrics['desde']} a {self.holdout_metrics['hasta']}):")
        print(f"  MAPE  : {mape:.2f}%")
        print(f"  Sesgo : {sesgo:+.2f}%")

        self.model = grid
        filename = f"{self.directorio}/xgb_{self.nombre_algoritmo}.pkl"
        _dump_pickle_atomic(grid, filename)


# ---- _refit_y_predecir polimórfico (usa clone, sirve para V1/V2/V3) ----
def _refit_y_predecir(template, kpi_df, fecha_train_inicio, fecha_corte,
                      horizonte_dias, usar_grid_existente=True):
    inicio_pred = pd.Timestamp(fecha_corte) + pd.Timedelta(days=1)
    fin_pred    = inicio_pred + pd.Timedelta(days=horizonte_dias - 1)

    fcst_class = type(template)
    init_kwargs = dict(
        nombre_algoritmo=template.nombre_algoritmo,
        meta=template.meta,
        fechas_entrenamiento=(str(fecha_train_inicio), str(fecha_corte)),
        fechas_proyeccion=(str(inicio_pred),
                           str(inicio_pred.replace(day=1)),
                           str(fin_pred)),
        directorio=template.directorio,
    )
    if isinstance(template, ForecastProductoV2):
        init_kwargs['holdout_dias'] = template.holdout_dias
    fcst = fcst_class(**init_kwargs)
    fcst._preparar_datos(kpi_df.copy())

    X = fcst.kpi_dummies.drop(['target_kpi', 'fecha'], axis=1)
    y = fcst.kpi_dummies['target_kpi'].clip(lower=0)

    pkl_path = f"{template.directorio}/xgb_{template.nombre_algoritmo}.pkl"
    loaded = pickle.load(open(pkl_path, 'rb'))
    model = clone(loaded.best_estimator_)   # respeta wrapper (TransformedTarget...)
    model.fit(X, y)

    fcst.model = model
    fcst.model_columns = X.columns.tolist()

    test = fcst.kpi_merge[
        (fcst.kpi_merge['fecha'] >= inicio_pred) &
        (fcst.kpi_merge['fecha'] <= fin_pred)
    ]
    test_dummies = (
        pd.get_dummies(test)
          .drop(columns=['fecha', 'target_kpi', 'real'], errors='ignore')
          .reindex(columns=fcst.model_columns, fill_value=0)
    )
    pred = model.predict(test_dummies)
    pred = np.maximum(pred, 0).astype(np.int64)  # clip a 0 por seguridad
    return pd.DataFrame({
        'fecha': test['fecha'].values,
        'pred': pred,
        'fecha_corte': pd.Timestamp(fecha_corte),
    })


## ForecastProductoV4: V3 + filtro `real == 1` (bugfix heredado)

**Bug detectado en `mau_paypal`**: la clase base hace `pd.date_range('2023-01-01','2026-12-31')` + `fillna(0)` en `_preparar_datos`, lo que mete al training días sin observar con `target=0`. Para KPIs cuya data llega hasta `fecha_train_final` no se nota (`mau_app`); para KPIs con data desfasada (`mau_paypal`), el holdout queda lleno de ceros falsos y `MAPE` crashea — y peor, el modelo aprende esos ceros falsos como señal real.

**Fix**: filtrar `real == 1` antes de armar X/y. Aplica a todos los KPIs y debería bajar el sesgo aún más en los que tienen data un poco desfasada.

También se redefine `_refit_y_predecir` para filtrar `real == 1` en cada refit del backtest, manteniendo la metodología consistente.

In [ ]:
class ForecastProductoV4(ForecastProductoV3):
    """V3 + filtra real==1 antes de entrenar (evita ceros falsos del fillna)."""

    def _entrenar_modelo(self):
        # Tomar real desde kpi_modelo (mismo orden que kpi_dummies antes de sort)
        df = self.kpi_dummies.copy()
        df['_real_flag'] = self.kpi_modelo['real'].values
        df = df.sort_values('fecha').reset_index(drop=True)
        df = df[df['_real_flag'] == 1].reset_index(drop=True)
        df = df.drop(columns='_real_flag')

        X = df.drop(['target_kpi', 'fecha'], axis=1)
        y = df['target_kpi'].clip(lower=0)
        fechas = df['fecha']
        self.model_columns = X.columns.tolist()

        n_h = self.holdout_dias
        if len(df) <= n_h:
            raise RuntimeError(
                f"Solo {len(df)} días observados; holdout={n_h} es >= que el dataset."
            )
        X_tr, X_h = X.iloc[:-n_h], X.iloc[-n_h:]
        y_tr, y_h = y.iloc[:-n_h], y.iloc[-n_h:]
        f_h       = fechas.iloc[-n_h:]

        base = xgb.XGBRegressor(random_state=123)
        modelo = TransformedTargetRegressor(
            regressor=base, func=np.log1p, inverse_func=np.expm1,
        )
        param_grid = {
            'regressor__learning_rate':   [0.05, 0.1],
            'regressor__max_depth':       [3, 4, 5],
            'regressor__n_estimators':    [500, 1000],
            'regressor__min_child_weight':[4],
            'regressor__subsample':       [0.8],
            'regressor__colsample_bytree':[0.8],
            'regressor__objective':       ['reg:squarederror'],
        }
        tscv = TimeSeriesSplit(n_splits=5)
        grid = GridSearchCV(
            modelo, param_grid, cv=tscv,
            scoring='neg_mean_absolute_percentage_error',
            n_jobs=5, verbose=True,
        )
        grid.fit(X_tr, y_tr)

        pred_h = grid.predict(X_h)
        mask = y_h > 0
        if mask.sum() == 0:
            print("WARN: holdout sin valores > 0. Saltando métricas.")
            self.holdout_metrics = {'mape_pct': None, 'sesgo_pct': None,
                                    'n_dias': 0,
                                    'desde': str(f_h.min().date()),
                                    'hasta': str(f_h.max().date())}
        else:
            mape  = mean_absolute_percentage_error(y_h[mask], pred_h[mask]) * 100
            sesgo = ((pred_h[mask] - y_h[mask]) / y_h[mask]).mean() * 100
            self.holdout_metrics = {
                'mape_pct': round(mape, 2),
                'sesgo_pct': round(sesgo, 2),
                'n_dias': int(mask.sum()),
                'desde': str(f_h.min().date()),
                'hasta': str(f_h.max().date()),
            }
            print(f"Best CV score (neg MAPE log-space): {grid.best_score_:.4f}")
            print(f"Best params: {grid.best_params_}")
            print(f"Holdout temporal ({self.holdout_metrics['desde']} a {self.holdout_metrics['hasta']}):")
            print(f"  MAPE  : {mape:.2f}%   Sesgo : {sesgo:+.2f}%")

        # V11: el holdout solo sirve para medir. Antes el modelo final quedaba entrenado
        # SIN los ultimos holdout_dias (= sin el mes mas reciente). Refit con todo.
        _final = clone(grid.best_estimator_)
        _final.fit(X, y)
        grid.best_estimator_ = _final
        print(f"  [V11] refit final con {len(X)} dias (incluye holdout)")

        self.model = grid
        filename = f"{self.directorio}/xgb_{self.nombre_algoritmo}.pkl"
        _dump_pickle_atomic(grid, filename)


# ---- _refit_y_predecir polimórfico CON filtro real==1 ----
def _refit_y_predecir(template, kpi_df, fecha_train_inicio, fecha_corte,
                      horizonte_dias, usar_grid_existente=True):
    inicio_pred = pd.Timestamp(fecha_corte) + pd.Timedelta(days=1)
    fin_pred    = inicio_pred + pd.Timedelta(days=horizonte_dias - 1)

    fcst_class = type(template)
    init_kwargs = dict(
        nombre_algoritmo=template.nombre_algoritmo,
        meta=template.meta,
        fechas_entrenamiento=(str(fecha_train_inicio), str(fecha_corte)),
        fechas_proyeccion=(str(inicio_pred),
                           str(inicio_pred.replace(day=1)),
                           str(fin_pred)),
        directorio=template.directorio,
    )
    if isinstance(template, ForecastProductoV2):
        init_kwargs['holdout_dias'] = template.holdout_dias
    fcst = fcst_class(**init_kwargs)
    fcst._preparar_datos(kpi_df.copy())

    # filtrar real==1 antes del refit
    df_fit = fcst.kpi_dummies.copy()
    df_fit['_real_flag'] = fcst.kpi_modelo['real'].values
    df_fit = df_fit[df_fit['_real_flag'] == 1].drop(columns='_real_flag')

    X = df_fit.drop(['target_kpi', 'fecha'], axis=1)
    y = df_fit['target_kpi'].clip(lower=0)

    pkl_path = f"{template.directorio}/xgb_{template.nombre_algoritmo}.pkl"
    loaded = pickle.load(open(pkl_path, 'rb'))
    model = clone(loaded.best_estimator_)
    model.fit(X, y)

    fcst.model = model
    fcst.model_columns = X.columns.tolist()

    test = fcst.kpi_merge[
        (fcst.kpi_merge['fecha'] >= inicio_pred) &
        (fcst.kpi_merge['fecha'] <= fin_pred)
    ]
    test_dummies = (
        pd.get_dummies(test)
          .drop(columns=['fecha', 'target_kpi', 'real'], errors='ignore')
          .reindex(columns=fcst.model_columns, fill_value=0)
    )
    pred = model.predict(test_dummies)
    pred = np.maximum(pred, 0).astype(np.int64)
    return pd.DataFrame({
        'fecha': test['fecha'].values,
        'pred': pred,
        'fecha_corte': pd.Timestamp(fecha_corte),
    })


## ForecastProductoV5: V4 + features de tendencia mensual

**Síntoma a corregir**: V4 corrige el sesgo del `last_estimated` (resuelve el problema original), pero proyecta plano contra una tendencia creciente sostenida (mayo cerró en 976,838 con MoM +0.7%, junio se proyectaba en 976,749 → ~0% MoM).

**Causa**: las features actuales (`dia_semana, dia, mes, year, feriado, irrenunciable, activacion_tc`) sólo capturan estacionalidad. El modelo no tiene cómo aprender 'el nivel del mes anterior fue X, esperaría continuidad o crecimiento'.

**Fix**: override de `_preparar_datos` para agregar dos features de tendencia:
1. `lag_mau_mes_anterior`: total del KPI del mes inmediatamente anterior (calculado sólo con días `real==1` para no contaminar con ceros falsos del `fillna`).
2. `ma_3m_mes_anterior`: media móvil del total mensual de los 3 meses anteriores.

El `_entrenar_modelo` se hereda de V4 sin cambios (`TimeSeriesSplit + scoring MAPE + log-transform + filtro real==1`).

In [ ]:
class ForecastProductoV5(ForecastProductoV4):
    """V4 + lag features mensuales para capturar tendencia."""

    def _preparar_datos(self, kpi):
        # 1. Llamar a la lógica base (date_range + feriados + temporales + activacion_tc)
        super()._preparar_datos(kpi)

        # 2. Calcular features de lag sobre el kpi_merge ya armado
        df = self.kpi_merge.copy()

        # Total mensual usando sólo días observados (real==1) → evita ceros falsos
        observados = df[df['real'] == 1].copy()
        observados['periodo'] = observados['fecha'].dt.to_period('M').astype(str)

        mensual = (observados.groupby('periodo')['target_kpi']
                              .sum().reset_index()
                              .rename(columns={'target_kpi': 'mau_mes_total'})
                              .sort_values('periodo').reset_index(drop=True))

        # Lag-1 (mes anterior) y MA de los 3 meses anteriores
        mensual['lag_mau_mes_anterior'] = mensual['mau_mes_total'].shift(1)
        mensual['ma_3m_mes_anterior']   = (mensual['mau_mes_total'].shift(1)
                                                    .rolling(3, min_periods=1).mean())

        # Fill NaN de primeros meses con bfill (usa el siguiente disponible)
        mensual['lag_mau_mes_anterior'] = mensual['lag_mau_mes_anterior'].bfill().fillna(0)
        mensual['ma_3m_mes_anterior']   = mensual['ma_3m_mes_anterior'].bfill().fillna(0)

        # 3. Merge por periodo
        df['periodo'] = df['fecha'].dt.to_period('M').astype(str)
        # V12: un mes SIN dias observados (dia 1, o datos atrasados) no tenia fila en `mensual` y sus
        # features de lag quedaban en 0 (el modelo se entreno con ~1e5-1e11) -> cierre ~50% bajo.
        # Ahora hereda el ultimo mes observado anterior. Meses CON datos: identicos a V11.
        _faltan = sorted(set(df['periodo']) - set(mensual['periodo']))
        if V12_FIXES.get('lag_mes_sin_datos', True) and _faltan:
            _filas = []
            for _p in _faltan:
                _prev = mensual.loc[mensual['periodo'] < _p, 'mau_mes_total']
                if len(_prev):
                    _filas.append({'periodo': _p, 'lag_mau_mes_anterior': float(_prev.iloc[-1]),
                                   'ma_3m_mes_anterior': float(_prev.tail(3).mean())})
            if _filas:
                mensual = pd.concat([mensual, pd.DataFrame(_filas)], ignore_index=True)
        df = df.merge(
            mensual[['periodo', 'lag_mau_mes_anterior', 'ma_3m_mes_anterior']],
            on='periodo', how='left',
        ).drop(columns='periodo')

        df['lag_mau_mes_anterior'] = df['lag_mau_mes_anterior'].fillna(0)
        df['ma_3m_mes_anterior']   = df['ma_3m_mes_anterior'].fillna(0)

        # 4. Re-armar kpi_modelo y kpi_dummies con las nuevas features incluidas
        self.kpi_merge = df
        self.kpi_modelo = df[
            (df['fecha'] >= self.fecha_train_inicio) &
            (df['fecha'] <= self.fecha_train_final)
        ]
        self.kpi_dummies = pd.get_dummies(self.kpi_modelo.drop(columns='real'))


In [ ]:
import os
import datetime
import pandas as pd
import pandas_gbq

print("\n=========================================================")
print(" INICIANDO PIPELINE DE RGU (EXTRACCIÓN, MODELO Y LE)")
print("=========================================================\n")

# -----------------------------------------------------------------------------
# PASO 1: EXTRACCIÓN HISTÓRICA INCREMENTAL (Solo "Nuevos" RGUs del mes por día)
# -----------------------------------------------------------------------------
print("Extrayendo historial de RGU Incremental...")

QUERY_RGU = """
WITH saldo AS (SELECT DISTINCT DATE_SUB(fecha, INTERVAL 1 DAY) AS fecha, user FROM `tenpo-bi-prod.balance.daily_balance` WHERE saldo_dia > 0),
economics AS (SELECT DISTINCT user, DATE(fecha) AS fecha FROM `tenpo-bi-prod.economics.economics` WHERE linea IN (SELECT * FROM `tenpo-bi-prod.kpitos.lineas_mau_app`) AND LOWER(IFNULL(nombre,"null")) NOT LIKE "%devolu%" AND LOWER(IFNULL(actividad_nac_cd,"null")) NOT LIKE "%refund%"),
revenue_fin AS (SELECT DISTINCT DATE(fecha) AS fecha, user FROM `tenpo-bi-prod.kpitos.revenue_tc_detalle`),
users_tc AS (SELECT CAST(individual_id AS STRING) AS id_persona, user_id FROM `tenpo-bi-prod.kpitos.tipo_users_tc` WHERE name_segment = "XS"),
base_saldos_tca AS (SELECT DISTINCT DATE(Fecha_Proceso) AS fecha, user_id AS user FROM `operacionestc-operaciones-prod.MAE.Cuentas` INNER JOIN users_tc USING(id_persona) WHERE Status_id NOT IN ('200','6','14') AND Cupo_Total <> '0' AND id_cuenta NOT IN ('1','2','1733','1734','1735','1736','1737')),
universo_rgu AS (
  SELECT fecha, user FROM saldo UNION DISTINCT SELECT fecha, user FROM economics UNION DISTINCT SELECT fecha, user FROM revenue_fin UNION DISTINCT SELECT fecha, user FROM base_saldos_tca
),
primera_actividad_mes AS (
  SELECT user, DATE_TRUNC(fecha, MONTH) AS mes, MIN(fecha) AS fecha_activacion FROM universo_rgu WHERE fecha >= '2023-01-01' AND fecha < CURRENT_DATE("America/Santiago") GROUP BY 1, 2
)
SELECT
  fecha_activacion AS fecha,
  FORMAT_DATE('%a', fecha_activacion) as dia_semana,
  EXTRACT(DAY FROM fecha_activacion) as dia,
  EXTRACT(MONTH FROM fecha_activacion) as mes,
  EXTRACT(YEAR FROM fecha_activacion) as year,
  1 AS real,
  COUNT(DISTINCT user) AS rgu
FROM primera_actividad_mes
GROUP BY 1, 2, 3, 4, 5
ORDER BY fecha DESC
"""

rgu_historico = pd.read_gbq(QUERY_RGU, use_bqstorage_api=True, project_id='tenpo-bi-prod', credentials=credentials, dialect='standard')
rgu_historico['fecha'] = pd.to_datetime(rgu_historico['fecha'], format="%Y-%m-%d")


# -----------------------------------------------------------------------------
# PASO 2: PROYECCIÓN DEL MODELO (¡CON BORRADO DE CACHÉ!)
# -----------------------------------------------------------------------------
print("Proyectando RGU...")

# ESTA ES LA CLAVE: Borramos el modelo viejo para obligar al algoritmo a re-entrenar con los datos incrementales
ruta_modelo_rgu = f"{directorio_salida}/xgb_{algoritmo_rgu}.pkl"
_bak_rgu = ruta_modelo_rgu + ".bak"
if os.path.exists(ruta_modelo_rgu):
    if V12_FIXES.get("rgu_pkl_respaldo", True):
        os.replace(ruta_modelo_rgu, _bak_rgu)   # V12: se respalda; si el re-entreno falla se restaura
    else:
        os.remove(ruta_modelo_rgu)
    print("🗑️ Modelo viejo eliminado de Drive. Re-entrenando modelo XGBoost...")

forecast_rgu = ForecastProducto(
    nombre_algoritmo=algoritmo_rgu,
    meta=meta_rgu,
    fechas_entrenamiento=fechas_entrenamiento,
    fechas_proyeccion=fechas_proyeccion,
    directorio=directorio_salida
)
try:
    forecast_rgu = forecast_rgu.proyectar(rgu_historico)
except BaseException:
    if os.path.exists(_bak_rgu) and not os.path.exists(ruta_modelo_rgu):
        os.replace(_bak_rgu, ruta_modelo_rgu)
    raise
else:
    if os.path.exists(_bak_rgu):
        os.remove(_bak_rgu)


# -----------------------------------------------------------------------------
# PASO 3: LATEST ESTIMATE (Acumulado Real + Proyección Cierre)
# -----------------------------------------------------------------------------
print("Calculando Latest Estimate Real...")

QUERY_MTD = """
WITH saldo AS (SELECT DISTINCT DATE_SUB(fecha, INTERVAL 1 DAY) AS fecha, user FROM `tenpo-bi-prod.balance.daily_balance` WHERE saldo_dia > 0),
economics AS (SELECT DISTINCT user, DATE(fecha) AS fecha FROM `tenpo-bi-prod.economics.economics` WHERE linea IN (SELECT * FROM `tenpo-bi-prod.kpitos.lineas_mau_app`) AND LOWER(IFNULL(nombre,"null")) NOT LIKE "%devolu%" AND LOWER(IFNULL(actividad_nac_cd,"null")) NOT LIKE "%refund%"),
revenue_fin AS (SELECT DISTINCT DATE(fecha) AS fecha, user FROM `tenpo-bi-prod.kpitos.revenue_tc_detalle`),
users_tc AS (SELECT CAST(individual_id AS STRING) AS id_persona, user_id FROM `tenpo-bi-prod.kpitos.tipo_users_tc` WHERE name_segment = "XS"),
base_saldos_tca AS (SELECT DISTINCT DATE(Fecha_Proceso) AS fecha, user_id AS user FROM `operacionestc-operaciones-prod.MAE.Cuentas` INNER JOIN users_tc USING(id_persona) WHERE Status_id NOT IN ('200','6','14') AND Cupo_Total <> '0' AND id_cuenta NOT IN ('1','2','1733','1734','1735','1736','1737')),
universo_rgu AS (
  SELECT fecha, user FROM saldo UNION DISTINCT SELECT fecha, user FROM economics UNION DISTINCT SELECT fecha, user FROM revenue_fin UNION DISTINCT SELECT fecha, user FROM base_saldos_tca
),
primera_actividad_mes AS (
  SELECT user, DATE_TRUNC(fecha, MONTH) AS mes, MIN(fecha) AS fecha_activacion FROM universo_rgu WHERE fecha < CURRENT_DATE("America/Santiago") GROUP BY 1, 2
),
nuevos_rgu_por_dia AS (
  SELECT fecha_activacion AS fecha, COUNT(DISTINCT user) AS rgu_nuevos_dia FROM primera_actividad_mes GROUP BY 1
)
SELECT
  fecha,
  SUM(rgu_nuevos_dia) OVER (PARTITION BY DATE_TRUNC(fecha, MONTH) ORDER BY fecha ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS rgu_acumulado
FROM nuevos_rgu_por_dia
WHERE fecha >= DATE_TRUNC(CURRENT_DATE("America/Santiago"), MONTH)   -- V11: solo mes en curso (el dia 1 queda vacio -> 0)
ORDER BY fecha DESC
LIMIT 1
"""

df_mtd = pd.read_gbq(QUERY_MTD, project_id='tenpo-bi-prod', use_bqstorage_api=True, credentials=credentials, dialect='standard')
rgu_acumulado_hoy = df_mtd['rgu_acumulado'].values[0] if len(df_mtd) else 0


def _le_rgu(forecast_rgu, rgu_acumulado_hoy):
    """V11: LE = acumulado real hasta AYER + prediccion del modelo desde HOY al fin de mes.
    Antes: acumulado (con el dia en curso parcial) + dias_restantes * 2500, y el dia
    ultimo del mes quedaba sin proyeccion (dias_restantes = 0)."""
    hoy_ts = pd.Timestamp(hoy_chile().date())
    col_pred = [c for c in forecast_rgu.columns
                if c.startswith('rgu_prediccion') and 'cumsum' not in c and 'meta' not in c][0]
    fut = forecast_rgu[pd.to_datetime(forecast_rgu['fecha']) >= hoy_ts]
    resto = float(fut[col_pred].sum())
    if len(fut) and resto <= 0:
        resto = len(fut) * RGU_RESPALDO_DIA
        print(f"[V11 RGU] !! modelo sin prediccion, respaldo {RGU_RESPALDO_DIA}/dia")
    le = int(rgu_acumulado_hoy + resto)
    print(f"[V11 RGU] acumulado<=ayer={rgu_acumulado_hoy:,.0f} | resto modelo ({len(fut)} dias)={resto:,.0f} | LE={le:,}")
    return le

cierre_estimado = _le_rgu(forecast_rgu, rgu_acumulado_hoy)
forecast_rgu['rgu_cierre_mensual_estimado'] = cierre_estimado


# -----------------------------------------------------------------------------
# PASO 4: LIMPIEZA Y SUBIDA A BIGQUERY
# -----------------------------------------------------------------------------
print("Limpiando y subiendo a BigQuery...")

# Limpieza de decimales
for col in forecast_rgu.columns:
    if pd.api.types.is_float_dtype(forecast_rgu[col]):
        forecast_rgu[col] = forecast_rgu[col].round(0)

# Subida final
project = 'tenpo-bi-prod'
pandas_gbq.to_gbq(
    forecast_rgu,
    'kpitos.forecasting_rgu',
    project_id=project,
    if_exists='replace',
)
print(f"✅ ¡RGU Completado y Subido! Acumulado real hoy: {rgu_acumulado_hoy:,.0f} | Cierre proyectado (LE): {cierre_estimado:,.0f}")

In [ ]:
forecast_mau_tc_fis = ForecastProductoV5(nombre_algoritmo=algoritmo_mau_tc_fis,meta=meta_mau_tc_fis,fechas_entrenamiento=fechas_entrenamiento_tc,fechas_proyeccion=fechas_proyeccion,directorio=directorio_salida, holdout_dias=30)
forecast_mau_tc_fis = forecast_mau_tc_fis.proyectar(mau_tc_fis)

forecast_mau_tc_vir = ForecastProductoV5(nombre_algoritmo=algoritmo_mau_tc_vir,meta=meta_mau_tc_vir,fechas_entrenamiento=fechas_entrenamiento_tc,fechas_proyeccion=fechas_proyeccion,directorio=directorio_salida, holdout_dias=30)
forecast_mau_tc_vir = forecast_mau_tc_vir.proyectar(mau_tc_vir)

forecast_mau_tc_tar = ForecastProductoV5(nombre_algoritmo=algoritmo_mau_tc_tar,meta=meta_mau_tc_tar,fechas_entrenamiento=fechas_entrenamiento_tc,fechas_proyeccion=fechas_proyeccion,directorio=directorio_salida, holdout_dias=30)
forecast_mau_tc_tar = forecast_mau_tc_tar.proyectar(mau_tc_tar)

forecast_mau_prp_fis = ForecastProductoV5(nombre_algoritmo=algoritmo_mau_prp_fis,meta=meta_mau_prp_fis,fechas_entrenamiento=fechas_entrenamiento,fechas_proyeccion=fechas_proyeccion,directorio=directorio_salida, holdout_dias=30)
forecast_mau_prp_fis = forecast_mau_prp_fis.proyectar(mau_prp_fis)

forecast_mau_prp_vir = ForecastProductoV5(nombre_algoritmo=algoritmo_mau_prp_vir,meta=meta_mau_prp_vir,fechas_entrenamiento=fechas_entrenamiento,fechas_proyeccion=fechas_proyeccion,directorio=directorio_salida, holdout_dias=30)
forecast_mau_prp_vir = forecast_mau_prp_vir.proyectar(mau_prp_vir)

forecast_mau_prp_uni = ForecastProductoV5(nombre_algoritmo=algoritmo_mau_prp_uni,meta=meta_mau_prp_uni,fechas_entrenamiento=fechas_entrenamiento,fechas_proyeccion=fechas_proyeccion,directorio=directorio_salida, holdout_dias=30)
forecast_mau_prp_uni = forecast_mau_prp_uni.proyectar(mau_prp_uni)

forecast_mau_p2p = ForecastProductoV5(nombre_algoritmo=algoritmo_mau_p2p,meta=meta_mau_p2p,fechas_entrenamiento=fechas_entrenamiento,fechas_proyeccion=fechas_proyeccion,directorio=directorio_salida, holdout_dias=30)
forecast_mau_p2p = forecast_mau_p2p.proyectar(mau_p2p)

forecast_mau_insurance = ForecastProductoV5(nombre_algoritmo=algoritmo_mau_insurance,meta=meta_mau_insurance,fechas_entrenamiento=fechas_entrenamiento,fechas_proyeccion=fechas_proyeccion,directorio=directorio_salida, holdout_dias=30)
forecast_mau_insurance = forecast_mau_insurance.proyectar(mau_insurance)

forecast_mau_crossborder = ForecastProductoV5(nombre_algoritmo=algoritmo_mau_crossborder,meta=meta_mau_crossborder,fechas_entrenamiento=fechas_entrenamiento,fechas_proyeccion=fechas_proyeccion,directorio=directorio_salida, holdout_dias=30)
forecast_mau_crossborder = forecast_mau_crossborder.proyectar(mau_crossborder)

forecast_mau_paypal = ForecastProductoV5(nombre_algoritmo=algoritmo_mau_paypal,meta=meta_mau_paypal,fechas_entrenamiento=fechas_entrenamiento,fechas_proyeccion=fechas_proyeccion,directorio=directorio_salida, holdout_dias=30)
forecast_mau_paypal = forecast_mau_paypal.proyectar(mau_paypal)

forecast_mau_paypal_abonos = ForecastProductoV5(nombre_algoritmo=algoritmo_mau_paypal_abonos,meta=meta_mau_paypal_abonos,fechas_entrenamiento=fechas_entrenamiento,fechas_proyeccion=fechas_proyeccion,directorio=directorio_salida, holdout_dias=30)
forecast_mau_paypal_abonos = forecast_mau_paypal_abonos.proyectar(mau_paypal_abonos)

forecast_mau_top_ups = ForecastProductoV5(nombre_algoritmo=algoritmo_mau_top_ups,meta=meta_mau_top_ups,fechas_entrenamiento=fechas_entrenamiento,fechas_proyeccion=fechas_proyeccion,directorio=directorio_salida, holdout_dias=30)
forecast_mau_top_ups = forecast_mau_top_ups.proyectar(mau_top_ups)

forecast_mau_pdc = ForecastProductoV5(nombre_algoritmo=algoritmo_mau_pdc,meta=meta_mau_pdc,fechas_entrenamiento=fechas_entrenamiento,fechas_proyeccion=fechas_proyeccion,directorio=directorio_salida, holdout_dias=30)
forecast_mau_pdc = forecast_mau_pdc.proyectar(mau_pdc)

num_cols = forecast_mau_paypal.select_dtypes(include=[np.number]).columns
forecast_mau_paypal[num_cols] = forecast_mau_paypal[num_cols].map(lambda x: 0)

num_cols = forecast_mau_paypal_abonos.select_dtypes(include=[np.number]).columns
forecast_mau_paypal_abonos[num_cols] = forecast_mau_paypal_abonos[num_cols].map(lambda x: 0)

dfs = [
    forecast_mau_tc_fis, forecast_mau_tc_vir,forecast_mau_tc_tar, forecast_mau_prp_fis, forecast_mau_prp_vir, forecast_mau_prp_uni,
    forecast_mau_pdc, forecast_mau_insurance, forecast_mau_crossborder, forecast_mau_paypal,
    forecast_mau_paypal_abonos, forecast_mau_p2p,forecast_mau_top_ups
]

mau_productos_merged = dfs[0]
for df in dfs[1:]:
    mau_productos_merged = mau_productos_merged.merge(df, on="fecha", how="outer")

In [ ]:
project = 'tenpo-bi-prod'
pandas_gbq.to_gbq(
    mau_productos_merged, 'kpitos.forecasting_mau_productos', project_id=project, if_exists='replace',
)

In [ ]:
# ==============================================================================
# V10: BLEND ADAPTATIVO -- helper y buffer de auditoria
# ==============================================================================
import numpy as np
import pandas as pd
from datetime import date

PACE_WEIGHT_OVERRIDE = {
    # "activo": True, "valor": 0.15,
    # "vigente_hasta": date(2026, 9, 30), "motivo": "override manual",
    "activo": False,
}


def _pace_weight_adaptativo(dia_mes, pace_val, model_val, kpi_label="MAU APP"):
    """Devuelve (peso_efectivo, diag_dict). Curva por dia_mes + detector de
    divergencia + override manual. Ver comentarios."""
    diag = {"kpi": kpi_label, "dia_mes": dia_mes,
            "pace_val": pace_val, "model_val": model_val,
            "peso_curva": None, "divergencia_pct": None,
            "cap_por_divergencia": False, "override_activo": False,
            "override_motivo": None, "peso_final": None,
            "fecha_calculo": hoy_chile()}

    if pace_val is None or pace_val <= 0 or model_val is None or model_val <= 0:
        diag["peso_final"] = 0.0
        diag["peso_curva"] = 0.0
        return 0.0, diag

    d = max(1, int(dia_mes))
    if d <= 2:
        peso_curva = 0.15
    elif d <= 7:
        peso_curva = 0.15 + (d - 2) * (0.25 - 0.15) / 5
    elif d <= 14:
        peso_curva = 0.25 + (d - 7) * (0.40 - 0.25) / 7
    elif d <= 21:
        peso_curva = 0.40 + (d - 14) * (0.50 - 0.40) / 7
    else:
        peso_curva = 0.50
    diag["peso_curva"] = round(peso_curva, 4)

    div = abs(pace_val - model_val) / model_val
    diag["divergencia_pct"] = round(div * 100, 2)

    peso_efectivo = peso_curva
    if div > 0.15:
        peso_efectivo = min(peso_curva, 0.15)
        diag["cap_por_divergencia"] = True
        print(f"[V10 GUARD {kpi_label}] divergencia pace/modelo = {div:.1%} > 15% "
              f"-> cap PACE_WEIGHT a {peso_efectivo:.3f} (era {peso_curva:.3f})")

    ov = PACE_WEIGHT_OVERRIDE
    if ov.get("activo") and ov.get("vigente_hasta") and hoy_chile().date() <= ov["vigente_hasta"]:
        peso_efectivo = float(ov["valor"])
        diag["override_activo"] = True
        diag["override_motivo"] = str(ov.get("motivo", ""))
        print(f"[V10 OVERRIDE {kpi_label}] PACE_WEIGHT forzado a {peso_efectivo:.3f} "
              f"por override manual: {diag['override_motivo']}")

    diag["peso_final"] = round(peso_efectivo, 4)
    return peso_efectivo, diag


_AUDIT_ROWS = []


def _flush_audit_a_bq():
    """V11: se define aqui y se llama al final (celda REPORTE LE). Antes estaba comentado
    y no quedaba historia de las corridas."""
    if not _AUDIT_ROWS:
        print("[V11 AUDIT] Nada que volcar.")
        return
    df_audit = pd.DataFrame(_AUDIT_ROWS)
    df_audit["fecha_calculo"] = pd.to_datetime(df_audit["fecha_calculo"])
    df_audit["ts_corrida"] = ahora_utc_naive()
    for col in ["pace_val", "model_val", "peso_curva", "divergencia_pct",
                "peso_final", "pace_target_efectivo", "factor_meta",
                "meta_ref", "actual_acumulado", "pred_bruta",
                "landing_final_publicado", "dias_restantes"]:
        if col in df_audit.columns:
            df_audit[col] = pd.to_numeric(df_audit[col], errors="coerce").astype(float)
    for col in ["kpi", "override_motivo"]:
        if col in df_audit.columns:
            df_audit[col] = df_audit[col].astype(str)
    for col in ["cap_por_divergencia", "override_activo"]:
        if col in df_audit.columns:
            df_audit[col] = df_audit[col].astype(bool)
    print(f"[V11 AUDIT] Volcando {len(df_audit)} filas a kpitos.pipeline_audit")
    print(df_audit.to_string(index=False))
    import pandas_gbq
    pandas_gbq.to_gbq(df_audit, "kpitos.pipeline_audit", project_id="tenpo-bi-prod",
                      if_exists="append", credentials=credentials)
    print("[V11 AUDIT] OK.")


print("[V10] helper y buffer inicializados. PACE_WEIGHT sera adaptativo desde ahora.")


In [ ]:
# 1. MAU APP
forecast_mau_app = ForecastProductoV5(nombre_algoritmo=algoritmo_mau_app, meta=meta_mau_app, fechas_entrenamiento=fechas_entrenamiento, fechas_proyeccion=fechas_proyeccion, directorio=directorio_salida, holdout_dias=30)
forecast_mau_app = forecast_mau_app.proyectar(mau_app)

mask_app = forecast_mau_app['mau_app'].isna()
# --- Ajuste MAU APP (General + Cyber) ---
fechas_cyber = pd.to_datetime(list(CYBER_UPLIFT_MAU))   # V11: desde CONFIG
mask_cyber = forecast_mau_app['fecha'].isin(fechas_cyber)

# --- PACE-BASED SCALING (V7): reflejando realidad con diagnostico ---
_hoy = hoy_chile()
_ini_mes_actual  = _hoy.replace(day=1)
_ayer            = _hoy - pd.Timedelta(days=1)
_dia_mes         = _ayer.day
_fin_mes_ant     = _ini_mes_actual - pd.Timedelta(days=1)
_ini_mes_ant     = _fin_mes_ant.replace(day=1)
_ayer_mes_ant    = _ini_mes_ant + pd.Timedelta(days=_dia_mes - 1)

print(f'[MAU APP PACE] Hoy={_hoy.date()}, ayer={_ayer.date()}, dia_mes={_dia_mes}')
print(f'[MAU APP PACE] Mes actual: {_ini_mes_actual.date()}..{_ayer.date()}')
print(f'[MAU APP PACE] Mes anterior: {_ini_mes_ant.date()}..{_fin_mes_ant.date()} (MTD homologo: {_ayer_mes_ant.date()})')

_mtd_actual = forecast_mau_app[
    (forecast_mau_app['fecha'] >= _ini_mes_actual) &
    (forecast_mau_app['fecha'] <= _ayer)
]['mau_app'].sum()

_mtd_ant = forecast_mau_app[
    (forecast_mau_app['fecha'] >= _ini_mes_ant) &
    (forecast_mau_app['fecha'] <= _ayer_mes_ant)
]['mau_app'].sum()

_cerrado_ant = forecast_mau_app[
    (forecast_mau_app['fecha'] >= _ini_mes_ant) &
    (forecast_mau_app['fecha'] <= _fin_mes_ant)
]['mau_app'].sum()

print(f'[MAU APP PACE] MTD_actual={_mtd_actual:,.0f} | MTD_ant={_mtd_ant:,.0f} | Cerrado_ant={_cerrado_ant:,.0f}')

# V10: BLEND ADAPTATIVO -- PACE_WEIGHT se calcula abajo, despues de
# que _model_landing este definido (no se puede setear aca porque
# la funcion adaptativa lo necesita como input).

_actual_precheck = forecast_mau_app.loc[~mask_app, 'mau_app'].sum()
_pred_precheck   = forecast_mau_app.loc[mask_app, 'mau_prediccion_app'].sum()
_model_landing   = _actual_precheck + _pred_precheck

# --- V10: BLEND ADAPTATIVO (reemplaza PACE_WEIGHT = 0.5 fijo original) ---
PACE_WEIGHT, _audit_diag_mau = _pace_weight_adaptativo(
    dia_mes=_dia_mes,
    pace_val=PACE_MAU_APP,
    model_val=_model_landing,
    kpi_label="MAU APP",
)
print(f"[V10 ADAPT MAU APP] dia_mes={_dia_mes} | pace={PACE_MAU_APP} | "
      f"crudo={_model_landing:,.0f} | "
      f"divergencia={_audit_diag_mau['divergencia_pct']}% | "
      f"peso_curva={_audit_diag_mau['peso_curva']} | "
      f"PACE_WEIGHT_final={PACE_WEIGHT:.3f}")

if PACE_MAU_APP is not None and PACE_MAU_APP > 0:
    _pace_target = int(PACE_WEIGHT * PACE_MAU_APP + (1 - PACE_WEIGHT) * _model_landing)
    print(f'[MAU APP BLEND] pace={PACE_MAU_APP:,} | model_raw={_model_landing:,.0f} | weight={PACE_WEIGHT} | target={_pace_target:,}')
else:
    _pace_target = int(_model_landing)
    print(f'[MAU APP BLEND] !! Sin pace. Usando solo model_raw={_pace_target:,}')

actual_acum = forecast_mau_app.loc[~mask_app, 'mau_app'].sum()
pred_bruta  = forecast_mau_app.loc[mask_app, 'mau_prediccion_app'].sum()
factor_meta = ((_pace_target - actual_acum) / pred_bruta) if pred_bruta > 0 else 1.0
print(f'[MAU APP] actual={actual_acum:,.0f} | pred_bruta={pred_bruta:,.0f} | pace_target={_pace_target:,.0f} | meta_ref={meta_mau_app:,.0f} | factor={factor_meta:.4f}')
if factor_meta < 0:
    print(f'  !! WARN: pace target ya superado. factor=1.0')
    factor_meta = 1.0
forecast_mau_app.loc[mask_app, 'mau_prediccion_app'] *= factor_meta
# Ajuste adicional por Cyber sobre los dias especificos
forecast_mau_app.loc[mask_app & mask_cyber, 'mau_prediccion_app'] *= 1
_aplicar_cyber(forecast_mau_app, 'mau_app', 'mau_prediccion_app', CYBER_UPLIFT_MAU, 'MAU App')

# Recalcular columnas dependientes
forecast_mau_app['mau_prediccion_app'] = forecast_mau_app['mau_prediccion_app'].round(0)
forecast_mau_app.loc[mask_app, 'mau_real_pred_app'] = forecast_mau_app.loc[mask_app, 'mau_prediccion_app']
forecast_mau_app['mau_prediccion_cumsum_app'] = forecast_mau_app['mau_prediccion_app'].cumsum()
forecast_mau_app.loc[mask_app, 'mau_real_pred_cumsum2_app'] = forecast_mau_app['mau_real_pred_app'].cumsum()[mask_app]


# 2. MAU CONCURRENT (Reactive)

# --- V10: registrar en buffer de auditoria antes de continuar con subcats ---
try:
    _audit_diag_mau.update({
        "pace_target_efectivo": int(_pace_target),
        "factor_meta": float(factor_meta),
        "meta_ref": float(meta_mau_app),
        "actual_acumulado": float(actual_acum),
        "pred_bruta": float(pred_bruta),
        "landing_final_publicado": int(_pace_target),
        "dias_restantes": DIAS_RESTANTES_MES,
    })
    _AUDIT_ROWS.append(_audit_diag_mau)
except NameError:
    pass

forecast_mau_concurrent = ForecastProductoV5(nombre_algoritmo=algoritmo_mau_concurrent, meta=meta_mau_concurrent, fechas_entrenamiento=fechas_entrenamiento, fechas_proyeccion=fechas_proyeccion, directorio=directorio_salida, holdout_dias=30)
forecast_mau_concurrent = forecast_mau_concurrent.proyectar(mau_concurrent)

mask_con = forecast_mau_concurrent['mau_concurrent'].isna()
forecast_mau_concurrent.loc[mask_con, 'mau_prediccion_concurrent'] *= 1
forecast_mau_concurrent['mau_prediccion_concurrent'] = forecast_mau_concurrent['mau_prediccion_concurrent'].round(0)
forecast_mau_concurrent.loc[mask_con, 'mau_real_pred_concurrent'] = forecast_mau_concurrent.loc[mask_con, 'mau_prediccion_concurrent']
forecast_mau_concurrent['mau_prediccion_cumsum_concurrent'] = forecast_mau_concurrent['mau_prediccion_concurrent'].cumsum()
forecast_mau_concurrent.loc[mask_con, 'mau_real_pred_cumsum2_concurrent'] = forecast_mau_concurrent['mau_real_pred_concurrent'].cumsum()[mask_con]


# 3. MAU M0 (Acquisition)
forecast_mau_m0 = ForecastProductoV5(nombre_algoritmo=algoritmo_mau_m0, meta=meta_mau_m0, fechas_entrenamiento=fechas_entrenamiento, fechas_proyeccion=fechas_proyeccion, directorio=directorio_salida, holdout_dias=30)
forecast_mau_m0 = forecast_mau_m0.proyectar(mau_m0)

mask_m0 = forecast_mau_m0['mau_m0'].isna()
forecast_mau_m0.loc[mask_m0, 'mau_prediccion_m0'] *= 1 # Cambiar factor si es necesario
forecast_mau_m0['mau_prediccion_m0'] = forecast_mau_m0['mau_prediccion_m0'].round(0)
forecast_mau_m0.loc[mask_m0, 'mau_real_pred_m0'] = forecast_mau_m0.loc[mask_m0, 'mau_prediccion_m0']
forecast_mau_m0['mau_prediccion_cumsum_m0'] = forecast_mau_m0['mau_prediccion_m0'].cumsum()
forecast_mau_m0.loc[mask_m0, 'mau_real_pred_cumsum2_m0'] = forecast_mau_m0['mau_real_pred_m0'].cumsum()[mask_m0]


# 4. MAU RESURRECTED
forecast_mau_resurrected = ForecastProductoV5(nombre_algoritmo=algoritmo_mau_resurrected, meta=meta_mau_resurrected, fechas_entrenamiento=fechas_entrenamiento, fechas_proyeccion=fechas_proyeccion, directorio=directorio_salida, holdout_dias=30)
forecast_mau_resurrected = forecast_mau_resurrected.proyectar(mau_resurrected)

mask_res = forecast_mau_resurrected['mau_resurrected'].isna()
forecast_mau_resurrected.loc[mask_res, 'mau_prediccion_resurrected'] *= 1
forecast_mau_resurrected['mau_prediccion_resurrected'] = forecast_mau_resurrected['mau_prediccion_resurrected'].round(0)
forecast_mau_resurrected.loc[mask_res, 'mau_real_pred_resurrected'] = forecast_mau_resurrected.loc[mask_res, 'mau_prediccion_resurrected']
forecast_mau_resurrected['mau_prediccion_cumsum_resurrected'] = forecast_mau_resurrected['mau_prediccion_resurrected'].cumsum()
forecast_mau_resurrected.loc[mask_res, 'mau_real_pred_cumsum2_resurrected'] = forecast_mau_resurrected['mau_real_pred_resurrected'].cumsum()[mask_res]


# 5. MAU LATE ACTIVE
forecast_mau_late_active = ForecastProductoV5(nombre_algoritmo=algoritmo_mau_late_active, meta=meta_mau_late_active, fechas_entrenamiento=fechas_entrenamiento, fechas_proyeccion=fechas_proyeccion, directorio=directorio_salida, holdout_dias=30)
forecast_mau_late_active = forecast_mau_late_active.proyectar(mau_late_active)

mask_late = forecast_mau_late_active['mau_late_active'].isna()
forecast_mau_late_active.loc[mask_late, 'mau_prediccion_late_active'] *= 1
forecast_mau_late_active['mau_prediccion_late_active'] = forecast_mau_late_active['mau_prediccion_late_active'].round(0)
forecast_mau_late_active.loc[mask_late, 'mau_real_pred_late_active'] = forecast_mau_late_active.loc[mask_late, 'mau_prediccion_late_active']
forecast_mau_late_active['mau_prediccion_cumsum_late_active'] = forecast_mau_late_active['mau_prediccion_late_active'].cumsum()
forecast_mau_late_active.loc[mask_late, 'mau_real_pred_cumsum2_late_active'] = forecast_mau_late_active['mau_real_pred_late_active'].cumsum()[mask_late]

# --- RGU ---
print("Proyectando RGU...")
forecast_rgu = ForecastProducto(
    nombre_algoritmo=algoritmo_rgu,
    meta=meta_rgu,
    fechas_entrenamiento=fechas_entrenamiento,
    fechas_proyeccion=fechas_proyeccion,
    directorio=directorio_salida
)
forecast_rgu = forecast_rgu.proyectar(rgu_historico)
print("Proyección RGU terminada.")

# --- UNIFICACIÓN FINAL ---
forecast_mau_app.rename(columns={'mau_prediccion_meta_app':'mau_prediccion_app_meta', 'mau_prediccion_meta_cumsum_app':'mau_prediccion_app_meta_cumsum', 'meta_app':'meta_mau_app'}, inplace=True)

# ==============================================================================
# V10: ANCLAJE ESTACIONAL DEL RECURRENTE (reemplaza el factor_subcats uniforme)
# El recurrente (concurrent) tiene curva MTD/cierre muy estable (~86-88% al dia 12,
# std ~1.9pp). Lo anclamos a SU PROPIA curva estacional (ventana movil de meses
# cerrados) en vez de recortarlo con el factor del total. Las 3 categorias volatiles
# (m0, resurrected, late_active) absorben el residuo contra el pace_target total,
# de modo que el TOTAL sigue aterrizando exactamente en _pace_target.
# Para volver al comportamiento V9: USAR_ANCLAJE_ESTACIONAL = False
# ==============================================================================
USAR_ANCLAJE_ESTACIONAL = True
VENTANA_MESES_RATIO      = 4   # meses cerrados recientes para la razon estacional

actual_total_subcats = (
      forecast_mau_concurrent.loc[~mask_con, 'mau_concurrent'].sum()
    + forecast_mau_m0.loc[~mask_m0, 'mau_m0'].sum()
    + forecast_mau_resurrected.loc[~mask_res, 'mau_resurrected'].sum()
    + forecast_mau_late_active.loc[~mask_late, 'mau_late_active'].sum()
)
pred_total_futuro = (
      forecast_mau_concurrent.loc[mask_con, 'mau_prediccion_concurrent'].sum()
    + forecast_mau_m0.loc[mask_m0, 'mau_prediccion_m0'].sum()
    + forecast_mau_resurrected.loc[mask_res, 'mau_prediccion_resurrected'].sum()
    + forecast_mau_late_active.loc[mask_late, 'mau_prediccion_late_active'].sum()
)

# V11: sin dias reales en el mes (corrida del dia 1) el anclaje daba factor 0 al
# concurrent y cargaba todo el MAU en las volatiles. Se desactiva en ese caso.
if forecast_mau_concurrent.loc[~mask_con, 'mau_concurrent'].sum() <= 0:
    print('[ANCLAJE] sin dias reales en el mes -> se usa factor uniforme')
    USAR_ANCLAJE_ESTACIONAL = False

if USAR_ANCLAJE_ESTACIONAL:
    # 1) Razon estacional del recurrente con los MISMOS DIAS RESTANTES (V11), sobre meses CERRADOS.
    #    ratio = MTD(dia)/cierre; usamos el promedio de la ventana movil reciente.
    _h = mau_concurrent[['fecha', 'mau']].copy()
    _h = _h[_h['fecha'] < _ini_mes_actual]                       # excluir mes en curso
    _h = _h.groupby('fecha', as_index=False)['mau'].sum().sort_values('fecha')
    _h['mes_h']   = _h['fecha'].values.astype('datetime64[M]')
    _h['dia_mes'] = _h['fecha'].dt.day
    _h['mtd']    = _h.groupby('mes_h')['mau'].cumsum()
    _h['cierre'] = _h.groupby('mes_h')['mau'].transform('sum')
    _h['ratio']  = _h['mtd'] / _h['cierre']
    # V11: alinear por dias restantes (antes: mismo numero de dia -> sesgo 30 vs 31 dias)
    _h['dia_equiv'] = _h['fecha'].dt.days_in_month - DIAS_RESTANTES_MES
    _r = (_h[_h['dia_mes'] == _h['dia_equiv']].sort_values('mes_h')
            .tail(VENTANA_MESES_RATIO)['ratio'])
    _ratio_estacional = float(_r.mean()) if len(_r) else None
    print(f'[ANCLAJE] ratio estacional dia {_dia_mes} (avg {len(_r)}m/ventana {VENTANA_MESES_RATIO}) = '
          f'{None if _ratio_estacional is None else round(_ratio_estacional, 4)}')

    if _ratio_estacional and _ratio_estacional > 0:
        # 2) Target del recurrente anclado a SU propia curva estacional
        _con_actual   = forecast_mau_concurrent.loc[~mask_con, 'mau_concurrent'].sum()
        _con_pred_fut = forecast_mau_concurrent.loc[mask_con, 'mau_prediccion_concurrent'].sum()
        _con_target   = _con_actual / _ratio_estacional
        factor_concurrent = ((_con_target - _con_actual) / _con_pred_fut) if _con_pred_fut > 0 else 1.0
        if factor_concurrent < 0:
            print('  !! WARN: factor_concurrent negativo (target ya superado). Se fija en 1.0')
            factor_concurrent = 1.0

        # 3) Las 3 volatiles absorben el residuo contra el pace_target TOTAL
        _otras_actual   = actual_total_subcats - _con_actual
        _otras_pred_fut = pred_total_futuro - _con_pred_fut
        _residuo_target = _pace_target - _con_target - _otras_actual
        factor_otras = (_residuo_target / _otras_pred_fut) if _otras_pred_fut > 0 else 1.0
        if factor_otras < 0:
            # V11: antes se fijaba en 0 y el total de subcats dejaba de cuadrar con el
            # MAU App (24-09: 925k vs 900k publicados). Ahora cae al factor uniforme.
            print('  !! WARN: anclaje inconsistente con el pace target -> fallback a factor uniforme')
            USAR_ANCLAJE_ESTACIONAL = False

        print(f'[ANCLAJE] con_actual={_con_actual:,.0f} | con_target={_con_target:,.0f} | factor_con={factor_concurrent:.4f}')
        print(f'[ANCLAJE] otras_actual={_otras_actual:,.0f} | residuo_target={_residuo_target:,.0f} | factor_otras={factor_otras:.4f}')

        if USAR_ANCLAJE_ESTACIONAL:
            forecast_mau_concurrent.loc[mask_con, 'mau_prediccion_concurrent']    *= factor_concurrent
            forecast_mau_m0.loc[mask_m0, 'mau_prediccion_m0']                     *= factor_otras
            forecast_mau_resurrected.loc[mask_res, 'mau_prediccion_resurrected']  *= factor_otras
            forecast_mau_late_active.loc[mask_late, 'mau_prediccion_late_active'] *= factor_otras
    else:
        print('[ANCLAJE] !! ratio no computable. Fallback a factor uniforme V9.')
        USAR_ANCLAJE_ESTACIONAL = False

if not USAR_ANCLAJE_ESTACIONAL:
    # --- V6-V9: factor comun a las 4 subcats (fallback) ---
    factor_subcats = ((_pace_target - actual_total_subcats) / pred_total_futuro) if pred_total_futuro > 0 else 1.0
    print(f'[SUBCATS SUM] raw_pred_futuro={pred_total_futuro:,.0f} | actual={actual_total_subcats:,.0f} | pace_target={_pace_target:,.0f} | factor={factor_subcats:.4f}')
    if factor_subcats < 0:
        print(f'  !! WARN: factor subcats negativo. Se mantiene en 1.0')
        factor_subcats = 1.0
    forecast_mau_concurrent.loc[mask_con, 'mau_prediccion_concurrent']       *= factor_subcats
    forecast_mau_m0.loc[mask_m0, 'mau_prediccion_m0']                        *= factor_subcats
    forecast_mau_resurrected.loc[mask_res, 'mau_prediccion_resurrected']     *= factor_subcats
    forecast_mau_late_active.loc[mask_late, 'mau_prediccion_late_active']    *= factor_subcats

for _fcst, _mask, _base in [
    (forecast_mau_concurrent,   mask_con,  'concurrent'),
    (forecast_mau_m0,           mask_m0,   'm0'),
    (forecast_mau_resurrected,  mask_res,  'resurrected'),
    (forecast_mau_late_active,  mask_late, 'late_active'),
]:
    _fcst[f'mau_prediccion_{_base}'] = _fcst[f'mau_prediccion_{_base}'].round(0)
    _fcst.loc[_mask, f'mau_real_pred_{_base}'] = _fcst.loc[_mask, f'mau_prediccion_{_base}']
    _fcst[f'mau_prediccion_cumsum_{_base}'] = _fcst[f'mau_prediccion_{_base}'].cumsum()
    _fcst.loc[_mask, f'mau_real_pred_cumsum2_{_base}'] = _fcst[f'mau_real_pred_{_base}'].cumsum()[_mask]

_verif = (
      forecast_mau_concurrent['mau_real_pred_cumsum2_concurrent'].iloc[-1]
    + forecast_mau_m0['mau_real_pred_cumsum2_m0'].iloc[-1]
    + forecast_mau_resurrected['mau_real_pred_cumsum2_resurrected'].iloc[-1]
    + forecast_mau_late_active['mau_real_pred_cumsum2_late_active'].iloc[-1]
)
print(f'[VERIF] proy_g_cumsum fin de mes = {_verif:,.0f} | pace_target = {_pace_target:,.0f} | meta = {meta_mau_app:,.0f} (ref) | dev_vs_meta = {(_verif-meta_mau_app)/meta_mau_app:+.2%}')

# V11: el MAU App publicado y la suma de subcategorias DEBEN cuadrar
_landing_app = forecast_mau_app['mau_real_pred_app'].sum()
_desc = abs(_verif - _landing_app) / _landing_app
print(f'[V11 CUADRE] MAU App={_landing_app:,.0f} | subcats={_verif:,.0f} | descuadre={_desc:.3%}')
assert _desc <= TOL_CUADRE_MAU, (
    f"MAU App ({_landing_app:,.0f}) y subcategorias ({_verif:,.0f}) no cuadran "
    f"({_desc:.2%} > {TOL_CUADRE_MAU:.1%}). No subir: revisar anclaje/pace.")

mau_merged = pd.merge(forecast_mau_app, forecast_mau_concurrent, on='fecha', how='outer')
mau_merged = pd.merge(mau_merged, forecast_mau_m0, on='fecha', how='outer')
mau_merged = pd.merge(mau_merged, forecast_mau_resurrected, on='fecha', how='outer')
mau_merged = pd.merge(mau_merged, forecast_mau_late_active, on='fecha', how='outer')

# MAU TOTAL reconstruido desde las partes ajustadas
mau_merged['mau_prediccion_app_2'] = (mau_merged['mau_prediccion_concurrent'] +
                                      mau_merged['mau_prediccion_resurrected'] +
                                      mau_merged['mau_prediccion_m0'] +
                                      mau_merged['mau_prediccion_late_active'])

In [ ]:
# ==============================================================================
# MAU REACTIVE
# ==============================================================================

QUERY_REACTIVE = """
WITH base_diaria AS (
  SELECT
    fecha,
    DATE_TRUNC(fecha, MONTH) AS mes,
    user,
    CASE
      WHEN tipo_mau_mes_2 = "late_active_user" THEN "A- Inactive"
      WHEN tipo_mau_mes_2 = "resurrected_user" THEN "B- Dormant"
      WHEN tipo_mau_mes_2 = "active_user_m0" THEN "C- Acquisition"
      WHEN tipo_mau_mes = "M1"  THEN "D- M1"
      WHEN tipo_mau_mes = "M2"  THEN "E- M2"
      WHEN tipo_mau_mes = "M3"  THEN "F- M3"
      WHEN tipo_mau_mes = "M4"  THEN "G- M4"
      WHEN tipo_mau_mes = "M5"  THEN "H- M5"
      WHEN tipo_mau_mes = "M6"  THEN "I- M6"
      WHEN tipo_mau_mes = "M7"  THEN "J- M7"
      WHEN tipo_mau_mes = "M8"  THEN "K- M8"
      WHEN tipo_mau_mes = "M9"  THEN "L- M9"
      WHEN tipo_mau_mes = "M10" THEN "M- M10"
      WHEN tipo_mau_mes = "M11" THEN "N- M11"
      WHEN tipo_mau_mes = "M12+" THEN "O- M12+"
      ELSE "ver"
    END AS categoria
  FROM `tenpo-bi-prod.kpitos.cohorts_current_date_nuevos`),

cierre_mes AS (
  SELECT
    user,
    mes,
    categoria AS categoria_cierre_mes
  FROM base_diaria
  QUALIFY ROW_NUMBER() OVER (PARTITION BY user, mes ORDER BY fecha DESC) = 1),

base_final AS (SELECT
  b.fecha,
  b.categoria,
  b.user,
  CASE
    WHEN b.categoria IN (
      "D- M1","E- M2","F- M3","G- M4","H- M5","I- M6","J- M7",
      "K- M8","L- M9","M- M10","N- M11","O- M12+") THEN COALESCE(p.categoria_cierre_mes, "P- N/A")
    ELSE "N/A"
  END AS mes_anterior
FROM base_diaria b
LEFT JOIN cierre_mes p
  ON p.user = b.user
 AND p.mes  = DATE_TRUNC(DATE_SUB(b.fecha, INTERVAL 1 MONTH), MONTH))

SELECT
  fecha,
  DATE_TRUNC(fecha,MONTH) AS mes,
  categoria,
  mes_anterior,
  COUNT(DISTINCT user) AS mau
FROM base_final
WHERE fecha >= "2023-01-01"
AND categoria NOT IN ("A- Inactive", "B- Dormant", "C- Acquisition")
GROUP BY 1,2,3,4
ORDER BY 1 DESC,2,3,4
"""

# 2. Cargar data
reactive_detail = pd.read_gbq(QUERY_REACTIVE, use_bqstorage_api=True, project_id='tenpo-bi-prod', credentials=credentials, dialect='standard')
reactive_detail['fecha'] = pd.to_datetime(reactive_detail['fecha'], format="%Y-%m-%d")

# 3. Crear identificador único de segmento para pivotear (Ej: "D_M1_from_A_Inactive")
# Limpiamos caracteres raros para evitar problemas en nombres de columnas
reactive_detail['segmento_id'] = (
    reactive_detail['categoria'].str.replace(r'[^a-zA-Z0-9]', '_', regex=True) +
    "_from_" +
    reactive_detail['mes_anterior'].str.replace(r'[^a-zA-Z0-9]', '_', regex=True)
)

# 4. Pivotear para tener series de tiempo por columnas
# Rellenamos con 0 los nulos (días sin usuarios en esa micro-categoría)
reactive_pivot = reactive_detail.pivot_table(
    index=['fecha'],
    columns='segmento_id',
    values='mau',
    aggfunc='sum'
).fillna(0).reset_index()

reactive_pivot['dia_semana'] = reactive_pivot['fecha'].dt.dayofweek
reactive_pivot['dia'] = reactive_pivot['fecha'].dt.day
reactive_pivot['mes'] = reactive_pivot['fecha'].dt.month
reactive_pivot['year'] = reactive_pivot['fecha'].dt.year
reactive_pivot['real'] = 1

print(f"Se encontraron {len(reactive_pivot.columns) - 5} sub-segmentos para entrenar.")

In [ ]:
# ==============================================================================
# AJUSTE TOTAL
# ==============================================================================

print("Iniciando ajuste total (Historia y Proyección)...")

# Create df_merged_reactive and proyecciones_subsegmentos ---
# Identify sub-segment columns from reactive_pivot
# These are columns other than 'fecha', 'dia_semana', 'dia', 'mes', 'year', 'real'
subsegment_cols = [col for col in reactive_pivot.columns if col not in ['fecha', 'dia_semana', 'dia', 'mes', 'year', 'real']]

all_forecasts_reactive = []
proyecciones_subsegmentos = [] # This list will hold the names of the 'pred_' columns for the adjustment

for col_name in subsegment_cols:
    # Prepare data for ForecastProducto. The ForecastProducto expects a 'mau' column for the target.
    df_subsegment_kpi = reactive_pivot[['fecha', 'dia_semana', 'dia', 'mes', 'year', 'real', col_name]].copy()
    df_subsegment_kpi = df_subsegment_kpi.rename(columns={col_name: 'mau'})

    # Create a unique algorithm name for this sub-segment.
    # The ForecastProducto will use this name to generate output columns.
    algoritmo_subsegmento = f"mau_reactive_{col_name}"

    # Instantiate and project the ForecastProducto
    # We use meta_mau_concurrent as a placeholder meta; the adjustment logic will correct the totals.
    forecast_subsegment_instance = ForecastProducto(
        nombre_algoritmo=algoritmo_subsegmento,
        meta=meta_mau_concurrent,
        fechas_entrenamiento=fechas_entrenamiento,
        fechas_proyeccion=fechas_proyeccion,
        directorio=directorio_salida
    )
    forecast_df = forecast_subsegment_instance.proyectar(df_subsegment_kpi)

    # The output columns from ForecastProducto will be like:
    # 'fecha', 'mau_reactive_D__M1_from_A__Inactive', 'mau_prediccion_reactive_D__M1_from_A__Inactive',
    # 'mau_real_pred_reactive_D__M1_from_A__Inactive', etc.

    # We need to extract the 'mau_prediccion_{algoritmo_subsegmento}' and 'mau_real_pred_{algoritmo_subsegmento}'
    # and rename them to the format expected by the subsequent adjustment block.
    # The adjustment block expects column names like 'pred_D__M1_from_A__Inactive' and 'D__M1_from_A__Inactive'

    # Fix: remove the redundant 'mau_' from algoritmo_subsegmento when forming the expected column names
    # The ForecastProducto already includes 'mau_' in the prefix 'mau_prediccion_' and 'mau_real_pred_'
    pred_col_name_from_forecast = f"mau_prediccion_{algoritmo_subsegmento[4:]}"
    real_pred_col_name_from_forecast = f"mau_real_pred_{algoritmo_subsegmento[4:]}"

    # Define the new names for these columns
    new_pred_col_name = f"pred_{col_name}" # Example: pred_D__M1_from_A__Inactive
    new_real_pred_col_name = col_name # Example: D__M1_from_A__Inactive

    # Select the relevant columns and rename them
    forecast_df_processed = forecast_df[['fecha', pred_col_name_from_forecast, real_pred_col_name_from_forecast]].copy()
    forecast_df_processed.rename(columns={
        pred_col_name_from_forecast: new_pred_col_name,
        real_pred_col_name_from_forecast: new_real_pred_col_name
    }, inplace=True)

    all_forecasts_reactive.append(forecast_df_processed)
    proyecciones_subsegmentos.append(new_pred_col_name) # Add the new 'pred_' prefixed column name to the list

# Merge all individual forecasts into df_merged_reactive
if all_forecasts_reactive:
    df_merged_reactive = all_forecasts_reactive[0]
    for i in range(1, len(all_forecasts_reactive)):
        df_merged_reactive = df_merged_reactive.merge(all_forecasts_reactive[i], on='fecha', how='outer')
else:
    # Fallback if no subsegments were processed (should not happen with reactive_pivot)
    df_merged_reactive = pd.DataFrame({'fecha': reactive_pivot['fecha'].unique()})

# Ensure the 'real' column is present for merging and setting values
df_merged_reactive = df_merged_reactive.merge(reactive_pivot[['fecha', 'real']], on='fecha', how='left')
df_merged_reactive['real'] = df_merged_reactive['real'].fillna(0).astype(int)

# --- END FIX ---

# 1. Definimos el Target: Usamos 'mau_real_pred_concurrent' que tiene la Historia Real + La Proyección Oficial
# Esto será nuestra "Verdad Absoluta" para todos los días.
target_mau = forecast_mau_concurrent[['fecha', 'mau_real_pred_concurrent']].copy()

# 2. Unimos: Usamos 'right' join para asegurar que tenemos TODAS las fechas del modelo oficial
df_final_reactive = df_merged_reactive.merge(target_mau, on='fecha', how='right')

# Rellenamos con 0 los nulos en las columnas de segmentos (por si faltaba alguna fecha en el desglose)
for col in proyecciones_subsegmentos:
    df_final_reactive[col] = df_final_reactive[col].fillna(0)

# --- INICIO LÓGICA DE AJUSTE ---

# 3. Calcular suma bruta original de los 15 segmentos
df_final_reactive['suma_bruta_submodelos'] = df_final_reactive[proyecciones_subsegmentos].sum(axis=1)
df_final_reactive['suma_bruta_submodelos'] = df_final_reactive['suma_bruta_submodelos'].replace(0, 1) # Evitar división por 0

# 4. Calcular factor de ajuste (Target / Suma Bruta)
# Esto forzará que la suma de los segmentos sea igual al total oficial, tanto en pasado como futuro
df_final_reactive['factor_ajuste'] = df_final_reactive['mau_real_pred_concurrent'] / df_final_reactive['suma_bruta_submodelos']

# 5. Aplicar factor (con decimales primero)
for col in proyecciones_subsegmentos:
    df_final_reactive[col] = df_final_reactive[col] * df_final_reactive['factor_ajuste']

# 6. Redondear y limpiar nombres
cols_ajustadas = []
for col in proyecciones_subsegmentos:
    nombre_final = col.replace("pred_", "") # Quitamos el prefijo temporal
    df_final_reactive[nombre_final] = df_final_reactive[col].fillna(0).round(0).astype('Int64')
    cols_ajustadas.append(nombre_final)

# 7. CORRECCIÓN DE RESIDUOS (El ajuste fino para eliminar diferencias de 1-5 unidades)
# Sumamos los valores ya redondeados
df_final_reactive['suma_redondeada'] = df_final_reactive[cols_ajustadas].sum(axis=1)

# Vemos cuánto nos falta o sobra para llegar al número oficial
df_final_reactive['diferencia'] = df_final_reactive['mau_real_pred_concurrent'] - df_final_reactive['suma_redondeada']

# Función para asignar la diferencia al segmento más grande del día
def ajustar_residuo(row):
    diff = row['diferencia']
    if diff != 0 and not pd.isna(diff):
        # Busca la columna con el valor máximo en esa fila
        col_max = row[cols_ajustadas].astype(float).idxmax()
        # Le suma (o resta) la diferencia para cuadrar a cero
        row[col_max] += diff
    return row

# Aplicamos la corrección fila por fila
print("  -> Aplicando corrección fina de residuos...")
df_final_reactive = df_final_reactive.apply(ajustar_residuo, axis=1)

# --- FIN LÓGICA DE AJUSTE ---

# 8. AGREGAR COLUMNA DE PROYECCION ORIGINAL (Solicitud anterior)
# Traemos la columna mau_prediccion_concurrent pura del modelo original
col_comparacion = forecast_mau_concurrent[['fecha', 'mau_prediccion_concurrent']].copy()
df_final_reactive = df_final_reactive.merge(col_comparacion, on='fecha', how='left')
df_final_reactive['mau_prediccion_concurrent'] = df_final_reactive['mau_prediccion_concurrent'].round(0).astype('Int64')

# 9. Selección final de columnas
cols_finales = ['fecha'] + cols_ajustadas + ['mau_prediccion_concurrent']
df_upload_reactive = df_final_reactive[cols_finales].copy()

# Validación final en consola
diff_total = (df_final_reactive['mau_real_pred_concurrent'] - df_final_reactive[cols_ajustadas].sum(axis=1)).abs().sum()
print(f"Diferencia total acumulada (Historia + Forecast): {diff_total} (Debería ser 0)")

# 10. Subida a BigQuery
project = 'tenpo-bi-prod'
pandas_gbq.to_gbq(
    df_upload_reactive,
    'kpitos.forecasting_mau_reactive_desglosado',
    project_id=project,
    if_exists='replace'
)
print("¡Tabla ajustada (Historia y Futuro) subida exitosamente!")

In [ ]:
# --- LIMPIEZA --------------
print("\n--- Limpiando la Data antes de subir a BigQuery ---")

# Itera sobre los nombres de columna
for col in mau_merged.columns:
    # Verifica si la columna es de tipo float
    if pd.api.types.is_float_dtype(mau_merged[col]):
        print(f"Procesando columna float: {col}")

        mau_merged[col] = mau_merged[col].round(0)

        try:
            mau_merged[col] = mau_merged[col].astype('Int64')
        except (ValueError, TypeError):
            print(f"  -> Advertencia: No se pudo convertir '{col}' a Int64, se mantiene como float redondeado.")

print("\n--- Tipos de datos finales, listos para subir ---")
mau_merged.info()
# --- LIMPIEZA ----------



In [ ]:

project = 'tenpo-bi-prod'
pandas_gbq.to_gbq(
        mau_merged, 'kpitos.forecasting_categorias', project_id=project, if_exists='replace',
    )
print("\n¡DataFrame subido a BigQuery exitosamente!")


In [ ]:
# --- forecast_gpv_app ---
forecast_gpv_app = ForecastProductoV5(nombre_algoritmo=algoritmo_gpv_app,meta=meta_gpv_app,fechas_entrenamiento=fechas_entrenamiento,fechas_proyeccion=fechas_proyeccion,directorio=directorio_salida, holdout_dias=30)
forecast_gpv_app = forecast_gpv_app.proyectar(gpv_app)

# =================================================================
# AJUSTE PUNTUAL CYBER DAYS (6, 7 y 8 de Octubre)
# =================================================================
# Define los días y el factor de ajuste para el evento Cyber.
# Puedes cambiar el 'factor_ajuste_gpv' según la expectativa de aumento (ej: 1.8 para un 80% más).
fechas_cyber = list(CYBER_UPLIFT_GPV)   # V11: desde CONFIG (ver _aplicar_cyber)
factor_ajuste_gpv = 1

# Crear una máscara booleana para los días del evento Cyber
mask_cyber_gpv = forecast_gpv_app['fecha'].isin(pd.to_datetime(fechas_cyber))

# Aplicar el ajuste solo a las predicciones de los días del Cyber
# Se usa 'mask' para asegurar que solo se modifiquen los días futuros (no los reales ya ocurridos)
mask = forecast_gpv_app['gpv_app'].isna()
forecast_gpv_app.loc[mask & mask_cyber_gpv, 'gpv_prediccion_app'] *= factor_ajuste_gpv
_aplicar_cyber(forecast_gpv_app, 'gpv_app', 'gpv_prediccion_app', CYBER_UPLIFT_GPV, 'GPV App')
# =================================================================


#--- V7: PACE-BASED SCALING para gpv_app (reemplaza *= 1.2) ---
mask = forecast_gpv_app['gpv_app'].isna()

# Compute pace target usando los mismos boundary dates que MAU
_mtd_gpv_actual = forecast_gpv_app[
    (forecast_gpv_app['fecha'] >= _ini_mes_actual) &
    (forecast_gpv_app['fecha'] <= _ayer)
]['gpv_app'].sum()
_mtd_gpv_ant = forecast_gpv_app[
    (forecast_gpv_app['fecha'] >= _ini_mes_ant) &
    (forecast_gpv_app['fecha'] <= _ayer_mes_ant)
]['gpv_app'].sum()
_cerrado_gpv_ant = forecast_gpv_app[
    (forecast_gpv_app['fecha'] >= _ini_mes_ant) &
    (forecast_gpv_app['fecha'] <= _fin_mes_ant)
]['gpv_app'].sum()

# V9: BLEND GPV entre PACE (realidad) y MODEL RAW (crecimiento predicho)
# Usa el mismo PACE_WEIGHT definido para MAU (arriba)
_gpv_actual_precheck = forecast_gpv_app.loc[~mask, 'gpv_app'].sum()
_gpv_pred_precheck   = forecast_gpv_app.loc[mask, 'gpv_prediccion_app'].sum()
_gpv_model_landing   = _gpv_actual_precheck + _gpv_pred_precheck   # CLP raw

# V11: peso propio para GPV (antes reusaba el del MAU, con SU chequeo de divergencia)
PACE_WEIGHT_GPV, _audit_diag_gpv = _pace_weight_adaptativo(
    dia_mes=_dia_mes,
    pace_val=None if PACE_GPV_APP is None else PACE_GPV_APP * 1_000_000,
    model_val=_gpv_model_landing,
    kpi_label="GPV APP",
)
if PACE_GPV_APP is not None and PACE_GPV_APP > 0:
    _pace_gpv_raw = PACE_GPV_APP * 1_000_000   # MM CLP -> CLP
    _pace_target_gpv = PACE_WEIGHT_GPV * _pace_gpv_raw + (1 - PACE_WEIGHT_GPV) * _gpv_model_landing
    print(f'[GPV APP BLEND] pace={PACE_GPV_APP:,.0f} MM | model_raw={_gpv_model_landing/1e6:,.0f} MM | weight={PACE_WEIGHT_GPV:.3f} | target={_pace_target_gpv/1e6:,.0f} MM')
else:
    _pace_target_gpv = _gpv_model_landing
    print(f'[GPV APP BLEND] !! Sin pace. Usando solo model_raw={_pace_target_gpv/1e6:,.0f} MM')

_gpv_actual_acum = forecast_gpv_app.loc[~mask, 'gpv_app'].sum()
_gpv_pred_bruta  = forecast_gpv_app.loc[mask, 'gpv_prediccion_app'].sum()
_gpv_factor = ((_pace_target_gpv - _gpv_actual_acum) / _gpv_pred_bruta) if _gpv_pred_bruta > 0 else 1.0
print(f'[GPV APP] actual={_gpv_actual_acum:,.0f} | pred_bruta={_gpv_pred_bruta:,.0f} | pace_target={_pace_target_gpv:,.0f} | meta_ref={meta_gpv_app:,.0f} | factor={_gpv_factor:.4f}')
if _gpv_factor < 0:
    print(f'  !! WARN: pace target ya superado por actual GPV. factor=1.0')
    _gpv_factor = 1.0
forecast_gpv_app.loc[mask, 'gpv_prediccion_app'] *= _gpv_factor

_audit_diag_gpv.update({
    "pace_target_efectivo": float(_pace_target_gpv),
    "factor_meta": float(_gpv_factor),
    "meta_ref": float(meta_gpv_app),
    "actual_acumulado": float(_gpv_actual_acum),
    "pred_bruta": float(_gpv_pred_bruta),
    "landing_final_publicado": float(_pace_target_gpv),
    "dias_restantes": DIAS_RESTANTES_MES,
})
_AUDIT_ROWS.append(_audit_diag_gpv)

forecast_gpv_app['gpv_prediccion_app'] = forecast_gpv_app['gpv_prediccion_app'].round(0)
forecast_gpv_app.loc[mask, 'gpv_real_pred_app'] = forecast_gpv_app.loc[mask, 'gpv_prediccion_app']
forecast_gpv_app['gpv_prediccion_cumsum_app'] = forecast_gpv_app['gpv_prediccion_app'].cumsum()
cumsum_real = forecast_gpv_app['gpv_real_pred_app'].cumsum()
forecast_gpv_app.loc[mask, 'gpv_real_pred_cumsum2_app'] = cumsum_real[mask]
#-------------------------------------

# --- forecast_gpv_tc_fis ---
forecast_gpv_tc_fis = ForecastProductoV5(nombre_algoritmo=algoritmo_gpv_tc_fis,meta=meta_gpv_tc_fis,fechas_entrenamiento=fechas_entrenamiento_tc,fechas_proyeccion=fechas_proyeccion,directorio=directorio_salida, holdout_dias=30)
forecast_gpv_tc_fis = forecast_gpv_tc_fis.proyectar(gpv_tc_fis)
#--- Ajuste y cálculo de acumulados ---
mask = forecast_gpv_tc_fis['gpv_tc_fis'].isna()
forecast_gpv_tc_fis.loc[mask, 'gpv_prediccion_tc_fis'] *= FACTORES_GPV['tc_fis']
_aplicar_cyber(forecast_gpv_tc_fis, 'gpv_tc_fis', 'gpv_prediccion_tc_fis', CYBER_UPLIFT_PROD['tc'], 'tc_fis')  # V10: 1.2
forecast_gpv_tc_fis['gpv_prediccion_tc_fis'] = forecast_gpv_tc_fis['gpv_prediccion_tc_fis'].round(0)
forecast_gpv_tc_fis.loc[mask, 'gpv_real_pred_tc_fis'] = forecast_gpv_tc_fis.loc[mask, 'gpv_prediccion_tc_fis']
forecast_gpv_tc_fis['gpv_prediccion_cumsum_tc_fis'] = forecast_gpv_tc_fis['gpv_prediccion_tc_fis'].cumsum()
cumsum_real = forecast_gpv_tc_fis['gpv_real_pred_tc_fis'].cumsum()
forecast_gpv_tc_fis.loc[mask, 'gpv_real_pred_cumsum2_tc_fis'] = cumsum_real[mask]
#-------------------------------------

# --- forecast_gpv_tc_vir ---
forecast_gpv_tc_vir = ForecastProductoV5(nombre_algoritmo=algoritmo_gpv_tc_vir,meta=meta_gpv_tc_vir,fechas_entrenamiento=fechas_entrenamiento_tc,fechas_proyeccion=fechas_proyeccion,directorio=directorio_salida, holdout_dias=30)
forecast_gpv_tc_vir = forecast_gpv_tc_vir.proyectar(gpv_tc_vir)
#--- Ajuste y cálculo de acumulados ---
mask = forecast_gpv_tc_vir['gpv_tc_vir'].isna()
forecast_gpv_tc_vir.loc[mask, 'gpv_prediccion_tc_vir'] *= FACTORES_GPV['tc_vir']
_aplicar_cyber(forecast_gpv_tc_vir, 'gpv_tc_vir', 'gpv_prediccion_tc_vir', CYBER_UPLIFT_PROD['tc'], 'tc_vir')  # V10: 1.2
forecast_gpv_tc_vir['gpv_prediccion_tc_vir'] = forecast_gpv_tc_vir['gpv_prediccion_tc_vir'].round(0)
forecast_gpv_tc_vir.loc[mask, 'gpv_real_pred_tc_vir'] = forecast_gpv_tc_vir.loc[mask, 'gpv_prediccion_tc_vir']
forecast_gpv_tc_vir['gpv_prediccion_cumsum_tc_vir'] = forecast_gpv_tc_vir['gpv_prediccion_tc_vir'].cumsum()
cumsum_real = forecast_gpv_tc_vir['gpv_real_pred_tc_vir'].cumsum()
forecast_gpv_tc_vir.loc[mask, 'gpv_real_pred_cumsum2_tc_vir'] = cumsum_real[mask]
#-------------------------------------

# --- forecast_gpv_prp_fis ---
forecast_gpv_prp_fis = ForecastProductoV5(nombre_algoritmo=algoritmo_gpv_prp_fis,meta=meta_gpv_prp_fis,fechas_entrenamiento=fechas_entrenamiento,fechas_proyeccion=fechas_proyeccion,directorio=directorio_salida, holdout_dias=30)
forecast_gpv_prp_fis = forecast_gpv_prp_fis.proyectar(gpv_prp_fis)
#--- Ajuste y cálculo de acumulados ---
mask = forecast_gpv_prp_fis['gpv_prp_fis'].isna()
forecast_gpv_prp_fis.loc[mask, 'gpv_prediccion_prp_fis'] *= FACTORES_GPV['prp_fis']
_aplicar_cyber(forecast_gpv_prp_fis, 'gpv_prp_fis', 'gpv_prediccion_prp_fis', CYBER_UPLIFT_PROD['prp'], 'prp_fis')  # V10: 1.2
forecast_gpv_prp_fis['gpv_prediccion_prp_fis'] = forecast_gpv_prp_fis['gpv_prediccion_prp_fis'].round(0)
forecast_gpv_prp_fis.loc[mask, 'gpv_real_pred_prp_fis'] = forecast_gpv_prp_fis.loc[mask, 'gpv_prediccion_prp_fis']
forecast_gpv_prp_fis['gpv_prediccion_cumsum_prp_fis'] = forecast_gpv_prp_fis['gpv_prediccion_prp_fis'].cumsum()
cumsum_real = forecast_gpv_prp_fis['gpv_real_pred_prp_fis'].cumsum()
forecast_gpv_prp_fis.loc[mask, 'gpv_real_pred_cumsum2_prp_fis'] = cumsum_real[mask]
#-------------------------------------

# --- forecast_gpv_prp_vir ---
forecast_gpv_prp_vir = ForecastProductoV5(nombre_algoritmo=algoritmo_gpv_prp_vir,meta=meta_gpv_prp_vir,fechas_entrenamiento=fechas_entrenamiento,fechas_proyeccion=fechas_proyeccion,directorio=directorio_salida, holdout_dias=30)
forecast_gpv_prp_vir = forecast_gpv_prp_vir.proyectar(gpv_prp_vir)
#--- Ajuste y cálculo de acumulados ---
mask = forecast_gpv_prp_vir['gpv_prp_vir'].isna()
forecast_gpv_prp_vir.loc[mask, 'gpv_prediccion_prp_vir'] *= FACTORES_GPV['prp_vir']
_aplicar_cyber(forecast_gpv_prp_vir, 'gpv_prp_vir', 'gpv_prediccion_prp_vir', CYBER_UPLIFT_PROD['prp'], 'prp_vir')  # V10: 1.2
forecast_gpv_prp_vir['gpv_prediccion_prp_vir'] = forecast_gpv_prp_vir['gpv_prediccion_prp_vir'].round(0)
forecast_gpv_prp_vir.loc[mask, 'gpv_real_pred_prp_vir'] = forecast_gpv_prp_vir.loc[mask, 'gpv_prediccion_prp_vir']
forecast_gpv_prp_vir['gpv_prediccion_cumsum_prp_vir'] = forecast_gpv_prp_vir['gpv_prediccion_prp_vir'].cumsum()
cumsum_real = forecast_gpv_prp_vir['gpv_real_pred_prp_vir'].cumsum()
forecast_gpv_prp_vir.loc[mask, 'gpv_real_pred_cumsum2_prp_vir'] = cumsum_real[mask]
#-------------------------------------

# --- forecast_gpv_paypal ---
forecast_gpv_paypal = ForecastProductoV5(nombre_algoritmo=algoritmo_gpv_paypal,meta=meta_gpv_paypal,fechas_entrenamiento=fechas_entrenamiento,fechas_proyeccion=fechas_proyeccion,directorio=directorio_salida, holdout_dias=30)
forecast_gpv_paypal = forecast_gpv_paypal.proyectar(gpv_paypal)
#--- Ajuste y cálculo de acumulados ---
mask = forecast_gpv_paypal['gpv_paypal'].isna()
forecast_gpv_paypal.loc[mask, 'gpv_prediccion_paypal'] *= FACTORES_GPV['paypal']  # V10: 1.2
forecast_gpv_paypal['gpv_prediccion_paypal'] = forecast_gpv_paypal['gpv_prediccion_paypal'].round(0)
forecast_gpv_paypal.loc[mask, 'gpv_real_pred_paypal'] = forecast_gpv_paypal.loc[mask, 'gpv_prediccion_paypal']
forecast_gpv_paypal['gpv_prediccion_cumsum_paypal'] = forecast_gpv_paypal['gpv_prediccion_paypal'].cumsum()
cumsum_real = forecast_gpv_paypal['gpv_real_pred_paypal'].cumsum()
forecast_gpv_paypal.loc[mask, 'gpv_real_pred_cumsum2_paypal'] = cumsum_real[mask]
#-------------------------------------

# --- forecast_gpv_paypal_abonos ---
forecast_gpv_paypal_abonos = ForecastProductoV5(nombre_algoritmo=algoritmo_gpv_paypal_abonos,meta=meta_gpv_paypal_abonos,fechas_entrenamiento=fechas_entrenamiento,fechas_proyeccion=fechas_proyeccion,directorio=directorio_salida, holdout_dias=30)
forecast_gpv_paypal_abonos = forecast_gpv_paypal_abonos.proyectar(gpv_paypal_abonos)
#--- Ajuste y cálculo de acumulados ---
mask = forecast_gpv_paypal_abonos['gpv_paypal_abonos'].isna()
forecast_gpv_paypal_abonos.loc[mask, 'gpv_prediccion_paypal_abonos'] *= FACTORES_GPV['paypal_abonos']  # V10: 1.2
forecast_gpv_paypal_abonos['gpv_prediccion_paypal_abonos'] = forecast_gpv_paypal_abonos['gpv_prediccion_paypal_abonos'].round(0)
forecast_gpv_paypal_abonos.loc[mask, 'gpv_real_pred_paypal_abonos'] = forecast_gpv_paypal_abonos.loc[mask, 'gpv_prediccion_paypal_abonos']
forecast_gpv_paypal_abonos['gpv_prediccion_cumsum_paypal_abonos'] = forecast_gpv_paypal_abonos['gpv_prediccion_paypal_abonos'].cumsum()
cumsum_real = forecast_gpv_paypal_abonos['gpv_real_pred_paypal_abonos'].cumsum()
forecast_gpv_paypal_abonos.loc[mask, 'gpv_real_pred_cumsum2_paypal_abonos'] = cumsum_real[mask]
#-------------------------------------

# --- forecast_gpv_pdc ---
forecast_gpv_pdc = ForecastProductoV5(nombre_algoritmo=algoritmo_gpv_pdc,meta=meta_gpv_pdc,fechas_entrenamiento=fechas_entrenamiento,fechas_proyeccion=fechas_proyeccion,directorio=directorio_salida, holdout_dias=30)
forecast_gpv_pdc = forecast_gpv_pdc.proyectar(gpv_pdc)
#--- Ajuste y cálculo de acumulados ---
mask = forecast_gpv_pdc['gpv_pdc'].isna()
forecast_gpv_pdc.loc[mask, 'gpv_prediccion_pdc'] *= FACTORES_GPV['pdc']  # V10: 1.2
forecast_gpv_pdc['gpv_prediccion_pdc'] = forecast_gpv_pdc['gpv_prediccion_pdc'].round(0)
forecast_gpv_pdc.loc[mask, 'gpv_real_pred_pdc'] = forecast_gpv_pdc.loc[mask, 'gpv_prediccion_pdc']
forecast_gpv_pdc['gpv_prediccion_cumsum_pdc'] = forecast_gpv_pdc['gpv_prediccion_pdc'].cumsum()
cumsum_real = forecast_gpv_pdc['gpv_real_pred_pdc'].cumsum()
forecast_gpv_pdc.loc[mask, 'gpv_real_pred_cumsum2_pdc'] = cumsum_real[mask]
#-------------------------------------

# --- forecast_gpv_crossborder ---
forecast_gpv_crossborder = ForecastProductoV5(nombre_algoritmo=algoritmo_gpv_crossborder,meta=meta_gpv_crossborder,fechas_entrenamiento=fechas_entrenamiento,fechas_proyeccion=fechas_proyeccion,directorio=directorio_salida, holdout_dias=30)
forecast_gpv_crossborder = forecast_gpv_crossborder.proyectar(gpv_crossborder)
#--- Ajuste y cálculo de acumulados ---
mask = forecast_gpv_crossborder['gpv_crossborder'].isna()
forecast_gpv_crossborder.loc[mask, 'gpv_prediccion_crossborder'] *= FACTORES_GPV['crossborder']  # V10: 1.2
forecast_gpv_crossborder['gpv_prediccion_crossborder'] = forecast_gpv_crossborder['gpv_prediccion_crossborder'].round(0)
forecast_gpv_crossborder.loc[mask, 'gpv_real_pred_crossborder'] = forecast_gpv_crossborder.loc[mask, 'gpv_prediccion_crossborder']
forecast_gpv_crossborder['gpv_prediccion_cumsum_crossborder'] = forecast_gpv_crossborder['gpv_prediccion_crossborder'].cumsum()
cumsum_real = forecast_gpv_crossborder['gpv_real_pred_crossborder'].cumsum()
forecast_gpv_crossborder.loc[mask, 'gpv_real_pred_cumsum2_crossborder'] = cumsum_real[mask]
#-------------------------------------

# --- forecast_gpv_insurance ---
forecast_gpv_insurance = ForecastProductoV5(nombre_algoritmo=algoritmo_gpv_insurance,meta=meta_gpv_insurance,fechas_entrenamiento=fechas_entrenamiento_tc,fechas_proyeccion=fechas_proyeccion,directorio=directorio_salida, holdout_dias=30)
forecast_gpv_insurance = forecast_gpv_insurance.proyectar(gpv_insurance)

if not np.issubdtype(forecast_gpv_insurance['fecha'].dtype, np.datetime64):
    forecast_gpv_insurance['fecha'] = pd.to_datetime(forecast_gpv_insurance['fecha'])

# ---- AJUSTE ESPECÍFICO INSURANCE ----
mask_null = forecast_gpv_insurance['gpv_insurance'].isna() # Máscara para todos los días futuros

# --- (INICIO) NUEVO AJUSTE GENERAL ---
# Aplicar ajuste general a todos los días futuros ANTES del ajuste puntual
forecast_gpv_insurance.loc[mask_null, 'gpv_prediccion_insurance'] *= 1 # <--- TU FACTOR DE AJUSTE
# --- (FIN) NUEVO AJUSTE GENERAL ---

# Ajuste puntual (Día 20)
mask_day20 = forecast_gpv_insurance['fecha'].dt.day == 20
mask_ajuste_dia_20 = mask_null & mask_day20

# Aplicar ajuste solo el día 20 (se aplica sobre el ajuste general que acabamos de hacer)
forecast_gpv_insurance.loc[mask_ajuste_dia_20, 'gpv_prediccion_insurance'] *= 1.32 # Ajustado del 1.5 original

# Redondear después de todos los ajustes
forecast_gpv_insurance['gpv_prediccion_insurance'] = forecast_gpv_insurance['gpv_prediccion_insurance'].round(0)

# Replicar la estructura de columnas para consistencia (usando mask_null para todas las fechas futuras)
mask = forecast_gpv_insurance['gpv_insurance'].isna() # 'mask' es lo mismo que 'mask_null'
forecast_gpv_insurance.loc[mask, 'gpv_real_pred_insurance'] = forecast_gpv_insurance.loc[mask, 'gpv_prediccion_insurance']
forecast_gpv_insurance['gpv_prediccion_cumsum_insurance'] = forecast_gpv_insurance['gpv_prediccion_insurance'].cumsum()
cumsum_real = forecast_gpv_insurance['gpv_real_pred_insurance'].cumsum()
# Corregido: Nombre de columna consistente
forecast_gpv_insurance.loc[mask, 'gpv_real_pred_cumsum2_insurance'] = cumsum_real[mask]
# -----------------------------

# --- forecast_gpv_top_ups ---
forecast_gpv_top_ups = ForecastProductoV5(nombre_algoritmo=algoritmo_gpv_top_ups,meta=meta_gpv_top_ups,fechas_entrenamiento=fechas_entrenamiento,fechas_proyeccion=fechas_proyeccion,directorio=directorio_salida, holdout_dias=30)
forecast_gpv_top_ups = forecast_gpv_top_ups.proyectar(gpv_top_ups)
#--- Ajuste y cálculo de acumulados ---
mask = forecast_gpv_top_ups['gpv_top_ups'].isna()
forecast_gpv_top_ups.loc[mask, 'gpv_prediccion_top_ups'] *= FACTORES_GPV['top_ups']  # V10: 1.1
forecast_gpv_top_ups['gpv_prediccion_top_ups'] = forecast_gpv_top_ups['gpv_prediccion_top_ups'].round(0)
forecast_gpv_top_ups.loc[mask, 'gpv_real_pred_top_ups'] = forecast_gpv_top_ups.loc[mask, 'gpv_prediccion_top_ups']
forecast_gpv_top_ups['gpv_prediccion_cumsum_top_ups'] = forecast_gpv_top_ups['gpv_prediccion_top_ups'].cumsum()
cumsum_real = forecast_gpv_top_ups['gpv_real_pred_top_ups'].cumsum()
forecast_gpv_top_ups.loc[mask, 'gpv_real_pred_cumsum2_top_ups'] = cumsum_real[mask]
#-------------------------------------

# --- forecast_gpv_p2p ---
forecast_gpv_p2p = ForecastProductoV5(nombre_algoritmo=algoritmo_gpv_p2p,meta=meta_gpv_p2p,fechas_entrenamiento=fechas_entrenamiento,fechas_proyeccion=fechas_proyeccion,directorio=directorio_salida, holdout_dias=30)
forecast_gpv_p2p = forecast_gpv_p2p.proyectar(gpv_p2p)
#--- Ajuste y cálculo de acumulados ---
mask = forecast_gpv_p2p['gpv_p2p'].isna()
forecast_gpv_p2p.loc[mask, 'gpv_prediccion_p2p'] *= FACTORES_GPV['p2p']  # V10: 1.1
forecast_gpv_p2p['gpv_prediccion_p2p'] = forecast_gpv_p2p['gpv_prediccion_p2p'].round(0)
forecast_gpv_p2p.loc[mask, 'gpv_real_pred_p2p'] = forecast_gpv_p2p.loc[mask, 'gpv_prediccion_p2p']
forecast_gpv_p2p['gpv_prediccion_cumsum_p2p'] = forecast_gpv_p2p['gpv_prediccion_p2p'].cumsum()
cumsum_real = forecast_gpv_p2p['gpv_real_pred_p2p'].cumsum()
forecast_gpv_p2p.loc[mask, 'gpv_real_pred_cumsum2_p2p'] = cumsum_real[mask]
#-------------------------------------

# --- forecast_gpv_cash_in_savings ---
forecast_gpv_cash_in_savings = ForecastProductoV5(nombre_algoritmo=algoritmo_gpv_cash_in_savings,meta=meta_gpv_top_ups,fechas_entrenamiento=fechas_entrenamiento,fechas_proyeccion=fechas_proyeccion,directorio=directorio_salida, holdout_dias=30)
forecast_gpv_cash_in_savings = forecast_gpv_cash_in_savings.proyectar(gpv_cash_in_savings)
#--- Ajuste y cálculo de acumulados ---
mask = forecast_gpv_cash_in_savings['gpv_cash_in_savings'].isna()
forecast_gpv_cash_in_savings.loc[mask, 'gpv_prediccion_cash_in_savings'] *= FACTORES_GPV['cash_in_savings']  # V10: 1.1
forecast_gpv_cash_in_savings['gpv_prediccion_cash_in_savings'] = forecast_gpv_cash_in_savings['gpv_prediccion_cash_in_savings'].round(0)
forecast_gpv_cash_in_savings.loc[mask, 'gpv_real_pred_cash_in_savings'] = forecast_gpv_cash_in_savings.loc[mask, 'gpv_prediccion_cash_in_savings']
forecast_gpv_cash_in_savings['gpv_prediccion_cumsum_cash_in_savings'] = forecast_gpv_cash_in_savings['gpv_prediccion_cash_in_savings'].cumsum()
cumsum_real = forecast_gpv_cash_in_savings['gpv_real_pred_cash_in_savings'].cumsum()
forecast_gpv_cash_in_savings.loc[mask, 'gpv_real_pred_cumsum2_cash_in_savings'] = cumsum_real[mask]
#-------------------------------------

# --- forecast_gpv_investment_tyba ---
forecast_gpv_investment_tyba = ForecastProductoV5(nombre_algoritmo=algoritmo_gpv_investment_tyba,meta=meta_gpv_top_ups,fechas_entrenamiento=fechas_entrenamiento,fechas_proyeccion=fechas_proyeccion,directorio=directorio_salida, holdout_dias=30)
forecast_gpv_investment_tyba = forecast_gpv_investment_tyba.proyectar(gpv_investment_tyba)
#--- Ajuste y cálculo de acumulados ---
mask = forecast_gpv_investment_tyba['gpv_investment_tyba'].isna()
forecast_gpv_investment_tyba.loc[mask, 'gpv_prediccion_investment_tyba'] *= FACTORES_GPV['investment_tyba']  # V10: 1.030
forecast_gpv_investment_tyba['gpv_prediccion_investment_tyba'] = forecast_gpv_investment_tyba['gpv_prediccion_investment_tyba'].round(0)
forecast_gpv_investment_tyba.loc[mask, 'gpv_real_pred_investment_tyba'] = forecast_gpv_investment_tyba.loc[mask, 'gpv_prediccion_investment_tyba']
forecast_gpv_investment_tyba['gpv_prediccion_cumsum_investment_tyba'] = forecast_gpv_investment_tyba['gpv_prediccion_investment_tyba'].cumsum()
cumsum_real = forecast_gpv_investment_tyba['gpv_real_pred_investment_tyba'].cumsum()
forecast_gpv_investment_tyba.loc[mask, 'gpv_real_pred_cumsum2_investment_tyba'] = cumsum_real[mask]
#-------------------------------------

# --- forecast_trx_ci_bestado ---
forecast_trx_ci_bestado = ForecastProductoV5(nombre_algoritmo=algoritmo_trx_ci_bestado,meta=meta_trx_ci_bestado,fechas_entrenamiento=fechas_entrenamiento,fechas_proyeccion=fechas_proyeccion,directorio=directorio_salida, holdout_dias=30)
forecast_trx_ci_bestado = forecast_trx_ci_bestado.proyectar(trx_ci_bestado)
#--- Ajuste y cálculo de acumulados ---
mask = forecast_trx_ci_bestado['trx_ci_bestado'].isna()
forecast_trx_ci_bestado.loc[mask, 'trx_prediccion_ci_bestado'] *= FACTORES_GPV['trx_ci_bestado']
forecast_trx_ci_bestado['trx_prediccion_ci_bestado'] = forecast_trx_ci_bestado['trx_prediccion_ci_bestado'].round(0)
forecast_trx_ci_bestado.loc[mask, 'trx_real_pred_ci_bestado'] = forecast_trx_ci_bestado.loc[mask, 'trx_prediccion_ci_bestado']
forecast_trx_ci_bestado['trx_prediccion_cumsum_ci_bestado'] = forecast_trx_ci_bestado['trx_prediccion_ci_bestado'].cumsum()
cumsum_real = forecast_trx_ci_bestado['trx_real_pred_ci_bestado'].cumsum()
forecast_trx_ci_bestado.loc[mask, 'trx_real_pred_cumsum2_ci_bestado'] = cumsum_real[mask]
#-------------------------------------

# --- forecast_trx_ci_obancos ---
forecast_trx_ci_obancos = ForecastProductoV5(nombre_algoritmo=algoritmo_trx_ci_obancos,meta=meta_trx_ci_obancos,fechas_entrenamiento=fechas_entrenamiento,fechas_proyeccion=fechas_proyeccion,directorio=directorio_salida, holdout_dias=30)
forecast_trx_ci_obancos = forecast_trx_ci_obancos.proyectar(trx_ci_obancos)
#--- Ajuste y cálculo de acumulados ---
mask = forecast_trx_ci_obancos['trx_ci_obancos'].isna()
forecast_trx_ci_obancos.loc[mask, 'trx_prediccion_ci_obancos'] *= FACTORES_GPV['trx_ci_obancos']
forecast_trx_ci_obancos['trx_prediccion_ci_obancos'] = forecast_trx_ci_obancos['trx_prediccion_ci_obancos'].round(0)
forecast_trx_ci_obancos.loc[mask, 'trx_real_pred_ci_obancos'] = forecast_trx_ci_obancos.loc[mask, 'trx_prediccion_ci_obancos']
forecast_trx_ci_obancos['trx_prediccion_cumsum_ci_obancos'] = forecast_trx_ci_obancos['trx_prediccion_ci_obancos'].cumsum()
cumsum_real = forecast_trx_ci_obancos['trx_real_pred_ci_obancos'].cumsum()
forecast_trx_ci_obancos.loc[mask, 'trx_real_pred_cumsum2_ci_obancos'] = cumsum_real[mask]



# --- forecast_trx_co_bestado ---
forecast_trx_co_bestado = ForecastProductoV5(nombre_algoritmo=algoritmo_trx_co_bestado,meta=meta_trx_co_bestado,fechas_entrenamiento=fechas_entrenamiento,fechas_proyeccion=fechas_proyeccion,directorio=directorio_salida, holdout_dias=30)
forecast_trx_co_bestado = forecast_trx_co_bestado.proyectar(trx_co_bestado)
#--- Ajuste y cálculo de acumulados ---
mask = forecast_trx_co_bestado['trx_co_bestado'].isna() # CORRECTED LINE
forecast_trx_co_bestado.loc[mask, 'trx_prediccion_co_bestado'] *= FACTORES_GPV['trx_co_bestado']
forecast_trx_co_bestado['trx_prediccion_co_bestado'] = forecast_trx_co_bestado['trx_prediccion_co_bestado'].round(0)
forecast_trx_co_bestado.loc[mask, 'trx_real_pred_co_bestado'] = forecast_trx_co_bestado.loc[mask, 'trx_prediccion_co_bestado']
forecast_trx_co_bestado['trx_prediccion_cumsum_co_bestado'] = forecast_trx_co_bestado['trx_prediccion_co_bestado'].cumsum()
cumsum_real = forecast_trx_co_bestado['trx_real_pred_co_bestado'].cumsum()
forecast_trx_co_bestado.loc[mask, 'trx_real_pred_cumsum2_co_bestado'] = cumsum_real[mask]
#-------------------------------------

# --- forecast_trx_co_obancos ---
forecast_trx_co_obancos = ForecastProductoV5(nombre_algoritmo=algoritmo_trx_co_obancos,meta=meta_trx_co_obancos,fechas_entrenamiento=fechas_entrenamiento,fechas_proyeccion=fechas_proyeccion,directorio=directorio_salida, holdout_dias=30) # CORRECTED LINE
forecast_trx_co_obancos = forecast_trx_co_obancos.proyectar(trx_co_obancos) # CORRECTED LINE
#--- Ajuste y cálculo de acumulados ---
mask = forecast_trx_co_obancos['trx_co_obancos'].isna() # CORRECTED LINE
forecast_trx_co_obancos.loc[mask, 'trx_prediccion_co_obancos'] *= FACTORES_GPV['trx_co_obancos']
forecast_trx_co_obancos['trx_prediccion_co_obancos'] = forecast_trx_co_obancos['trx_prediccion_co_obancos'].round(0)
forecast_trx_co_obancos.loc[mask, 'trx_real_pred_co_obancos'] = forecast_trx_co_obancos.loc[mask, 'trx_prediccion_co_obancos']
forecast_trx_co_obancos['trx_prediccion_cumsum_co_obancos'] = forecast_trx_co_obancos['trx_prediccion_co_obancos'].cumsum()
cumsum_real = forecast_trx_co_obancos['trx_real_pred_co_obancos'].cumsum()
forecast_trx_co_obancos.loc[mask, 'trx_real_pred_cumsum2_co_obancos'] = cumsum_real[mask]
#-------------------------------------

# =================================================================
# 1. FORECAST: REVENUE ATM INTERNACIONAL Y NACIONAL (MOVILIZADOS AQUÍ)
# =================================================================
forecast_rev_atm_int = ForecastProducto(
    nombre_algoritmo=algoritmo_rev_atm_int,
    meta=meta_rev_atm_int,
    fechas_entrenamiento=fechas_entrenamiento,
    fechas_proyeccion=fechas_proyeccion,
    directorio=directorio_salida
)
forecast_rev_atm_int = forecast_rev_atm_int.proyectar(df_rev_atm_int)

# --- Ajuste y cálculo de acumulados ---
mask_int = forecast_rev_atm_int['rev_atm_int'].isna()
forecast_rev_atm_int.loc[mask_int, 'rev_prediccion_atm_int'] *= 1.0 # Puedes poner un factor como 1.1 si esperas crecimiento
forecast_rev_atm_int['rev_prediccion_atm_int'] = forecast_rev_atm_int['rev_prediccion_atm_int'].round(0)
forecast_rev_atm_int.loc[mask_int, 'rev_real_pred_atm_int'] = forecast_rev_atm_int.loc[mask_int, 'rev_prediccion_atm_int']
forecast_rev_atm_int['rev_prediccion_cumsum_atm_int'] = forecast_rev_atm_int['rev_prediccion_atm_int'].cumsum()
cumsum_real_int = forecast_rev_atm_int['rev_real_pred_atm_int'].cumsum()
forecast_rev_atm_int.loc[mask_int, 'rev_real_pred_cumsum2_atm_int'] = cumsum_real_int[mask_int]

# =================================================================
# FORECAST: REVENUE ATM NACIONAL (REDBANK)
# =================================================================
forecast_rev_atm_nac = ForecastProducto(
    nombre_algoritmo=algoritmo_rev_atm_nac,
    meta=meta_rev_atm_nac,
    fechas_entrenamiento=fechas_entrenamiento,
    fechas_proyeccion=fechas_proyeccion,
    directorio=directorio_salida
)
forecast_rev_atm_nac = forecast_rev_atm_nac.proyectar(df_rev_atm_nac)

# --- Ajuste y cálculo de acumulados ---
mask_nac = forecast_rev_atm_nac['rev_atm_nac'].isna()
forecast_rev_atm_nac.loc[mask_nac, 'rev_prediccion_atm_nac'] *= 1.0
forecast_rev_atm_nac['rev_prediccion_atm_nac'] = forecast_rev_atm_nac['rev_prediccion_atm_nac'].round(0)
forecast_rev_atm_nac.loc[mask_nac, 'rev_real_pred_atm_nac'] = forecast_rev_atm_nac.loc[mask_nac, 'rev_prediccion_atm_nac']
forecast_rev_atm_nac['rev_prediccion_cumsum_atm_nac'] = forecast_rev_atm_nac['rev_prediccion_atm_nac'].cumsum()
cumsum_real_nac = forecast_rev_atm_nac['rev_real_pred_atm_nac'].cumsum()
forecast_rev_atm_nac.loc[mask_nac, 'rev_real_pred_cumsum2_atm_nac'] = cumsum_real_nac[mask_nac]


# =================================================================
# 2. LISTA DFS (CON TODOS LOS FORECAST YA CREADOS)
# =================================================================
dfs = [
    forecast_gpv_tc_fis, forecast_gpv_tc_vir, forecast_gpv_prp_fis, forecast_gpv_prp_vir,
    forecast_gpv_pdc, forecast_gpv_insurance, forecast_gpv_crossborder, forecast_gpv_paypal,
    forecast_gpv_paypal_abonos, forecast_gpv_p2p,forecast_gpv_top_ups, forecast_gpv_cash_in_savings, forecast_gpv_investment_tyba,
    forecast_rev_atm_int, forecast_rev_atm_nac
]

num_cols = forecast_gpv_paypal.select_dtypes(include=[np.number]).columns
forecast_gpv_paypal[num_cols] = forecast_gpv_paypal[num_cols].map(lambda x: 0)

num_cols = forecast_gpv_paypal_abonos.select_dtypes(include=[np.number]).columns
forecast_gpv_paypal_abonos[num_cols] = forecast_gpv_paypal_abonos[num_cols].map(lambda x: 0)


gpv_merged = dfs[0]
for df in dfs[1:]:
    gpv_merged = gpv_merged.merge(df, on="fecha", how="outer")
# V11: solo columnas de GPV (antes entraban rev_real_pred_atm_* al prorrateo)
gpv_merged_para_ajustar = gpv_merged[[col for col in gpv_merged.columns if col == "fecha" or (col.startswith("gpv_real_pred") and "cumsum" not in col)]]

forecast_gpv_app.columns = forecast_gpv_app.columns.str.replace('_app', '', regex=False)
df_adjust = gpv_merged_para_ajustar.merge(
    forecast_gpv_app[['fecha','gpv_real_pred']], on='fecha', how='left'
)

forecast_gpv_app.rename(columns={'meta':'meta_gpv_app'},inplace=True)

# 2. Obtener la lista de columnas de productos (excluyendo "fecha" y "gpv_real_pred")
cols_productos = [col for col in df_adjust.columns if col not in ['fecha', 'gpv_real_pred']]

# # Calcular el total de GPV de los productos en cada fila
df_adjust['total_productos'] = df_adjust[cols_productos].sum(axis=1)

col_fijo = 'gpv_real_pred_insurance'

# 2. Calcular el total disponible para prorratear (es decir, lo que queda después de dejar fija la columna)
df_adjust['total_ajustable'] = df_adjust['gpv_real_pred'] - df_adjust[col_fijo]

# 3. Calcular el total original de los otros productos (sin contar insurance)
otros_productos = [col for col in cols_productos if col != col_fijo]
df_adjust['suma_otros_productos'] = df_adjust[otros_productos].sum(axis=1)

# 4. Ajustar cada producto excepto insurance -- V11: SOLO dias futuros.
#    Antes tambien reescalaba los dias con real, asi que el "real" por producto
#    publicado no era el real sino un prorrateo al GPV App.
_mask_fut_adj = pd.to_datetime(df_adjust['fecha']) > _ayer
_fac_adj = (df_adjust['total_ajustable'] / df_adjust['suma_otros_productos'].replace(0, np.nan)).fillna(1.0)
for col in otros_productos:
    df_adjust.loc[_mask_fut_adj, col] = df_adjust.loc[_mask_fut_adj, col] * _fac_adj[_mask_fut_adj]

# 5. Eliminar columnas auxiliares si no se necesitan
df_adjust.drop(columns=['total_ajustable', 'suma_otros_productos'], inplace=True)


# Opcional: Verificar que la suma de los productos ajustados sea igual a gpv_real_pred
df_adjust['total_ajustado'] = df_adjust[[col for col in cols_productos]].sum(axis=1)

df_adjust = df_adjust.sort_values("fecha")

# Definir los prefijos para las columnas originales y las acumuladas
original_prefix = "gpv_real_pred_"
cumsum_prefix = "gpv_real_pred_cumsum2_"

# Iterar sobre las columnas que empiecen con el prefijo original
for col in df_adjust.columns:
    if col.startswith(original_prefix) and not col.startswith(cumsum_prefix):
        # Extraer el nombre del producto que sigue al prefijo
        product_name = col[len(original_prefix):]
        new_col = cumsum_prefix + product_name

        # Crear la columna acumulada
        df_adjust[new_col] = df_adjust[col].cumsum()

# --- INICIO DE LA CORRECCIÓN ---
print(f"Forma original de df_adjust (potencialmente 126 filas): {df_adjust.shape}")
print(f"Forma de gpv_merged (debe ser 30 filas): {gpv_merged.shape}")

# 1. De-duplicar df_adjust (126 filas) quedándonos con la última entrada por fecha
#    Esto es necesario porque tiene fechas duplicadas.
df_adjust_deduped = df_adjust.drop_duplicates(subset=['fecha'], keep='last')

# 2. Ahora sí, alinear con gpv_merged por si faltan fechas
df_adjust_aligned = pd.merge(
    gpv_merged[['fecha']],   # DataFrame base con 30 fechas únicas
    df_adjust_deduped,       # DataFrame de datos de-duplicado
    on='fecha',
    how='left'               # Nos aseguramos de tener 30 filas
)

print(f"Nueva forma de df_adjust (alineada y única): {df_adjust_aligned.shape}")

# 3. Definir las columnas comunes DESPUÉS de alinear
#    Usamos el df_adjust_aligned (30 filas) para la comparación
common_cols = [col for col in gpv_merged.columns if col in df_adjust_aligned.columns]
print(f"Se copiarán {len(common_cols)} columnas comunes.")

# 4. Ejecutar el bucle SÓLO sobre las columnas comunes
for col in common_cols:
    # Usamos el df_adjust_aligned (30 filas)
    gpv_merged[col] = np.where(gpv_merged[col].isnull(), gpv_merged[col], df_adjust_aligned[col])

print("Copia de datos de df_adjust a gpv_merged completada.")
# --- FIN DE LA CORRECCIÓN v2 ---

if V12_FIXES.get("guardas_idempotencia", True) and globals().get("_V12_GPV_ESCALADO"):
    raise RuntimeError("[V12] gpv_merged ya se dividio por 1e6 en esta sesion: re-ejecuta el notebook desde el inicio (no solo esta celda).")
gpv_merged.iloc[:, 1:] = gpv_merged.iloc[:, 1:] / 1000000
_V12_GPV_ESCALADO = True

forecast_trx_ci_bestado.rename(
    columns=lambda col: col if col.startswith("trx_ci") else col.replace("trx", "trx_ci", 1),
    inplace=True
)
forecast_trx_ci_obancos.rename(
    columns=lambda col: col if col.startswith("trx_ci") else col.replace("trx", "trx_ci", 1),
    inplace=True
)

###########REVISAR
forecast_trx_co_bestado.rename(
    columns=lambda col: col if col.startswith("trx_co") else col.replace("trx", "trx_co", 1),
    inplace=True
)
forecast_trx_co_obancos.rename( # CORRECTED LINE
    columns=lambda col: col if col.startswith("trx_co") else col.replace("trx", "trx_co", 1),
    inplace=True
)


gpv_merged = gpv_merged.merge(forecast_trx_ci_bestado,on='fecha',how='left')
gpv_merged = gpv_merged.merge(forecast_trx_ci_obancos,on='fecha',how='left')
gpv_merged = gpv_merged.merge(forecast_trx_co_bestado,on='fecha',how='left')
gpv_merged = gpv_merged.merge(forecast_trx_co_obancos,on='fecha',how='left')

# --- FIX: Volver a agregar la columna meta_gpv_app al DataFrame final ---
gpv_merged = gpv_merged.merge(forecast_gpv_app[['fecha', 'meta_gpv_app']], on='fecha', how='left')

In [ ]:
# --- CORRECCIÓN SEGURA DE REVENUE/COSTO (EVITA DOBLE CÁLCULO) ---

# Definición de Tarifas
TARIFA_CI_BESTADO = 0.01
TARIFA_CI_OBANCOS = 0.0028
TARIFA_CO_BESTADO = 0.01
TARIFA_CO_OBANCOS = 0.0028

print("--- INICIANDO TRANSFORMACIÓN SEGURA A PESOS (CLP) ---")

# 1. Preparar UF
if 'valor_uf' not in gpv_merged.columns:
    gpv_merged = gpv_merged.merge(df_uf, on='fecha', how='left')
    gpv_merged['valor_uf'] = gpv_merged['valor_uf'].ffill().fillna(ultimo_valor_uf)

# 2. Función con PROTECCIÓN contra doble ejecución
def convertir_a_clp_seguro(df, col_name, tarifa):
    if col_name not in df.columns:
        print(f"⚠️ La columna {col_name} no existe.")
        return df

    # Tomamos una muestra promedio para ver la magnitud
    promedio = df[col_name].mean()

    # Si el promedio es mayor a 500.000, asumimos que YA ES DINERO y no tocamos nada.
    if promedio > 500000:
        print(f"aaa 🛑 {col_name} ya parece estar en pesos (Promedio: {promedio:,.0f}). No se modifica.")
    else:
        print(f"🔄 Convirtiendo {col_name} a CLP...")
        df[col_name] = df[col_name] * tarifa * df['valor_uf']

    return df

# --- EJECUTAR CONVERSIÓN ---

# IMPORTANTE: Primero recargamos gpv_merged si es posible.
# Si no puedes recargar las celdas anteriores, tendrás que dividir manualmente por la UF para corregir,
# pero lo ideal es volver a correr la celda 20 (donde se crea gpv_merged).

# Cash In
gpv_merged = convertir_a_clp_seguro(gpv_merged, 'trx_ci_bestado', TARIFA_CI_BESTADO)
gpv_merged = convertir_a_clp_seguro(gpv_merged, 'trx_ci_obancos', TARIFA_CI_OBANCOS)

# Cash Out
gpv_merged = convertir_a_clp_seguro(gpv_merged, 'trx_co_bestado', TARIFA_CO_BESTADO)
gpv_merged = convertir_a_clp_seguro(gpv_merged, 'trx_co_obancos', TARIFA_CO_OBANCOS)

print("✅ Proceso finalizado.")

# V11: la conversion de arriba solo pasa a CLP la columna REAL (la prediccion queda
# en n de trx y el ultimo dia sale <NA>). Se agregan columnas rev_* coherentes
# (real + prediccion, en CLP) a partir de trx_*_real_pred_*. Las columnas viejas
# no se tocan para no romper consumidores.
for _pref, _seg, _tar in [("ci", "bestado", TARIFA_CI_BESTADO), ("ci", "obancos", TARIFA_CI_OBANCOS),
                          ("co", "bestado", TARIFA_CO_BESTADO), ("co", "obancos", TARIFA_CO_OBANCOS)]:
    _src = f"trx_{_pref}_real_pred_{_pref}_{_seg}"
    if _src in gpv_merged.columns:
        gpv_merged[f"rev_{_pref}_{_seg}_real_pred"] = gpv_merged[_src].astype(float) * _tar * gpv_merged['valor_uf']
        print(f"[V11 REV] rev_{_pref}_{_seg}_real_pred = {gpv_merged[f'rev_{_pref}_{_seg}_real_pred'].sum()/1e6:,.1f} MM en el mes")
    else:
        print(f"[V11 REV] !! falta {_src}")

# --- MUESTRA FINAL ---
col_test = 'trx_ci_bestado'
if col_test in gpv_merged.columns:
    print(f"\nValor final en {col_test} (debe ser aprox 25 millones):")
    print(f"${gpv_merged[col_test].iloc[-1]:,.0f}")

In [ ]:
gpv_merged.filter(regex="insurance")

In [ ]:
forecast_gpv_insurance_credito = ForecastProductoV5(nombre_algoritmo=algoritmo_gpv_insurance_credito,meta=meta_gpv_insurance_credito,fechas_entrenamiento=fechas_entrenamiento,fechas_proyeccion=fechas_proyeccion,directorio=directorio_salida, holdout_dias=30)
forecast_gpv_insurance_credito = forecast_gpv_insurance_credito.proyectar(gpv_insurance_credito)

forecast_gpv_insurance_no_credito = ForecastProductoV5(nombre_algoritmo=algoritmo_gpv_insurance_no_credito,meta=meta_gpv_insurance_no_credito,fechas_entrenamiento=fechas_entrenamiento,fechas_proyeccion=fechas_proyeccion,directorio=directorio_salida, holdout_dias=30)
forecast_gpv_insurance_no_credito = forecast_gpv_insurance_no_credito.proyectar(gpv_insurance_no_credito)

In [ ]:
forecast_gpv_insurance_credito[['fecha','gpv_real_pred_insurance_credito']]
forecast_gpv_insurance_no_credito[['fecha','gpv_real_pred_insurance_no_credito']]

porc_insurance = pd.merge(forecast_gpv_insurance_credito[['fecha','gpv_real_pred_insurance_credito']], forecast_gpv_insurance_no_credito[['fecha','gpv_real_pred_insurance_no_credito']], on='fecha', how='outer')
porc_insurance['porc_insurance_credito'] = porc_insurance['gpv_real_pred_insurance_credito'] / (porc_insurance['gpv_real_pred_insurance_credito'] + porc_insurance['gpv_real_pred_insurance_no_credito'])
porc_insurance['porc_insurance_no_credito'] = porc_insurance['gpv_real_pred_insurance_no_credito'] / (porc_insurance['gpv_real_pred_insurance_credito'] + porc_insurance['gpv_real_pred_insurance_no_credito'])
porc_insurance = porc_insurance[['fecha', 'porc_insurance_credito', 'porc_insurance_no_credito']]

In [ ]:
forecast_gpv_ci_bestado = ForecastProductoV5(nombre_algoritmo=algoritmo_gpv_ci_bestado,meta=meta_gpv_ci_bestado,fechas_entrenamiento=fechas_entrenamiento,fechas_proyeccion=fechas_proyeccion,directorio=directorio_salida, holdout_dias=30)
forecast_gpv_ci_bestado = forecast_gpv_ci_bestado.proyectar(gpv_ci_bestado)

forecast_gpv_ci_obancos = ForecastProductoV5(nombre_algoritmo=algoritmo_gpv_ci_obancos,meta=meta_gpv_ci_obancos,fechas_entrenamiento=fechas_entrenamiento,fechas_proyeccion=fechas_proyeccion,directorio=directorio_salida, holdout_dias=30)
forecast_gpv_ci_obancos = forecast_gpv_ci_obancos.proyectar(gpv_ci_obancos)

gpv_ci_merged = forecast_gpv_ci_bestado.merge(forecast_gpv_ci_obancos, on='fecha', how='outer')


########NUEVO REVISAR
forecast_gpv_co_bestado = ForecastProductoV5(nombre_algoritmo=algoritmo_gpv_co_bestado,meta=meta_gpv_co_bestado,fechas_entrenamiento=fechas_entrenamiento,fechas_proyeccion=fechas_proyeccion,directorio=directorio_salida, holdout_dias=30)
forecast_gpv_co_bestado = forecast_gpv_co_bestado.proyectar(gpv_co_bestado)

forecast_gpv_co_obancos = ForecastProductoV5(nombre_algoritmo=algoritmo_gpv_co_obancos,meta=meta_gpv_co_obancos,fechas_entrenamiento=fechas_entrenamiento,fechas_proyeccion=fechas_proyeccion,directorio=directorio_salida, holdout_dias=30)
forecast_gpv_co_obancos = forecast_gpv_co_obancos.proyectar(gpv_co_obancos)

gpv_co_merged = forecast_gpv_co_bestado.merge(forecast_gpv_co_obancos, on='fecha', how='outer')

In [ ]:
# --- LIMPIEZA para gpv_merged --------------
print("\n--- Limpiando gpv_merged antes de subir a BigQuery ---")

# Itera sobre los nombres de columna en el DataFrame gpv_merged
for col in gpv_merged.columns:
    # Verifica si la columna es de tipo float (número decimal)
    # ¡Asegúrate de NO convertir columnas que DEBEN ser float (ej: montos con decimales)!
    # Puedes añadir condiciones aquí para saltar columnas específicas si es necesario.
    # Por ejemplo: if 'meta' in col or 'porc' in col: continue
    if pd.api.types.is_float_dtype(gpv_merged[col]):
        # Solo intenta convertir si parece un entero después de redondear
        # (para evitar convertir montos con decimales importantes a enteros)
        if (gpv_merged[col].round(0) == gpv_merged[col]).all():
            print(f"Procesando columna float (potencialmente entera): {col}")
            # 1. Redondea
            gpv_merged[col] = gpv_merged[col].round(0)
            # 2. Intenta convertir a Int64
            try:
                gpv_merged[col] = gpv_merged[col].astype('Int64')
            except (ValueError, TypeError):
                print(f"  -> Advertencia: No se pudo convertir '{col}' a Int64, se mantiene como float redondeado.")
        else:
             print(f"Saltando columna float (probablemente decimales importantes): {col}")


print("\n--- Tipos de datos finales de gpv_merged ---")
gpv_merged.info()
# --- FIN LIMPIEZA gpv_merged ----------

# ... luego viene la celda con pandas_gbq.to_gbq(gpv_merged, ...) ...

In [ ]:
project = 'tenpo-bi-prod'
pandas_gbq.to_gbq(
    gpv_merged, 'kpitos.forecasting_gpv_productos', project_id=project, if_exists='replace',
)



project = 'tenpo-bi-prod'
pandas_gbq.to_gbq(
    forecast_gpv_app, 'kpitos.forecasting_gpv_app', project_id=project, if_exists='replace',
)



project = 'tenpo-bi-prod'
pandas_gbq.to_gbq(
    gpv_ci_merged, 'kpitos.forecasting_gpv_cca_ci', project_id=project, if_exists='replace',
)


project = 'tenpo-bi-prod'
pandas_gbq.to_gbq(
    gpv_co_merged, 'kpitos.forecasting_gpv_cca_co', project_id=project, if_exists='replace',
)


project = 'tenpo-bi-prod'
pandas_gbq.to_gbq(
    porc_insurance, 'kpitos.porc_insurance', project_id=project, if_exists='replace',
)

In [ ]:
# ==============================================================================
# V11: calibracion OB TC. El modelo sobreproyectaba 3-4x en medium/small (sep-26:
# 200-380/dia vs 50-70 reales) y encima se multiplicaba por 1.4 / 1.7 a mano.
# Factor = real / prediccion del modelo en los ultimos OB_CALIB_DIAS dias con real,
# acotado a OB_CALIB_CLIP. Ademas se prohiben negativos.
# ==============================================================================
import json as _json_ob, os as _os_ob
_OB_CALIB_FILE = f"{directorio_salida}/ob_calib_v11.json"


def _factor_calib_ob(df, seg):
    """Con >= OB_CALIB_MIN_DIAS dias reales en el mes calibra y guarda el factor en Drive;
    con menos (inicio de mes) reutiliza el ultimo guardado (antes: 1.0 = modelo crudo)."""
    guardado = {}
    if _os_ob.path.exists(_OB_CALIB_FILE):
        guardado = _json_ob.load(open(_OB_CALIB_FILE))
    real = df[f'ob_{seg}']
    ult = df[real.notna()].tail(OB_CALIB_DIAS)
    if len(ult) < OB_CALIB_MIN_DIAS or ult[f'ob_prediccion_{seg}'].sum() <= 0:
        f_c = float(guardado.get(seg, OB_CALIB_DEFAULT.get(seg, 1.0)))
        print(f"[V11 OB {seg}] {len(ult)} dias reales en el mes -> uso factor guardado {f_c:.3f}")
        return f_c
    f = ult[f'ob_{seg}'].sum() / ult[f'ob_prediccion_{seg}'].sum()
    f_c = min(max(f, OB_CALIB_CLIP[0]), OB_CALIB_CLIP[1])
    print(f"[V11 OB {seg}] real {len(ult)}d={ult[f'ob_{seg}'].sum():,.0f} | modelo={ult[f'ob_prediccion_{seg}'].sum():,.0f} "
          f"| factor={f:.3f} -> {f_c:.3f}")
    guardado[seg] = round(f_c, 4)
    _json_ob.dump(guardado, open(_OB_CALIB_FILE, "w"))
    return f_c

# --- Bloque para 'large' ---
forecast_ob_tc_large = ForecastProductoV5(nombre_algoritmo=algoritmo_ob_tc_large, meta=meta_ob_tc_large, fechas_entrenamiento=fechas_entrenamiento_tc, fechas_proyeccion=fechas_proyeccion, directorio=directorio_salida, holdout_dias=30)
forecast_ob_tc_large = forecast_ob_tc_large.proyectar(ob_tc_large)

# Bloque de ajuste completo para 'large'
mask = forecast_ob_tc_large['ob_large'].isna()
forecast_ob_tc_large.loc[mask, 'ob_prediccion_large'] *= _factor_calib_ob(forecast_ob_tc_large, 'large')  # V10: x1.4
forecast_ob_tc_large['ob_prediccion_large'] = forecast_ob_tc_large['ob_prediccion_large'].clip(lower=0).round(0)
forecast_ob_tc_large.loc[mask, 'ob_real_pred_large'] = forecast_ob_tc_large.loc[mask, 'ob_prediccion_large']
forecast_ob_tc_large['ob_prediccion_cumsum_large'] = forecast_ob_tc_large['ob_prediccion_large'].cumsum()
cumsum_real = forecast_ob_tc_large['ob_real_pred_large'].cumsum()
forecast_ob_tc_large.loc[mask, 'ob_real_pred_cumsum2_large'] = cumsum_real[mask]
forecast_ob_tc_large['ob_prediccion_large'] = forecast_ob_tc_large['ob_prediccion_large'].astype(np.int64, errors='ignore')
forecast_ob_tc_large['ob_real_pred_large'] = forecast_ob_tc_large['ob_real_pred_large'].astype(np.int64, errors='ignore')
forecast_ob_tc_large['ob_prediccion_cumsum_large'] = forecast_ob_tc_large['ob_prediccion_cumsum_large'].astype(np.int64, errors='ignore')

# --- Bloque para 'medium' ---
forecast_ob_tc_medium = ForecastProductoV5(nombre_algoritmo=algoritmo_ob_tc_medium, meta=meta_ob_tc_medium, fechas_entrenamiento=fechas_entrenamiento_tc, fechas_proyeccion=fechas_proyeccion, directorio=directorio_salida, holdout_dias=30)
forecast_ob_tc_medium = forecast_ob_tc_medium.proyectar(ob_tc_medium)

# Bloque de ajuste completo para 'medium'
mask = forecast_ob_tc_medium['ob_medium'].isna() # Asegúrate que el nombre de la columna sea correcto
forecast_ob_tc_medium.loc[mask, 'ob_prediccion_medium'] *= _factor_calib_ob(forecast_ob_tc_medium, 'medium')  # V10: x1.4
forecast_ob_tc_medium['ob_prediccion_medium'] = forecast_ob_tc_medium['ob_prediccion_medium'].clip(lower=0).round(0)
forecast_ob_tc_medium.loc[mask, 'ob_real_pred_medium'] = forecast_ob_tc_medium.loc[mask, 'ob_prediccion_medium']
forecast_ob_tc_medium['ob_prediccion_cumsum_medium'] = forecast_ob_tc_medium['ob_prediccion_medium'].cumsum()
cumsum_real = forecast_ob_tc_medium['ob_real_pred_medium'].cumsum()
forecast_ob_tc_medium.loc[mask, 'ob_real_pred_cumsum2_medium'] = cumsum_real[mask]
forecast_ob_tc_medium['ob_prediccion_medium'] = forecast_ob_tc_medium['ob_prediccion_medium'].astype(np.int64, errors='ignore')
forecast_ob_tc_medium['ob_real_pred_medium'] = forecast_ob_tc_medium['ob_real_pred_medium'].astype(np.int64, errors='ignore')
forecast_ob_tc_medium['ob_prediccion_cumsum_medium'] = forecast_ob_tc_medium['ob_prediccion_cumsum_medium'].astype(np.int64, errors='ignore')

# --- Bloque para 'small' ---
forecast_ob_tc_small = ForecastProductoV5(nombre_algoritmo=algoritmo_ob_tc_small, meta=meta_ob_tc_small, fechas_entrenamiento=fechas_entrenamiento_tc, fechas_proyeccion=fechas_proyeccion, directorio=directorio_salida, holdout_dias=30)
forecast_ob_tc_small = forecast_ob_tc_small.proyectar(ob_tc_small)

# Bloque de ajuste completo para 'small'
mask = forecast_ob_tc_small['ob_small'].isna()
forecast_ob_tc_small.loc[mask, 'ob_prediccion_small'] *= _factor_calib_ob(forecast_ob_tc_small, 'small')  # V10: x1.7
forecast_ob_tc_small['ob_prediccion_small'] = forecast_ob_tc_small['ob_prediccion_small'].clip(lower=0).round(0)
forecast_ob_tc_small.loc[mask, 'ob_real_pred_small'] = forecast_ob_tc_small.loc[mask, 'ob_prediccion_small']
forecast_ob_tc_small['ob_prediccion_cumsum_small'] = forecast_ob_tc_small['ob_prediccion_small'].cumsum()
cumsum_real = forecast_ob_tc_small['ob_real_pred_small'].cumsum()
forecast_ob_tc_small.loc[mask, 'ob_real_pred_cumsum2_small'] = cumsum_real[mask]
forecast_ob_tc_small['ob_prediccion_small'] = forecast_ob_tc_small['ob_prediccion_small'].astype(np.int64, errors='ignore')
forecast_ob_tc_small['ob_real_pred_small'] = forecast_ob_tc_small['ob_real_pred_small'].astype(np.int64, errors='ignore')
forecast_ob_tc_small['ob_prediccion_cumsum_small'] = forecast_ob_tc_small['ob_prediccion_cumsum_small'].astype(np.int64, errors='ignore')

# --- Bloque para 'xs' ---
forecast_ob_tc_xs = ForecastProductoV5(nombre_algoritmo=algoritmo_ob_tc_xs, meta=meta_ob_tc_xs, fechas_entrenamiento=fechas_entrenamiento_tc_xs, fechas_proyeccion=fechas_proyeccion, directorio=directorio_salida, holdout_dias=30)
forecast_ob_tc_xs = forecast_ob_tc_xs.proyectar(ob_tc_xs)

# Bloque de ajuste completo para 'xs'
mask = forecast_ob_tc_xs['ob_xs'].isna() # Asegúrate que el nombre de la columna sea correcto
forecast_ob_tc_xs.loc[mask, 'ob_prediccion_xs'] *= _factor_calib_ob(forecast_ob_tc_xs, 'xs')
forecast_ob_tc_xs['ob_prediccion_xs'] = forecast_ob_tc_xs['ob_prediccion_xs'].clip(lower=0).round(0)
forecast_ob_tc_xs.loc[mask, 'ob_real_pred_xs'] = forecast_ob_tc_xs.loc[mask, 'ob_prediccion_xs']
forecast_ob_tc_xs['ob_prediccion_cumsum_xs'] = forecast_ob_tc_xs['ob_prediccion_xs'].cumsum()
cumsum_real = forecast_ob_tc_xs['ob_real_pred_xs'].cumsum()
forecast_ob_tc_xs.loc[mask, 'ob_real_pred_cumsum2_xs'] = cumsum_real[mask]
forecast_ob_tc_xs['ob_prediccion_xs'] = forecast_ob_tc_xs['ob_prediccion_xs'].astype(np.int64, errors='ignore')
forecast_ob_tc_xs['ob_real_pred_xs'] = forecast_ob_tc_xs['ob_real_pred_xs'].astype(np.int64, errors='ignore')
forecast_ob_tc_xs['ob_prediccion_cumsum_xs'] = forecast_ob_tc_xs['ob_prediccion_cumsum_xs'].astype(np.int64, errors='ignore')

# --- Fusión de todos los resultados ---
ob_tc_merged = pd.merge(forecast_ob_tc_large, forecast_ob_tc_medium, on='fecha', how='outer')
ob_tc_merged = pd.merge(ob_tc_merged, forecast_ob_tc_small, on='fecha', how='outer')
ob_tc_merged = pd.merge(ob_tc_merged, forecast_ob_tc_xs, on='fecha', how='outer')

In [ ]:
project = 'tenpo-bi-prod'
pandas_gbq.to_gbq(
    ob_tc_merged, 'kpitos.forecasting_ob_tc', project_id=project, if_exists='replace',
)

In [ ]:
forecast_ob_tenpo = ForecastProductoV5(nombre_algoritmo=algoritmo_ob_tenpo,meta=meta_ob_tenpo,fechas_entrenamiento=fechas_entrenamiento,fechas_proyeccion=fechas_proyeccion,directorio=directorio_salida, holdout_dias=30)
forecast_ob_tenpo = forecast_ob_tenpo.proyectar(ob_tenpo)

In [ ]:
project = 'tenpo-bi-prod'
pandas_gbq.to_gbq(
    forecast_ob_tenpo, 'kpitos.ob_tenpo', project_id=project, if_exists='replace',
)

In [ ]:
import datetime

# =============================================================================
# EXTRACCIÓN DATA HISTÓRICA RGU DIARIO (ENTRENAMIENTO INCREMENTAL)
# =============================================================================
print("Extrayendo historial de RGU...")

QUERY_RGU = """
WITH saldo AS (SELECT DISTINCT DATE_SUB(fecha, INTERVAL 1 DAY) AS fecha, user FROM `tenpo-bi-prod.balance.daily_balance` WHERE saldo_dia > 0),
economics AS (SELECT DISTINCT user, DATE(fecha) AS fecha FROM `tenpo-bi-prod.economics.economics` WHERE linea IN (SELECT * FROM `tenpo-bi-prod.kpitos.lineas_mau_app`) AND LOWER(IFNULL(nombre,"null")) NOT LIKE "%devolu%" AND LOWER(IFNULL(actividad_nac_cd,"null")) NOT LIKE "%refund%"),
revenue_fin AS (SELECT DISTINCT DATE(fecha) AS fecha, user FROM `tenpo-bi-prod.kpitos.revenue_tc_detalle`),
users_tc AS (SELECT CAST(individual_id AS STRING) AS id_persona, user_id FROM `tenpo-bi-prod.kpitos.tipo_users_tc` WHERE name_segment = "XS"),
base_saldos_tca AS (SELECT DISTINCT DATE(Fecha_Proceso) AS fecha, user_id AS user FROM `operacionestc-operaciones-prod.MAE.Cuentas` INNER JOIN users_tc USING(id_persona) WHERE Status_id NOT IN ('200','6','14') AND Cupo_Total <> '0' AND id_cuenta NOT IN ('1','2','1733','1734','1735','1736','1737')),
universo_rgu AS (
  SELECT fecha, user FROM saldo UNION DISTINCT SELECT fecha, user FROM economics UNION DISTINCT SELECT fecha, user FROM revenue_fin UNION DISTINCT SELECT fecha, user FROM base_saldos_tca
),
-- MAGIA: Identificamos el PRIMER DÍA del mes en que cada usuario se vuelve RGU
primera_actividad_mes AS (
  SELECT user, DATE_TRUNC(fecha, MONTH) AS mes, MIN(fecha) AS fecha_activacion
  FROM universo_rgu WHERE fecha < CURRENT_DATE("America/Santiago") GROUP BY 1, 2
)
-- Extraemos los RGUs "Nuevos" por día para que el modelo los acumule
SELECT
  fecha_activacion AS fecha,
  FORMAT_DATE('%a', fecha_activacion) as dia_semana,
  EXTRACT(DAY FROM fecha_activacion) as dia,
  EXTRACT(MONTH FROM fecha_activacion) as mes,
  EXTRACT(YEAR FROM fecha_activacion) as year,
  1 AS real,
  COUNT(DISTINCT user) AS rgu
FROM primera_actividad_mes
WHERE fecha_activacion >= '2023-01-01'
GROUP BY 1, 2, 3, 4, 5
ORDER BY fecha DESC
"""

rgu_historico = pd.read_gbq(QUERY_RGU, use_bqstorage_api=True, project_id='tenpo-bi-prod', credentials=credentials, dialect='standard')
rgu_historico['fecha'] = pd.to_datetime(rgu_historico['fecha'], format="%Y-%m-%d")
print("✅ Historial de RGU descargado.")

# Leemos el acumulado real del mes en curso
df_mtd = pd.read_gbq(QUERY_MTD, project_id='tenpo-bi-prod', use_bqstorage_api=True, credentials=credentials, dialect='standard')
rgu_acumulado_hoy = df_mtd['rgu_acumulado'].values[0] if len(df_mtd) else 0

# 2. CÁLCULO DEL LE (V11: real hasta ayer + modelo desde hoy; ver _le_rgu en la celda RGU)
cierre_estimado = _le_rgu(forecast_rgu, rgu_acumulado_hoy)

# Asignamos la proyección al dataframe diario (para que la curva tenga contra qué compararse)
forecast_rgu['rgu_cierre_mensual_estimado'] = cierre_estimado

# 3. LIMPIEZA DE FLOTANTES
for col in forecast_rgu.columns:
    if pd.api.types.is_float_dtype(forecast_rgu[col]):
        forecast_rgu[col] = forecast_rgu[col].round(0)

# 4. SUBIDA A BIGQUERY
project = 'tenpo-bi-prod'
pandas_gbq.to_gbq(
    forecast_rgu,
    'kpitos.forecasting_rgu',
    project_id=project,
    if_exists='replace',
)
print(f"✅ ¡RGU Subido! Acumulado real: {rgu_acumulado_hoy:,.0f} | Cierre proyectado: {cierre_estimado:,.0f}")

In [ ]:
import pandas as pd
import numpy as np
import pandas_gbq
from datetime import date

# A. Total mensual del LE
QUERY_LE = """SELECT * FROM `tenpo-bi-prod.kpitos.le_revenue_financiero`"""
le_rev_financiero = pd.read_gbq(QUERY_LE, use_bqstorage_api=True, project_id='tenpo-bi-prod', credentials=credentials, dialect='standard')
total = le_rev_financiero.iloc[0, 0]

# Override manual del LE (evoluciona a diario; actualizar cuando cambie)
# Comentar la línea para volver a usar el valor de la tabla le_revenue_financiero.
# V11: el override queda en CONFIG y solo aplica en el mes indicado (antes 2552
# quedaba pegado y se arrastraba al mes siguiente).
if REV_FIN_OVERRIDE and REV_FIN_OVERRIDE.get("mes") == hoy_chile().date().strftime("%Y-%m"):
    total = REV_FIN_OVERRIDE["valor"]
    print(f"[V11 REV FIN] override manual {total} ({REV_FIN_OVERRIDE['mes']})")
else:
    print(f"[V11 REV FIN] sin override vigente: uso le_revenue_financiero = {total}")

# B. Historia real TC Financiera del mes en curso
QUERY_TC = """SELECT
    fecha,
    SUM(valor) AS revenue_tc_financiero,
    SUM(SUM(valor)) OVER (ORDER BY fecha) AS revenue_acumulado
FROM `tenpo-bi-prod.kpitos.revenue_tc_detalle`
WHERE tipo_revenue = "revenue_financiero"
    AND DATE_TRUNC(fecha, MONTH) = DATE_TRUNC(DATE_SUB(CURRENT_DATE("America/Santiago"),INTERVAL 1 DAY), MONTH)
GROUP BY 1
ORDER BY 1"""
tc_financiero = pd.read_gbq(QUERY_TC, use_bqstorage_api=True, project_id='tenpo-bi-prod', credentials=credentials, dialect='standard')

# C. Calendario del mes
hoy = pd.Timestamp(hoy_chile().date())
inicio_mes = hoy.replace(day=1)
fin_mes = inicio_mes + pd.offsets.MonthEnd(0)
calendario = pd.DataFrame({'fecha': pd.date_range(start=inicio_mes, end=fin_mes)})

# ---------------------------------------------------------
# D. CURVA INTRA-MES (picos día 5 y día 20)
# ---------------------------------------------------------
# Curva basada en patrón observado: picos de liquidación en día 5 (~12.4%) y día 20 (~49.3%).
# Si esos días caen sáb/dom, el pico se corre al siguiente día hábil.
porcentajes_base = [
    0.0103, 0.0094, 0.0107, 0.0107, 0.1244,  # Días 1-5   (pico día 5)
    0.0131, 0.0148, 0.0152, 0.0107, 0.0111,  # Días 6-10
    0.0115, 0.0123, 0.0123, 0.0127, 0.0131,  # Días 11-15
    0.0131, 0.0123, 0.0127, 0.0127, 0.4932,  # Días 16-20 (pico día 20)
    0.0185, 0.0168, 0.0177, 0.0099, 0.0177,  # Días 21-25
    0.0086, 0.0148, 0.0156, 0.0140, 0.0131,  # Días 26-30
    0.0168                                    # Día 31
]

# Renormalizar para asegurar que sumen 1.0
_s = sum(porcentajes_base)
porcentajes_base = [p / _s for p in porcentajes_base]

def ajustar_picos_fin_de_semana(pct, inicio_mes, dias_pico=(5, 20), pct_normal=0.013):
    """Si un día pico cae en fin de semana, transfiere el exceso al siguiente día hábil."""
    pct = list(pct)
    n = len(pct)
    for dp in dias_pico:
        if dp > n:
            continue
        wd = inicio_mes.replace(day=dp).weekday()  # 0=lun ... 5=sáb, 6=dom
        if wd < 5:
            continue  # día hábil, no se corre
        offset = 2 if wd == 5 else 1  # sáb→+2 (lun), dom→+1 (lun)
        dest = dp + offset
        if dest > n:
            continue  # se sale del mes
        exceso = pct[dp - 1] - pct_normal
        pct[dest - 1] += exceso
        pct[dp - 1] = pct_normal
    return pct

porcentajes_fijos = ajustar_picos_fin_de_semana(porcentajes_base, inicio_mes)

# ---------------------------------------------------------
# E. MERGE CALENDARIO + REAL
# ---------------------------------------------------------
tc_financiero['fecha'] = pd.to_datetime(tc_financiero['fecha'])
tc_financiero = calendario.merge(tc_financiero, on='fecha', how='left')

# ---------------------------------------------------------
# F. FORECAST DIARIO Y ACUMULADO
# ---------------------------------------------------------
tc_financiero['dia_mes'] = tc_financiero['fecha'].dt.day
tc_financiero['porcentaje_diario'] = tc_financiero['dia_mes'].apply(
    lambda d: porcentajes_fijos[d-1] if d <= len(porcentajes_fijos) else 0
)

# Multiplicamos por 1 millón para volver a pesos
tc_financiero['revenue_diario_forecast'] = tc_financiero['porcentaje_diario'] * total * 1000000

# Rellenamos nulos del pasado real
tc_financiero['revenue_tc_financiero'] = tc_financiero['revenue_tc_financiero'].fillna(0)

# Regla: pasado = real, hoy en adelante = forecast
tc_financiero['revenue_diario_final'] = np.where(
    tc_financiero['fecha'] < hoy,
    tc_financiero['revenue_tc_financiero'],
    tc_financiero['revenue_diario_forecast']
)

tc_financiero['revenue_acumulado'] = tc_financiero['revenue_diario_final'].cumsum()

# ---------------------------------------------------------
# G. REVISIÓN Y SUBIDA A BIGQUERY
# ---------------------------------------------------------
print("--- DATOS QUE SE VAN A SUBIR A BQ (ÚLTIMOS 10 DÍAS) ---")
print(tc_financiero[['fecha', 'revenue_tc_financiero', 'revenue_diario_forecast', 'revenue_diario_final', 'revenue_acumulado']].tail(10))
print("------------------------------------------------------")

final_upload = tc_financiero[['fecha', 'revenue_tc_financiero', 'revenue_acumulado']].copy()

project = 'tenpo-bi-prod'
pandas_gbq.to_gbq(
    final_upload, 'kpitos.forecasting_revenue_financiero', project_id=project, if_exists='replace',
)

In [ ]:
import os
api_key_cmf = os.environ.get("CMF_API_KEY")
if not api_key_cmf:
    try:
        from google.colab import userdata
        api_key_cmf = userdata.get("CMF_API_KEY")
    except Exception:
        api_key_cmf = None
if not api_key_cmf:
    raise RuntimeError("Falta CMF_API_KEY (variable de entorno o Secretos de Colab). La key ya NO va en el notebook.")

_anios = range(2025, hoy_chile().year + 1) if V12_FIXES.get("tmc_anios", True) else [2025]
_tmcs = []
for _y in _anios:
    try:
        _r = requests.get(f'https://api.cmfchile.cl/api-sbifv3/recursos_api/tmc/{_y}?apikey={api_key_cmf}&formato=json', timeout=60)
        _r.raise_for_status()
        _tmcs += _r.json()['TMCs']
    except Exception as _e:
        if _y == _anios[0]:
            raise
        print(f"[V12 TMC] !! no se pudo leer {_y}: {type(_e).__name__}: {str(_e)[:80].replace(api_key_cmf, '***')}")
response = pd.json_normalize(_tmcs)
response = response.drop_duplicates(subset=['Fecha', 'Hasta', 'SubTitulo', 'Tipo'] if 'Tipo' in response.columns else ['Fecha', 'Hasta', 'SubTitulo'])
response['Valor'] = response['Valor'].astype(np.float64)
response['Valor'] = response['Valor']/100
response['Fecha'] = pd.to_datetime(response['Fecha'], format='%Y-%m-%d')
response['Hasta'] = pd.to_datetime(response['Hasta'], format='%Y-%m-%d')
response = response[response['Fecha'] >= '2025-01-01']
response = response[response['SubTitulo'].isin(['Inferiores o iguales al equivalente de 50 unidades de fomento','Inferiores o iguales al equivalente de 200 unidades de fomento y superiores al equivalente de 50 unidades de fomento','Inferiores o iguales al equivalente de 5.000 unidades de fomento y superiores al equivalente de 200 unidades de fomento'])]
map_subtitulos = {
    'Inferiores o iguales al equivalente de 50 unidades de fomento': '≤ 50 UF',
    'Inferiores o iguales al equivalente de 200 unidades de fomento y superiores al equivalente de 50 unidades de fomento': '> 50 y <=200 UF',
    'Inferiores o iguales al equivalente de 5.000 unidades de fomento y superiores al equivalente de 200 unidades de fomento': '>200 y <=5000 UF'
}
response['SubTitulo'] = response['SubTitulo'].replace(map_subtitulos)
response = response.sort_values(by=['Fecha','Valor'])
response.rename(columns={'Valor':'tmc_anual'},inplace=True)

response['tmc_anual_mensual'] = response['tmc_anual']/12
response['tmc_anual_mensual'] = np.trunc(response['tmc_anual_mensual'] * 10000) / 10000
response['tasa_pizarra_avances'] = response['tmc_anual_mensual'] - 0.0008

project = 'tenpo-bi-prod'
pandas_gbq.to_gbq(
    response, 'kpitos.tmc', project_id=project, if_exists='replace',
)

In [ ]:
response

In [ ]:
QUERY = """SELECT * FROM `tenpo-bi-prod.kpitos.one_page_cuenta_rem`"""
one_page_cta_rem = pd.read_gbq(QUERY,use_bqstorage_api=True, project_id='tenpo-bi-prod', credentials=credentials, dialect='standard')
one_page_cta_rem['fecha'] = pd.to_datetime(one_page_cta_rem['fecha'], format="%Y-%m-%d")

In [ ]:
project = 'tenpo-bi-prod'
pandas_gbq.to_gbq(
    one_page_cta_rem, 'kpitos.one_page_cuenta_remunerada', project_id=project, if_exists='replace',
)

In [ ]:
gpv_merged.filter(regex="trx_ci")

In [ ]:
# --- SCRIPT VALIDADOR DE CÁLCULO ---
import pandas as pd

def validar_transformacion(df, col_name, tarifa, nombre_producto):
    print(f"\n🔎 VALIDANDO: {nombre_producto} (Columna: {col_name})")
    print("-" * 60)

    # 1. Verificar si la columna existe
    if col_name not in df.columns:
        print(f"❌ ERROR: La columna '{col_name}' no existe en el dataframe.")
        return

    # 2. Tomar una muestra (última fila con datos mayores a 0)
    try:
        sample = df[df[col_name] > 0].iloc[-1]
    except IndexError:
        print("⚠️ Advertencia: La columna existe pero solo tiene ceros o nulos.")
        return

    fecha_muestra = sample['fecha']
    valor_en_tabla = sample[col_name]

    # 3. Obtener UF de esa fecha (intenta sacarla de la fila, sino del df_uf global)
    if 'valor_uf' in sample and pd.notnull(sample['valor_uf']):
        uf_usada = sample['valor_uf']
    else:
        # Buscar en la tabla df_uf si no está en la fila
        try:
            uf_usada = df_uf[df_uf['fecha'] == fecha_muestra]['valor_uf'].values[0]
        except:
            uf_usada = 38000 # Valor referencia si todo falla
            print("⚠️ Usando UF referencial ($38.000) para validación")

    # 4. CÁLCULO INVERSO (La prueba de fuego)
    # Si Valor_Tabla = Trx * Tarifa * UF  ==>  Trx = Valor_Tabla / (Tarifa * UF)
    trx_implicitas = valor_en_tabla / (tarifa * uf_usada)

    # 5. REPORTE
    print(f"📅 Fecha Muestra: {fecha_muestra.date()}")
    print(f"💰 Valor en Tabla: {valor_en_tabla:,.0f}")
    print(f"📊 UF Fecha:       ${uf_usada:,.2f}")
    print(f"🏷️ Tarifa Usada:   {tarifa} UF")
    print(f"🔄 Trx Implícitas: {trx_implicitas:,.0f} (Cálculo inverso)")
    print("-" * 60)

    # 6. VEREDICTO AUTOMÁTICO
    # Si el valor en tabla es similar a las trx implicitas, NO se multiplicó (Falso)
    # Si el valor en tabla es millones y las trx son miles, SÍ se multiplicó (Verdadero)

    if valor_en_tabla > (trx_implicitas * 10000):
        print("✅ RESULTADO: CORRECTO. El valor en tabla es DINERO (CLP).")
        print("   (El número es grande, consistente con Revenue/Costo).")
    elif abs(valor_en_tabla - trx_implicitas) < trx_implicitas * 0.1:
        print("❌ RESULTADO: INCORRECTO. El valor en tabla siguen siendo TRANSACCIONES.")
        print("   (El script de transformación no sobreescribió la columna).")
    else:
        print("⚠️ RESULTADO: INDETERMINADO. Revisa los números manualmente.")

# --- EJECUTAR VALIDACIONES ---
# Ajusta los nombres de columnas ('trx_ci_bestado', etc.) si en tu gpv_merged se llaman distinto
print(">>> INICIANDO VALIDACIÓN DE REVENUE Y COSTOS <<<")

validar_transformacion(gpv_merged, 'trx_ci_bestado', 0.01,   'Cash In - Banco Estado')
validar_transformacion(gpv_merged, 'trx_ci_obancos', 0.0028, 'Cash In - Otros Bancos')
# validar_transformacion(gpv_merged, 'trx_co_bestado', 0.01,   'Cash Out - Banco Estado') # Descomentar si existe

In [ ]:
# --- VALIDACIÓN ACUMULADA DEL MES EN CURSO ---
from datetime import datetime

def validar_mes_actual(df, col_name, tarifa, nombre_producto):
    print(f"\n📅 REVISIÓN ACUMULADA MES ACTUAL: {nombre_producto}")
    print("=" * 70)

    # 1. Filtrar Mes Actual (febrero 2026 según tu contexto)
    # Usamos la fecha máxima del dataframe para saber en qué "presente" estamos
    fecha_max = df['fecha'].max()
    mes_actual = fecha_max.month
    anio_actual = fecha_max.year

    df_mes = df[(df['fecha'].dt.month == mes_actual) & (df['fecha'].dt.year == anio_actual)].copy()

    if df_mes.empty:
        print(f"⚠️ No hay datos para el mes {mes_actual}/{anio_actual}.")
        return

    # 2. Asegurar que tenemos UF (si no está, pegarla)
    if 'valor_uf' not in df_mes.columns:
        # Intento de rescate simple si faltara la columna
        df_mes['valor_uf'] = 39688 # Valor ref

    # 3. CÁLCULOS ACUMULADOS
    total_dinero = df_mes[col_name].sum()

    # Reconstruimos las transacciones día por día para ser exactos
    # Trx_dia = Dinero_dia / (Tarifa * UF_dia)
    df_mes['trx_recalculadas'] = df_mes[col_name] / (tarifa * df_mes['valor_uf'])
    total_trx_estimadas = df_mes['trx_recalculadas'].sum()

    # 4. REPORTE
    print(f"Mes Analizado:      {fecha_max.strftime('%B %Y')}")
    print(f"Días con data:      {len(df_mes)} días")
    print("-" * 30)
    print(f"💰 REVENUE/COSTO TOTAL:  ${total_dinero:,.0f}")
    print(f"🔢 TRX TOTALES ESTIMADAS: {total_trx_estimadas:,.0f}")
    print("-" * 30)

    # 5. VALIDACIÓN DE SENTIDO
    if total_dinero > 100000000: # Si es mayor a 100 Millones
        print("✅ EL MONTO TIENE SENTIDO (Escala de Millones).")
    elif total_dinero < 1000000:
        print("⚠️ ALERTA: El monto acumulado es muy bajo. ¿Seguro que está en pesos?")
    else:
        print("ℹ️ El monto parece razonable, pero verifícalo.")

# --- EJECUTAR ---
# Cash In
validar_mes_actual(gpv_merged, 'trx_ci_bestado', 0.01,   'Cash In - Banco Estado')
validar_mes_actual(gpv_merged, 'trx_ci_obancos', 0.0028, 'Cash In - Otros Bancos')

# Cash Out (Si ya lo calculaste)
# validar_mes_actual(gpv_merged, 'trx_co_bestado', 0.01,   'Cash Out - Banco Estado')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def validar_forecast_produccion(df):
    print("=========================================================")
    print("🔍 INICIANDO VALIDACIÓN DE DATA QUALITY (FORECAST)")
    print("=========================================================\n")

    errores = 0
    advertencias = 0

    # 1. Validación de Fechas y Filas (Completitud del mes)
    if not pd.api.types.is_datetime64_any_dtype(df['fecha']):
        df['fecha'] = pd.to_datetime(df['fecha'])

    dias_esperados = df['fecha'].dt.daysinmonth.iloc[0]
    dias_actuales = len(df)

    if dias_actuales != dias_esperados:
        print(f"❌ [ERROR] Filas incorrectas: El DF tiene {dias_actuales} filas, pero el mes tiene {dias_esperados} días.")
        errores += 1
    else:
        print(f"✅ [OK] Días del mes completos ({dias_actuales}/{dias_esperados} filas).")

    # 2. Validación de Estructura ATM
    columnas_clave = [
        'rev_atm_int', 'rev_prediccion_atm_int',
        'rev_atm_nac', 'rev_prediccion_atm_nac'
    ]
    faltantes = [col for col in columnas_clave if col not in df.columns]

    if faltantes:
        print(f"❌ [ERROR] Faltan columnas críticas en el DF: {faltantes}")
        errores += 1
    else:
        print("✅ [OK] Estructura de columnas ATM integrada correctamente.")

    # 3. Validación de Nulos en Predicciones
    cols_prediccion = [col for col in df.columns if 'prediccion' in col and 'meta' not in col and 'cumsum' not in col]
    nulos_prediccion = df[cols_prediccion].isnull().sum().sum()

    if nulos_prediccion > 0:
        print(f"❌ [ERROR] Existen {nulos_prediccion} valores nulos en las columnas de predicción.")
        errores += 1
    else:
        print(f"✅ [OK] Predicciones completas al 100% (0 nulos evaluados en {len(cols_prediccion)} modelos).")

    # 4. Validación de Escala (Millones)
    if 'gpv_prediccion_tc_fis' in df.columns:
        max_val_tc = df['gpv_prediccion_tc_fis'].max()
        if max_val_tc > 100000:
            print(f"⚠️ [WARNING] Los valores parecen estar en escala bruta (Ej. GPV TC max: {max_val_tc:.0f}). ¿Faltó dividir por 1,000,000?")
            advertencias += 1
        else:
            print("✅ [OK] Escala financiera validada (Valores normalizados en millones).")

    # 5. Validación Lógica Financiera
    if not faltantes:
        sum_int = df['rev_prediccion_atm_int'].sum()
        sum_nac = df['rev_prediccion_atm_nac'].sum()

        if sum_nac > sum_int:
            print(f"⚠️ [WARNING] Revenue Nacional ATM ({sum_nac:.2f}) superó al Internacional ({sum_int:.2f}). Validar comportamiento atípico.")
            advertencias += 1
        else:
            print(f"✅ [OK] Lógica ATM correcta: Revenue Internacional ({sum_int:.1f}M) > Nacional ({sum_nac:.1f}M).")

    # ---- RESULTADO FINAL ----
    print("\n=========================================================")
    print(f"📊 RESUMEN: {errores} Errores | {advertencias} Advertencias")
    print("=========================================================")

    # =========================================================
    # 📈 NUEVO: GENERACIÓN DE GRÁFICO VISUAL (REAL VS PREDICCIÓN)
    # =========================================================
    if not faltantes:
        print("\nGenerando gráfico de validación visual...")
        fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

        # Panel 1: ATM Internacional
        axes[0].plot(df['fecha'], df['rev_prediccion_atm_int'], label='Predicción Modelo (Mes Completo)', linestyle='--', color='darkorange', linewidth=2)
        axes[0].plot(df['fecha'], df['rev_atm_int'], label='Dato Real (Hasta Hoy)', marker='o', color='royalblue', linewidth=2.5, markersize=6)
        axes[0].set_title('Validación Visual: Revenue ATM Internacional', fontsize=14, fontweight='bold')
        axes[0].set_ylabel('Revenue Diario (Millones de $)', fontsize=11)
        axes[0].legend(loc='upper left')
        axes[0].grid(True, linestyle=':', alpha=0.7)

        # Panel 2: ATM Nacional
        axes[1].plot(df['fecha'], df['rev_prediccion_atm_nac'], label='Predicción Modelo (Mes Completo)', linestyle='--', color='darkorange', linewidth=2)
        axes[1].plot(df['fecha'], df['rev_atm_nac'], label='Dato Real (Hasta Hoy)', marker='o', color='forestgreen', linewidth=2.5, markersize=6)
        axes[1].set_title('Validación Visual: Revenue ATM Nacional (Redbanc)', fontsize=14, fontweight='bold')
        axes[1].set_xlabel('Fecha', fontsize=11)
        axes[1].set_ylabel('Revenue Diario (Millones de $)', fontsize=11)
        axes[1].legend(loc='upper left')
        axes[1].grid(True, linestyle=':', alpha=0.7)

        plt.tight_layout()
        plt.show()

    if errores == 0:
        print("🚀 APROBADO: El script está sano y listo para inyectarse a BigQuery.")
        return True
    else:
        print("🛑 RECHAZADO: Revisa los errores antes de subir los datos a producción.")
        return False

# ==========================================
# CÓMO USARLO EN TU NOTEBOOK
# ==========================================
# Ejecutar la validación (esto imprimirá el texto y mostrará el gráfico)
aprobado = validar_forecast_produccion(gpv_merged)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import numpy as np
from calendar import monthrange
import pandas_gbq

# ==========================================
# 1. TARGETS MENSUALES
# ==========================================
targets = {
    pd.Timestamp('2026-03-31'): 135_703,
    pd.Timestamp('2026-04-30'): 152_112,
    pd.Timestamp('2026-05-31'): 161_800,
    pd.Timestamp('2026-06-30'): 172_616,
    pd.Timestamp('2026-07-31'): 186_978,
    pd.Timestamp('2026-08-31'): 201_901,
    pd.Timestamp('2026-09-30'): 212_680,
    pd.Timestamp('2026-10-31'): 223_788,
    pd.Timestamp('2026-11-30'): 232_374,
    pd.Timestamp('2026-12-31'): 232_469,
}

# ==========================================
# 2. QUERY BIGQUERY (misma que antes)
# ==========================================
QUERY_COLOCACION = """
with cuentas as(
  Select
    id_cuenta, id_producto, id_persona,
    safe_cast(id_persona as numeric) as id_persona_num,
    Status_Cuenta,
    safe_cast(Cupo_Maximo as numeric) as cupo_total,
    safe_cast(total_disponible as numeric) as total_disponible,
    safe_cast(credit_balance as numeric) as credit_balance,
    safe_cast(funded_balance as numeric) as funded_balance,
    safe_cast(current_balance as numeric) as current_balance,
    safe_cast(future_installments_amount_with_fee as numeric) as future_installments_amount_with_fee,
    safe_cast(future_installments_amount_no_fee as numeric) as future_installments_amount_no_fee,
    safe_cast(unearned_interest_installment as numeric) as unearned_interest_installment,
    safe_cast(creliq_balance as numeric) as creliq_balance,
    safe_cast(loss_balance as numeric) as loss_balance,
    safe_cast(post_settlement_balance as numeric) as post_settlement_balance,
    safe_cast(revolving_charges_unearned_amount as numeric) as revolving_charges_unearned_amount,
    Fecha_Proceso
  from `operacionestc-operaciones-prod.MAE.Cuentas`
    where Status_id not in ('200','6','14')
    and Cupo_Total <> '0'
    and id_cuenta not in ('1','2','1733','1734','1735','1736','1737')
)
, saldos as(
  select
    fecha_proceso,
    round(credit_balance + current_balance + future_installments_amount_with_fee + future_installments_amount_no_fee + unearned_interest_installment + funded_balance +
    creliq_balance + revolving_charges_unearned_amount + post_settlement_balance,0) as Deuda_diaria_batch,
  from cuentas
)
, sumatoria as(
  select fecha_proceso, sum(Deuda_diaria_batch) as Deuda_diaria_batch
  from saldos
  group by 1
)
Select
  fecha_proceso as fecha,
  EXTRACT(DAY FROM DATE(fecha_proceso)) AS dia,
  EXTRACT(MONTH FROM DATE(fecha_proceso)) AS mes,
  Deuda_diaria_batch/1000000 as colocaciones,
from sumatoria
WHERE DATE(fecha_proceso) >= "2025-09-01"
ORDER BY 1
"""

print("Consultando BigQuery...")
df = pd.read_gbq(QUERY_COLOCACION, project_id='tenpo-bi-prod')

# ==========================================
# 3. PREPARACIÓN BASE
# ==========================================
df['colocaciones'] = df['colocaciones'].astype(float)
df['fecha']        = pd.to_datetime(df['fecha'])
df = df.sort_values('fecha').reset_index(drop=True)

metric     = 'colocaciones'
fecha_max  = df['fecha'].max()
valor_seed = float(df[metric].iloc[-1])

# Patrón histórico — últimos 3 meses
fecha_corte = fecha_max - pd.DateOffset(months=3)
df_hist     = df[df['fecha'] >= fecha_corte].copy().sort_values('fecha').reset_index(drop=True)
df_hist['tasa_diaria']  = df_hist[metric].pct_change() + 1
df_hist['dia_del_mes']  = df_hist['fecha'].dt.day
patron_media = df_hist.groupby('dia_del_mes')['tasa_diaria'].mean().fillna(1.0)
patron_std   = df_hist.groupby('dia_del_mes')['tasa_diaria'].std().fillna(0.0)

# Rango de fechas: desde mañana hasta fin del último mes target
fecha_inicio = fecha_max + pd.Timedelta(days=1)
fecha_fin    = max(targets.keys())
if fecha_fin < fecha_inicio:
    raise RuntimeError(f"[V12] Los targets mensuales hardcodeados terminan en {fecha_fin.date()} y el ultimo real es "
                       f"{fecha_max.date()}: cargar los targets del periodo siguiente (celda 'TARGETS MENSUALES').")
fechas_proy  = pd.date_range(start=fecha_inicio, end=fecha_fin, freq='D')

# ==========================================
# 4. PROYECCIÓN NATURAL + IC
# ==========================================
nat_vals, ic_sup_vals, ic_inf_vals, tasa_vals = [], [], [], []
v_nat = v_sup = v_inf = valor_seed

for f in fechas_proy:
    dia = f.day
    mu  = patron_media.get(dia, 1.0)
    sd  = patron_std.get(dia, 0.0)
    v_nat *= mu
    v_sup *= (mu + sd)
    v_inf *= max(mu - sd, 0.0001)
    nat_vals.append(v_nat)
    ic_sup_vals.append(v_sup)
    ic_inf_vals.append(v_inf)
    tasa_vals.append(mu)

df_nat = pd.DataFrame({
    'fecha'           : fechas_proy,
    'valor_natural'   : np.round(nat_vals,   2),
    'ic_superior'     : np.round(ic_sup_vals, 2),
    'ic_inferior'     : np.round(ic_inf_vals, 2),
    'tasa_crecimiento': np.round(tasa_vals,   6),
})

# ==========================================
# 5. PROYECCIÓN CON FACTOR MULTIPLICATIVO
#    → llega exacto al target de cada fin de mes
# ==========================================
rows_target = []
v_prev      = valor_seed   # valor de arranque (se actualiza mes a mes)

# Ordenamos los targets cronológicamente
targets_sorted = dict(sorted(targets.items()))

for fecha_target, target_val in targets_sorted.items():
    anio = fecha_target.year
    mes  = fecha_target.month
    dias_mes = monthrange(anio, mes)[1]

    # Días del mes que caen dentro de nuestro rango de proyección
    fechas_mes = pd.date_range(
        start=pd.Timestamp(anio, mes, 1),
        end=fecha_target,
        freq='D'
    )
    # Solo proyectamos días futuros al último real
    fechas_mes = fechas_mes[fechas_mes > fecha_max]
    if len(fechas_mes) == 0:
        continue

    # 1) Simulamos el cierre natural del mes partiendo de v_prev
    v_tmp = v_prev
    for f in fechas_mes:
        v_tmp *= patron_media.get(f.day, 1.0)
    val_natural_cierre = v_tmp

    # 2) Factor de ajuste para que el último día == target
    n_dias = len(fechas_mes)
    factor_ajuste = (target_val / val_natural_cierre) ** (1 / n_dias)

    # 3) Generamos la curva ajustada día a día
    v_cur = v_prev
    for f in fechas_mes:
        mu    = patron_media.get(f.day, 1.0)
        v_cur *= mu * factor_ajuste
        rows_target.append({
            'fecha'           : f,
            'valor_target'    : round(v_cur, 2),
            'target_mes'      : target_val,
            'factor_ajuste'   : round(factor_ajuste, 6),
        })

    # El cierre de este mes es la semilla del siguiente
    v_prev = v_cur

df_target = pd.DataFrame(rows_target)

# ==========================================
# 6. DATAFRAME CONSOLIDADO
# ==========================================
df_proj = (
    df_nat
    .merge(df_target[['fecha','valor_target','target_mes','factor_ajuste']],
           on='fecha', how='left')
    .rename(columns={
        'valor_natural': 'valor_target',
        'valor_target':  'valor_target_rf',
    })
)

print(df_proj.head(15).to_string(index=False))

# ==========================================
# 7. GRÁFICO
# ==========================================
sns.set_theme(style='whitegrid')
fmt_miles = ticker.FuncFormatter(lambda x, p: f'{int(x):,}'.replace(',', '.'))

fig, ax = plt.subplots(figsize=(18, 8))

# --- Histórico real ---
df_real_plot = df[df['fecha'] >= fecha_corte]
ax.plot(df_real_plot['fecha'], df_real_plot[metric],
        color='steelblue', linewidth=2.2, marker='o', markersize=3.5,
        label='Real (últ. 3 meses)', zorder=4)

# Línea vertical de corte
ax.axvline(x=fecha_max, color='gray', linestyle='--', linewidth=1, alpha=0.6)
ax.text(fecha_max + pd.Timedelta(days=1), ax.get_ylim()[0],
        'Último\nreal', color='gray', fontsize=7.5, va='bottom')

# --- IC ±1σ (proyección natural) ---
ax.fill_between(df_proj['fecha'], df_proj['ic_inferior'], df_proj['ic_superior'],
                color='steelblue', alpha=0.10, label='IC ±1σ (natural)', zorder=1)

# --- Proyección natural ---
ax.plot(df_proj['fecha'], df_proj['valor_target'],
        color='steelblue', linewidth=1.8, linestyle=':', alpha=0.7,
        label='Proyección natural', zorder=3)

# --- Proyección con factor (ruta crítica a targets) ---
ax.plot(df_proj['fecha'], df_proj['valor_target_rf'],
        color='coral', linewidth=2.5, linestyle='--',
        label='Ruta crítica (targets)', zorder=5)

# --- Puntos y anotaciones en cada fin de mes ---
for fecha_t, grp in df_target.groupby('target_mes'):
    ultimo = grp.iloc[-1]
    f, v = ultimo['fecha'], ultimo['valor_target']
    ax.scatter(f, v, color='darkred', s=60, zorder=6)
    ax.annotate(
        f"{int(ultimo['target_mes']):,}".replace(',', '.'),
        xy=(f, v),
        xytext=(0, 14), textcoords='offset points',
        ha='center', fontsize=7.5, color='darkred', fontweight='bold',
        arrowprops=dict(arrowstyle='->', color='darkred', lw=0.7)
    )

ax.set_title(
    'Colocaciones — Proyección natural vs. Ruta crítica a targets mensuales\n'
    '(patrón últimos 3 meses | factor multiplicativo diario por mes)',
    fontsize=14, fontweight='bold', pad=15
)
ax.set_xlabel('Fecha')
ax.set_ylabel('Colocaciones (MM)')
ax.yaxis.set_major_formatter(fmt_miles)
ax.xaxis.set_major_locator(plt.MaxNLocator(14))
plt.xticks(rotation=35)
ax.legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig('proyeccion_targets_2026.png', dpi=150)
plt.show()

# ==========================================
# 8. EXPORT EXCEL — 3 hojas
# ==========================================
df_real_export = df[df['fecha'] >= fecha_corte][['fecha', metric]].copy()
df_real_export.columns = ['fecha', 'colocaciones_real']

# with pd.ExcelWriter('proyeccion_targets_2026.xlsx', engine='openpyxl') as writer:
#     df_proj.to_excel(writer,           sheet_name='Proyeccion_consolidada', index=False)
#     df_real_export.to_excel(writer,    sheet_name='Historico_3m',           index=False)
#     pd.DataFrame(targets_sorted.items(),
#                  columns=['fecha_target','target_colocaciones']
#                 ).to_excel(writer,     sheet_name='Targets_mensuales',      index=False)

df_real_export = df_real_export.rename(columns={'colocaciones_real': 'valor_target'})
df_real_export['estado_total'] = 'R'
df_proj['estado_total'] = 'F'

df_union = pd.concat([df_real_export, df_proj], ignore_index=True).sort_values('fecha')

project = 'tenpo-bi-prod'
pandas_gbq.to_gbq(
    df_union, 'kpitos.colocacion_90', project_id=project, if_exists='replace',
)

# ==========================================
# MES EN CURSO — Real + Forecast
# día 1 del mes → muestra mes anterior (t-1)
# día 2 en adelante → muestra mes actual
# ==========================================
hoy     = hoy_chile()
mes_ref = (hoy - pd.DateOffset(months=1)) if hoy.day == 1 else hoy
mes_ref = mes_ref.replace(day=1)

le_mes = (
    df_union[
        (df_union['fecha'].dt.year  == mes_ref.year) &
        (df_union['fecha'].dt.month == mes_ref.month)
    ][['fecha', 'estado_total', 'valor_target', 'valor_target_rf']]
    .copy()
    .reset_index(drop=True)
)

print(f"Mes en curso: {mes_ref.strftime('%B %Y')} "
      f"| Reales: {(le_mes['estado_total']=='R').sum()} días "
      f"| Forecast: {(le_mes['estado_total']=='F').sum()} días")

project = 'tenpo-bi-prod'
pandas_gbq.to_gbq(
    le_mes, 'kpitos.le_colocacion', project_id=project, if_exists='replace',
)

## Calibración: comparativa Real vs Forecast (días transcurridos del mes)

Mide el **gap entre lo que el modelo proyectó y lo que realmente pasó**, usando solo
los días del mes que ya tienen dato real. Sirve para mostrar que el empalme
(el "salto" real → proyección) está bien calibrado: si la predicción del modelo
sigue de cerca al real en los días recientes, la transición a la proyección es suave.

- **MAPE diario**: error porcentual promedio día a día.
- **Sesgo**: + sobre-estima, − sub-estima.
- **Gap acumulado (MTD)**: diferencia entre el acumulado proyectado y el real del mes en curso.
- **Gap último día real**: qué tan cerca quedó el modelo justo antes del salto a proyección.


In [ ]:
# =====================================================================
# CALIBRACION: COMPARATIVA REAL vs FORECAST (dias ya transcurridos del mes)
# =====================================================================
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def cols_kpi(df, contiene=""):
    """Ayuda para descubrir nombres de columnas: cols_kpi(forecast_mau_app, 'mau')."""
    return [c for c in df.columns if contiene in c]

def comparar_real_vs_forecast(df, col_real, col_pred, label, escala=1.0, plot=True):
    """
    Compara, en los dias con dato real, la prediccion del modelo vs el real.
      - col_real : columna con el dato real (NaN en dias futuros).
      - col_pred : columna con la prediccion del modelo (sin la meta, sin cumsum).
      - escala   : divisor para mostrar (ej. 1e6 -> millones). El gap % no cambia.
    Devuelve un dict con metricas de calibracion (o None si no hay datos).
    """
    faltan = [c for c in [col_real, col_pred] if c not in df.columns]
    if faltan:
        print(f"[{label}] No existen columnas {faltan}. Disponibles parecidas:",
              cols_kpi(df, col_real.split('_')[0]))
        return None

    d = df[["fecha", col_real, col_pred]].copy()
    d = d[d[col_real].notna() & d[col_pred].notna()].sort_values("fecha")
    if d.empty:
        print(f"[{label}] Sin dias con real+pred para comparar.")
        return None

    real = d[col_real].astype(float) / escala
    pred = d[col_pred].astype(float) / escala
    err  = pred - real

    ape       = (err.abs() / real.replace(0, np.nan)).dropna()
    mape      = ape.mean() * 100 if len(ape) else np.nan
    sesgo     = (err.sum() / real.sum() * 100) if real.sum() else np.nan
    real_acum = real.sum(); pred_acum = pred.sum()
    gap_acum  = (pred_acum - real_acum) / real_acum * 100 if real_acum else np.nan
    gap_ult   = (pred.iloc[-1] - real.iloc[-1]) / real.iloc[-1] * 100 if real.iloc[-1] else np.nan

    g = abs(gap_acum)
    verd = "OK Bien calibrado" if g <= 3 else ("Aceptable" if g <= 7 else "Revisar")

    print(f"[{label}] dias={len(d)} | MAPE diario={mape:4.1f}% | sesgo={sesgo:+5.1f}% | "
          f"real MTD={real_acum:,.0f} | pred MTD={pred_acum:,.0f} | "
          f"gap acum={gap_acum:+5.1f}% | gap ult.dia real={gap_ult:+5.1f}% -> {verd}")

    if plot:
        fig, ax = plt.subplots(1, 2, figsize=(15, 4.2))
        ax[0].plot(d["fecha"], real, marker="o", lw=2.4, color="#0E9E88", label="Real")
        ax[0].plot(d["fecha"], pred, marker="o", lw=2.0, ls="--", color="#F5B700", label="Forecast modelo")
        ax[0].set_title(f"{label} -- Real vs Forecast (diario)", fontweight="bold")
        ax[0].legend(); ax[0].grid(alpha=.3)
        ax[1].plot(d["fecha"], real.cumsum(), lw=2.6, color="#0E9E88", label="Real acumulado")
        ax[1].plot(d["fecha"], pred.cumsum(), lw=2.2, ls="--", color="#F5B700", label="Forecast acumulado")
        ax[1].set_title(f"{label} -- Acumulado MTD (gap {gap_acum:+.1f}%)", fontweight="bold")
        ax[1].legend(); ax[1].grid(alpha=.3)
        for a in ax: a.tick_params(axis="x", rotation=35)
        plt.tight_layout(); plt.show()

    return {"kpi": label, "dias": len(d), "mape_diario_%": round(mape, 1),
            "sesgo_%": round(sesgo, 1), "real_mtd": round(real_acum, 0),
            "pred_mtd": round(pred_acum, 0), "gap_acum_%": round(gap_acum, 1),
            "gap_ult_dia_%": round(gap_ult, 1), "veredicto": verd}

def resumen_calibracion(comparaciones, subir_bq=False):
    """Tabla resumen de calibracion. subir_bq=True la versiona en kpitos.calibracion_forecast."""
    filas = [c for c in comparaciones if c]
    if not filas:
        print("Sin comparaciones validas."); return pd.DataFrame()
    res = pd.DataFrame(filas)
    print("\n=============== RESUMEN DE CALIBRACION (MES EN CURSO) ===============")
    print(res.to_string(index=False))
    if subir_bq:
        res2 = res.copy(); res2["fecha_corrida"] = hoy_chile()
        pandas_gbq.to_gbq(res2, "kpitos.calibracion_forecast",
                          project_id="tenpo-bi-prod", if_exists="append")
        print("\nResumen versionado en kpitos.calibracion_forecast")
    return res

# ---------------------------------------------------------------------
# EJEMPLOS  (verifica los nombres de columna con cols_kpi(df, 'mau') etc.)
# ---------------------------------------------------------------------
comparaciones = []

# MAU App  -> real='mau_app', pred='mau_prediccion_app'
comparaciones.append(
    comparar_real_vs_forecast(forecast_mau_app, "mau_app", "mau_prediccion_app", "MAU App"))

# GPV App  -> OJO: en la celda de GPV se le quita el sufijo '_app' a las columnas.
#   Si ya corriste esa celda, las columnas son 'gpv' y 'gpv_prediccion' (en pesos -> escala=1e6).
comparaciones.append(
    comparar_real_vs_forecast(forecast_gpv_app, "gpv", "gpv_prediccion", "GPV App", escala=1e6))

# GPV TC Fisico (desde gpv_merged, que ya esta en millones -> escala=1)
comparaciones.append(
    comparar_real_vs_forecast(gpv_merged, "gpv_tc_fis", "gpv_prediccion_tc_fis", "GPV TC Fisico"))

# Resumen consolidado (pon subir_bq=True para versionarlo en BigQuery)
resumen_calib = resumen_calibracion(comparaciones, subir_bq=False)


## Backtesting rolling-origin (paso 1 del replanteo de entrenamiento)

Evalua el metodo actual sobre los ultimos 6 meses:
- Para cada fecha de corte, refittea XGB con la ventana hasta ese corte usando los hiperparametros guardados en el `.pkl` actual (no re-busca grilla).
- Predice los siguientes 30 dias y compara contra el real.
- Devuelve MAPE/sesgo por corte y por dia de horizonte, mas el grafico de curvas.

Sirve para detectar si el problema es **estabilidad entre cortes** (apunta a ventana de training) o **degradacion con el horizonte** (apunta a frecuencia de reentrenamiento).

Requisitos: `ForecastProducto`, `mau_app`, `algoritmo_mau_app`, `meta_mau_app`, `fechas_entrenamiento`, `fechas_proyeccion`, `directorio_salida` ya en memoria, y el `.pkl` del modelo (`xgb_mau_app.pkl`) generado por una corrida previa de `_entrenar_modelo`.

In [ ]:
# =============================================================================
# Paso 1 - Backtesting rolling-origin
# Evalúa el método de entrenamiento sobre los últimos 6 meses.
# NOTA: _refit_y_predecir está definida en v4-code-clase (versión polimórfica
# con filtro real==1). Acá solo definimos backtest_rolling y el bloque de uso.
# =============================================================================
import pickle
import numpy as np
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt


def backtest_rolling(template, kpi_df, target_col, fecha_train_inicio,
                     cortes, horizonte_dias=30):
    real_df = (kpi_df[['fecha', target_col, 'real']]
               .rename(columns={target_col: 'real_val'})
               .copy())
    real_df['fecha'] = pd.to_datetime(real_df['fecha'])
    real_df = real_df[real_df['real'] == 1]    # solo días observados

    bloques = []
    for corte in cortes:
        print(f"-> corte {corte}")
        pred = _refit_y_predecir(template, kpi_df, fecha_train_inicio,
                                 corte, horizonte_dias)
        m = pred.merge(real_df[['fecha', 'real_val']], on='fecha', how='left')
        m['dias_adelante'] = (m['fecha'] - pd.Timestamp(corte)).dt.days
        m['ape_pct']   = (m['pred'] - m['real_val']).abs() / m['real_val'] * 100
        m['sesgo_pct'] = (m['pred'] - m['real_val']) / m['real_val'] * 100
        bloques.append(m)
    return pd.concat(bloques, ignore_index=True)


# --------------------------- USO ---------------------------
KPI_PILOTO         = 'mau_app'
TARGET_COL         = 'mau'
FECHA_TRAIN_INICIO = '2024-01-01'
HORIZONTE_DIAS     = 30

cortes = (pd.date_range('2025-12-01', '2026-05-01', freq='MS')
            .strftime('%Y-%m-%d').tolist())

template = ForecastProducto(
    nombre_algoritmo=algoritmo_mau_app,
    meta=meta_mau_app,
    fechas_entrenamiento=fechas_entrenamiento,
    fechas_proyeccion=fechas_proyeccion,
    directorio=directorio_salida,
)

bt = backtest_rolling(template, mau_app, TARGET_COL,
                      FECHA_TRAIN_INICIO, cortes, HORIZONTE_DIAS)

# ---- resumen por corte ----
resumen = (bt.dropna(subset=['real_val'])
             .groupby('fecha_corte')
             .agg(mape_pct=('ape_pct', 'mean'),
                  sesgo_pct=('sesgo_pct', 'mean'),
                  n_dias=('real_val', 'size'))
             .round(2))
print('\nResumen por corte:')
print(resumen)

# ---- resumen por horizonte ----
deg = (bt.dropna(subset=['real_val'])
         .groupby('dias_adelante')
         .agg(mape_pct=('ape_pct', 'mean'),
              sesgo_pct=('sesgo_pct', 'mean'))
         .round(2))
print('\nMAPE/sesgo por día de horizonte:')
print(deg.head(15))

# ---- curvas ----
fig, ax = plt.subplots(figsize=(13, 5))
real_plot = (bt[['fecha', 'real_val']].drop_duplicates('fecha')
               .sort_values('fecha'))
ax.plot(real_plot['fecha'], real_plot['real_val'], 'k-', lw=2, label='real')
for corte, grp in bt.groupby('fecha_corte'):
    ax.plot(grp['fecha'], grp['pred'], '--', alpha=0.75,
            label=f'pred desde {corte.date()}')
ax.legend(loc='best', fontsize=8)
ax.set_title(f'Backtest rolling - {KPI_PILOTO}')
ax.set_ylabel(TARGET_COL)
plt.tight_layout()
plt.show()


## Diagnóstico de outliers (después del backtest V1)

Mira los peores días y desglosa por día del mes / por corte para entender
si el MAPE alto viene de un evento único (ej. feb-2026) o de días sistemáticamente mal capturados.

In [ ]:
# ==== Diagnóstico de outliers sobre el backtest V1 (bt) ====
import pandas as pd

bt_obs = bt.dropna(subset=['real_val']).copy()
bt_obs['dia_del_mes'] = bt_obs['fecha'].dt.day
bt_obs['dia_semana'] = bt_obs['fecha'].dt.day_name()
bt_obs['mes'] = bt_obs['fecha'].dt.to_period('M').astype(str)

print('--- Top 15 peores días absolutos ---')
print(bt_obs.nlargest(15, 'ape_pct')[
    ['fecha_corte','fecha','dia_semana','dias_adelante',
     'real_val','pred','ape_pct','sesgo_pct']
].to_string(index=False))

print('\n--- MAPE día +14 por corte (¿es un único mes el que rompe?) ---')
print(bt[bt['dias_adelante']==14][
    ['fecha_corte','fecha','real_val','pred','ape_pct','sesgo_pct']
].to_string(index=False))

print('\n--- MAPE / sesgo por día del mes ---')
print(bt_obs.groupby('dia_del_mes')[['ape_pct','sesgo_pct']]
      .mean().round(1).to_string())

print('\n--- MAPE / sesgo por día de la semana ---')
orden = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
print(bt_obs.groupby('dia_semana')[['ape_pct','sesgo_pct']]
      .mean().round(1).reindex(orden).to_string())

print('\n--- MAPE / sesgo por mes proyectado ---')
print(bt_obs.groupby('mes')[['ape_pct','sesgo_pct']]
      .mean().round(1).to_string())


## Entrenar V2 (mau_app) y comparar contra V1 con backtest rolling

Pasos:
1. Instancia `ForecastProductoV2` con `nombre_algoritmo = 'mau_app_v2'` para no pisar el `.pkl` de V1.
2. Entrena → genera `xgb_mau_app_v2.pkl` y reporta MAPE/sesgo en holdout temporal.
3. Corre `backtest_rolling` con el template V2 (apples-to-apples con el `bt` de V1).
4. Compara V1 vs V2 lado a lado y grafica curvas.

In [ ]:
# ==== 1. Entrenar V2 para mau_app ====
algoritmo_mau_app_v2 = algoritmo_mau_app + '_v2'   # 'mau_app_v2'

forecast_v2_train = ForecastProductoV2(
    nombre_algoritmo=algoritmo_mau_app_v2,
    meta=meta_mau_app,
    fechas_entrenamiento=fechas_entrenamiento,
    fechas_proyeccion=fechas_proyeccion,
    directorio=directorio_salida,
    holdout_dias=30,
)
forecast_v2_train._preparar_datos(mau_app.copy())
forecast_v2_train._entrenar_modelo()

# ==== 2. Backtest rolling con V2 ====
template_v2 = ForecastProductoV2(
    nombre_algoritmo=algoritmo_mau_app_v2,
    meta=meta_mau_app,
    fechas_entrenamiento=fechas_entrenamiento,
    fechas_proyeccion=fechas_proyeccion,
    directorio=directorio_salida,
    holdout_dias=30,
)
bt_v2 = backtest_rolling(template_v2, mau_app, TARGET_COL,
                         FECHA_TRAIN_INICIO, cortes, HORIZONTE_DIAS)

# ==== 3. Tabla comparativa V1 vs V2 ====
def _resumen(df, etiqueta):
    return (df.dropna(subset=['real_val'])
              .groupby('fecha_corte')
              .agg(**{f'mape_{etiqueta}': ('ape_pct', 'mean'),
                      f'sesgo_{etiqueta}': ('sesgo_pct', 'mean')})
              .round(2))

cmp = _resumen(bt, 'v1').join(_resumen(bt_v2, 'v2'))
cmp['delta_mape'] = (cmp['mape_v2'] - cmp['mape_v1']).round(2)
cmp['delta_sesgo'] = (cmp['sesgo_v2'] - cmp['sesgo_v1']).round(2)
print('=== V1 vs V2 (por corte) ===')
print(cmp.to_string())

print(f"\nMAPE promedio  V1: {bt['ape_pct'].mean():.2f}%   V2: {bt_v2['ape_pct'].mean():.2f}%")
print(f"Sesgo promedio V1: {bt['sesgo_pct'].mean():+.2f}%   V2: {bt_v2['sesgo_pct'].mean():+.2f}%")
print(f"Holdout temporal V2 (out-of-grid): {forecast_v2_train.holdout_metrics}")

# ==== 4. Curvas V1 vs V2 ====
fig, ax = plt.subplots(figsize=(13, 5))
real_plot = (bt[['fecha','real_val']].drop_duplicates('fecha')
                                     .sort_values('fecha'))
ax.plot(real_plot['fecha'], real_plot['real_val'], 'k-', lw=2, label='real')
for corte, grp in bt.groupby('fecha_corte'):
    ax.plot(grp['fecha'], grp['pred'], '--', alpha=0.45, color='tab:red')
for corte, grp in bt_v2.groupby('fecha_corte'):
    ax.plot(grp['fecha'], grp['pred'], '--', alpha=0.75, color='tab:blue')
# leyenda manual (solo una entrada por modelo)
from matplotlib.lines import Line2D
ax.legend(handles=[
    Line2D([0],[0], color='black', lw=2, label='real'),
    Line2D([0],[0], color='tab:red', ls='--', label='pred V1'),
    Line2D([0],[0], color='tab:blue', ls='--', label='pred V2'),
], loc='best')
ax.set_title('Backtest rolling - mau_app  (V1 vs V2)')
ax.set_ylabel(TARGET_COL)
plt.tight_layout()
plt.show()


## Comparativa V1 vs V2 vs V3 (backtest rolling)

Entrena V3 para mau_app y corre el backtest. Si V3 baja sesgo a ≈ 0 y MAPE a la zona 10-15%, el log-transform resolvió el síntoma de `last_estimated` inflado al inicio del mes.

In [ ]:
# ==== Entrenar V3 ====
algoritmo_mau_app_v3 = algoritmo_mau_app + '_v3'

forecast_v3_train = ForecastProductoV3(
    nombre_algoritmo=algoritmo_mau_app_v3,
    meta=meta_mau_app,
    fechas_entrenamiento=fechas_entrenamiento,
    fechas_proyeccion=fechas_proyeccion,
    directorio=directorio_salida,
    holdout_dias=30,
)
forecast_v3_train._preparar_datos(mau_app.copy())
forecast_v3_train._entrenar_modelo()

# ==== Backtest V3 ====
template_v3 = ForecastProductoV3(
    nombre_algoritmo=algoritmo_mau_app_v3,
    meta=meta_mau_app,
    fechas_entrenamiento=fechas_entrenamiento,
    fechas_proyeccion=fechas_proyeccion,
    directorio=directorio_salida,
    holdout_dias=30,
)
bt_v3 = backtest_rolling(template_v3, mau_app, TARGET_COL,
                         FECHA_TRAIN_INICIO, cortes, HORIZONTE_DIAS)

# ==== Tabla comparativa V1 / V2 / V3 ====
def _resumen(df, et):
    return (df.dropna(subset=['real_val'])
              .groupby('fecha_corte')
              .agg(**{f'mape_{et}':  ('ape_pct',   'mean'),
                      f'sesgo_{et}': ('sesgo_pct', 'mean')})
              .round(2))

cmp3 = _resumen(bt, 'v1').join(_resumen(bt_v2, 'v2')).join(_resumen(bt_v3, 'v3'))
print('=== V1 vs V2 vs V3 (por corte) ===')
print(cmp3.to_string())

print('\n=== Promedios globales ===')
for name, df in [('V1', bt), ('V2', bt_v2), ('V3', bt_v3)]:
    d = df.dropna(subset=['real_val'])
    print(f"  {name}:  MAPE = {d['ape_pct'].mean():5.2f}%   sesgo = {d['sesgo_pct'].mean():+6.2f}%")

print(f"\nHoldout temporal (out-of-grid) V2: {forecast_v2_train.holdout_metrics}")
print(f"Holdout temporal (out-of-grid) V3: {forecast_v3_train.holdout_metrics}")

# ==== Curvas V1 vs V3 (V2 omitido para no saturar) ====
fig, ax = plt.subplots(figsize=(13, 5))
real_plot = (bt[['fecha','real_val']].drop_duplicates('fecha')
                                     .sort_values('fecha'))
ax.plot(real_plot['fecha'], real_plot['real_val'], 'k-', lw=2, label='real')
for corte, grp in bt.groupby('fecha_corte'):
    ax.plot(grp['fecha'], grp['pred'], '--', alpha=0.4, color='tab:red')
for corte, grp in bt_v3.groupby('fecha_corte'):
    ax.plot(grp['fecha'], grp['pred'], '--', alpha=0.85, color='tab:green')
from matplotlib.lines import Line2D
ax.legend(handles=[
    Line2D([0],[0], color='black',     lw=2,    label='real'),
    Line2D([0],[0], color='tab:red',   ls='--', label='pred V1'),
    Line2D([0],[0], color='tab:green', ls='--', label='pred V3 (log)'),
], loc='best')
ax.set_title('Backtest rolling - mau_app  (V1 vs V3 con log-transform)')
ax.set_ylabel(TARGET_COL)
plt.tight_layout()
plt.show()


In [ ]:
# =============================================================================
# APE_2/7/14/21 -- ultimos 2 meses cerrados (jul-26 y ago-26)
# Version reducida de celda_backtest_ape_modelo_mau.py: mismos requisitos, misma
# logica, solo cambia MESES. Corre en 15-25 min (8 refits de XGB en total).
# =============================================================================
import os, numpy as np, pandas as pd

FECHA_TRAIN_INICIO = "2024-01-01"
MESES              = pd.date_range("2026-07-01", "2026-08-01", freq="MS")  # solo 2
CORTES_DIA         = [2, 7, 14, 21]
TARGET_COL         = "mau"

template_v5 = ForecastProductoV5(
    nombre_algoritmo   = algoritmo_mau_app,
    meta               = meta_mau_app,
    fechas_entrenamiento = fechas_entrenamiento,
    fechas_proyeccion    = fechas_proyeccion,
    directorio           = directorio_salida,
    holdout_dias         = 30,
)

_pkl = f"{directorio_salida}/xgb_{algoritmo_mau_app}.pkl"
assert os.path.exists(_pkl), (
    f"Falta {_pkl}. Corre antes la celda 42 (MAU APP) para generarlo.")

real_df = (mau_app[["fecha", TARGET_COL, "real"]]
             .rename(columns={TARGET_COL: "real_val"})
             .assign(fecha=lambda d: pd.to_datetime(d["fecha"]))
             .query("real == 1")
             .reset_index(drop=True))

rows = []
for mes in MESES:
    dias_mes   = mes.days_in_month
    fin_mes    = mes + pd.Timedelta(days=dias_mes - 1)
    total_real = real_df.loc[(real_df["fecha"] >= mes) &
                             (real_df["fecha"] <= fin_mes), "real_val"].sum()
    dias_reales_mes = real_df[(real_df["fecha"] >= mes) &
                              (real_df["fecha"] <= fin_mes)].shape[0]
    if total_real == 0 or pd.isna(total_real):
        print(f"skip {mes.date()}: sin real"); continue
    print(f"\n[{mes.strftime('%Y-%m')}] real={total_real:,.0f} "
          f"({dias_reales_mes}/{dias_mes} dias observados)")

    for corte_dia in CORTES_DIA:
        fecha_corte = mes + pd.Timedelta(days=corte_dia - 1)
        horizonte   = (fin_mes - fecha_corte).days
        if horizonte <= 0:
            continue

        pred = _refit_y_predecir(
            template_v5, mau_app, FECHA_TRAIN_INICIO,
            fecha_corte.strftime("%Y-%m-%d"), horizonte,
        )
        pred["fecha"] = pd.to_datetime(pred["fecha"])

        mtd_real   = real_df.loc[(real_df["fecha"] >= mes) &
                                 (real_df["fecha"] <= fecha_corte), "real_val"].sum()
        rest_pred  = pred.loc[(pred["fecha"] >  fecha_corte) &
                              (pred["fecha"] <= fin_mes), "pred"].sum()
        total_pred = mtd_real + rest_pred

        ape   = abs(total_pred - total_real) / total_real * 100
        sesgo = (total_pred - total_real) / total_real * 100

        rows.append({"mes": mes.strftime("%Y-%m"), "corte_dia": corte_dia,
                     "real_total_mes": total_real,
                     "pred_total_mes": total_pred,
                     "APE_pct": round(ape, 3),
                     "sesgo_pct": round(sesgo, 3)})
        print(f"  corte={corte_dia:>2}  APE={ape:5.2f}%   sesgo={sesgo:+5.2f}%")

bt = pd.DataFrame(rows)

print("\n============= RESUMEN ULTIMOS 2 MESES =============")
print(bt.pivot(index="mes", columns="corte_dia",
               values="APE_pct").round(2).to_string())

bt.to_csv("/content/backtest_ultimos_2_meses.csv", index=False)
print("\nGuardado: /content/backtest_ultimos_2_meses.csv")

In [ ]:
# ==============================================================================
# V11: _flush_audit_a_bq() ahora se define en la celda helper V10 y se ejecuta
# al final de la celda REPORTE LE. Esta celda queda solo como referencia.
# ==============================================================================


In [ ]:
# ==============================================================================
# V10: BACKTEST DEL CAMBIO CONTRA LOS 8 MESES HISTORICOS
# Requiere resultados_final.csv en el directorio de trabajo.
# ==============================================================================

def backtest_v10_vs_pipeline_actual(archivo="resultados_final.csv"):
    import os
    if not os.path.exists(archivo):
        print(f"Falta {archivo} en el directorio.")
        return None

    df = pd.read_csv(archivo)
    d = df[(df["kpi"] == "mau") & (df["hierarchy_level"] == "agregado")
           & (df["series"] == "mau_total")].copy()
    d_crudo = (df[(df["kpi"] == "mau") & (df["hierarchy_level"] == "total")
                  & (df["series"] == "mau_app")]
               .set_index(["month", "cutoff_day"])["model_crudo"])
    d = d.set_index(["month", "cutoff_day"]).join(
        d_crudo.rename("model_crudo"), how="inner").reset_index()

    filas = []
    for _, r in d.iterrows():
        w_v10, _ = _pace_weight_adaptativo(
            dia_mes=int(r["cutoff_day"]),
            pace_val=float(r["le_baseline"]),
            model_val=float(r["model_crudo"]),
            kpi_label=f"mau_{r['month']}")
        le_v10 = w_v10 * r["le_baseline"] + (1 - w_v10) * r["model_crudo"]
        le_actual = r["le_pred"]; real = r["le_real"]
        filas.append({
            "month": r["month"], "cutoff_day": int(r["cutoff_day"]),
            "real": real,
            "le_actual_w05": le_actual, "le_v10_adaptivo": le_v10,
            "w_v10": w_v10,
            "APE_actual_pct": abs(le_actual - real) / real * 100,
            "APE_v10_pct": abs(le_v10 - real) / real * 100,
        })
    out = pd.DataFrame(filas).sort_values(["month", "cutoff_day"])
    out["mejora_pp"] = (out["APE_actual_pct"] - out["APE_v10_pct"]).round(2)

    print("\n=== APE actual vs APE V10, por mes y corte ===")
    print(out[["month", "cutoff_day", "APE_actual_pct", "APE_v10_pct",
               "mejora_pp", "w_v10"]].round(2).to_string(index=False))

    print("\n=== Mediana APE por corte (positivo = V10 mejora) ===")
    med = out.groupby("cutoff_day")[["APE_actual_pct", "APE_v10_pct"]].median().round(2)
    med["mejora_pp"] = (med["APE_actual_pct"] - med["APE_v10_pct"]).round(2)
    print(med.to_string())

    out.to_csv("backtest_v10_vs_pipeline_actual.csv", index=False)
    print("\nGuardado: backtest_v10_vs_pipeline_actual.csv")
    return out


# backtest_v10_vs_pipeline_actual()


In [ ]:
# ==============================================================================
# REPORTE LE PARA CONTROL TOWER (Rodrigo + Lukas)
# ==============================================================================
# Extrae los numeros clave del LE del mes en curso, arma un mensaje listo para
# pegar en Slack, y appendea a un CSV historico para reconstruir el Control
# Tower al cierre del mes.
#
# Se ejecuta automatico al final del pipeline. Usa el dia del mes actual para
# etiquetar el corte (d2, d7, d14, d21). El CSV se acumula corrida tras corrida.
# ==============================================================================
from datetime import date
import os

def _fmt_int(v, default="N/A"):
    return f"{int(v):,}" if v is not None else default


def _get_var(name, default=None):
    """Lee variable global del notebook si existe, sino devuelve default."""
    return globals().get(name, default)


def armar_reporte_le():
    hoy = hoy_chile().date()
    corte_dia = hoy.day
    mes_str = hoy.strftime("%Y-%m")

    # LE del pipeline (variables globales que dejaron MAU APP y GPV APP)
    le_mau_total = _get_var("_pace_target")
    le_gpv_total_clp = _get_var("_pace_target_gpv")

    # MTD conocido
    mtd_mau = _get_var("actual_acum")
    mtd_gpv = _get_var("_gpv_actual_acum")

    # Config V10 aplicada (del buffer de auditoria)
    audit = _get_var("_AUDIT_ROWS", [])
    mau_row = next((r for r in audit if r.get("kpi") == "MAU APP"), {})
    peso_mau = mau_row.get("peso_final")
    regimen_mau = mau_row.get("regimen", "n/a")
    div_mau = mau_row.get("divergencia_pct")

    reporte = {
        "fecha_corrida":         hoy.isoformat(),
        "le_rgu":                _get_var("cierre_estimado"),
        "dias_restantes":        _get_var("DIAS_RESTANTES_MES"),
        "mes":                   mes_str,
        "corte_dia":             corte_dia,
        "le_mau_total":          int(le_mau_total) if le_mau_total is not None else None,
        "le_gpv_total_mm_clp":   round(le_gpv_total_clp / 1e6, 0) if le_gpv_total_clp else None,
        "mtd_mau_conocido":      int(mtd_mau) if mtd_mau is not None else None,
        "mtd_gpv_conocido_mm":   round(mtd_gpv / 1e6, 0) if mtd_gpv else None,
        "peso_pace_v10_mau":     peso_mau,
        "regimen_v10_mau":       regimen_mau,
        "divergencia_pct_mau":   div_mau,
    }

    # --- Print resumen legible ---------------------------------------------
    print("\n" + "=" * 72)
    print(f"REPORTE LE {mes_str} - corte d{corte_dia} (corrida {hoy.isoformat()})")
    print("=" * 72)
    print(f"LE MAU total {mes_str}:  {_fmt_int(le_mau_total):>18s}")
    print(f"  MTD MAU conocido:  {_fmt_int(mtd_mau):>18s}")
    if le_gpv_total_clp:
        print(f"LE GPV total {mes_str}:  {int(le_gpv_total_clp/1e6):>15,} MM CLP")
        if mtd_gpv:
            print(f"  MTD GPV conocido:  {int(mtd_gpv/1e6):>15,} MM CLP")
    print()
    print(f"V10 config aplicada (MAU): peso={peso_mau} | regimen={regimen_mau} | "
          f"divergencia={div_mau}%")
    print("=" * 72)

    # --- Guardar a CSV historico -------------------------------------------
    import pandas as pd
    # V11: en Drive (antes /content -> se perdia al cerrar la sesion de Colab)
    archivo = f"{directorio_salida}/reporte_le_control_tower.csv"
    df_new = pd.DataFrame([reporte])
    if os.path.exists(archivo):
        df_hist = pd.read_csv(archivo)
        # Si ya existe una fila con la misma (mes, corte_dia), la reemplazamos
        df_hist = df_hist[~((df_hist["mes"] == mes_str) &
                            (df_hist["corte_dia"] == corte_dia))]
        df_all = pd.concat([df_hist, df_new], ignore_index=True)
    else:
        df_all = df_new
    df_all = df_all.sort_values(["mes", "corte_dia"]).reset_index(drop=True)
    df_all.to_csv(archivo, index=False)
    print(f"\nCSV historico: {archivo} ({len(df_all)} filas acumuladas)")

    # --- Mensaje listo para pegar en Slack ---------------------------------
    _le_mau_str = _fmt_int(le_mau_total)
    _mtd_mau_str = _fmt_int(mtd_mau)
    _le_gpv_str = f"{int(le_gpv_total_clp/1e6):,} MM CLP" if le_gpv_total_clp else "N/A"
    _mtd_gpv_str = f"{int(mtd_gpv/1e6):,} MM CLP" if mtd_gpv else "N/A"
    _pct_mtd = f"{mtd_mau/le_mau_total*100:.1f}%" if (mtd_mau and le_mau_total) else "N/A"

    mensaje = f"""LE de {mes_str} al corte d{corte_dia} (corrida del {hoy.isoformat()})

- LE MAU total {mes_str}: {_le_mau_str}
- LE GPV total {mes_str}: {_le_gpv_str}
- MTD MAU conocido:       {_mtd_mau_str} ({_pct_mtd} del LE)
- MTD GPV conocido:       {_mtd_gpv_str}
- Config V10 (MAU):       peso pace = {peso_mau}, regimen = {regimen_mau}, divergencia = {div_mau}%

CSV historico se va acumulando; al cierre de {mes_str} armamos las 4 comparaciones vs real."""

    print("\n" + "=" * 72)
    print("MENSAJE LISTO PARA SLACK (copiar y pegar):")
    print("=" * 72)
    print(mensaje)
    print("=" * 72)

    return reporte, mensaje


# --- Ejecucion automatica al final del pipeline -----------------------------
reporte_dict, mensaje_slack = armar_reporte_le()

# V11: guardar la auditoria de la corrida en BQ (kpitos.pipeline_audit)
try:
    _flush_audit_a_bq()
except Exception as e:
    print(f"[V11 AUDIT] !! no se pudo volcar la auditoria: {e}")


In [ ]:
# ==============================================================================
# V12 — PUBLICACIÓN FINAL (ejecutar al FINAL). Valida y publica los grupos de tablas.
# stage: aquí se escribe en BigQuery, grupo por grupo. Un grupo con error se retiene completo
#        (se conserva la versión anterior y consistente) y al final se levanta PublishError.
# dry_run: solo reporte. live: ya se publicó tabla por tabla; esto solo imprime el reporte.
# ==============================================================================
try:
    GATE.report_dir = f"{directorio_salida}/reportes_publicacion"
except NameError:
    pass
GATE.commit_all()
if GATE.mode == "dry_run":
    GATE.compare_with_bq()   # solo lectura: compara lo que se publicaría vs lo vigente en BigQuery
